# Palatini gravity: neutron stars, muonic atoms and the infrared sector

Jupyter export of the marimo notebook `palatini_jeans_window.py`. The marimo file is the canonical, reactive
version (run it with `marimo edit palatini_jeans_window.py` or on molab). In Jupyter, run the cells from top
to bottom; interactive buttons (`mo.ui.run_button`) do not trigger in Jupyter, so set the environment
variables listed in the README (for example `RUN_HEAVY=1`) before starting the kernel to run the heavy stages,
or restore the published checkpoints into `pj_ckpt/` so that every stage loads instead of recomputing.
See README.md for the mathematics, the data sources and the checkpoint layout.


In [ ]:
# Install dependencies if needed (uncomment):
# %pip install marimo numpy scipy matplotlib psutil camb torch huggingface_hub


In [ ]:
# ---- SETUP: environment, imports, budgets, helpers (from the molab template) --------------
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import ctypes
import json
import math
import time
import hashlib
import types
import marimo as mo
import numpy as np

try:
    import psutil
except ImportError:
    psutil = None
import torch

torch.set_default_dtype(torch.float64)   # cosmology needs float64 (chi^2 differences of O(0.1) on O(1600))

RAM_CAP_GB = 30.0
RAM_SOFT_GB = 26.0
VRAM_CAP_GB = 70.0

def rss_gb(include_children=True):
    if psutil is not None:
        p = psutil.Process()
        total = p.memory_info().rss
        if include_children:
            for c in p.children(recursive=True):
                try:
                    total += c.memory_info().rss
                except psutil.Error:
                    pass
        return total / 2**30
    with open("/proc/self/status") as f:
        for line in f:
            if line.startswith("VmRSS:"):
                return int(line.split()[1]) / 2**20
    return float("nan")

def ram_guard(extra_gb, what="allocation"):
    used = rss_gb()
    if used + extra_gb > RAM_SOFT_GB:
        raise MemoryError(f"{what}: need ~{extra_gb:.1f} GB on top of {used:.1f} GB; soft limit {RAM_SOFT_GB:.0f} GB")

def trim_ram():
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except OSError:
        pass
    return rss_gb()

def gpu_setup():
    if not torch.cuda.is_available():
        return {"device": "cpu", "torch": torch.__version__}
    _free, _total = torch.cuda.mem_get_info()
    _total_gb = _total / 2**30
    _cap_gb = min(VRAM_CAP_GB, _total_gb * 0.95)
    torch.cuda.set_per_process_memory_fraction(min(1.0, _cap_gb / _total_gb))
    return {"device": "cuda", "name": torch.cuda.get_device_name(0), "total_gb": round(_total_gb, 1),
            "cap_gb": round(_cap_gb, 1), "torch": torch.__version__}

def gpu_free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()

def vram_report(reset_peak=False):
    if not torch.cuda.is_available():
        return {}
    out = {"allocated_gb": round(torch.cuda.memory_allocated() / 2**30, 2),
           "peak_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2)}
    if reset_peak:
        torch.cuda.reset_peak_memory_stats()
    return out

def atomic_write(path, writer):
    base, ext = os.path.splitext(path)
    tmp = f"{base}.tmp{ext}"
    writer(tmp)
    os.replace(tmp, path)

def save_json(path, obj):
    atomic_write(path, lambda p: open(p, "w").write(json.dumps(obj, indent=1, default=float)))

# The Palatini–Jeans Window
### A stability-selected, background-dark gravitational sector that can only *lower* late-time clustering — and where it must act

**What this notebook does.** It derives four structural results for the whole class of *auxiliary-field* (Palatini-type)
curvature corrections, turns them into a one-parameter model with a sharp, falsifiable prediction, and confronts that
prediction with real data: DESI DR2 BAO, Pantheon+ (uncalibrated), Planck 2018 compressed CMB, SDSS DR16 + 6dFGS
$f\sigma_8$, DES Y6 and KiDS-Legacy cosmic shear, and ACT DR6 + Planck CMB lensing.

**One-sentence claim under test.** *In any Palatini-type theory, linear stability forces the curvature sector to act as a
Jeans pressure on matter that (i) is invisible to the expansion history at the $10^{-5}$ level, (ii) cannot raise
clustering, and (iii) — for any correction that grows toward low curvature — switches on only around the matter–$\Lambda$
transition $z_J\simeq 0.2$–$0.6$, so it lowers galaxy-lensing $S_8$ while leaving the CMB, CMB lensing and large-scale
RSD almost untouched.*

Run order: everything above the **Run analysis** button is cheap (data download, validation, theory figures).
The button runs the profile likelihood and MCMC (resumable, checkpointed). `SMOKE=1` shrinks everything for a quick test;
`RUN_HEAVY=1` presses the button headlessly.

## 1. Theory

### 1.1 Setting
Palatini $f(R)$: $S=\frac{1}{2\kappa}\int\! d^4x\sqrt{-g}\,f\big(g^{\mu\nu}R_{\mu\nu}(\Gamma)\big)+S_m[g,\psi]$.
Varying $\Gamma$ makes it the Levi-Civita connection of $F g_{\mu\nu}$, $F\equiv f'(R)$; the trace equation
$FR-2f=\kappa T$ is **algebraic**, so $R=R(T)$ and $F=F(\rho)$ for dust. The theory is Brans–Dicke with $\omega=-3/2$:
no new propagating degree of freedom, hence no ghost (the defect of metric $R_{\mu\nu}R^{\mu\nu}$ actions noted in the gap
analysis). In the Jordan frame
$$F G_{\mu\nu}=\kappa T_{\mu\nu}-\tfrac12 g_{\mu\nu}(RF-f)+\nabla_\mu\nabla_\nu F-g_{\mu\nu}\Box F-\tfrac{3}{2F}\Big[\nabla_\mu F\nabla_\nu F-\tfrac12 g_{\mu\nu}(\nabla F)^2\Big].$$

### 1.2 Result 1 — exact quasi-static response (derived here, validated numerically below)
With $ds^2=-(1+2\Psi)dt^2+a^2(1-2\Phi)d\mathbf x^2$, $k\gg aH$, and $\delta F=F_\rho\,\delta\rho$ (algebraic), the $00$ and
traceless $ij$ equations give $2\nabla^2\Phi=\kappa a^2\delta\rho/F+\nabla^2\delta F/F$ and $\Phi-\Psi=\delta F/F$. Hence
$$k^2\Psi=-4\pi G a^2\bar\rho\,\delta\;\frac{1-k^2/k_J^2}{F},\qquad
  k^2\Phi=-4\pi G a^2\bar\rho\,\delta\;\frac{1+k^2/k_J^2}{F},\qquad
  k_J^2(a)\equiv\frac{\kappa a^2}{-F_\rho}.$$
So $\mu=(1-k^2/k_J^2)/F$, $\;\Sigma=(\Phi+\Psi)/2\text{-response}=1/F$, $\;\eta=\Phi/\Psi=(1+x)/(1-x)$, $x=k^2/k_J^2$, and
$$\delta''+\Big(2+\frac{d\ln H}{d\ln a}\Big)\delta'-\frac32\Omega_m(a)\,\mu(k,a)\,\delta=0 ,
\qquad c_{\rm eff}^2=-\frac{\rho F_\rho}{2F}.$$
The sound-speed term itself is known (Koivisto 2006; Li, Barrow & Mota 2007). Sanity check of the sign conventions:
the same two equations with the metric-$f(R)$ relation $\nabla^2\delta F=-\kappa a^2\delta\rho/3$ reproduce the textbook
$\Psi\to\frac43\Psi_{\rm GR}$, $\Phi\to\frac23\Phi_{\rm GR}$.

### 1.3 Result 2 — stability selects the sign (*clustering can only go down*)
If $F_\rho>0$ anywhere, $c^2_{\rm eff}<0$ and linear modes grow as $\exp(k|c_{\rm eff}|t/a)$ with no UV bound: the theory
is linearly unstable. **Viability therefore requires $F_\rho\le0$**, which makes $\mu\le\Sigma$ at every $k$: the whole
stable Palatini class can only *suppress* growth below the Jeans length, while the lensing response to a given
$\delta$ stays exactly $1/F\simeq1$. The direction of the $S_8$ discrepancy is thus a *consequence of stability*, not a fit choice.

### 1.4 Result 3 — background darkness (a parameter-free consistency relation)
For dust, $\epsilon\equiv d\ln F/d\ln a=-3\rho F_\rho/F$, and combining with the definition of $k_J$:
$$\boxed{\;\epsilon(a)=9\,\Omega_m(a)\left(\frac{aH}{k_J}\right)^2\;}$$
The Palatini Friedmann equation $(H+\dot F/2F)^2=(\kappa\rho+f)/(6F)$ deviates from GR at $\mathcal O(\epsilon)$.
A Jeans scale that touches linear LSS ($k_J\gtrsim0.1\,h\,$Mpc$^{-1}$) forces $\epsilon\lesssim10^{-5}$, i.e.
$|\Delta H/H|\lesssim10^{-5}$ at all $z$.
**Corollaries.** (a) A stable Palatini sector that touches clustering cannot move $H_0$ or produce DESI's
$w_0w_a$ signal — those need a separate field (for the PRBQ programme: the quintessence/braiding sector must carry $H_0$;
the Palatini sector is purely an $S_8$ sector). (b) Gap 1 of the proposal is *inverted* for this class: background data are
structurally blind to it; only growth and lensing can test it.

### 1.5 Result 4 — the Jeans window is pinned to the matter–$\Lambda$ transition
Take any correction that grows toward low curvature, $f=R-2\Lambda-\mu^{2n+2}R^{-n}$ ($n>0$). Then
$-F_\rho\propto R^{-(n+2)}$ with $R\simeq\kappa\rho+4\Lambda$, so
$$k_J^2(a)\propto a^2\left(\Omega_m a^{-3}+4\Omega_\Lambda\right)^{n+2},\qquad
  (1+z_J)^3=\frac{8\,\Omega_\Lambda}{(3n+4)\,\Omega_m}\quad(\text{minimum of }k_J).$$
For $\Omega_m\simeq0.31$: $z_J=0.36$ ($n=1$), $0.24$ ($n=2$), $\to0.64$ ($n\to0$). Early on $k_J\propto a^{-(3n+4)/2}\to\infty$
(CMB untouched); after $z_J$, the $4\Lambda$ term freezes $R$ and $k_J\propto a$ (the effect switches off in the future).
The window sits exactly where the galaxy-lensing kernels peak, while the CMB-lensing kernel ($z\sim2$) sees a
$k_J$ several times larger. By contrast Palatini $R^2$ ($F_\rho$=const) has $k_J\propto a$ at all times — largest
suppression *early* — which is why it is excluded; the selection rule is: **viable Palatini-class corrections must be
infrared (decaying with curvature).**

### 1.6 Built-in density screening
Locally $k_{J,\rm phys}^2\propto R_{\rm loc}^{\,n+2}$, so the suppression in an environment with contrast $\delta_{\rm env}$ scales
as $S=[(\Omega_m a^{-3}+4\Omega_\Lambda)/(\Omega_m a^{-3}(1+\delta_{\rm env})+4\Omega_\Lambda)]^{n+2}$: enhanced in voids,
off in halos and in the Solar System. Non-linearly, $\Psi=\Psi_{\rm GR}-\delta F/2F$ with $|\delta F|\le F(\rho\to0)-1\sim\epsilon$.
*Open issue we flag rather than hide:* Palatini theories with $f_{RR}(R_{\min})\neq0$ inherit the Barausse–Sotiriou–Miller
surface-curvature issue for idealised polytropes; here its coefficient is $\propto\mu^{2n+2}$ and real stars sit in media
with $\rho\gg\bar\rho$, but a dedicated stellar-structure check belongs in the paper.

### 1.7 Pre-registered predictions and falsifiers
| # | Prediction (no extra freedom once $k_{J*}$ is fitted) | Kills the model if |
|---|---|---|
| P1 | Probe ordering $S_8^{\rm shear}<S_8^{\rm CMB\,lens}\lesssim S_8^{\rm CMB}$ | shear $S_8$ ≥ CMB-lensing $S_8$ at high significance |
| P2 | $f(k)$ drops and $E_G(k)$ rises for $k\gtrsim0.3k_J$ at $z\sim0.3$–$0.6$ | DESI full-shape finds scale-independent $f$ at the forecast level |
| P3 | Effective $S_8$ rises with source redshift | inverted tomographic trend |
| P4 | Lensing deficit larger in troughs/voids, absent in cluster cores | the opposite environmental trend |
| P5 | $H(z)$, BAO, SNe, $H_0$ identical to ΛCDM to $10^{-5}$ | any background preference attributed to this sector |
| P6 | CMB primary unchanged | — |

**Novelty status (honest).** The sound-speed term is known; what we have not found in a limited literature search is
(i) the stability-sign theorem framed as a prediction of the *direction* of $S_8$, (ii) the background-darkness relation as a
no-go for $H_0$, (iii) the pinning of the Jeans window to the matter–$\Lambda$ transition of $R$, and (iv) the probe-ordering
reading of the DES-vs-KiDS/ACT "bifurcation". Treat these as candidate results until a full literature audit is done.

In [ ]:
HW = gpu_setup()
mo.md(f"**Hardware:** `{HW}`  \n**RAM in use at start:** {rss_gb():.2f} GB")

In [ ]:
# ---- CONFIG: every size, count and path in one place ------------------------------------
SMOKE = os.environ.get("SMOKE", "0") == "1"
CFG = {
    "seed": 20261018,
    "device": HW["device"],
    "data_dir": "pj_data",
    "ckpt_dir": "pj_ckpt_smoke" if SMOKE else "pj_ckpt",
    "deadline_hours": 11.0,
    # physics / numerics
    "n_index": 1.0,                      # f = R - 2 Lambda - mu^(2n+2) R^-n ; n=1 is the minimal IR correction
    "prescriptions": ["ratio", "linear_only"],   # non-linear bracket: upper / lower bound on suppression
    "nk_ode": 48 if SMOKE else 96,
    "nk": 300 if SMOKE else 600,
    "nzl": 32 if SMOKE else 64,
    "n_coarse": 60 if SMOKE else 120,
    "n_fine": 200 if SMOKE else 400,
    "shear_zmean": {"DESY6_shear": 0.63, "KiDS_Legacy": 0.75},   # Smail-n(z) means (approximation; see text)
    # profile likelihood (primary statistic)
    "u_grid": [-0.6, 0.6] if SMOKE else [-0.9, -0.7, -0.5, -0.35, -0.2, -0.05, 0.1, 0.25, 0.45, 0.7, 1.0, 1.5],
    "anneal_walkers": 8 if SMOKE else 64,
    "anneal_iters": 6 if SMOKE else 300,
    "anneal_T_final": 1e-3,
    # MCMC (secondary: posteriors, DIC)
    "mcmc_walkers": 8 if SMOKE else 128,
    "mcmc_iters": 8 if SMOKE else 4000,
    "mcmc_chunk": 4 if SMOKE else 250,
    "mcmc_burn_frac": 0.3,
    # robustness (separate button)
    "robust_drop": ["rsd", "DESY6_shear", "KiDS_Legacy", "ACT_Planck_CMBL"],
    "robust_n": [2.0] if SMOKE else [2.0, 3.0],
}
CFG["run_mcmc"] = os.environ.get("RUN_MCMC", "1") == "1"
CFG.update(json.loads(os.environ.get("PJ_OVERRIDES", "{}")))   # e.g. '{"anneal_iters": 200}'
CFG["tag"] = hashlib.sha1(json.dumps({k: v for k, v in CFG.items() if k not in ("device",)},
                                     sort_keys=True).encode()).hexdigest()[:10]
os.makedirs(CFG["ckpt_dir"], exist_ok=True)
os.makedirs(CFG["data_dir"], exist_ok=True)

In [ ]:
# ---- MEMORY PLAN: peak tensors per likelihood call (batch = half the walkers) -------------
_B = max(CFG["mcmc_walkers"], CFG["anneal_walkers"]) // 2
_M = 2 * (CFG["n_coarse"] + CFG["n_fine"] + 120) + 1          # RK4 nodes + midpoints
_ode = _B * (CFG["nk_ode"] + 1) * _M * 8 * 2 / 2**30           # coefficients + temporaries
_hf = _B * 200 * CFG["nk"] * 8 / 2**30                         # sigma(R) scan
_nl = _B * CFG["nzl"] * CFG["nk"] * 8 * 6 / 2**30              # NL spectra + Limber temporaries
_sn = 1590**2 * 8 / 2**30
mo.md(f"""
| stage | peak (GB) | budget |
|---|---|---|
| growth ODE coefficients (B={_B}) | {_ode:.3f} | VRAM 70 |
| halofit sigma(R) scan | {_hf:.3f} | VRAM 70 |
| non-linear spectra + Limber | {_nl:.3f} | VRAM 70 |
| Pantheon+ inverse covariance | {_sn:.3f} | RAM 30 / VRAM 70 |

Everything is < 1 GB: the analysis is compute-bound, not memory-bound, so the batch is sized for throughput.
""")

In [ ]:
# ---- PHYSICS CORE (tested module, embedded verbatim; exported as one namespace PJ) --------
def _build_core():
    # Palatini-Jeans core physics (development module; embedded verbatim into the notebook)
    import math
    import numpy as np
    import torch

    C_KMS = 299792.458
    CH0 = 2997.92458                 # c/H0 in Mpc/h
    T_CMB = 2.7255
    OMG_H2 = 2.47282e-5 * (T_CMB / 2.7255) ** 4   # photon density omega_gamma
    NEFF = 3.046
    NU_RATIO = 0.22710731766         # (7/8)(4/11)^(4/3)
    M_NU = 0.06                      # eV, one massive state (minimal normal ordering)
    KT_NU0 = 8.617333262e-5 * T_CMB * (4.0 / 11.0) ** (1.0 / 3.0)
    Y_NU0 = M_NU / KT_NU0            # m/T_nu today (~358)
    OMNU_H2 = 0.000644 * (M_NU / 0.06)   # late-time omega_nu (= sum m / 93.14 eV)

    PNAMES = ["h", "ombh2", "omch2", "lnAs", "ns", "u", "nidx"]
    PRIOR_LO = torch.tensor([0.60, 0.0200, 0.100, 2.80, 0.920, -1.0, 0.25], dtype=torch.float64)
    PRIOR_HI = torch.tensor([0.80, 0.0250, 0.140, 3.30, 1.000, 1.5, 4.00], dtype=torch.float64)


    def _fnu(y):
        """Massive-neutrino energy density relative to massless (Komatsu+11 fit, 0.3% accurate)."""
        return (1.0 + (0.3173 * y) ** 1.83) ** (1.0 / 1.83)


    def unpack(theta):
        """theta (B,7) -> dict of (B,1) tensors and derived densities (flat universe)."""
        d = {k: theta[:, i:i + 1] for i, k in enumerate(PNAMES)}
        h = d["h"]
        d["omcb"] = d["ombh2"] + d["omch2"]
        d["omm"] = d["omcb"] + OMNU_H2
        d["Ocb"] = d["omcb"] / h**2
        d["Om"] = d["omm"] / h**2
        d["Og"] = OMG_H2 / h**2
        d["Onu_rel_unit"] = NU_RATIO * (NEFF / 3.0) * d["Og"]          # per neutrino state, massless
        _onu0 = d["Onu_rel_unit"] * (2.0 + _fnu(torch.as_tensor(Y_NU0, dtype=theta.dtype)))
        d["OL"] = 1.0 - d["Ocb"] - d["Og"] - _onu0
        return d


    def E2_of_a(d, a):
        """H^2/H0^2 at scale factor a (broadcasts: a is (...,) or (B,N))."""
        rad = d["Og"] * a**-4 + d["Onu_rel_unit"] * a**-4 * (2.0 + _fnu(Y_NU0 * a))
        return d["Ocb"] * a**-3 + rad + d["OL"]


    def dlnE_dlna(d, a, eps=1e-4):
        lp = torch.log(E2_of_a(d, a * math.exp(eps)))
        lm = torch.log(E2_of_a(d, a * math.exp(-eps)))
        return 0.25 * (lp - lm) / eps


    # ---------------------------------------------------------------- distances
    def make_zgrid(zmax=1400.0, n=6000, dtype=torch.float64, device="cpu"):
        lnz = torch.linspace(0.0, math.log1p(zmax), n, dtype=dtype, device=device)
        return torch.expm1(lnz), lnz


    def comoving_chi(d, zg, lnzg):
        """Dimensionless comoving distance chi(z) = int dz/E on fixed grid. Returns (B,N)."""
        a = 1.0 / (1.0 + zg)
        integrand = (1.0 + zg) / torch.sqrt(E2_of_a(d, a))   # dz/E = (1+z)/E dln(1+z)
        dl = lnzg[1:] - lnzg[:-1]
        inc = 0.5 * (integrand[:, 1:] + integrand[:, :-1]) * dl
        return torch.cat([torch.zeros_like(inc[:, :1]), torch.cumsum(inc, dim=1)], dim=1)


    def interp_fixed(xg, yg, xq):
        """Linear interpolation on a fixed 1-D grid xg (N,), batched yg (B,N); xq (M,) or (B,M)."""
        if xq.dim() == 1:
            xq = xq.unsqueeze(0).expand(yg.shape[0], -1)
        idx = torch.searchsorted(xg, xq.contiguous()).clamp(1, xg.numel() - 1)
        x0, x1 = xg[idx - 1], xg[idx]
        y0 = torch.gather(yg, 1, idx - 1)
        y1 = torch.gather(yg, 1, idx)
        w = (xq - x0) / (x1 - x0)
        return y0 + w * (y1 - y0)


    def r_drag(d):
        """Aubourg et al. 2015 eq.16 (0.06% vs CAMB for standard physics), Mpc."""
        return 55.154 * math.exp(-72.3 * (OMNU_H2 + 0.0006) ** 2) / (
            d["omcb"] ** 0.25351 * d["ombh2"] ** 0.12807)


    def z_star(d):
        wb, wm = d["ombh2"], d["omm"]
        g1 = 0.0783 * wb ** -0.238 / (1.0 + 39.5 * wb ** 0.763)
        g2 = 0.560 / (1.0 + 21.1 * wb ** 1.81)
        return 1048.0 * (1.0 + 0.00124 * wb ** -0.738) * (1.0 + g1 * wm ** g2)


    def sound_horizon(d, astar, n=3000):
        """r_s(a*) in Mpc: int_0^{a*} c_s da/(a^2 H)."""
        t = torch.linspace(math.log(1e-9), 0.0, n, dtype=astar.dtype, device=astar.device)
        a = astar * torch.exp(t).unsqueeze(0)                       # (B,n)
        Rb = 3.0 * d["ombh2"] / (4.0 * OMG_H2) * a
        cs = 1.0 / torch.sqrt(3.0 * (1.0 + Rb))
        integ = cs / (a * torch.sqrt(E2_of_a(d, a)))               # dln a
        val = torch.trapezoid(integ, t.unsqueeze(0).expand_as(integ), dim=1).unsqueeze(1)
        return CH0 / d["h"] * val


    # ---------------------------------------------------------------- linear matter power
    def eh_nowiggle_T(d, k):
        """Eisenstein & Hu 1998 zero-baryon-wiggle transfer, k in h/Mpc, (B,Nk)."""
        h, wm, wb = d["h"], d["omcb"], d["ombh2"]
        fb = wb / wm
        th = T_CMB / 2.7
        s = 44.5 * torch.log(9.83 / wm) / torch.sqrt(1.0 + 10.0 * wb ** 0.75)    # Mpc
        aG = 1.0 - 0.328 * torch.log(431.0 * wm) * fb + 0.38 * torch.log(22.3 * wm) * fb**2
        kMpc = k * h
        Geff = d["Ocb"] * h * (aG + (1.0 - aG) / (1.0 + (0.43 * kMpc * s) ** 4))
        q = k * th**2 / Geff
        L0 = torch.log(2.0 * math.e + 1.8 * q)
        C0 = 14.2 + 731.0 / (1.0 + 62.5 * q)
        return L0 / (L0 + C0 * q**2)


    def delta2_lin0(d, k, D0, calib=None):
        """Dimensionless Delta^2_cb(k, z=0) for LCDM-normalised growth D0 (growth with D->a early)."""
        T = eh_nowiggle_T(d, k)
        As = torch.exp(d["lnAs"]) * 1e-10
        kMpc = k * d["h"]
        prim = As * (kMpc / 0.05) ** (d["ns"] - 1.0)
        out = (4.0 / 25.0) * (k * CH0) ** 4 / d["Ocb"] ** 2 * T**2 * D0**2 * prim
        if calib is not None:
            out = out * calib
        return out


    def tophat_W(x):
        x = torch.clamp(x, min=1e-6)
        return 3.0 * (torch.sin(x) - x * torch.cos(x)) / x**3


    def sigma_R(lnk, D2, R=8.0):
        W = tophat_W(torch.exp(lnk) * R)
        return torch.sqrt(torch.trapezoid(D2 * W**2, lnk, dim=-1))


    # ---------------------------------------------------------------- Palatini-Jeans sector
    def jeans_shape(d, a):
        """S(a) = a^2 R(a)^{n+2} with R/(3H0^2) = Om a^-3 + 4 OL. k_J^2(a) proportional to S(a)."""
        n = d["nidx"]
        return a**2 * (d["Om"] * a**-3 + 4.0 * d["OL"]) ** (n + 2.0)


    def jeans_amin(d):
        """Epoch of minimum comoving Jeans wavenumber: Om a*^-3 = 8 OL/(3n+4) (clipped to a<=1)."""
        n = d["nidx"]
        astar = (d["Om"] * (3.0 * n + 4.0) / (8.0 * d["OL"])) ** (1.0 / 3.0)
        return torch.clamp(astar, max=1.0)


    def kJ2_of_a(d, a):
        """Comoving Jeans wavenumber^2 [(h/Mpc)^2]; parameter u = log10 of its minimum over a<=1."""
        am = jeans_amin(d)
        kJs = 10.0 ** d["u"]
        return kJs**2 * jeans_shape(d, a) / jeans_shape(d, am)


    def mu_palatini(d, k, a, model):
        """mu(k,a) multiplying 4piG rho delta in the growth equation (quasi-static, k>>aH)."""
        if model == "lcdm":
            return torch.ones_like(k * a)
        x = k**2 / kJ2_of_a(d, a)
        Fm1 = (a * a / CH0**2) * 3.0 * (d["Om"] * a**-3 + 4.0 * d["OL"]) / ((d["nidx"] + 1.0) * kJ2_of_a(d, a))
        return (1.0 - x) / (1.0 + Fm1)


    def eps_background(d, a):
        """epsilon = dlnF/dlna = 9 Omega_m(a) (aH/k_J)^2 (consistency relation)."""
        E2 = E2_of_a(d, a)
        Om_a = d["Om"] * a**-3 / E2
        aH = a * torch.sqrt(E2) / CH0         # h/Mpc
        return 9.0 * Om_a * aH**2 / kJ2_of_a(d, a)


    def growth_xgrid(z_out, a_init=0.02, a_switch=0.2, n_coarse=120, n_fine=400, dtype=torch.float64, device="cpu"):
        """Non-uniform ln a grid: coarse while the Jeans scale is far in the UV, fine at late times.
        Output redshifts are inserted as exact nodes. Returns (xg, out_idx)."""
        xc = torch.linspace(math.log(a_init), math.log(a_switch), n_coarse + 1, dtype=dtype, device=device)
        xf = torch.linspace(math.log(a_switch), 0.0, n_fine + 1, dtype=dtype, device=device)
        x_out = -torch.log1p(z_out.to(dtype=dtype, device=device))
        xg, inv = torch.unique(torch.cat([xc, xf, x_out]), sorted=True, return_inverse=True)
        return xg, inv[xc.numel() + xf.numel():]


    def solve_growth(d, kode, z_out, model, a_init=0.02, a_switch=0.2, n_coarse=120, n_fine=400):
        """RK4 in x = ln a for  delta'' + (2 + dlnH/dlna) delta' - 1.5 Omega_cb(a) mu(k,a) delta = 0.
        Column 0 is k = 0 (the GR reference, mu = 1/F -> 1). Growing-mode ICs delta = delta' = a_init.
        All background coefficients are precomputed on nodes and midpoints, so the loop is pure
        element-wise algebra (few kernel launches per step on GPU).
        Returns D_GR (B,Nz), G = D/D_GR (B,Nk,Nz), Gp = D'/D_GR (B,Nk,Nz), f_GR (B,Nz)."""
        dev, dt = kode.device, kode.dtype
        B = d["h"].shape[0]
        kk = kode if model != "lcdm" else kode[:0]
        kk = torch.cat([torch.zeros(1, dtype=dt, device=dev), kk]).view(1, -1, 1)      # (1,Nk+1,1)
        xg, out_idx = growth_xgrid(z_out, a_init, a_switch, n_coarse, n_fine, dt, dev)
        xm = 0.5 * (xg[1:] + xg[:-1])
        xall = torch.stack([xg[:-1], xm], dim=1).reshape(-1)
        xall = torch.cat([xall, xg[-1:]])                                             # node,mid,node,...
        a = torch.exp(xall).unsqueeze(0)                                               # (1,M)
        E2 = E2_of_a(d, a)
        Acoef = (2.0 + dlnE_dlna(d, a)).unsqueeze(1)                                   # (B,1,M)
        Bcoef = (1.5 * d["Ocb"] * a**-3 / E2).unsqueeze(1)
        if model == "lcdm":
            Ccoef = Bcoef.expand(B, kk.shape[1], -1)
        else:
            kJ2 = kJ2_of_a(d, a).unsqueeze(1)                                          # (B,1,M)
            Fm1 = (a * a / CH0**2).unsqueeze(1) * 3.0 * (d["Om"].unsqueeze(1) * a.unsqueeze(1)**-3
                   + 4.0 * d["OL"].unsqueeze(1)) / ((d["nidx"].unsqueeze(1) + 1.0) * kJ2)
            Ccoef = Bcoef * (1.0 - kk**2 / kJ2) / (1.0 + Fm1)                          # (B,Nk+1,M)
        y = torch.full((B, kk.shape[1]), a_init, dtype=dt, device=dev)
        yp = y.clone()
        nout = len(out_idx)
        _oi = out_idx.tolist()
        Ys = [None] * nout
        YPs = [None] * nout
        def _store(i, y_, yp_):
            for n, j in enumerate(_oi):
                if j == i:
                    Ys[n], YPs[n] = y_, yp_
        _store(0, y, yp)
        hs = (xg[1:] - xg[:-1]).tolist()
        for i, h_ in enumerate(hs):
            A0, Am, A1 = Acoef[..., 2 * i], Acoef[..., 2 * i + 1], Acoef[..., 2 * i + 2]
            C0, Cm, C1 = Ccoef[..., 2 * i], Ccoef[..., 2 * i + 1], Ccoef[..., 2 * i + 2]
            k1y, k1p = yp, C0 * y - A0 * yp
            y2, p2 = y + 0.5 * h_ * k1y, yp + 0.5 * h_ * k1p
            k2y, k2p = p2, Cm * y2 - Am * p2
            y3, p3 = y + 0.5 * h_ * k2y, yp + 0.5 * h_ * k2p
            k3y, k3p = p3, Cm * y3 - Am * p3
            y4, p4 = y + h_ * k3y, yp + h_ * k3p
            k4y, k4p = p4, C1 * y4 - A1 * p4
            y = y + (h_ / 6.0) * (k1y + 2.0 * k2y + 2.0 * k3y + k4y)
            yp = yp + (h_ / 6.0) * (k1p + 2.0 * k2p + 2.0 * k3p + k4p)
            _store(i + 1, y, yp)
        Y = torch.stack(Ys, dim=-1)
        YP = torch.stack(YPs, dim=-1)
        DGR = Y[:, 0, :]
        G = Y[:, 1:, :] / DGR.unsqueeze(1)          # delta(k)/D_GR
        Gp = YP[:, 1:, :] / DGR.unsqueeze(1)        # delta'(k)/D_GR  (theta-delta cross uses G*Gp)
        fGR = YP[:, 0, :] / DGR
        return DGR, G, Gp, fGR


    def interp_logk(kode, Gk, kq):
        """Interpolate (B,Nk,Nz) in ln k onto kq (Nq,); constant extrapolation (G->1 at low k)."""
        lk, lq = torch.log(kode), torch.log(kq).clamp(min=math.log(kode[0].item()), max=math.log(kode[-1].item()))
        idx = torch.searchsorted(lk, lq).clamp(1, lk.numel() - 1)
        w = ((lq - lk[idx - 1]) / (lk[idx] - lk[idx - 1])).view(1, -1, 1)
        return Gk[:, idx - 1, :] + w * (Gk[:, idx, :] - Gk[:, idx - 1, :])


    # ---------------------------------------------------------------- halofit (Takahashi+2012)
    def _halofit_formula(lnk, D2lin, Rs, I1, I2, Om_z):
        k = torch.exp(lnk)
        neff = -3.0 + 2.0 * I1
        C = 4.0 * (I1 - I2 + I1**2)
        n, n2, n3, n4 = neff, neff**2, neff**3, neff**4
        an = 10 ** (1.5222 + 2.8553 * n + 2.3706 * n2 + 0.9903 * n3 + 0.2250 * n4 - 0.6038 * C)
        bn = 10 ** (-0.5642 + 0.5864 * n + 0.5716 * n2 - 1.5474 * C)
        cn = 10 ** (0.3698 + 2.0404 * n + 0.8161 * n2 + 0.5869 * C)
        gn = 0.1971 - 0.0843 * n + 0.8460 * C
        alpn = torch.abs(6.0835 + 1.3373 * n - 0.1959 * n2 - 5.5274 * C)
        betn = 2.0379 - 0.7354 * n + 0.3157 * n2 + 1.2490 * n3 + 0.3980 * n4 - 0.1682 * C
        nun = 10 ** (5.2105 + 3.6902 * n)
        f1, f2, f3 = Om_z ** -0.0307, Om_z ** -0.0585, Om_z ** 0.0743
        ex = lambda t: t.unsqueeze(-1)
        yk = k.view(1, 1, -1) * Rs
        fy = yk / 4.0 + yk**2 / 8.0
        DQ = D2lin * (1.0 + D2lin) ** ex(betn) / (1.0 + ex(alpn) * D2lin) * torch.exp(-fy)
        DHp = ex(an) * yk ** (3.0 * ex(f1)) / (1.0 + ex(bn) * yk ** ex(f2) + (ex(cn) * ex(f3) * yk) ** (3.0 - ex(gn)))
        DH = DHp / (1.0 + ex(nun) / yk**2)
        return DQ + DH


    def _rsigma_moments(lnk, D2lin, ls2, lnR):
        """Root of ln sigma^2(R)=0 on the R grid, then the spectral moments at R_sigma."""
        k = torch.exp(lnk)
        neg = (ls2 < 0).to(torch.int64)
        i1 = torch.clamp(torch.argmax(neg, dim=-1), 1, lnR.numel() - 1)
        l0 = torch.gather(ls2, 2, (i1 - 1).unsqueeze(-1)).squeeze(-1)
        l1 = torch.gather(ls2, 2, i1.unsqueeze(-1)).squeeze(-1)
        lnRs = lnR[i1 - 1] + (lnR[i1] - lnR[i1 - 1]) * l0 / (l0 - l1)
        Rs = torch.exp(lnRs).unsqueeze(-1)
        y2 = (k.view(1, 1, -1) * Rs) ** 2
        e = torch.exp(-y2)
        I0 = torch.trapezoid(D2lin * e, lnk, dim=-1)
        I1 = torch.trapezoid(D2lin * y2 * e, lnk, dim=-1) / I0
        I2 = torch.trapezoid(D2lin * y2**2 * e, lnk, dim=-1) / I0
        return Rs, I1, I2


    def halofit_lcdm(d, lnk, D2lin0, gz, Om_z, OL_z):
        """Takahashi+2012 halofit for the LCDM reference. D2lin0 (B,Nk) at z=0, gz=D(z)/D(0) (B,Nz).
        sigma^2(R,z) = gz^2 sigma^2(R,0) because the reference growth is scale independent, so the
        R-scan is done once at z=0. Returns (Delta^2_NL, Delta^2_lin), each (B,Nz,Nk). w = -1."""
        k = torch.exp(lnk)
        lnR = torch.linspace(math.log(1e-3), math.log(40.0), 200, dtype=lnk.dtype, device=lnk.device)
        g = torch.exp(-(k.view(1, -1) * torch.exp(lnR).view(-1, 1)) ** 2)        # (NR,Nk)
        s20 = torch.trapezoid(D2lin0.unsqueeze(1) * g.unsqueeze(0), lnk, dim=-1)  # (B,NR)
        ls2 = torch.log(s20).unsqueeze(1) + 2.0 * torch.log(gz).unsqueeze(-1)     # (B,Nz,NR)
        D2lin = D2lin0.unsqueeze(1) * gz.unsqueeze(-1) ** 2
        Rs, I1, I2 = _rsigma_moments(lnk, D2lin, ls2, lnR)
        return _halofit_formula(lnk, D2lin, Rs, I1, I2, Om_z), D2lin


    # ---------------------------------------------------------------- lensing (Limber)
    def smail_nz(z, zmean):
        z0 = zmean / 1.50458
        return z**2 * torch.exp(-(z / z0) ** 1.5)


    def lensing_kernels(d, zl, chil, nz_list, z_cmb_chi=None):
        """Shear/CMB-kappa lensing efficiencies on z nodes. chil (B,Nz) in Mpc/h."""
        a = 1.0 / (1.0 + zl)
        pref = 1.5 * d["Om"] / CH0**2 * chil / a                              # (B,Nz)
        out = []
        dz = torch.gradient(zl)[0]
        for nz in nz_list:
            nzn = nz / torch.sum(nz * dz)
            ratio = 1.0 - chil.unsqueeze(2) / chil.unsqueeze(1)               # [i,j] = 1 - chi_i/chi_j (lens i, source j)
            _ar = torch.arange(zl.numel(), device=zl.device)
            mask = (_ar.view(1, -1) >= _ar.view(-1, 1)).to(zl.dtype)
            g = torch.sum(ratio.clamp(min=0) * mask * (nzn * dz).view(1, 1, -1), dim=-1)
            out.append(pref * g)
        if z_cmb_chi is not None:
            out.append(pref * (1.0 - chil / z_cmb_chi).clamp(min=0))
        return out


    def limber_cl(d, ells, zl, chil, kern, lnk, D2nl):
        """C_ell = int dchi q^2/chi^2 P(k=(l+1/2)/chi). D2nl (B,Nz,Nk) dimensionless."""
        E = torch.sqrt(E2_of_a(d, 1.0 / (1.0 + zl)))
        dchi_dz = CH0 / E                                                     # Mpc/h
        kq = (ells.view(1, 1, -1) + 0.5) / chil.unsqueeze(-1)                 # (B,Nz,Nl)
        lq = torch.log(kq).clamp(lnk[0].item(), lnk[-1].item())
        idx = torch.searchsorted(lnk, lq.contiguous()).clamp(1, lnk.numel() - 1)
        lD = torch.log(D2nl.clamp(min=1e-300))
        l0 = torch.gather(lD, 2, idx - 1)
        l1 = torch.gather(lD, 2, idx)
        w = (lq - lnk[idx - 1]) / (lnk[idx] - lnk[idx - 1])
        D2q = torch.exp(l0 + w * (l1 - l0))
        Pq = D2q * 2.0 * math.pi**2 / kq**3
        integ = (kern**2 / chil**2 * dchi_dz).unsqueeze(-1) * Pq
        return torch.trapezoid(integ, zl, dim=1)                              # (B,Nl)


    # ================================================================ data
    BAO_BASE = "https://raw.githubusercontent.com/CobayaSampler/bao_data/master/"
    PP_BASE = "https://raw.githubusercontent.com/PantheonPlusSH0ES/DataRelease/main/Pantheon%2B_Data/4_DISTANCES_AND_COVAR/"
    DATA_FILES = {
        "desi_mean": BAO_BASE + "desi_bao_dr2/desi_gaussian_bao_ALL_GCcomb_mean.txt",
        "desi_cov": BAO_BASE + "desi_bao_dr2/desi_gaussian_bao_ALL_GCcomb_cov.txt",
        "lrg16_mean": BAO_BASE + "sdss_DR16_BAOplus_LRG_FSBAO_DMDHfs8.dat",
        "lrg16_cov": BAO_BASE + "sdss_DR16_BAOplus_LRG_FSBAO_DMDHfs8_covtot.txt",
        "qso16_mean": BAO_BASE + "sdss_DR16_BAOplus_QSO_FSBAO_DMDHfs8.dat",
        "qso16_cov": BAO_BASE + "sdss_DR16_BAOplus_QSO_FSBAO_DMDHfs8_covtot.txt",
        "pp_dat": PP_BASE + "Pantheon%2BSH0ES.dat",
        "pp_cov": PP_BASE + "Pantheon%2BSH0ES_STAT%2BSYS.cov",
    }

    # Published summary likelihoods (values transcribed from the cited papers; see notebook text)
    PLANCK_DP_MEAN = [1.7502, 301.471, 0.02236, 0.9649]          # Chen, Huang & Wang 2019, TT,TE,EE+lowE
    PLANCK_DP_SIG = [0.0046, 0.0895, 0.00015, 0.0043]
    PLANCK_DP_CORR = [[1.0, 0.46, -0.66, -0.74], [0.46, 1.0, -0.33, -0.35],
                      [-0.66, -0.33, 1.0, 0.46], [-0.74, -0.35, 0.46, 1.0]]
    PLANCK_LNAS = (3.045, 0.016)                                   # Planck 2018 VI, TT,TE,EE+lowE
    PLANCK_MEAN_THETA = [0.6727, 0.02236, 0.1202, 3.045, 0.9649]   # cosmology used to calibrate (R, l_A) conventions
    FS8_6DF = (0.067, 0.423, 0.055)                                # Beutler et al. 2012
    LENSING = {   # name: (value, sigma_plus, sigma_minus, kind)
        "DESY6_shear": (0.783, 0.019, 0.015, "shear"),             # DES Y6 cosmic shear, LCDM S8
        "KiDS_Legacy": (0.815, 0.016, 0.021, "shear"),             # KiDS-Legacy cosmic shear, LCDM S8
        "ACT_Planck_CMBL": (0.813, 0.018, 0.018, "cmbl"),          # ACT DR6 + Planck lensing, S8^CMBL
    }


    def fetch(name, cache_dir, url=None):
        import os, urllib.request
        url = url or DATA_FILES[name]
        path = os.path.join(cache_dir, url.split("/")[-1].replace("%2B", "+"))
        if not os.path.exists(path):
            tmp = path + ".part"
            urllib.request.urlretrieve(url, tmp)
            os.replace(tmp, path)
        return path


    def load_data(cache_dir, zmin_sn=0.01):
        import os
        os.makedirs(cache_dir, exist_ok=True)
        out = {}
        # DESI DR2 BAO
        rows = [l.split() for l in open(fetch("desi_mean", cache_dir)) if l.strip() and not l.startswith("#")]
        out["bao_z"] = np.array([float(r[0]) for r in rows])
        out["bao_val"] = np.array([float(r[1]) for r in rows])
        out["bao_kind"] = [r[2] for r in rows]
        out["bao_cov"] = np.loadtxt(fetch("desi_cov", cache_dir))
        # RSD f sigma8: 6dFGS + SDSS DR16 (LRG incl. DR12 z=0.38,0.51; QSO). Marginal fs8 blocks.
        fz, fv, blocks = [FS8_6DF[0]], [FS8_6DF[1]], [np.array([[FS8_6DF[2] ** 2]])]
        for tag in ["lrg16", "qso16"]:
            rr = [l.split() for l in open(fetch(tag + "_mean", cache_dir)) if l.strip() and not l.startswith("#")]
            cv = np.atleast_2d(np.loadtxt(fetch(tag + "_cov", cache_dir)))
            idx = [i for i, r in enumerate(rr) if r[2] == "f_sigma8"]
            fz += [float(rr[i][0]) for i in idx]
            fv += [float(rr[i][1]) for i in idx]
            blocks.append(cv[np.ix_(idx, idx)])
        nf = sum(b.shape[0] for b in blocks)
        C = np.zeros((nf, nf)); o = 0
        for b in blocks:
            C[o:o + b.shape[0], o:o + b.shape[0]] = b; o += b.shape[0]
        out["fs8_z"], out["fs8_val"], out["fs8_cov"] = np.array(fz), np.array(fv), C
        # Pantheon+ (uncalibrated: SH0ES Cepheid distances NOT used; absolute magnitude marginalised)
        p = fetch("pp_dat", cache_dir)
        hdr = open(p).readline().split()
        tab = np.genfromtxt(p, names=hdr, dtype=None, encoding=None, skip_header=1)
        cov = np.loadtxt(fetch("pp_cov", cache_dir))
        n = int(cov[0]); cov = cov[1:].reshape(n, n)
        m = tab["zHD"] > zmin_sn
        out["sn_zhd"], out["sn_zhel"], out["sn_mb"] = tab["zHD"][m], tab["zHEL"][m], tab["m_b_corr"][m]
        out["sn_cov"] = cov[np.ix_(m, m)]
        return out


    def split_normal_chi2(x, mean, sp, sm):
        s = torch.where(x > mean, torch.as_tensor(sp, dtype=x.dtype, device=x.device),
                        torch.as_tensor(sm, dtype=x.dtype, device=x.device))
        return ((x - mean) / s) ** 2


    # ================================================================ pipeline
    class PalatiniJeansLikelihood:
        """Batched likelihood for LCDM and the Palatini-Jeans sector on a common, exact background."""

        def __init__(self, data, device="cpu", nk_ode=96, nk=600, nzl=64, prescription="ratio",
                     shear_zmean=None, use=None, calib_k=None, calib_ratio=None, n_coarse=120, n_fine=400):
            self.dev, self.dt = torch.device(device), torch.float64
            T = lambda x: torch.as_tensor(np.asarray(x), dtype=self.dt, device=self.dev)
            self.T = T
            self.use = dict(bao=True, sn=True, cmb=True, rsd=True, DESY6_shear=True, KiDS_Legacy=True,
                            ACT_Planck_CMBL=True) | (use or {})
            self.prescription = prescription
            self.n_coarse, self.n_fine = n_coarse, n_fine
            self.zg, self.lnzg = make_zgrid(dtype=self.dt, device=self.dev)
            # BAO
            self.bao_z = T(data["bao_z"]); self.bao_val = T(data["bao_val"])
            self.bao_kind = data["bao_kind"]
            self.bao_icov = T(np.linalg.inv(data["bao_cov"]))
            # RSD
            self.fs8_z = T(data["fs8_z"]); self.fs8_val = T(data["fs8_val"])
            self.fs8_icov = T(np.linalg.inv(data["fs8_cov"]))
            # SN
            self.sn_zhd = T(data["sn_zhd"]); self.sn_zhel = T(data["sn_zhel"]); self.sn_mb = T(data["sn_mb"])
            self.sn_icov = T(np.linalg.inv(data["sn_cov"]))
            self.sn_isum = self.sn_icov.sum()
            # CMB compressed
            sig = np.array(PLANCK_DP_SIG)
            self.dp_mean = T(PLANCK_DP_MEAN)
            self.dp_icov = T(np.linalg.inv(np.array(PLANCK_DP_CORR) * np.outer(sig, sig)))
            # grids
            self.kode = torch.logspace(-3, 1, nk_ode, dtype=self.dt, device=self.dev)
            self.lnk = torch.linspace(math.log(1e-4), math.log(200.0), nk, dtype=self.dt, device=self.dev)
            self.k = torch.exp(self.lnk)
            self.zl = torch.expm1(torch.linspace(math.log(1.01), math.log(13.0), nzl, dtype=self.dt, device=self.dev))
            self.z_out = torch.cat([T([0.0]), self.fs8_z, self.zl])
            self.i_rsd = slice(1, 1 + self.fs8_z.numel())
            self.i_l = slice(1 + self.fs8_z.numel(), None)
            zm = shear_zmean or {"DESY6_shear": 0.63, "KiDS_Legacy": 0.75}
            self.nz = {kname: smail_nz(self.zl, zm[kname]) for kname in ["DESY6_shear", "KiDS_Legacy"]}
            self.ell_shear = torch.logspace(math.log10(100), math.log10(1500), 20, dtype=self.dt, device=self.dev)
            self.ell_cmbl = torch.logspace(math.log10(40), math.log10(763), 16, dtype=self.dt, device=self.dev)
            nbar = 6.0 * (180 * 60 / math.pi) ** 2                     # 6 gal/arcmin^2 -> per sr
            self.noise = {"DESY6_shear": 0.26**2 / nbar, "KiDS_Legacy": 0.26**2 / nbar, "ACT_Planck_CMBL": 5e-8}
            self.calib = None
            if calib_k is not None:
                self.calib = torch.exp(torch.as_tensor(np.interp(np.log(self.k.cpu().numpy()), np.log(calib_k),
                                                                 np.log(calib_ratio)), dtype=self.dt, device=self.dev)).unsqueeze(0)
            self.cmb_scale = torch.ones(2, dtype=self.dt, device=self.dev)
            self.gamma = {"DESY6_shear": 1.0, "KiDS_Legacy": 1.0, "ACT_Planck_CMBL": 1.0}
            self._calibrate()

        # -------------------------------------------------------- calibrations at the Planck mean
        def _calibrate(self):
            th = self.T([PLANCK_MEAN_THETA + [1.5, 1.0]])
            d = unpack(th)
            R, lA = self._cmb_raw(d)
            self.cmb_scale = torch.stack([self.dp_mean[0] / R[0, 0], self.dp_mean[1] / lA[0, 0]])
            # exponents gamma = dln C / dln sigma8 for each lensing probe (LCDM, fixed Omega_m)
            amps = []
            for dl in (-0.02, 0.02):
                t2 = th.clone(); t2[0, 3] += dl
                amps.append(self._lensing_amp(t2, "lcdm", raw=True))
            for kname in self.gamma:
                self.gamma[kname] = float((torch.log(amps[1][kname]) - torch.log(amps[0][kname])).item() / 0.02)

        def _cmb_raw(self, d):
            chi = comoving_chi(d, self.zg, self.lnzg)
            zs = z_star(d)
            chis = interp_fixed(self.lnzg, chi, torch.log1p(zs))
            rs = sound_horizon(d, 1.0 / (1.0 + zs))
            R = torch.sqrt(d["Om"]) * chis
            lA = math.pi * (CH0 / d["h"]) * chis / rs
            return R, lA

        # -------------------------------------------------------- linear + nonlinear spectra
        def _spectra(self, d, model):
            DGR, G, Gp, fGR = solve_growth(d, self.kode, self.z_out, model,
                                           n_coarse=self.n_coarse, n_fine=self.n_fine)
            D0 = DGR[:, :1]
            D2lin0 = delta2_lin0(d, self.k.unsqueeze(0), D0, self.calib)              # (B,Nk)
            gz = DGR / D0                                                              # (B,Nz_out)
            if model == "lcdm":
                Gk = torch.ones(DGR.shape[0], self.k.numel(), DGR.shape[1], dtype=self.dt, device=self.dev)
                Gpk = Gk * fGR.unsqueeze(1)
            else:
                Gk = interp_logk(self.kode, G, self.k)
                Gpk = interp_logk(self.kode, Gp, self.k)
                lowk = (self.k < self.kode[0]).view(1, -1, 1)
                Gk = torch.where(lowk, torch.ones_like(Gk), Gk)
                Gpk = torch.where(lowk, fGR.unsqueeze(1).expand_as(Gpk), Gpk)
            return D2lin0, gz, Gk, Gpk, fGR

        def _lensing_amp(self, theta, model, raw=False, D=None):
            d = unpack(theta) if D is None else D
            D2lin0, gz, Gk, Gpk, fGR = self._spectra(d, model)
            return self._lensing_from_spectra(d, D2lin0, gz, Gk, model, raw)

        def _lensing_from_spectra(self, d, D2lin0, gz, Gk, model, raw=False):
            zl = self.zl
            gl = gz[:, self.i_l]
            E2 = E2_of_a(d, 1.0 / (1.0 + zl))
            Omz = d["Om"] * (1.0 + zl) ** 3 / E2
            OLz = d["OL"] / E2
            D2nl_ref, D2lin_ref = halofit_lcdm(d, self.lnk, D2lin0, gl, Omz, OLz)
            chi = comoving_chi(d, self.zg, self.lnzg)
            chil = interp_fixed(self.lnzg, chi, torch.log1p(zl)) * CH0
            zs = z_star(d)
            chis = interp_fixed(self.lnzg, chi, torch.log1p(zs)) * CH0
            kerns = lensing_kernels(d, zl, chil, [self.nz["DESY6_shear"], self.nz["KiDS_Legacy"]], z_cmb_chi=chis)
            names = ["DESY6_shear", "KiDS_Legacy", "ACT_Planck_CMBL"]
            ells = [self.ell_shear, self.ell_shear, self.ell_cmbl]
            if raw:
                return {nm: limber_cl(d, el, zl, chil, kn, self.lnk, D2nl_ref).mean(dim=1)
                        for nm, el, kn in zip(names, ells, kerns)}
            G2 = Gk[:, :, self.i_l].transpose(1, 2) ** 2                              # (B,Nz,Nk)
            if model == "lcdm":
                D2nl_mod = D2nl_ref
            elif self.prescription == "ratio":       # upper bracket: whole nonlinear spectrum suppressed
                D2nl_mod = D2nl_ref * G2
            else:   # "linear_only": the one-halo (screened) part is left untouched
                D2nl_mod = (D2nl_ref + D2lin_ref * (G2 - 1.0)).clamp(min=1e-30)
            ratio = {}
            for nm, el, kn in zip(names, ells, kerns):
                Cr = limber_cl(d, el, zl, chil, kn, self.lnk, D2nl_ref)
                Cm = Cr if model == "lcdm" else limber_cl(d, el, zl, chil, kn, self.lnk, D2nl_mod)
                w = (2 * el + 1).unsqueeze(0) * Cr**2 / (Cr + self.noise[nm]) ** 2
                ratio[nm] = (w * Cm / Cr).sum(1) / w.sum(1)
            return ratio

        def ratio_for_nz(self, theta, model, zmeans):
            """Weighted shear-amplitude ratio C_mod/C_LCDM for Smail n(z) with the given mean redshifts,
            converted to an effective S8 with the DES exponent. Returns (B, len(zmeans))."""
            d = unpack(theta)
            D2lin0, gz, Gk, Gpk, fGR = self._spectra(d, model)
            zl = self.zl
            E2 = E2_of_a(d, 1.0 / (1.0 + zl))
            Omz = d["Om"] * (1.0 + zl) ** 3 / E2
            D2nl_ref, D2lin_ref = halofit_lcdm(d, self.lnk, D2lin0, gz[:, self.i_l], Omz, d["OL"] / E2)
            G2 = Gk[:, :, self.i_l].transpose(1, 2) ** 2
            if model == "lcdm":
                D2nl_mod = D2nl_ref
            elif self.prescription == "ratio":
                D2nl_mod = D2nl_ref * G2
            else:
                D2nl_mod = (D2nl_ref + D2lin_ref * (G2 - 1.0)).clamp(min=1e-30)
            chi = comoving_chi(d, self.zg, self.lnzg)
            chil = interp_fixed(self.lnzg, chi, torch.log1p(zl)) * CH0
            kerns = lensing_kernels(d, zl, chil, [smail_nz(zl, zm) for zm in zmeans])
            W8 = tophat_W(self.k * 8.0).unsqueeze(0) ** 2
            S8_ref = torch.sqrt(torch.trapezoid(D2lin0 * W8, self.lnk, dim=1)) * torch.sqrt(d["Om"][:, 0] / 0.3)
            outs = []
            el = self.ell_shear
            for kn in kerns:
                Cr = limber_cl(d, el, zl, chil, kn, self.lnk, D2nl_ref)
                Cm = limber_cl(d, el, zl, chil, kn, self.lnk, D2nl_mod)
                w = (2 * el + 1).unsqueeze(0) * Cr**2 / (Cr + self.noise["DESY6_shear"]) ** 2
                r = (w * Cm / Cr).sum(1) / w.sum(1)
                outs.append(S8_ref * r ** (1.0 / self.gamma["DESY6_shear"]))
            return torch.stack(outs, dim=1)

        # -------------------------------------------------------- full evaluation
        def evaluate(self, theta, model="palatini", want_extra=False):
            d = unpack(theta)
            B = theta.shape[0]
            out = {}
            chi = comoving_chi(d, self.zg, self.lnzg)
            # BAO
            rd = r_drag(d)
            cz = interp_fixed(self.lnzg, chi, torch.log1p(self.bao_z))
            DM = CH0 / d["h"] * cz
            DH = CH0 / (d["h"] * torch.sqrt(E2_of_a(d, 1.0 / (1.0 + self.bao_z))))
            DV = (self.bao_z * DM**2 * DH) ** (1.0 / 3.0)
            pred = torch.stack([{"DM_over_rs": DM, "DH_over_rs": DH, "DV_over_rs": DV}[kd][:, i]
                                for i, kd in enumerate(self.bao_kind)], dim=1) / rd
            r = pred - self.bao_val
            out["chi2_bao"] = torch.einsum("bi,ij,bj->b", r, self.bao_icov, r)
            # SN (analytic marginalisation of the absolute magnitude)
            csn = interp_fixed(self.lnzg, chi, torch.log1p(self.sn_zhd))
            mu = 5.0 * torch.log10((1.0 + self.sn_zhel) * csn * CH0 / d["h"]) + 25.0
            r = self.sn_mb - mu
            Ar = r @ self.sn_icov
            out["chi2_sn"] = (Ar * r).sum(1) - Ar.sum(1) ** 2 / self.sn_isum
            # CMB compressed (+ amplitude)
            R, lA = self._cmb_raw(d)
            v = torch.cat([R * self.cmb_scale[0], lA * self.cmb_scale[1], d["ombh2"], d["ns"]], dim=1) - self.dp_mean
            out["chi2_cmb"] = torch.einsum("bi,ij,bj->b", v, self.dp_icov, v) + \
                ((d["lnAs"][:, 0] - PLANCK_LNAS[0]) / PLANCK_LNAS[1]) ** 2
            # growth, sigma8, f sigma8
            D2lin0, gz, Gk, Gpk, fGR = self._spectra(d, model)
            W8 = tophat_W(self.k * 8.0).unsqueeze(0) ** 2
            s8_ref0 = torch.sqrt(torch.trapezoid(D2lin0 * W8, self.lnk, dim=1))
            s8_mod0 = torch.sqrt(torch.trapezoid(D2lin0 * W8 * Gk[:, :, 0] ** 2, self.lnk, dim=1))
            gr = gz[:, self.i_rsd]                                                     # (B,Nr)
            D2r = D2lin0.unsqueeze(-1) * gr.unsqueeze(1) ** 2                         # (B,Nk,Nr)
            Gr, Gpr = Gk[:, :, self.i_rsd], Gpk[:, :, self.i_rsd]
            s8z = torch.sqrt(torch.trapezoid(D2r * Gr**2 * W8.unsqueeze(-1), self.lnk, dim=1))
            fs8 = torch.trapezoid(D2r * Gr * Gpr * W8.unsqueeze(-1), self.lnk, dim=1) / s8z
            r = fs8 - self.fs8_val
            out["chi2_rsd"] = torch.einsum("bi,ij,bj->b", r, self.fs8_icov, r)
            # lensing
            ratio = self._lensing_from_spectra(d, D2lin0, gz, Gk, model)
            S8_ref = s8_ref0 * torch.sqrt(d["Om"][:, 0] / 0.3)
            S8c_ref = s8_ref0 * (d["Om"][:, 0] / 0.3) ** 0.25
            out["S8_eff_DESY6_shear"] = S8_ref * ratio["DESY6_shear"] ** (1.0 / self.gamma["DESY6_shear"])
            out["S8_eff_KiDS_Legacy"] = S8_ref * ratio["KiDS_Legacy"] ** (1.0 / self.gamma["KiDS_Legacy"])
            out["S8_eff_ACT_Planck_CMBL"] = S8c_ref * ratio["ACT_Planck_CMBL"] ** (1.0 / self.gamma["ACT_Planck_CMBL"])
            for nm, (val, sp, sm, _) in LENSING.items():
                out["chi2_" + nm] = split_normal_chi2(out["S8_eff_" + nm], val, sp, sm)
            tot = torch.zeros(B, dtype=self.dt, device=self.dev)
            for key in ["bao", "sn", "cmb", "rsd", "DESY6_shear", "KiDS_Legacy", "ACT_Planck_CMBL"]:
                if self.use.get(key, True):
                    tot = tot + out["chi2_" + key]
            out["chi2"] = tot
            out["sigma8_lin_LCDMequiv"] = s8_ref0
            out["sigma8_lin_true"] = s8_mod0
            out["S8_LCDMequiv"] = S8_ref
            out["Om"] = d["Om"][:, 0]
            out["H0"] = 100 * d["h"][:, 0]
            out["rd"] = rd[:, 0]
            if want_extra:
                out["fs8_pred"] = fs8
                out["Gk"], out["Gpk"], out["gz"] = Gk, Gpk, gz
            return out


    # ================================================================ samplers
    def in_prior(x, lo, hi):
        return torch.all((x >= lo) & (x <= hi), dim=1)


    def stretch_sampler(logp_fn, x0, n_iter, a=2.0, gen=None, temp_fn=None, callback=None, start_iter=0):
        """Goodman & Weare affine-invariant stretch move, batched halves (emcee's red-blue scheme).
        logp_fn(X) -> log posterior (B,). temp_fn(it) -> temperature (annealing); None = 1.
        Returns chain (n_iter, W, d), logp chain (n_iter, W), acceptance fraction."""
        W, dim = x0.shape
        x = x0.clone()
        lp = logp_fn(x)
        chain = torch.empty((n_iter, W, dim), dtype=x.dtype, device=x.device)
        lpc = torch.empty((n_iter, W), dtype=x.dtype, device=x.device)
        acc = torch.zeros(W, dtype=x.dtype, device=x.device)
        half = W // 2
        for it in range(n_iter):
            T = 1.0 if temp_fn is None else temp_fn(start_iter + it)
            for s in (0, 1):
                act = slice(0, half) if s == 0 else slice(half, W)
                oth = slice(half, W) if s == 0 else slice(0, half)
                xa, xo = x[act], x[oth]
                nA = xa.shape[0]
                zz = ((a - 1.0) * torch.rand(nA, 1, dtype=x.dtype, device=x.device, generator=gen) + 1.0) ** 2 / a
                j = torch.randint(0, xo.shape[0], (nA,), device=x.device, generator=gen)
                prop = xo[j] + zz * (xa - xo[j])
                lpp = logp_fn(prop)
                logr = (dim - 1) * torch.log(zz[:, 0]) + (lpp - lp[act]) / T
                u = torch.log(torch.rand(nA, dtype=x.dtype, device=x.device, generator=gen))
                ok = u < logr
                x[act] = torch.where(ok.unsqueeze(1), prop, xa)
                lp[act] = torch.where(ok, lpp, lp[act])
                acc[act] += ok.to(x.dtype)
            chain[it], lpc[it] = x, lp
            if callback is not None:
                callback(it, x, lp)
        return chain, lpc, acc / max(n_iter, 1), x, lp


    def autocorr_time(chain_1d, c=5.0):
        """Integrated autocorrelation time (Sokal windowing) of a (N, W) chain, walker-averaged ACF."""
        x = chain_1d - chain_1d.mean(0, keepdims=True)
        n = x.shape[0]
        f = np.fft.rfft(x, n=2 * n, axis=0)
        acf = np.fft.irfft(f * np.conj(f), axis=0)[:n].mean(1)
        acf /= acf[0]
        taus = 2.0 * np.cumsum(acf) - 1.0
        m = np.arange(len(taus)) < c * taus
        w = np.argmin(m) if np.any(~m) else len(taus) - 1
        return float(taus[w])


    def split_rhat(chain_1d, ngroups=4):
        """Gelman-Rubin R-hat on split walker groups (N, W)."""
        n, W = chain_1d.shape
        g = np.array_split(np.arange(W), ngroups)
        seqs = []
        for gi in g:
            s = chain_1d[:, gi].reshape(-1)
            h = len(s) // 2
            seqs += [s[:h], s[h:2 * h]]
        seqs = np.array(seqs)
        m, L = seqs.shape
        Bv = L * seqs.mean(1).var(ddof=1)
        Wv = seqs.var(1, ddof=1).mean()
        var = (L - 1) / L * Wv + Bv / L
        return float(np.sqrt(var / Wv))


    # ================================================================ exact Palatini background (check)
    def palatini_background_check(d, a, iters=6):
        """Exact Palatini Friedmann equation (H + Fdot/2F)^2 = (kappa rho + f)/(6F) for
        f = R - 2 Lambda - M R^-n, late-time dust + Lambda, units H0 = c = 1.
        M is fixed by the Jeans scale (k_J^2 = a^2 R^{n+2}/(n(n+1)M) at its minimum).
        Returns H/H_LCDM - 1, epsilon(a) numerical, epsilon from the consistency relation, F-1."""
        n = d["nidx"]
        Om, OL = d["Om"], d["OL"] + (1.0 - d["Om"] - d["OL"])     # dust + Lambda closure for this check
        Lam = 3.0 * OL
        am = jeans_amin(d)
        Ram = 3.0 * Om * am**-3 + 4.0 * Lam
        kJs = (10.0 ** d["u"]) * CH0                              # in H0/c units
        M = am**2 * Ram ** (n + 2.0) / (n * (n + 1.0) * kJs**2)

        def solve(aa):
            krho = 3.0 * Om * aa**-3
            R = krho + 4.0 * Lam
            for _ in range(iters):                                 # Newton on R - (n+2) M R^-n = krho + 4 Lam
                g = R - (n + 2.0) * M * R ** (-n) - krho - 4.0 * Lam
                dg = 1.0 + n * (n + 2.0) * M * R ** (-n - 1.0)
                R = R - g / dg
            F = 1.0 + n * M * R ** (-n - 1.0)
            f = R - 2.0 * Lam - M * R ** (-n)
            return krho, R, F, f

        krho, R, F, f = solve(a)
        h = 1e-4
        _, _, Fp, _ = solve(a * math.exp(h))
        _, _, Fm, _ = solve(a * math.exp(-h))
        eps = (torch.log1p(Fp - 1.0) - torch.log1p(Fm - 1.0)) / (2 * h)
        H2 = (krho + f) / (6.0 * F * (1.0 + 0.5 * eps) ** 2)
        H2_lcdm = Om * a**-3 + OL
        E2 = H2_lcdm
        eps_rel = 9.0 * (Om * a**-3 / E2) * (a * torch.sqrt(E2)) ** 2 / (kJs**2 * jeans_shape(d, a) / jeans_shape(d, am))
        return torch.sqrt(H2 / H2_lcdm) - 1.0, eps, eps_rel, F - 1.0


    def screening_factor(d, a, delta_env):
        """Relative Jeans-suppression strength in an environment of density contrast delta_env:
        (k_J,mean / k_J,local)^2 = [(Om a^-3 + 4 OL)/(Om a^-3 (1+delta) + 4 OL)]^(n+2)."""
        u = d["Om"] * a**-3
        L = 4.0 * d["OL"]
        return ((u + L) / (u * (1.0 + delta_env) + L)) ** (d["nidx"] + 2.0)
    return types.SimpleNamespace(**{_k: _v for _k, _v in locals().items() if not _k.startswith("__")})

PJ = _build_core()

In [ ]:
# ---- DATA: downloaded once into data_dir (public GitHub releases), then read from disk ------
obs = PJ.load_data(CFG["data_dir"])
_rows = [
    ("DESI DR2 BAO", f"{len(obs['bao_z'])} points, z={obs['bao_z'].min():.2f}-{obs['bao_z'].max():.2f}", "CobayaSampler/bao_data"),
    ("Pantheon+ (no SH0ES calibration)", f"{len(obs['sn_zhd'])} SNe, z>0.01, STAT+SYS cov, M marginalised", "PantheonPlusSH0ES/DataRelease"),
    ("f sigma8", ", ".join(f"{z:.3f}:{v:.3f}+-{s:.3f}" for z, v, s in zip(obs['fs8_z'], obs['fs8_val'], np.sqrt(np.diag(obs['fs8_cov'])))), "6dFGS + SDSS DR16 (marginal fs8 blocks)"),
    ("Planck 2018 compressed", "R, l_A, omega_b, n_s (Chen+19) + ln10^10As=3.045+-0.016", "transcribed"),
    ("DES Y6 cosmic shear", "S8 = 0.783 +0.019 -0.015", "DES Collab. 2026"),
    ("KiDS-Legacy cosmic shear", "S8 = 0.815 +0.016 -0.021", "Wright+2025"),
    ("ACT DR6 + Planck CMB lensing", "S8^CMBL = 0.813 +- 0.018", "Madhavacheril+/Qu+ 2024"),
]
mo.md("| dataset | content | source |\n|---|---|---|\n" + "\n".join(f"| {a} | {b} | {c} |" for a, b, c in _rows))

In [ ]:
# ---- CAMB CALIBRATION of the fast linear spectrum (once, cached) ---------------------------
# Eisenstein-Hu (no-wiggle) x [P_CAMB/P_EH at the Planck-mean cosmology]. The ratio carries BAO
# wiggles, neutrino and transfer-function corrections; it varies only weakly across the posterior.
_path = os.path.join(CFG["data_dir"], "camb_calibration.npz")
calib = None
_msg = ""
if os.path.exists(_path):
    _z = np.load(_path)
    calib = {"k": _z["k"], "ratio": _z["ratio"]}
    _msg = f"loaded `{_path}`"
else:
    try:
        import camb as _camb
        _th = PJ.PLANCK_MEAN_THETA
        _pars = _camb.set_params(H0=100 * _th[0], ombh2=_th[1], omch2=_th[2], mnu=0.06, num_massive_neutrinos=1,
                                 As=math.exp(_th[3]) * 1e-10, ns=_th[4], tau=0.0544)
        _pars.set_matter_power(redshifts=[0.0], kmax=250.0, nonlinear=False)
        _res = _camb.get_results(_pars)
        _kh, _zz, _pk = _res.get_linear_matter_power_spectrum(var1="delta_nonu", var2="delta_nonu",
                                                              hubble_units=True, k_hunit=True)
        _k = torch.logspace(-4, math.log10(200.0), 800)
        _d = PJ.unpack(torch.tensor([_th + [1.5, 1.0]]))
        _DGR = PJ.solve_growth(_d, torch.logspace(-3, 1, 8), torch.tensor([0.0]), "lcdm")[0]
        _peh = (PJ.delta2_lin0(_d, _k.unsqueeze(0), _DGR) * 2 * math.pi**2 / _k**3)[0].numpy()
        _pc = np.exp(np.interp(np.log(_k.numpy()), np.log(_kh), np.log(_pk[0])))
        calib = {"k": _k.numpy(), "ratio": _pc / _peh}
        np.savez(_path, **calib)
        _msg = f"computed with CAMB {_camb.__version__}; sigma8(cb) CAMB = {float(np.sqrt(np.trapezoid(_pc*_k.numpy()**3/(2*np.pi**2)*PJ.tophat_W(_k*8).numpy()**2, np.log(_k.numpy())))):.4f}"
    except Exception as _e:   # CAMB missing: fall back to pure EH (~1-2% in sigma8; flagged)
        _msg = f"CAMB unavailable ({type(_e).__name__}); using uncalibrated Eisenstein-Hu (sigma8 good to ~1.5%)"
mo.md(f"**Linear-spectrum calibration:** {_msg}")

In [ ]:
# ---- LIKELIHOOD OBJECTS: one per non-linear prescription (same data, same background) -----
def make_lik(prescription, use=None):
    return PJ.PalatiniJeansLikelihood(
        obs, device=CFG["device"], nk_ode=CFG["nk_ode"], nk=CFG["nk"], nzl=CFG["nzl"],
        prescription=prescription, shear_zmean=CFG["shear_zmean"], use=use,
        calib_k=None if calib is None else calib["k"], calib_ratio=None if calib is None else calib["ratio"],
        n_coarse=CFG["n_coarse"], n_fine=CFG["n_fine"])

LIKS = {_p: make_lik(_p) for _p in CFG["prescriptions"]}
_lk = LIKS[CFG["prescriptions"][0]]
_th = torch.tensor([PJ.PLANCK_MEAN_THETA + [1.5, CFG["n_index"]]], device=_lk.dev)
_o = _lk.evaluate(_th, "lcdm")
mo.md(
    f"""**Sanity at the Planck-mean cosmology (ΛCDM):** total χ² = {_o['chi2'].item():.2f};
    BAO {_o['chi2_bao'].item():.2f} (13 pts), SN {_o['chi2_sn'].item():.1f} (1590), CMB {_o['chi2_cmb'].item():.2f},
    RSD {_o['chi2_rsd'].item():.2f}, DES {_o['chi2_DESY6_shear'].item():.2f}, KiDS {_o['chi2_KiDS_Legacy'].item():.2f},
    CMBL {_o['chi2_ACT_Planck_CMBL'].item():.2f}; σ8 = {_o['sigma8_lin_true'].item():.4f}, S8 = {_o['S8_LCDMequiv'].item():.4f}.
    Lensing exponents γ = dlnC/dlnσ8: {', '.join(f'{k}: {v:.3f}' for k, v in _lk.gamma.items())}.
    (R, l_A) convention offsets: {_lk.cmb_scale.tolist()} (calibrated to the Chen+19 pipeline at the Planck mean).
    """)

In [ ]:
# ---- VALIDATION: solver convergence, exact background, consistency relation ---------------
_lk = LIKS[CFG["prescriptions"][0]]
_dev = _lk.dev
_th = torch.tensor([PJ.PLANCK_MEAN_THETA + [_u, CFG["n_index"]] for _u in (-0.9, 0.0, 1.0)], device=_dev)
_thc = torch.tensor([PJ.PLANCK_MEAN_THETA + [_u, CFG["n_index"]] for _u in (-0.9, -0.5, 0.0)], device=_dev)
_d = PJ.unpack(_th)
_kk = torch.logspace(-3, 1, 64, device=_dev)
_zz = torch.tensor([0.0, 0.5, 1.0], device=_dev)
_G1 = PJ.solve_growth(_d, _kk, _zz, "palatini", n_coarse=CFG["n_coarse"], n_fine=CFG["n_fine"])[1]
_G2 = PJ.solve_growth(_d, _kk, _zz, "palatini", n_coarse=8 * CFG["n_coarse"], n_fine=8 * CFG["n_fine"])[1]
ode_err = (_G1 - _G2).abs().max().item()
_a = torch.logspace(math.log10(0.02), 0, 400, device=_dev).unsqueeze(0)
_dH, _eps, _epsr, _Fm1 = PJ.palatini_background_check(_d, _a)
# consistency relation: checked where epsilon is resolvable by finite differences (>1e-8)
_dc, _ec, _erc, _ = PJ.palatini_background_check(PJ.unpack(_thc), _a)
_m = _erc > 1e-3 * _erc.max(1, keepdim=True).values
cons_err = max(((_ec[i] / _erc[i]) - 1)[_m[i]].abs().max().item() for i in range(3))
bg_dev = _dH.abs().max(1).values.tolist()
zJ = (1.0 / PJ.jeans_amin(_d) - 1.0)[:, 0].tolist()
VALID = {"ode_max_abs_err_G": ode_err, "consistency_rel_err": cons_err,
         "max_abs_dH_over_H": bg_dev, "z_J": zJ}
mo.md(f"""
| check | result | pass criterion |
|---|---|---|
| RK4 growth: default vs 8x finer grid, max abs error in $D/D_{{GR}}$ | {ode_err:.1e} | < 1e-3 |
| consistency relation $\\epsilon=9\\Omega_m(a)(aH/k_J)^2$ vs exact $d\\ln F/d\\ln a$ ($k_{{J*}}$ = 0.13, 0.3, 1) | {cons_err:.1e} rel. | < 1e-2 |
| exact Palatini Friedmann: max $|H/H_{{\\Lambda\\rm CDM}}-1|$ for $k_{{J*}}$ = 0.13, 1, 10 h/Mpc | {', '.join(f'{x:.1e}' for x in bg_dev)} | < 1e-4 (background-dark) |
| Jeans-window epoch $z_J$ (n={CFG['n_index']:g}) | {zJ[0]:.3f} | analytic $(1+z_J)^3=8\\Omega_\\Lambda/((3n+4)\\Omega_m)$ |

Offline checks done while building this notebook (against CAMB 1.x/2.x): distances to 2e-5, growth ratios to 1e-3,
our Takahashi halofit to 0.4% up to k = 30 h/Mpc, Limber CMB-lensing $C_L^{{\\kappa\\kappa}}$ to 0.5% for L = 40–763.
""")

In [ ]:
# ---- THEORY FIGURE: where the sector acts (k_J(z), suppression map, screening) -------------
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

_lk = LIKS[CFG["prescriptions"][0]]
_dev = _lk.dev
_fig, _ax = plt.subplots(1, 3, figsize=(15, 4.2))
_a = torch.logspace(math.log10(0.05), 0, 300, device=_dev).unsqueeze(0)
_z = (1 / _a - 1)[0].cpu().numpy()
for _n, _c in [(1.0, "C0"), (2.0, "C1"), (3.0, "C2")]:
    _d = PJ.unpack(torch.tensor([PJ.PLANCK_MEAN_THETA + [0.0, _n]], device=_dev))
    _kj = torch.sqrt(PJ.kJ2_of_a(_d, _a))[0].cpu().numpy()
    _zj = (1 / PJ.jeans_amin(_d) - 1).item()
    _ax[0].plot(_z, _kj, _c, label=f"n={_n:g}, $z_J$={_zj:.2f}")
    _ax[0].axvline(_zj, color=_c, ls=":")
_ax[0].set(xscale="log", yscale="log", xlabel="z", ylabel="$k_J$ [h/Mpc] (min = 1)", title="Jeans window (k_J* = 1 h/Mpc)")
_ax[0].axvspan(0.2, 0.8, color="0.9", zorder=0, label="galaxy-lensing kernels")
_ax[0].axvspan(1.0, 4.0, color="0.97", zorder=0, label="CMB-lensing kernel")
_ax[0].legend(fontsize=8)
_d = PJ.unpack(torch.tensor([PJ.PLANCK_MEAN_THETA + [0.0, CFG["n_index"]]], device=_dev))
_kk = torch.logspace(-2, 1, 120, device=_dev)
_zo = torch.linspace(0, 3, 60, device=_dev)
_G = PJ.solve_growth(_d, _kk, _zo, "palatini")[1][0].cpu().numpy()
_im = _ax[1].pcolormesh(_kk.cpu().numpy(), _zo.cpu().numpy(), (_G**2).T, shading="auto", vmin=0, vmax=1, cmap="viridis")
_ax[1].set(xscale="log", xlabel="k [h/Mpc]", ylabel="z", title="$P_{lin}/P_{lin}^{\\Lambda CDM}=G^2$ (k_J*=1)")
plt.colorbar(_im, ax=_ax[1])
_dl = torch.linspace(-0.9, 10, 200, device=_dev)
for _zz, _c in [(0.0, "k"), (0.36, "C3"), (1.0, "C4")]:
    _s = PJ.screening_factor(_d, torch.tensor(1 / (1 + _zz), device=_dev), _dl)[0].cpu().numpy()
    _ax[2].plot(_dl.cpu().numpy() + 1, _s, _c, label=f"z={_zz}")
_ax[2].set(xscale="log", yscale="log", xlabel="$1+\\delta_{env}$", ylabel="relative suppression strength", title="density screening")
_ax[2].legend()
_fig.tight_layout()
theory_fig = _fig
plt.close(_fig)
theory_fig

In [ ]:
# ---- SAMPLER UTILITIES: parameter maps, annealed ensemble optimiser, MCMC driver ----------
PRIOR_LO, PRIOR_HI = PJ.PRIOR_LO, PJ.PRIOR_HI
PLANCK0 = torch.tensor(PJ.PLANCK_MEAN_THETA)
SCATTER0 = torch.tensor([0.01, 0.0002, 0.002, 0.02, 0.005])

def full_theta(x, u=None, n=None):
    """x (B,5 or 6) -> (B,7). u fixed (profile) or taken from x[:,5]; n fixed."""
    B = x.shape[0]
    uu = x[:, 5:6] if u is None else torch.full((B, 1), float(u), dtype=x.dtype, device=x.device)
    nn = torch.full((B, 1), float(CFG["n_index"] if n is None else n), dtype=x.dtype, device=x.device)
    return torch.cat([x[:, :5], uu, nn], dim=1)

def make_logp(lik, model, u=None, n=None):
    lo = PRIOR_LO[: 6 if (model != "lcdm" and u is None) else 5].to(lik.dev)
    hi = PRIOR_HI[: lo.numel()].to(lik.dev)
    def logp(x):
        ok = PJ.in_prior(x, lo, hi)
        out = torch.full((x.shape[0],), -math.inf, dtype=x.dtype, device=x.device)
        if ok.any():
            th = full_theta(x[ok], u=(u if u is not None else (1.5 if model == "lcdm" else None)), n=n)
            out[ok] = -0.5 * lik.evaluate(th, model)["chi2"]
        return out
    return logp, lo, hi

def init_walkers(W, dim, gen, dev, center=None, scale=1.0):
    c = (PLANCK0 if center is None else center[:5].cpu()).to(dev)
    x = c + scale * SCATTER0.to(dev) * torch.randn(W, 5, generator=gen, device=dev)
    if dim == 6:
        u = PRIOR_LO[5] + (PRIOR_HI[5] - PRIOR_LO[5]) * torch.rand(W, 1, generator=gen, device=dev)
        if center is not None and center.numel() > 5:
            u = center[5].to(dev) + 0.15 * torch.randn(W, 1, generator=gen, device=dev)
        x = torch.cat([x, u], dim=1)
    lo, hi = PRIOR_LO[:dim].to(dev), PRIOR_HI[:dim].to(dev)
    return torch.minimum(torch.maximum(x, lo + 1e-9), hi - 1e-9)

def anneal_min(lik, model, u=None, n=None, seed=0, center=None):
    """Annealed affine-invariant ensemble: robust global minimiser, batched on the GPU."""
    gen = torch.Generator(device=lik.dev).manual_seed(int(seed))
    logp, lo, hi = make_logp(lik, model, u=u, n=n)
    dim = lo.numel()
    W, N = CFG["anneal_walkers"], CFG["anneal_iters"]
    x0 = init_walkers(W, dim, gen, lik.dev, center=center, scale=1.0)
    Tf = CFG["anneal_T_final"]
    tfun = lambda it: Tf ** (min(it, N - 1) / max(N - 1, 1))
    best = {"lp": -math.inf, "x": None}
    def cb(it, x, lp):
        i = int(torch.argmax(lp))
        if lp[i].item() > best["lp"]:
            best["lp"], best["x"] = lp[i].item(), x[i].clone()
    PJ.stretch_sampler(logp, x0, N, gen=gen, temp_fn=tfun, callback=cb)
    th = full_theta(best["x"].unsqueeze(0), u=(u if u is not None else (1.5 if model == "lcdm" else None)), n=n)
    comps = {k: float(v[0]) for k, v in lik.evaluate(th, model).items() if v.dim() == 1}
    return {"chi2": -2 * best["lp"], "theta": th[0].tolist(), "x": best["x"].tolist(), "components": comps}

def chernoff_Z(dchi2):
    """Significance for a parameter on the boundary (k_J* -> infinity is LCDM): p = 1/2 P(chi2_1 > dchi2)."""
    from scipy.stats import chi2 as _c2, norm as _nm
    dchi2 = max(float(dchi2), 0.0)
    p = 0.5 * _c2.sf(dchi2, 1)
    return float(_nm.isf(p)) if p < 0.5 else 0.0, float(p)

In [ ]:
run_btn = mo.ui.run_button(label="Run analysis (profile likelihood + MCMC)")
run_btn

In [ ]:
# ---- HEAVY 1: PROFILE LIKELIHOOD in k_J* for both non-linear prescriptions (resumable) -----
mo.stop(not (run_btn.value or os.environ.get("RUN_HEAVY") == "1"), mo.md("*Press **Run analysis** to start.*"))
_t0 = time.time()
PROFILE = {}
for _pres, _lk in LIKS.items():
    _p0 = os.path.join(CFG["ckpt_dir"], f"lcdm_{_pres}_{CFG['tag']}.json")
    if os.path.exists(_p0):
        _lc = json.load(open(_p0))
    else:
        _lc = anneal_min(_lk, "lcdm", seed=CFG["seed"])
        save_json(_p0, _lc)
    _pts, _center = [], None
    for _i, _u in enumerate(mo.status.progress_bar(CFG["u_grid"], title=f"profile [{_pres}]")):
        _pp = os.path.join(CFG["ckpt_dir"], f"prof_{_pres}_{_i:02d}_{CFG['tag']}.json")
        if os.path.exists(_pp):
            _r = json.load(open(_pp))
        else:
            if (time.time() - _t0) / 3600 > CFG["deadline_hours"]:
                break
            import torch as _torch
            _cen = None if _center is None else _torch.tensor(_center)
            _r = anneal_min(_lk, "palatini", u=_u, seed=CFG["seed"] + 100 * _i, center=_cen)
            _r["u"] = _u
            save_json(_pp, _r)
        _center = _r["x"]
        _pts.append(_r)
        gpu_free()
    _best = min(_pts, key=lambda r: r["chi2"]) if _pts else None
    _dchi2 = _lc["chi2"] - _best["chi2"] if _best else float("nan")
    _Z, _pval = chernoff_Z(_dchi2)
    PROFILE[_pres] = {"lcdm": _lc, "points": _pts, "best": _best, "dchi2": _dchi2, "Z": _Z, "p": _pval}
save_json(os.path.join(CFG["ckpt_dir"], f"PROFILE_{CFG['tag']}.json"), PROFILE)
mo.md("**Profile likelihood done** in {:.1f} min: ".format((time.time() - _t0) / 60) + "; ".join(
    f"{k}: Δχ² = {v['dchi2']:.2f}, Z = {v['Z']:.2f}σ at k_J* = {10**v['best']['u']:.3g} h/Mpc" for k, v in PROFILE.items()))

In [ ]:
# ---- HEAVY 2: MCMC posteriors (LCDM and Palatini-Jeans, each prescription), chunked --------
def _run_chain(lik, model, name, center):
    dim = 5 if model == "lcdm" else 6
    logp, lo, hi = make_logp(lik, model)
    W, NI, CH = CFG["mcmc_walkers"], CFG["mcmc_iters"], CFG["mcmc_chunk"]
    files = sorted(f for f in os.listdir(CFG["ckpt_dir"]) if f.startswith(f"mcmc_{name}_{CFG['tag']}_"))
    chains, lps = [], []
    x = None
    for f in files:
        z = np.load(os.path.join(CFG["ckpt_dir"], f))
        chains.append(z["chain"]); lps.append(z["lp"]); x = torch.tensor(z["x_last"], device=lik.dev)
    done = sum(c.shape[0] for c in chains)
    if x is None:
        gen0 = torch.Generator(device=lik.dev).manual_seed(CFG["seed"] + 7)
        x = init_walkers(W, dim, gen0, lik.dev, center=torch.tensor(center), scale=0.5)
    t0 = time.time()
    while done < NI:
        if (time.time() - t0) / 3600 > CFG["deadline_hours"]:
            break
        gen = torch.Generator(device=lik.dev).manual_seed(CFG["seed"] + 1000 + done)
        n = min(CH, NI - done)
        ch, lp, acc, x, _ = PJ.stretch_sampler(logp, x, n, gen=gen)
        fn = os.path.join(CFG["ckpt_dir"], f"mcmc_{name}_{CFG['tag']}_{done:06d}.npz")
        np.savez(fn + ".tmp.npz", chain=ch.cpu().numpy(), lp=lp.cpu().numpy(), x_last=x.cpu().numpy(), acc=acc.cpu().numpy())
        os.replace(fn + ".tmp.npz", fn)
        chains.append(ch.cpu().numpy()); lps.append(lp.cpu().numpy())
        done += n
        gpu_free()
    return np.concatenate(chains, 0), np.concatenate(lps, 0)

CHAINS = {}
for _pres, _lk in (LIKS.items() if CFG["run_mcmc"] else []):
    _bl = PROFILE[_pres]["lcdm"]["x"]
    _bp = PROFILE[_pres]["best"]["x"] + [PROFILE[_pres]["best"]["u"]]
    if _pres == CFG["prescriptions"][0]:
        CHAINS["lcdm"] = _run_chain(_lk, "lcdm", "lcdm", _bl)
    CHAINS[f"pj_{_pres}"] = _run_chain(_lk, "palatini", f"pj_{_pres}", _bp)
mo.md("**MCMC skipped (RUN_MCMC=0).**" if not CHAINS else "**MCMC done:** " + ", ".join(f"{k}: {v[0].shape[0]} iters x {v[0].shape[1]} walkers" for k, v in CHAINS.items()))

In [ ]:
# ---- RESULTS: model comparison, per-dataset fit, posteriors, convergence ------------------
_N = 13 + 1590 + 5 + 5 + 3
_keys = ["bao", "sn", "cmb", "rsd", "DESY6_shear", "KiDS_Legacy", "ACT_Planck_CMBL"]
_rows, STATS = [], {}
for _pres, _P in PROFILE.items():
    _lc, _b = _P["lcdm"]["components"], _P["best"]["components"]
    STATS[_pres] = {"dchi2": _P["dchi2"], "Z": _P["Z"], "dAIC": -_P["dchi2"] + 2, "dBIC": -_P["dchi2"] + math.log(_N),
                    "kJ_best": 10 ** _P["best"]["u"],
                    "per_dataset_dchi2": {k: _b["chi2_" + k] - _lc["chi2_" + k] for k in _keys},
                    "H0": (_lc["H0"], _b["H0"]),
                    "S8_eff_best": {k: _b["S8_eff_" + k] for k in ["DESY6_shear", "KiDS_Legacy", "ACT_Planck_CMBL"]},
                    "S8_LCDMequiv_best": _b["S8_LCDMequiv"], "S8_lcdm_best": _lc["S8_LCDMequiv"]}
    # 95% one-sided lower bound on k_J* from the profile (delta chi2 = 2.71 above the minimum)
    _us = np.array([p["u"] for p in _P["points"]]); _cs = np.array([p["chi2"] for p in _P["points"]])
    _o = np.argsort(_us); _us, _cs = _us[_o], _cs[_o] - min(_cs.min(), _P["lcdm"]["chi2"])
    _ib = int(np.argmin(_cs)); _lb = float("nan")
    for _j in range(_ib, 0, -1):
        if _cs[_j - 1] > 2.71 >= _cs[_j]:
            _lb = 10 ** np.interp(2.71, [_cs[_j], _cs[_j - 1]], [_us[_j], _us[_j - 1]]); break
    STATS[_pres]["kJ_lower95"] = _lb
    _rows.append(f"| {_pres} | {_P['dchi2']:.2f} | {_P['Z']:.2f} | {STATS[_pres]['dAIC']:.2f} | {STATS[_pres]['dBIC']:.2f} | "
                 f"{STATS[_pres]['kJ_best']:.3g} | {_lb:.3g} | " +
                 " | ".join(f"{STATS[_pres]['per_dataset_dchi2'][k]:+.2f}" for k in _keys) + " |")
# posteriors
_post = {}
for _name, (_ch, _lp) in CHAINS.items():
    _nb = int(CFG["mcmc_burn_frac"] * _ch.shape[0])
    _c = _ch[_nb:]
    _flat = _c.reshape(-1, _c.shape[-1])
    _taus = [PJ.autocorr_time(_c[:, :, j]) for j in range(_c.shape[-1])]
    _rh = [PJ.split_rhat(_c[:, :, j]) for j in range(_c.shape[-1])]
    _chi = -2 * _lp[_nb:].reshape(-1)
    _lk = LIKS[CFG["prescriptions"][0] if _name == "lcdm" else _name[3:]]
    _mean = torch.tensor(_flat.mean(0), device=_lk.dev).unsqueeze(0)
    _model = "lcdm" if _name == "lcdm" else "palatini"
    _chim = _lk.evaluate(full_theta(_mean, u=(1.5 if _model == "lcdm" else None)), _model)["chi2"].item()
    _pD = _chi.mean() - _chim
    _post[_name] = {"mean": _flat.mean(0).tolist(), "std": _flat.std(0).tolist(), "tau": _taus, "rhat": _rh,
                    "DIC": float(_chi.mean() + _pD), "n_eff": float(_c.shape[0] * _c.shape[1] / max(max(_taus), 1.0)),
                    "flat": _flat}
STATS["posterior"] = {k: {kk: vv for kk, vv in v.items() if kk != "flat"} for k, v in _post.items()}
_names = ["h", "ombh2", "omch2", "lnAs", "ns", "log10 kJ*"]
_prow = []
for _k, _v in _post.items():
    _prow.append(f"| {_k} | " + " | ".join(f"{m:.4g} ± {s:.2g}" for m, s in zip(_v["mean"], _v["std"])) +
                 ("" if len(_v["mean"]) == 6 else " | — ") + f" | {max(_v['tau']):.0f} | {max(_v['rhat']):.3f} | {_v['DIC']:.1f} |")
# profile figure
_fig, _ax = plt.subplots(1, 2, figsize=(12, 4))
for _pres, _P in PROFILE.items():
    _us = [p["u"] for p in _P["points"]]; _cs = [p["chi2"] - _P["lcdm"]["chi2"] for p in _P["points"]]
    _ax[0].plot([10**u for u in _us], _cs, "o-", label=_pres)
_ax[0].axhline(0, color="k", lw=0.8); _ax[0].set(xscale="log", xlabel="$k_{J*}$ [h/Mpc]", ylabel="$\\chi^2_{\\rm prof}-\\chi^2_{\\Lambda CDM}$", title="profile likelihood")
_ax[0].legend()
for _k, _v in _post.items():
    if _v["flat"].shape[1] == 6:
        _ax[1].hist(_v["flat"][:, 5], bins=50, histtype="step", density=True, label=_k)
_ax[1].set(xlabel="log10 k_J*", title="marginal posterior (prior-volume dependent; profile is primary)"); _ax[1].legend()
_fig.tight_layout(); results_fig = _fig; plt.close(_fig)
_hdr = "| prescription | Δχ² (ΛCDM−PJ) | Z (Chernoff) | ΔAIC | ΔBIC | best k_J* | k_J* > (95%) | " + " | ".join(f"Δχ² {k}" for k in _keys) + " |"
mo.vstack([
    mo.md("### Model comparison (profile likelihood, primary)\n" + _hdr + "\n|" + "---|" * (7 + len(_keys)) + "\n" + "\n".join(_rows)),
    mo.md("### Posteriors (secondary)\n| chain | " + " | ".join(_names) + " | max τ | max R̂ | DIC |\n|" + "---|" * (len(_names) + 4) + "\n" + "\n".join(_prow)),
    results_fig])

In [ ]:
# ---- PREDICTIONS at the best fit of each prescription (P1-P4) -----------------------------
PRED = {}
_fig, _ax = plt.subplots(1, 3, figsize=(15, 4.2))
for _pres, _lk in LIKS.items():
    _th = torch.tensor([PROFILE[_pres]["best"]["theta"]], device=_lk.dev)
    _o = _lk.evaluate(_th, "palatini", want_extra=True)
    _zm = [0.3, 0.45, 0.6, 0.75, 0.9, 1.1, 1.3, 1.5]
    _s8t = _lk.ratio_for_nz(_th, "palatini", _zm)[0].tolist()
    _ir = _lk.i_rsd
    _k = _lk.k.cpu().numpy()
    _G = _o["Gk"][0, :, _ir].cpu().numpy(); _Gp = _o["Gpk"][0, :, _ir].cpu().numpy()
    _fk = _Gp / np.where(np.abs(_G) > 1e-8, _G, np.nan)
    _fgr = _Gp[0] / _G[0]
    _iz = 2                                        # z = 0.51
    _sel = (_k > 1e-3) & (_k < 0.5)
    _k02 = int(np.argmin(np.abs(_k - 0.2)))
    PRED[_pres] = {"S8_eff": STATS[_pres]["S8_eff_best"], "S8_LCDMequiv": STATS[_pres]["S8_LCDMequiv_best"],
                   "S8_vs_source_zmean": dict(zip(_zm, _s8t)),
                   "f_ratio_k0.2_z0.51": float(_fk[_k02, _iz] / _fgr[_iz]),
                   "z_J": float(1 / PJ.jeans_amin(PJ.unpack(_th)).item() - 1)}
    _ax[0].plot(_zm, _s8t, "o-", label=f"shear, {_pres}")
    _ax[0].axhline(STATS[_pres]["S8_eff_best"]["ACT_Planck_CMBL"], ls="--", color=_ax[0].lines[-1].get_color(), lw=0.8)
    _ax[1].plot(_k[_sel], _fk[_sel, _iz] / _fgr[_iz], label=_pres)
    _ax[2].plot(_k[_sel], _fgr[_iz] / _fk[_sel, _iz], label=_pres)
_ax[0].axhline(PRED[CFG["prescriptions"][0]]["S8_LCDMequiv"], color="k", lw=0.8, label="CMB-normalised (ΛCDM-equiv.)")
_ax[0].set(xlabel="mean source redshift", ylabel="effective $S_8$", title="P1/P3: probe ordering + tomographic trend"); _ax[0].legend(fontsize=8)
_ax[1].set(xscale="log", xlabel="k [h/Mpc]", ylabel="$f(k)/f_{GR}$ at z=0.51", title="P2: scale-dependent growth rate"); _ax[1].legend()
_ax[2].set(xscale="log", xlabel="k [h/Mpc]", ylabel="$E_G(k)/E_G^{\\Lambda CDM}$ at z=0.51", title="P2: $E_G$ rises (Σ=1, μ<1)"); _ax[2].legend()
_fig.tight_layout(); pred_fig = _fig; plt.close(_fig)
mo.vstack([mo.md("### Predictions at the best fit\n" + "\n".join(
    f"- **{p}**: S8_eff DES {v['S8_eff']['DESY6_shear']:.3f}, KiDS {v['S8_eff']['KiDS_Legacy']:.3f}, CMB-lensing "
    f"{v['S8_eff']['ACT_Planck_CMBL']:.3f}, CMB-normalised {v['S8_LCDMequiv']:.3f}; f(k=0.2)/f_GR at z=0.51 = "
    f"{v['f_ratio_k0.2_z0.51']:.4f}; z_J = {v['z_J']:.2f}" for p, v in PRED.items())), pred_fig])

In [ ]:
robust_btn = mo.ui.run_button(label="Run robustness suite (drop-one datasets, n = 2, 3)")
robust_btn

In [ ]:
# ---- ROBUSTNESS: re-minimise LCDM and PJ (u free around the best fit) per variation -------
mo.stop(not (robust_btn.value or os.environ.get("RUN_ROBUST") == "1"), mo.md("*Robustness suite not run.*"))
ROBUST = {}
for _pres in CFG["prescriptions"]:
    _ub = PROFILE[_pres]["best"]["u"]
    _vars = [("drop_" + d, {d: False}, CFG["n_index"]) for d in CFG["robust_drop"]] + \
            [(f"n={n:g}", None, n) for n in CFG["robust_n"]]
    for _name, _use, _n in _vars:
        _p = os.path.join(CFG["ckpt_dir"], f"robust_{_pres}_{_name}_{CFG['tag']}.json")
        if os.path.exists(_p):
            ROBUST[(_pres, _name)] = json.load(open(_p)); continue
        _lk = make_lik(_pres, use=_use)
        _lc = anneal_min(_lk, "lcdm", seed=CFG["seed"] + 5)
        _best = min((anneal_min(_lk, "palatini", u=_ub + du, n=_n, seed=CFG["seed"] + 9) for du in (-0.3, 0.0, 0.3)),
                    key=lambda r: r["chi2"])
        _d = _lc["chi2"] - _best["chi2"]
        _r = {"dchi2": _d, "Z": chernoff_Z(_d)[0], "u": _best["theta"][5]}
        save_json(_p, _r); ROBUST[(_pres, _name)] = _r
mo.md("| prescription | variation | Δχ² | Z | log10 k_J* |\n|---|---|---|---|---|\n" + "\n".join(
    f"| {k[0]} | {k[1]} | {v['dchi2']:.2f} | {v['Z']:.2f} | {v['u']:.2f} |" for k, v in ROBUST.items()))

In [ ]:
# ---- VERDICT against the pre-registered criteria -----------------------------------------
_z = {p: STATS[p]["Z"] for p in CFG["prescriptions"]}
_zmin, _zmax = min(_z.values()), max(_z.values())
if _zmin >= 3:
    _v = "EVIDENCE (≥3σ) for a finite Jeans scale, robust across the non-linear bracket."
elif _zmax >= 3:
    _v = "CONDITIONAL EVIDENCE: ≥3σ only under the upper-bracket prescription — the claim hinges on non-linear screening; N-body runs are required before any claim."
elif _zmax >= 2:
    _v = "HINT (2–3σ). Not a detection."
else:
    _v = "NO PREFERENCE. The data constrain the sector; quote the 95% lower bound on k_J*."
_checks = []
for _p in CFG["prescriptions"]:
    _worse = {k: v for k, v in STATS[_p]["per_dataset_dchi2"].items() if v > 6.25}
    _s = PRED[_p]["S8_eff"]
    _order = _s["DESY6_shear"] < _s["ACT_Planck_CMBL"]
    _dH0 = abs(STATS[_p]["H0"][1] - STATS[_p]["H0"][0])
    _checks.append(f"- **{_p}**: Z = {_z[_p]:.2f}; datasets degraded by >2.5σ: {list(_worse) or 'none'}; "
                   f"probe ordering S8(shear) < S8(CMB lensing): {_order}; |ΔH0| between best fits = {_dH0:.2f} km/s/Mpc "
                   f"(theorem predicts ≈ 0: any shift is driven by the growth data through Ω_m, not by H(z)); "
                   f"95% bound k_J* > {STATS[_p]['kJ_lower95']:.3g} h/Mpc")
mo.md(f"""
## Verdict
**{_v}**

{chr(10).join(_checks)}

Validation: ODE error {VALID['ode_max_abs_err_G']:.1e}; consistency relation {VALID['consistency_rel_err']:.1e};
max |ΔH/H| {max(VALID['max_abs_dH_over_H']):.1e}.

**What would make this Nature-Astronomy-grade (not done here):** (1) replace the S8 summary likelihoods by the DES Y6 /
KiDS-Legacy / HSC data vectors with official n(z) and scale cuts; (2) replace the compressed CMB by Planck+ACT+SPT spectra
through a modified Boltzmann code (MGCAMB/hi_class with μ(k,a), Σ=1); (3) N-body simulations with the screened force law
to close the non-linear bracket; (4) DESI DR2 full-shape f(k) and E_G as the decisive P2 test; (5) density-split/void lensing
for P4; (6) full literature audit of the novelty claims.
""")

In [ ]:
# ---- EXPORT: small JSON summary + figures (session files may not survive shutdown) --------
_out = os.path.join(CFG["ckpt_dir"], f"RESULTS_{CFG['tag']}.json")
save_json(_out, {"config": CFG, "validation": VALID, "stats": STATS, "predictions": PRED})
results_fig.savefig(os.path.join(CFG["ckpt_dir"], "profile_posterior.png"), dpi=160)
pred_fig.savefig(os.path.join(CFG["ckpt_dir"], "predictions.png"), dpi=160)
mo.md(f"Saved `{_out}`, `profile_posterior.png`, `predictions.png` in `{CFG['ckpt_dir']}/`. "
      "Download them before the session ends (molab session files are temporary).")

---
# Part II — The Conformal Wall
### Gravity cannot tell a cosmic void from a conformal neutron-star core

**Why Part II exists.** Part I found *no preference* for the Palatini–Jeans sector in cosmological data
($k_{J*}>1.47\,h$/Mpc at 95%). Going deeper exposed two facts. One is a weakness: infrared Palatini
theories respond to the *microscopic* matter trace, which Olmo (2008) showed destabilises hydrogen. They
survive only if the field equations are applied to coarse-grained matter, the assumption Part I made.
The other is new: a *macroscopic* regime exists where coarse-graining cannot help, namely dense matter
whose trace turns positive. Part II derives what happens there and tests it against neutron-star data.

## II.1 Theorem 5 — branch continuity and the non-analytic GR limit
For $f(R)=R-2\Lambda-\mu^{2n+2}R^{-n}$ ($n>0$), the Palatini trace equation $FR-2f=\kappa T$ reads
$$g(R)\equiv -R+4\Lambda+(n+2)\,\mu^{2n+2}R^{-n}=\kappa T .$$
On $R>0$, $g$ is strictly decreasing from $+\infty$ to $-\infty$, so there is exactly one positive root
for every $T$, and it depends continuously on $T$. Ordinary matter and vacuum live on this branch, so a
smooth star stays on it. For superconformal matter ($p>\varepsilon/3$, i.e. trace anomaly
$\Delta\equiv\tfrac13-p/\varepsilon<0$, so $T=-3\varepsilon\Delta>0$) the root is forced to
$R\simeq\big[(n+2)\mu^{2n+2}/\kappa T\big]^{1/n}\to0^+$ and
$$F=1+n\,\mu^{2n+2}R^{-(n+1)}\;\simeq\;n\,(n+2)^{-\frac{n+1}{n}}\;\mu^{-\frac{2n+2}{n}}\,(\kappa T)^{\frac{n+1}{n}} .$$
**F grows without bound as $\mu\to0$.** The weaker the cosmological modification, the larger the
breakdown inside superconformal matter. The GR limit is non-analytic: $\mu\to0$ does not recover GR there.
For a 700 MeV/fm$^3$ core with $\Delta=-0.05$: $F\sim10^{91}$ at the Part-I bound and $\sim10^{86}$ for
classic Palatini $1/R$ dark energy.

## II.2 Theorem 6 — void–core duality
In every Palatini (auxiliary-field) theory the gravitational scalar is a function of the local trace $T$
alone. The cosmic void ($T\to0^-$ from dilution) and the conformal point of dense QCD matter
($T\to0$ from $p\to\varepsilon/3$) are therefore the **same point** of the theory. A modification
engineered to act in voids necessarily acts at the conformal point of dense matter. Infrared-singular
theories ($F\to\infty$ as $R\to0^+$) explode on the far side of it. UV terms ($\alpha R^2$ with the
stability-required $\alpha<0$) are finite there and instead cap the density at $\rho_{\max}=1/(2|\alpha|\kappa)$.

## II.3 Hypotheses (stated boldly, tested below)
- **H1 — Conformal Wall.** In an infrared-Palatini universe, no gravitating body contains superconformal
  matter: the stellar sequence ends where the central trace anomaly reaches zero, $M_{\max}=M(\Delta_c=0)$,
  not where $dM/d\varepsilon_c=0$ as in GR. *(Assumption: a region with $F\gtrsim10^{80}$ admits no stable
  static star. A full Palatini TOV across the conformal surface is the theory follow-up.)*
- **H2 — Parameter-free test.** By Theorem 5, H1 holds for **every** $\mu>0$ and every $n>0$. The neutron-star
  test of the infrared-Palatini class has no free gravitational parameter, unlike any cosmological test.
- **H3 — Gravitational conformality principle.** If the dark sector is infrared-Palatini, the bound
  $p\le\varepsilon/3$ in all gravitating matter is a *law of gravity*, not a property of QCD.
- **H4 — Signature of the wall.** The heaviest neutron star then has an *exactly conformal centre*
  ($p_c=\varepsilon_c/3$), and the M–R curve ends abruptly with $dM/dR\neq0$, instead of turning over.

## II.4 What is tested here
A GPU-batched Bayesian inference of the dense-matter equation of state, built from:
- an SLy crust and a chiral-EFT-like pressure band at $1.1n_0$;
- a causal, thermodynamically consistent piecewise-linear $c_s^2(n)$ up to $12n_0$;
- exact TOV and tidal Love-number integration for every draw.

It is confronted with NICER (J0030, J0740, J0437), heavy pulsars (J0348, J1614; optionally the black-widow
J0952) and GW170817. Two worlds share the same prior: **GR** (sequence to the first maximum) and the
**Conformal Wall** (sequence truncated at $\Delta_c=0$). The Bayes factor $B_{\rm CW/GR}$ is the verdict on H1.

**Approximations (to upgrade for the paper):**
- Mass–radius and $\tilde\Lambda$ likelihoods are split-normal summaries of the published posteriors;
  replace them with the public NICER/LVK samples.
- The chiral-EFT band is a log-uniform pressure range at $1.1n_0$.
- No pQCD likelihood is included by default.
- Bayes factors depend on the $c_s^2$ prior and on the mass-prior normalisation. Both are reported.

In [ ]:
# ---- CONFORMAL-WALL CORE (tested module, embedded verbatim; exported as namespace NS) -------
def _build_ns():
    # Conformal-wall core: dense-matter EOS prior, batched TOV + tidal Love numbers, Palatini trace map,
    # neutron-star likelihoods. Development module; embedded verbatim into the notebook as namespace NS.
    import math
    import numpy as np
    import torch

    N0 = 0.16                      # saturation density [fm^-3]
    MEVFM3_TO_KM2 = 1.3234e-6      # G/c^4 x (MeV/fm^3)  ->  km^-2
    MSUN_KM = 1.476625             # G M_sun / c^2 [km]
    M_U = 931.494                  # MeV (atomic mass unit)
    G_CC_TO_MEVFM3 = 5.6096e-13    # (g/cm^3) * c^2  ->  MeV/fm^3

    # ---- SLy crust as a 4-piece polytrope (Read, Lackey, Owen & Friedman 2009, Table II) ----------
    # p/c^2 = K rho^Gamma in g/cm^3 units. Constants verified here by pressure continuity at the joins.
    CRUST_RHO = [0.0, 2.44034e7, 3.78358e11, 2.62780e12]
    CRUST_G = [1.58425, 1.28733, 0.62223, 1.35692]
    CRUST_K = [6.80110e-9, 1.06186e-6, 5.36765e1, 3.99874e-8]


    def crust_table(n_max, npts=400):
        """SLy crust (rho < 2.6e12 g/cc) + last SLy piece up to n_max [fm^-3]. Returns n, eps, p (MeV/fm^3)."""
        rho_hi = n_max * M_U / G_CC_TO_MEVFM3            # g/cc
        rho = np.logspace(3, np.log10(rho_hi), npts)
        p = np.zeros_like(rho); eps = np.zeros_like(rho)
        a = 0.0
        for i in range(4):
            lo = CRUST_RHO[i]; hi = CRUST_RHO[i + 1] if i < 3 else np.inf
            if i > 0:   # continuity of energy density fixes a_i
                r0 = CRUST_RHO[i]
                Kp, Gp = CRUST_K[i - 1], CRUST_G[i - 1]
                e_prev = (1 + a_prev) * r0 + Kp * r0**Gp / (Gp - 1)
                a = e_prev / r0 - 1 - CRUST_K[i] * r0 ** (CRUST_G[i] - 1) / (CRUST_G[i] - 1)
            m = (rho >= lo) & (rho < hi)
            p[m] = CRUST_K[i] * rho[m] ** CRUST_G[i]
            eps[m] = (1 + a) * rho[m] + CRUST_K[i] * rho[m] ** CRUST_G[i] / (CRUST_G[i] - 1)
            a_prev = a
        n = rho * G_CC_TO_MEVFM3 / M_U
        return n, eps * G_CC_TO_MEVFM3, p * G_CC_TO_MEVFM3


    # ---- EOS prior: crust -> chiral-EFT-like polytrope -> piecewise-linear sound speed -------------
    CS_NODES = np.array([1.1, 1.5, 2.0, 3.0, 4.5, 7.0, 12.0]) * N0


    def build_eos_batch(theta, n_grid, crust, p_band=(2.2, 4.4), n_join=0.5 * N0, n_chi=1.1 * N0):
        """theta (B, 1+K): [s in (0,1) = position inside the ChEFT pressure band at 1.1 n0,
        c_s^2 at nodes 2..K (node 1 fixed by continuity)].  n_grid (N,) fm^-3 common to all.
        Returns eps, p, cs2, mu (B,N) in MeV/fm^3 (mu in MeV) -- thermodynamically consistent and causal."""
        dev, dt = theta.device, theta.dtype
        B = theta.shape[0]
        nc, ec, pc = (torch.as_tensor(x, dtype=dt, device=dev) for x in crust)
        # join point (end of SLy, n_join)
        e_j = torch.as_tensor(np.interp(n_join, crust[0], crust[1]), dtype=dt, device=dev)
        p_j = torch.as_tensor(np.interp(n_join, crust[0], crust[2]), dtype=dt, device=dev)
        lo, hi = math.log(p_band[0]), math.log(p_band[1])
        p_chi = torch.exp(lo + (hi - lo) * theta[:, 0])                        # (B,)
        Gam = torch.log(p_chi / p_j) / math.log(n_chi / n_join)               # polytrope p = p_j (n/n_j)^Gam
        n = n_grid.view(1, -1)
        # region 1: crust (n <= n_join): interpolate table
        lnn = torch.log(n_grid)
        e_cr = torch.exp(torch.as_tensor(np.interp(np.log(n_grid.cpu().numpy()), np.log(crust[0]), np.log(crust[1])), dtype=dt, device=dev))
        p_cr = torch.exp(torch.as_tensor(np.interp(np.log(n_grid.cpu().numpy()), np.log(crust[0]), np.log(crust[2])), dtype=dt, device=dev))
        # region 2: polytrope n_join < n <= n_chi : eps from first law  d(eps/n) = p/n^2 dn
        G_ = Gam.view(-1, 1)
        p2 = p_j * (n / n_join) ** G_
        e2 = (e_j / n_join) * n + (p2 - p_j * (n / n_join)) / (G_ - 1.0)
        # values at n_chi
        pchi = p_chi.view(-1, 1)
        echi = (e_j / n_join) * n_chi + (pchi - p_j * (n_chi / n_join)) / (G_ - 1.0)
        cs2_chi = G_ * pchi / (echi + pchi)
        mu_chi = (echi + pchi) / n_chi
        # region 3: sound-speed nodes (linear in n), integrate ln mu = ln mu_chi + int cs2/n dn
        nodes = torch.as_tensor(CS_NODES, dtype=dt, device=dev)
        cs_vals = torch.cat([cs2_chi, theta[:, 1:]], dim=1)                    # (B,K)
        nn = torch.clamp(n, min=n_chi)
        idx = torch.clamp(torch.searchsorted(nodes, nn.expand(B, -1).contiguous()), 1, nodes.numel() - 1)
        x0, x1 = nodes[idx - 1], nodes[idx]
        y0 = torch.gather(cs_vals, 1, idx - 1); y1 = torch.gather(cs_vals, 1, idx)
        w = ((nn - x0) / (x1 - x0)).clamp(0, 1)
        cs2_3 = torch.where(nn > nodes[-1], cs_vals[:, -1:].expand(B, nn.shape[1]), y0 + w * (y1 - y0))
        # cumulative integrals on the grid (trapezoid in n), restricted to n >= n_chi
        m3 = (n_grid >= n_chi).view(1, -1)
        f = torch.where(m3, cs2_3 / n, torch.zeros_like(cs2_3))
        dn = (n_grid[1:] - n_grid[:-1]).view(1, -1)
        I = torch.cat([torch.zeros(B, 1, dtype=dt, device=dev), torch.cumsum(0.5 * (f[:, 1:] + f[:, :-1]) * dn, 1)], 1)
        # anchor exactly at n_chi (grid contains n_chi by construction)
        i_chi = int(torch.argmin((n_grid - n_chi).abs()))
        mu3 = mu_chi * torch.exp(I - I[:, i_chi:i_chi + 1])
        g = torch.where(m3, mu3, torch.zeros_like(mu3))
        J = torch.cat([torch.zeros(B, 1, dtype=dt, device=dev), torch.cumsum(0.5 * (g[:, 1:] + g[:, :-1]) * dn, 1)], 1)
        e3 = echi + (J - J[:, i_chi:i_chi + 1])
        p3 = n * mu3 - e3
        r1 = (n_grid <= n_join).view(1, -1)
        r2 = ((n_grid > n_join) & (n_grid <= n_chi)).view(1, -1)
        eps = torch.where(r1, e_cr.view(1, -1).expand(B, -1), torch.where(r2, e2, e3))
        p = torch.where(r1, p_cr.view(1, -1).expand(B, -1), torch.where(r2, p2, p3))
        # sound speed by finite differences of the assembled table (used by the tidal equation)
        cs2 = torch.gradient(p, spacing=(n_grid,), dim=1)[0] / torch.gradient(eps, spacing=(n_grid,), dim=1)[0]
        cs2 = torch.where(n > n_chi, cs2_3, cs2).clamp(1e-6, 1.0)
        mu = (eps + p) / n
        return eps, p, cs2, mu


    def make_n_grid(n_max=12.0 * N0, npts=700, dtype=torch.float64, device="cpu"):
        """Log grid from the outer crust to n_max that contains n_join and n_chi as exact nodes."""
        g = np.unique(np.concatenate([np.logspace(np.log10(1e-10), np.log10(n_max), npts), [0.5 * N0, 1.1 * N0], CS_NODES]))
        keep = np.concatenate([[True], np.diff(np.log(g)) > 1e-6])   # drop float-duplicate nodes
        return torch.as_tensor(g[keep], dtype=dtype, device=device)


    # ---- trace anomaly ----------------------------------------------------------------------------
    def trace_anomaly(eps, p):
        """Delta = 1/3 - p/eps ; T = -eps + 3p = -3 eps Delta  (T > 0  <=>  Delta < 0: superconformal)."""
        return 1.0 / 3.0 - p / eps


    # ---- batched TOV in enthalpy form (Lindblom 1992) + tidal Love number ------------------------
    def tov_batch(eps, p, cs2, mu, n_grid, nc_vals, n_h=320):
        """For each EOS (B) and each central density (Nc): mass [Msun], radius [km], Lambda, and
        central energy density. Integration in log-enthalpy h = ln(mu/mu_surface) from h_c to 0."""
        B, N = eps.shape
        dt, dev = eps.dtype, eps.device
        h_tab = torch.log(mu / mu[:, :1])                                     # (B,N), monotone
        E = eps * MEVFM3_TO_KM2; P = p * MEVFM3_TO_KM2
        lnn = torch.log(n_grid)
        hc = torch.stack([interp_rows(lnn.expand(B, -1).contiguous(), h_tab, torch.full((B,), math.log(v), dtype=dt, device=dev))
                          for v in nc_vals], 1)                               # (B,Nc)
        S = hc.reshape(-1)
        Hrows = h_tab.repeat_interleave(len(nc_vals), 0).contiguous()
        Erows = E.repeat_interleave(len(nc_vals), 0); Prows = P.repeat_interleave(len(nc_vals), 0)
        Crows = cs2.repeat_interleave(len(nc_vals), 0)

        def eos_at(hq):   # one search, three gathers
            xq = hq.reshape(-1, 1)
            idx = torch.clamp(torch.searchsorted(Hrows, xq.contiguous()), 1, Hrows.shape[1] - 1)
            x0 = torch.gather(Hrows, 1, idx - 1); x1 = torch.gather(Hrows, 1, idx)
            w = ((xq - x0) / (x1 - x0).clamp(min=1e-300)).clamp(0, 1)
            out = []
            for T_ in (Erows, Prows, Crows):
                y0 = torch.gather(T_, 1, idx - 1); y1 = torch.gather(T_, 1, idx)
                out.append((y0 + w * (y1 - y0)).reshape(-1))
            return out

        ec, pc, _ = eos_at(S)
        dh0 = 1e-3 * S
        r = torch.sqrt(3 * dh0 / (2 * math.pi * (ec + 3 * pc)))
        m = 4 * math.pi / 3 * ec * r**3
        y = torch.full_like(r, 2.0)
        hs = torch.linspace(0, 1, n_h + 1, dtype=dt, device=dev)

        def rhs(h, r, m, y):
            e, pp, c2 = eos_at(h)
            den = m + 4 * math.pi * r**3 * pp
            drdh = -r * (r - 2 * m) / den
            dmdh = 4 * math.pi * r**2 * e * drdh
            el = 1.0 / (1 - 2 * m / r)
            nup = 2 * den / (r * (r - 2 * m))
            Q = 4 * math.pi * el * (5 * e + 9 * pp + (e + pp) / c2) - 6 * el / r**2 - nup**2
            dydr = -(y**2 + y * el * (1 + 4 * math.pi * r**2 * (pp - e)) + r**2 * Q) / r
            return drdh, dmdh, dydr * drdh

        h_start = S - dh0
        for i in range(n_h):                                                  # h from h_start -> 0 (uniform)
            a0 = h_start * (1 - hs[i]); a1 = h_start * (1 - hs[i + 1]); st = a1 - a0
            k1 = rhs(a0, r, m, y)
            k2 = rhs(a0 + st / 2, r + st / 2 * k1[0], m + st / 2 * k1[1], y + st / 2 * k1[2])
            k3 = rhs(a0 + st / 2, r + st / 2 * k2[0], m + st / 2 * k2[1], y + st / 2 * k2[2])
            k4 = rhs(a1, r + st * k3[0], m + st * k3[1], y + st * k3[2])
            r = r + st / 6 * (k1[0] + 2 * k2[0] + 2 * k3[0] + k4[0])
            m = m + st / 6 * (k1[1] + 2 * k2[1] + 2 * k3[1] + k4[1])
            y = y + st / 6 * (k1[2] + 2 * k2[2] + 2 * k3[2] + k4[2])
        C = m / r
        k2 = love_k2(C, y)
        Lam = 2.0 / 3.0 * k2 / C**5
        sh = (B, len(nc_vals))
        return (m / MSUN_KM).reshape(sh), r.reshape(sh), Lam.reshape(sh), (ec / MEVFM3_TO_KM2).reshape(sh)


    def love_k2(C, y):
        num = 8 * C**5 / 5 * (1 - 2 * C) ** 2 * (2 + 2 * C * (y - 1) - y)
        den = (2 * C * (6 - 3 * y + 3 * C * (5 * y - 8))
               + 4 * C**3 * (13 - 11 * y + C * (3 * y - 2) + 2 * C**2 * (1 + y))
               + 3 * (1 - 2 * C) ** 2 * (2 - y + 2 * C * (y - 1)) * torch.log(1 - 2 * C))
        return num / den


    def interp_rows(X, Y, xq):
        """Row-wise linear interpolation: X,Y (R,N) with X increasing per row; xq (R,)."""
        xq = xq.reshape(-1, 1)
        idx = torch.clamp(torch.searchsorted(X.contiguous(), xq.contiguous()), 1, X.shape[1] - 1)
        x0 = torch.gather(X, 1, idx - 1); x1 = torch.gather(X, 1, idx)
        y0 = torch.gather(Y, 1, idx - 1); y1 = torch.gather(Y, 1, idx)
        w = ((xq - x0) / (x1 - x0).clamp(min=1e-300)).clamp(0, 1)
        return (y0 + w * (y1 - y0)).reshape(-1)


    # ---- stable branch, conformal truncation, likelihoods -------------------------------------------
    def stable_mask(M):
        """Stable branch: monotonically rising M(eps_c) up to the first maximum."""
        dM = M[:, 1:] - M[:, :-1]
        rising = torch.cat([torch.ones_like(M[:, :1], dtype=torch.bool), dM > 0], 1)
        return torch.cumprod(rising.to(torch.int64), 1).bool()


    def conformal_mask(ec, eps, p, n_grid):
        """True where the star's whole interior is subconformal: max over eps<=eps_c of (p/eps) <= 1/3.
        (Delta depends on eps only; the core has the highest eps, so the condition is cumulative.)"""
        B = eps.shape[0]
        sup = (p / eps > 1.0 / 3.0)
        run = torch.cummax(sup.to(torch.int64), 1).values.bool()             # superconformal at or below this eps
        first = torch.where(run.any(1), torch.argmax(run.to(torch.int64), 1), torch.full((B,), eps.shape[1] - 1, device=eps.device))
        e_wall = torch.gather(eps, 1, first.view(-1, 1))                     # first eps with Delta<0
        has = run.any(1).view(-1, 1)
        return torch.where(has, ec < e_wall, torch.ones_like(ec, dtype=torch.bool)), torch.where(has, e_wall, torch.full_like(e_wall, float("inf")))


    def split_norm_logpdf(x, mu, sp, sm):
        s = torch.where(x > mu, torch.as_tensor(sp, dtype=x.dtype, device=x.device), torch.as_tensor(sm, dtype=x.dtype, device=x.device))
        return -0.5 * ((x - mu) / s) ** 2 - math.log(math.sqrt(2 * math.pi) * 0.5 * (sp + sm))


    NICER = {   # name: (M, sM+, sM-, R, sR+, sR-)  -- values as quoted in Brandes et al. 2024 (arXiv:2412.05923)
        "J0030": (1.34, 0.15, 0.16, 12.71, 1.14, 1.19),
        "J0740": (2.07, 0.07, 0.07, 12.49, 1.28, 0.88),
        "J0437": (1.418, 0.037, 0.037, 11.36, 0.95, 0.63),
    }
    HEAVY = {"J0348": (2.01, 0.04), "J1614": (1.908, 0.016)}
    HEAVY_BW = {"J0952": (2.35, 0.17)}
    GW170817 = {"Mc": 1.186, "Lt": 300.0, "sp": 420.0 / 1.645, "sm": 230.0 / 1.645}


    def log_likelihoods(M, R, Lam, allowed, use_bw=False, m_lo=1.0, norm="population", m_hi_fixed=3.0):
        """Population-style likelihood per EOS: each observed star is drawn from the allowed sequence with
        a flat mass prior on [m_lo, M_max_allowed]. Returns dict of per-EOS log-likelihoods (B,)."""
        neg = torch.tensor(-1e30, dtype=M.dtype, device=M.device)
        Mm = torch.where(allowed, M, torch.zeros_like(M))
        Mmax = Mm.max(1).values
        span = (Mmax - m_lo).clamp(min=1e-3)
        if norm == "fixed":            # mass prior fixed on [m_lo, m_hi_fixed]: no Occam factor from M_max
            span = torch.full_like(span, m_hi_fixed - m_lo)
        out = {}
        # order the allowed branch by mass for quadrature
        for name, (mu, sp, sm, Ro, rp, rm) in NICER.items():
            lp = split_norm_logpdf(M, mu, sp, sm) + split_norm_logpdf(R, Ro, rp, rm)
            lp = torch.where(allowed & (M >= m_lo), lp, neg)
            out[name] = seq_logint(M, lp, allowed) - torch.log(span)
        pulsars = dict(HEAVY) | (HEAVY_BW if use_bw else {})
        for name, (mu, s) in pulsars.items():
            # P(mass measurement | M_max) with flat mass prior: (Phi((Mmax-mu)/s) - Phi((m_lo-mu)/s)) / span
            cdf = 0.5 * (torch.erf((Mmax - mu) / (s * math.sqrt(2))) - math.erf((m_lo - mu) / (s * math.sqrt(2))))
            cdf = torch.where(Mmax > m_lo, cdf, torch.zeros_like(cdf))
            out[name] = torch.log(cdf.clamp(min=1e-300)) - torch.log(span)
        out["GW170817"] = gw_loglike(M, Lam, allowed)
        return out


    def seq_logint(M, lp, allowed):
        """log of int dM exp(lp) along the allowed (rising) branch, trapezoid on the M(eps_c) samples."""
        dM = (M[:, 1:] - M[:, :-1]).clamp(min=0) * (allowed[:, 1:] & allowed[:, :-1])
        mx = lp.max(1, keepdim=True).values
        f = torch.exp(lp - mx)
        s = (0.5 * (f[:, 1:] + f[:, :-1]) * dM).sum(1)
        return torch.log(s.clamp(min=1e-300)) + mx[:, 0]


    def gw_loglike(M, Lam, allowed, nq=24):
        Mc = GW170817["Mc"]
        q = torch.linspace(0.73, 1.0, nq, dtype=M.dtype, device=M.device)
        m1 = Mc * (1 + q) ** 0.2 / q**0.6; m2 = q * m1
        logL = torch.log(Lam.clamp(min=1e-3))
        Mm = torch.where(allowed, M, torch.full_like(M, -1.0))
        Mmax = Mm.max(1).values
        def lam_at(mq):
            # interpolate log Lambda vs M on the allowed branch (M increasing there)
            Mx = torch.where(allowed, M, torch.full_like(M, 1e9))
            Ms, o = torch.sort(Mx, 1)
            Ls = torch.gather(logL, 1, o)
            val = interp_rows(Ms, Ls, mq.expand(M.shape[0]).contiguous())
            return torch.exp(val)
        acc = []
        for i in range(nq):
            L1, L2 = lam_at(m1[i]), lam_at(m2[i])
            Lt = 16.0 / 13.0 * ((m1[i] + 12 * m2[i]) * m1[i] ** 4 * L1 + (m2[i] + 12 * m1[i]) * m2[i] ** 4 * L2) / (m1[i] + m2[i]) ** 5
            ok = (m1[i] <= Mmax)
            acc.append(torch.where(ok, split_norm_logpdf(Lt, GW170817["Lt"], GW170817["sp"], GW170817["sm"]),
                                   torch.full_like(Lt, -1e30)))
        A = torch.stack(acc, 1)
        return torch.logsumexp(A, 1) - math.log(nq)


    # ---- Palatini trace map (theory check) -----------------------------------------------------------
    def palatini_R_of_T(kT, M, n=1.0, L=3 * 0.69, iters=400):
        """Physical (positive) root of  -R + 4 Lambda + (n+2) M R^-n = kappa T  (H0 = c = 1 units).
        The left side is strictly decreasing in R>0 and spans (+inf, -inf), so the root is unique and
        continuous in T: the R>0 branch is the only one connected to ordinary matter and vacuum."""
        kT = torch.as_tensor(kT, dtype=torch.float64)
        lo = torch.full_like(kT, -700.0); hi = torch.log(torch.abs(kT) + 8 * L + 10.0)
        for _ in range(iters):
            mid = 0.5 * (lo + hi); R = torch.exp(mid)
            g = -R + 4 * L + (n + 2) * M * R ** (-n) - kT
            lo = torch.where(g > 0, mid, lo); hi = torch.where(g > 0, hi, mid)
        return torch.exp(0.5 * (lo + hi))


    def palatini_F(kT, M, n=1.0, L=3 * 0.69):
        R = palatini_R_of_T(kT, M, n, L)
        return 1.0 + n * M * R ** (-(n + 1.0)), R
    return types.SimpleNamespace(**{_k: _v for _k, _v in locals().items() if not _k.startswith("__")})

NS = _build_ns()

In [ ]:
# ---- PART II CONFIG -------------------------------------------------------------------------
NSCFG = {
    "device": CFG["device"],
    "seed": 1729,
    "n_samples": 400 if SMOKE else 1_000_000,     # prior draws of the EOS
    "batch": 200 if SMOKE else 5000,              # EOS per GPU batch (x n_central ODEs)
    "n_central": 20 if SMOKE else 36,
    "n_h": 120 if SMOKE else 240,
    "prior": "uniform",                           # c_s^2 nodes ~ U(0,1); "loguniform": 10^U(-2,0)
    "keep_curves": 400 if SMOKE else 20000,       # M-R curves stored for plotting
    "ckpt_dir": os.path.join(CFG["ckpt_dir"], "part2"),
}
NSCFG.update(json.loads(os.environ.get("NS_OVERRIDES", "{}")))
os.makedirs(NSCFG["ckpt_dir"], exist_ok=True)
_gb = NSCFG["batch"] * NSCFG["n_central"] * 720 * 8 * 4 * 2 / 2**30
NS_MEM_GB = _gb

In [ ]:
# ---- THEOREM 5 CHECK: the physical root, F across the conformal point, non-analytic limit -----
_eps_core = 3 * 1.25e18 / 8.5e-27                 # kappa*eps/H0^2 for eps = 700 MeV/fm^3
_deltas = np.linspace(0.06, -0.08, 141)
_fig, _ax = plt.subplots(1, 3, figsize=(15, 4.2))
for _M, _lab in [(1.6e-5, "Jeans sector at Part-I bound"), (1.0, "Palatini 1/R dark energy (μ~H0)"), (1e-12, "μ⁴ = 1e-12 H0⁴")]:
    _kT = torch.tensor(-3 * _deltas * _eps_core)
    _F, _ = NS.palatini_F(_kT, _M)
    _ax[0].plot(_deltas, torch.log10(_F - 1 + 1e-300).numpy(), label=_lab)
_ax[0].axvline(0, color="k", lw=0.8); _ax[0].invert_xaxis()
_ax[0].set(xlabel="trace anomaly Δ = 1/3 − p/ε", ylabel="log10 (F − 1)", title="F across the conformal point (n=1)")
_ax[0].legend(fontsize=8)
_Ms = np.logspace(-30, 2, 60)
_Fd = [NS.palatini_F(torch.tensor([0.15 * _eps_core]), _m)[0].item() for _m in _Ms]
_ax[1].loglog(_Ms, _Fd, "C3")
_ax[1].set(xlabel="μ⁴ / H0⁴  (cosmological strength)", ylabel="F in a Δ = −0.05 core", title="non-analytic GR limit: weaker ⇒ stronger")
_slope = np.polyfit(np.log(_Ms), np.log(_Fd), 1)[0]
# UV comparison: alpha R^2 with alpha<0 has F = 1 + 2 alpha R, R = -kT + 4 Lambda (finite through Delta=0)
_alpha = -1e-46                                    # in H0^-2 units: a solar-system-safe UV scale
_R = -torch.tensor(-3 * _deltas * _eps_core) + 4 * 2.07
_ax[2].plot(_deltas, (1 + 2 * _alpha * _R).numpy(), "C2")
_ax[2].axvline(0, color="k", lw=0.8); _ax[2].invert_xaxis()
_ax[2].set(xlabel="Δ", ylabel="F", title="UV αR² (α<0): finite through Δ=0")
_fig.tight_layout(); trace_fig = _fig; plt.close(_fig)
mo.vstack([mo.md(f"""**Theorem 5, numerically.** In a Δ = −0.05, 700 MeV/fm³ core the fitted scaling is
F ∝ μ^(4·{_slope:.3f}). Theory predicts F ∝ μ^(−4) for n = 1, i.e. an exponent of −1 in μ⁴. F is O(1) for
every Δ ≥ 0 and explodes immediately below Δ = 0, for every μ. The UV R² term stays finite through the conformal point."""), trace_fig])

In [ ]:
# ---- VALIDATION: batched enthalpy-form TOV + Love numbers vs an independent r-integration -----
from scipy.integrate import solve_ivp as _ivp
_K = 1.0e-4
_ng = torch.logspace(-8, 0.3, 3000, dtype=torch.float64)
_rho = _ng * NS.M_U
_pp = _K * _rho**2; _ee = _rho + _pp; _cs = 2 * _K * _rho / (1 + 2 * _K * _rho); _mu = (_ee + _pp) / _ng
_ncs = [0.3, 0.6, 1.0]
_M, _R, _L, _ = NS.tov_batch(_ee.unsqueeze(0), _pp.unsqueeze(0), _cs.unsqueeze(0), _mu.unsqueeze(0), _ng, _ncs, n_h=NSCFG["n_h"])
_k = NS.MEVFM3_TO_KM2

def _ref(nc):
    pc = _K * (nc * NS.M_U) ** 2 * _k
    def e_of_p(p):
        r = math.sqrt(max(p, 0) / (_K * _k)); return (r + _K * r * r) * _k, 2 * _K * r / (1 + 2 * _K * r)
    def f(r, Y):
        m, p, y = Y
        if p <= 0: return [0, 0, 0]
        e, c2 = e_of_p(p); den = m + 4 * math.pi * r**3 * p; el = 1 / (1 - 2 * m / r); nup = 2 * den / (r * (r - 2 * m))
        Q = 4 * math.pi * el * (5 * e + 9 * p + (e + p) / c2) - 6 * el / r**2 - nup**2
        return [4 * math.pi * r * r * e, -(e + p) * den / (r * (r - 2 * m)), -(y * y + y * el * (1 + 4 * math.pi * r * r * (p - e)) + r * r * Q) / r]
    ev = lambda r, Y: Y[1] - 1e-14 * pc; ev.terminal = True
    e0, _ = e_of_p(pc)
    s = _ivp(f, [1e-6, 60], [4 / 3 * math.pi * e0 * 1e-18, pc, 2.0], events=ev, rtol=1e-10, atol=1e-30, max_step=0.01)
    m, _, y = s.y[:, -1]; r = s.t[-1]; C = m / r
    return m / NS.MSUN_KM, r, 2 / 3 * NS.love_k2(torch.tensor(C), torch.tensor(y)).item() / C**5

_rows = []
TOV_ERR = 0.0
for _i, _nc in enumerate(_ncs):
    _rm, _rr, _rl = _ref(_nc)
    _e = max(abs(_M[0, _i].item() / _rm - 1), abs(_R[0, _i].item() / _rr - 1), abs(_L[0, _i].item() / _rl - 1))
    TOV_ERR = max(TOV_ERR, _e)
    _rows.append(f"| {_nc} | {_M[0,_i].item():.4f} / {_rm:.4f} | {_R[0,_i].item():.3f} / {_rr:.3f} | {_L[0,_i].item():.1f} / {_rl:.1f} |")
mo.md("**TOV validation** (batched / independent scipy, Γ = 2 polytrope): max relative error "
      f"{TOV_ERR:.1e}\n\n| n_c | M | R [km] | Λ |\n|---|---|---|---|\n" + "\n".join(_rows))

In [ ]:
ns_btn = mo.ui.run_button(label=f"Run Part II: conformal-wall EOS inference (~{NS_MEM_GB:.1f} GB VRAM per batch)")
ns_btn

In [ ]:
# ---- HEAVY 3: prior draws -> EOS -> TOV -> GR and Conformal-Wall likelihoods (resumable) ---
mo.stop(not (ns_btn.value or os.environ.get("RUN_NS") == "1"), mo.md("*Press the Part II button to start.*"))
_dev = NSCFG["device"]
_crust = NS.crust_table(0.5 * NS.N0)
_ngr = NS.make_n_grid(device=_dev)
_ncv = list(np.logspace(np.log10(1.0 * NS.N0), np.log10(11.5 * NS.N0), NSCFG["n_central"]))
_nb = -(-NSCFG["n_samples"] // NSCFG["batch"])
_keys = ["GR_pop", "CW_pop", "GR_fix", "CW_fix", "GR_pop_bw", "CW_pop_bw"]
_t0 = time.time()
_kept = 0
for _b in mo.status.progress_bar(range(_nb), title="Part II: EOS batches"):
    _f = os.path.join(NSCFG["ckpt_dir"], f"batch_{_b:05d}.npz")
    if os.path.exists(_f):
        continue
    _g = torch.Generator(device=_dev).manual_seed(NSCFG["seed"] + _b)
    _th = torch.rand(NSCFG["batch"], 7, generator=_g, device=_dev, dtype=torch.float64)
    if NSCFG["prior"] == "loguniform":
        _th[:, 1:] = 10 ** (-2 + 2 * _th[:, 1:])
    _eps, _p, _cs2, _mu = NS.build_eos_batch(_th, _ngr, _crust)
    _M, _R, _L, _ec = NS.tov_batch(_eps, _p, _cs2, _mu, _ngr, _ncv, n_h=NSCFG["n_h"])
    _ok = torch.isfinite(_M) & torch.isfinite(_R) & torch.isfinite(_L) & (_R > 3)
    _st = NS.stable_mask(torch.where(_ok, _M, torch.zeros_like(_M))) & _ok
    _cw, _ewall = NS.conformal_mask(_ec, _eps, _p, _ngr)
    _out = {}
    for _name, _allowed in [("GR", _st), ("CW", _st & _cw)]:
        for _norm, _bw, _tag in [("population", False, "pop"), ("fixed", False, "fix"), ("population", True, "pop_bw")]:
            _ll = NS.log_likelihoods(_M, _R, _L, _allowed, use_bw=_bw, norm=_norm)
            _out[f"{_name}_{_tag}"] = sum(_ll.values()).cpu().numpy()
    _Mg = torch.where(_st, _M, torch.zeros_like(_M)); _Mc = torch.where(_st & _cw, _M, torch.zeros_like(_M))
    _ig = torch.argmax(_Mg, 1, keepdim=True); _icw = torch.argmax(_Mc, 1, keepdim=True)
    def _at(X, m):   # radius of the allowed branch at mass m (NaN if not reached)
        Mx = torch.where(X, _M, torch.full_like(_M, 1e9)); Ms, o = torch.sort(Mx, 1)
        Rs = torch.gather(_R, 1, o)
        v = NS.interp_rows(Ms.contiguous(), Rs.contiguous(), torch.full((_M.shape[0],), m, dtype=_M.dtype, device=_M.device))
        return torch.where(torch.where(X, _M, torch.zeros_like(_M)).max(1).values >= m, v, torch.full_like(v, float("nan")))
    _sv = dict(theta=_th.cpu().numpy(), **_out,
               Mmax_GR=_Mg.max(1).values.cpu().numpy(), Mmax_CW=_Mc.max(1).values.cpu().numpy(),
               ec_GRmax=torch.gather(_ec, 1, _ig)[:, 0].cpu().numpy(), e_wall=_ewall[:, 0].cpu().numpy(),
               R14_GR=_at(_st, 1.4).cpu().numpy(), R20_GR=_at(_st, 2.0).cpu().numpy(),
               R14_CW=_at(_st & _cw, 1.4).cpu().numpy(), R20_CW=_at(_st & _cw, 2.0).cpu().numpy(),
               Rmax_GR=torch.gather(_R, 1, _ig)[:, 0].cpu().numpy(), Rmax_CW=torch.gather(_R, 1, _icw)[:, 0].cpu().numpy())
    if _kept < NSCFG["keep_curves"]:
        _sv.update(M=_M.cpu().numpy().astype(np.float32), R=_R.cpu().numpy().astype(np.float32),
                   st=_st.cpu().numpy(), cw=_cw.cpu().numpy(), ec=_ec.cpu().numpy().astype(np.float32))
        _kept += NSCFG["batch"]
    np.savez(_f + ".tmp.npz", **_sv); os.replace(_f + ".tmp.npz", _f)
    del _eps, _p, _cs2, _mu, _M, _R, _L, _ec
    gpu_free()
NS_FILES = sorted(os.path.join(NSCFG["ckpt_dir"], f) for f in os.listdir(NSCFG["ckpt_dir"]) if f.startswith("batch_") and ".tmp" not in f)
mo.md(f"**Part II inference:** {len(NS_FILES)} batches on disk ({len(NS_FILES)*NSCFG['batch']:,} EOS); this session {(time.time()-_t0)/60:.1f} min.")

In [ ]:
# ---- PART II RESULTS: evidences, Bayes factors (bootstrap), posteriors, figures -------------
_cols = ["GR_pop", "CW_pop", "GR_fix", "CW_fix", "GR_pop_bw", "CW_pop_bw", "Mmax_GR", "Mmax_CW", "ec_GRmax",
         "e_wall", "R14_GR", "R20_GR", "R14_CW", "R20_CW", "Rmax_GR", "Rmax_CW"]
_D = {c: [] for c in _cols}
_curves = []
for _f in NS_FILES:
    _z = np.load(_f)
    for _cc in _cols:
        _D[_cc].append(_z[_cc])
    if "M" in _z.files:
        _curves.append({k: _z[k] for k in ("M", "R", "st", "cw", "GR_pop", "CW_pop")})
NS_D = {_k: np.concatenate(_v) for _k, _v in _D.items()}
_N = len(NS_D["GR_pop"])

def _logZ(l):
    l = np.where(np.isfinite(l), l, -1e30); m = l.max(); return m + np.log(np.mean(np.exp(l - m)))

def _bf(a, b, nboot=200, seed=0):
    r = np.random.default_rng(seed); out = []
    for _ in range(nboot):
        i = r.integers(0, _N, _N); out.append(_logZ(NS_D[a][i]) - _logZ(NS_D[b][i]))
    return _logZ(NS_D[a]) - _logZ(NS_D[b]), np.std(out)

NS_STATS = {}
for _tag, _lab in [("pop", "population mass prior"), ("fix", "fixed mass prior"), ("pop_bw", "population + J0952")]:
    _lb, _se = _bf(f"CW_{_tag}", f"GR_{_tag}")
    _w = np.exp(NS_D[f"GR_{_tag}"] - NS_D[f"GR_{_tag}"].max()); _w /= _w.sum()
    _wc = np.exp(NS_D[f"CW_{_tag}"] - NS_D[f"CW_{_tag}"].max()); _wc /= _wc.sum()
    _super = (NS_D["e_wall"] < NS_D["ec_GRmax"]).astype(float)
    def _q(x, w):
        m = np.isfinite(x) & (w > 0); o = np.argsort(x[m]); c = np.cumsum(w[m][o]); c /= c[-1]
        return [float(np.interp(p, c, x[m][o])) for p in (0.16, 0.5, 0.84)]
    NS_STATS[_tag] = {"label": _lab, "log10_BF_CW_GR": _lb / math.log(10), "log10_BF_err": _se / math.log(10),
                      "ESS_GR": float(1 / np.sum(_w**2)), "ESS_CW": float(1 / np.sum(_wc**2)),
                      "P_superconformal_core_GR": float(np.sum(_w * _super)),
                      "Mmax_GR": _q(NS_D["Mmax_GR"], _w), "Mmax_CW": _q(NS_D["Mmax_CW"], _wc),
                      "R14_GR": _q(NS_D["R14_GR"], _w), "R14_CW": _q(NS_D["R14_CW"], _wc),
                      "R20_GR": _q(NS_D["R20_GR"], _w), "R20_CW": _q(NS_D["R20_CW"], _wc),
                      "eps_wall_CW": _q(NS_D["e_wall"], _wc)}
_rows = "\n".join(f"| {v['label']} | {v['log10_BF_CW_GR']:+.2f} ± {v['log10_BF_err']:.2f} | {v['ESS_GR']:.0f} / {v['ESS_CW']:.0f} | "
                   f"{v['P_superconformal_core_GR']:.2f} | {v['Mmax_GR'][1]:.2f} / {v['Mmax_CW'][1]:.2f} | "
                   f"{v['R20_GR'][1]:.2f} / {v['R20_CW'][1]:.2f} |" for v in NS_STATS.values())
# figures
_fig, _ax = plt.subplots(1, 3, figsize=(15, 4.4))
if _curves:
    _c = _curves[0]
    for _key, _col, _lab, _mask in [("GR_pop", "C0", "GR", "st"), ("CW_pop", "C3", "Conformal Wall", "cw")]:
        _ll = _c[_key]; _w = np.exp(_ll - _ll.max()); _w /= _w.sum()
        _idx = np.random.default_rng(1).choice(len(_w), size=min(150, len(_w)), p=_w)
        for _j in _idx:
            _m = _c["st"][_j] & (_c["cw"][_j] if _mask == "cw" else True)
            _ax[0].plot(_c["R"][_j][_m], _c["M"][_j][_m], color=_col, alpha=0.05, lw=0.8)
        _ax[0].plot([], [], color=_col, label=_lab)
for (_mm, _sp, _sm, _rr, _rp, _rm), _n in [(v, k) for k, v in {"J0030": (1.34, .15, .16, 12.71, 1.14, 1.19), "J0740": (2.07, .07, .07, 12.49, 1.28, .88), "J0437": (1.418, .037, .037, 11.36, .95, .63)}.items()]:
    _ax[0].errorbar(_rr, _mm, xerr=[[_rm], [_rp]], yerr=[[_sm], [_sp]], fmt="ks", ms=3)
_ax[0].set(xlabel="R [km]", ylabel="M [M☉]", xlim=(9, 16), ylim=(0.8, 3.0), title="posterior M–R: GR vs Conformal Wall"); _ax[0].legend()
_w = np.exp(NS_D["GR_pop"] - NS_D["GR_pop"].max()); _w /= _w.sum()
_wc = np.exp(NS_D["CW_pop"] - NS_D["CW_pop"].max()); _wc /= _wc.sum()
_ax[1].hist(NS_D["Mmax_GR"], bins=60, range=(1.8, 3.2), weights=_w, histtype="step", density=True, label="GR")
_ax[1].hist(NS_D["Mmax_CW"], bins=60, range=(1.8, 3.2), weights=_wc, histtype="step", density=True, label="Conformal Wall")
_ax[1].set(xlabel="M_max [M☉]", title="maximum mass"); _ax[1].legend()
_fin = np.isfinite(NS_D["e_wall"])
_ax[2].hist(np.where(_fin, NS_D["e_wall"], 5000)[_fin], bins=60, range=(200, 2500), weights=_w[_fin], histtype="step", density=True, label="ε where Δ first < 0 (GR post.)")
_ax[2].hist(NS_D["ec_GRmax"], bins=60, range=(200, 2500), weights=_w, histtype="step", density=True, label="ε_c of the GR M_max star")
_ax[2].set(xlabel="energy density [MeV/fm³]", title="where the wall sits vs where stars reach"); _ax[2].legend(fontsize=8)
_fig.tight_layout(); ns_fig = _fig; plt.close(_fig)
mo.vstack([mo.md(f"""### Part II results ({_N:,} prior EOS draws)
| likelihood choice | log₁₀ B(CW/GR) | ESS GR / CW | P(superconformal core in GR M_max star) | median M_max GR / CW | median R(2.0) GR / CW [km] |
|---|---|---|---|---|---|
{_rows}
"""), ns_fig])

In [ ]:
# ---- PART II VERDICT (pre-registered) and the forecast that decides it ---------------------
_lbs = [v["log10_BF_CW_GR"] for v in NS_STATS.values()]
_lo, _hi = min(_lbs), max(_lbs)
if _hi < -1:
    _v = ("**The Conformal Wall is strongly disfavoured (B < 1/10 for every likelihood choice).** Because the test "
          "has no gravitational parameter (H2), this disfavours the *entire* infrared-Palatini class, for every μ and n, "
          "independently of cosmology. That also closes the coarse-graining loophole left by Olmo/Flanagan.")
elif _lo > 1:
    _v = ("**The data prefer the Conformal Wall (B > 10 for every likelihood choice).** Neutron-star data would then "
          "favour a sequence ending at an exactly conformal centre. That would be a gravitational, not nuclear, origin "
          "for the maximum mass, and it demands a full Palatini TOV and independent EOS-prior checks before any claim.")
else:
    _v = ("**Current data cannot decide (1/10 < B < 10).** The Conformal Wall survives as a sharp, parameter-free "
          "prediction: the heaviest neutron star must have an exactly conformal centre, and its M–R curve must end "
          "without turning over. The decisive observable is given below.")
_p = NS_STATS["pop"]
_dM = _p["Mmax_GR"][1] - _p["Mmax_CW"][1]
_dR = _p["R20_CW"][1] - _p["R20_GR"][1]
if abs(_dM) / 0.05 >= abs(_dR) / 0.3:
    _disc = (f"the maximum mass. The wall predicts M_max = {_p['Mmax_CW'][1]:.2f} "
             f"(68%: {_p['Mmax_CW'][0]:.2f}–{_p['Mmax_CW'][2]:.2f}) M☉ versus {_p['Mmax_GR'][1]:.2f} M☉ in GR. A pulsar "
             f"measured above ~{_p['Mmax_CW'][2]:.2f} M☉ to ±0.05 M☉ falsifies the whole infrared-Palatini class, for any μ. "
             "Compare the population + J0952 row: the black-widow mass is already the sharpest lever.")
else:
    _disc = (f"the radius of a 2 M☉ star: {_p['R20_CW'][1]:.2f} km (wall) versus {_p['R20_GR'][1]:.2f} km (GR); "
             "a ±0.3 km measurement separates them.")
mo.md(f"""
## Part II verdict
{_v}

- **Range of log₁₀ B(CW/GR)** across likelihood choices: [{_lo:+.2f}, {_hi:+.2f}].
- **Superconformal cores in GR:** under ordinary GR, P(the heaviest star has a superconformal core) =
  {_p['P_superconformal_core_GR']:.2f}. This is a statement about the EOS. It becomes a test of gravity only
  through the Bayes factor.
- **Decisive observable (data-driven):** {_disc}
- **Before any claim:**
  - replace the summary likelihoods with the public NICER and LVK samples;
  - add the pQCD constraint;
  - repeat with `NS_OVERRIDES='{{"prior":"loguniform"}}'`;
  - solve the Palatini TOV across Δ = 0 to turn H1 from hypothesis into theorem.
""")

In [ ]:
_out = os.path.join(NSCFG["ckpt_dir"], "PART2_RESULTS.json")
save_json(_out, {"config": NSCFG, "tov_validation_max_rel_err": TOV_ERR, "stats": NS_STATS})
ns_fig.savefig(os.path.join(NSCFG["ckpt_dir"], "conformal_wall_results.png"), dpi=160)
trace_fig.savefig(os.path.join(NSCFG["ckpt_dir"], "theorem5_trace_map.png"), dpi=160)
mo.md(f"Part II saved to `{_out}` and two figures in `{NSCFG['ckpt_dir']}/`. Download before the session ends.")

---
# Part III — From hypothesis to theorem, and a hardened test

Part II found that current data cannot decide ($-0.06\le\log_{10}B_{\rm CW/GR}\le+0.15$). It also
rested on an assumption (H1): no static star can hold superconformal matter in an infrared-Palatini
universe. Part III removes that assumption with an exact result. It then repeats the inference with the
upgrades a referee will demand.

## III.1 Theorem 7 — Palatini stars are GR stars of an effective fluid
Because the independent connection is the Levi-Civita connection of $h_{\mu\nu}=F g_{\mu\nu}$, and
$h^{\mu\nu}R_{\mu\nu}(h)=R/F$, the Palatini equations $F R_{\mu\nu}(h)-\tfrac12 f g_{\mu\nu}=\kappa T_{\mu\nu}$ can be
rewritten **exactly** as Einstein's equations for $h$:
$$G_{\mu\nu}(h)=\frac{\kappa}{F}T_{\mu\nu}-\frac{RF-f}{2F^2}\,h_{\mu\nu}
\quad\Longrightarrow\quad
\tilde\varepsilon=\frac{\varepsilon}{F^2}+\frac{RF-f}{2\kappa F^2},\qquad
\tilde p=\frac{p}{F^2}-\frac{RF-f}{2\kappa F^2}.$$
$F$, $R$ and $f$ are algebraic functions of $T=-\varepsilon+3p$, so $(\tilde\varepsilon,\tilde p)$ is a *local*
equation of state. A static Palatini star is therefore an ordinary TOV star in the $h$ frame, and
$\nabla^{(h)}_\mu\tilde T^{\mu\nu}=0$ holds by the Bianchi identity. (GR check: $F=1$ gives
$\tilde\varepsilon=\varepsilon+\Lambda/\kappa$, $\tilde p=p-\Lambda/\kappa$.)

## III.2 Corollary — the Conformal Wall is forced
- **Below the wall** (subconformal matter): $F-1\lesssim10^{-7}$, so $\tilde p=p$ to machine precision.
- **Beyond the wall** (superconformal, $T>0$): $F\simeq(\kappa T)^2/9\mu^4$, so
  $\tilde p\simeq(\varepsilon/3)/F^2\to0$ and $\tilde\varepsilon\simeq(2\varepsilon/3+p)/F^2\to0$.
- **Hence $\tilde p(p)$ has its global maximum exactly at the conformal point** $p=\varepsilon/3$.
- **Hydrostatic equilibrium forbids crossing it.** Equilibrium needs $\tilde p$ to rise monotonically
  inward from the surface, where matter is subconformal. To reach superconformal matter a star would have to
  push $\tilde p$ past its *first* maximum, which is the conformal point. If $\Delta$ later turns positive
  again at higher density, those states stay unreachable behind the wall.

**No static, spherically symmetric perfect-fluid star in an infrared-Palatini theory can contain matter
with $p>\varepsilon/3$**, for any $\mu>0$ and $n>0$. The stellar sequence ends at $\Delta_c=0$ (H1). Beyond
it no equilibrium exists. An accreting star whose centre reaches the conformal point must collapse:
a *conformal collapse*, triggered by QCD conformality rather than by the GR turning point.

## III.3 What Part III adds to the test
1. **Cell 34** verifies Theorem 7 numerically on posterior equations of state from Part II, for
   $\mu^4/H_0^4=10^{-30},\,1.6\times10^{-5},\,1$.
2. **pQCD** (Komoltsev–Kurkela 2022 integral constraint with the Fraga–Kurkela–Vuorinen pressure at
   $\mu_B=2.6$ GeV, $X\in[1,4]$), applied separately to the GR and the conformal-wall maximum-mass star.
3. **Prior robustness:** uniform and log-uniform $c_s^2$ priors.
4. **Public posterior samples:** if files are placed in `ns_samples/`, NICER and GW170817 enter through
   KDEs of the real posteriors instead of split-normal summaries.
5. **A forest plot** of $\log_{10}B_{\rm CW/GR}$ over every variant, and a verdict that is robust only if
   all variants agree.

In [ ]:
# ---- PART III CORE (tested module, embedded verbatim; exported as namespace NS3) ------------
def _build_ns3():
    # Part III core: exact Palatini stars via the Einstein frame, pQCD (Komoltsev-Kurkela) constraint,
    # sample-based (KDE) likelihoods. Uses functions of NS (Part II) passed in as `ns`.
    import math
    import os
    import numpy as np
    import torch

    HBARC = 197.327                              # MeV fm
    H_LITTLE = 0.677
    RHO_CRIT_C2 = 1.87847e-26 * H_LITTLE**2 * 2.99792458e8**2 / 1.602176634e-13 / 1e45   # MeV/fm^3
    KAPPA_H = 3.0 / RHO_CRIT_C2                  # kappa*eps/H0^2 per (MeV/fm^3)

    # ---- Theorem 7: Einstein-frame effective fluid -------------------------------------------------
    def palatini_effective(ns, eps, p, M, n=1.0, OL=0.69):
        """Palatini f(R)=R-2L-M R^-n.  G_mn(h) = (kappa/F) T_mn - (RF-f)/(2F^2) h_mn with h = F g gives an
        exact GR star in h with  eps_t = eps/F^2 + (RF-f)/(2 kappa F^2),  p_t = p/F^2 - (RF-f)/(2 kappa F^2).
        Inputs/outputs in MeV/fm^3; also returns F."""
        L = 3.0 * OL
        kT = KAPPA_H * (-eps + 3.0 * p)
        F, R = ns.palatini_F(kT, M, n, L)
        f = R - 2.0 * L - M * R ** (-n)
        RFf = (R * F - f) / KAPPA_H
        return eps / F**2 + RFf / (2 * F**2), p / F**2 - RFf / (2 * F**2), F

    # ---- pQCD: Fraga-Kurkela-Vuorinen (2014) fit + Komoltsev-Kurkela (2022) integral constraint -----
    def p_qcd(mu, X):
        """mu_B in MeV, X renormalisation-scale parameter. MeV/fm^3. (Benchmark: mu=2600, X=2 -> 3825.)"""
        muq = mu / 3.0
        psb = 3.0 * muq**4 / (4 * math.pi**2) / HBARC**3
        return psb * (0.9008 - 0.5034 * X ** (-0.3553) / (mu / 1000.0 - 1.452 * X ** (-0.9101)))

    def n_qcd(mu, X, h=1.0):
        return (p_qcd(mu + h, X) - p_qcd(mu - h, X)) / (2 * h)

    def pqcd_logweight(muL, nL, pL, mu_H=2600.0, nX=16):
        """log of the fraction of X in [1,4] (log-uniform) for which a causal, stable, thermodynamically
        consistent EOS can connect (muL,nL,pL) at the TOV point to pQCD at mu_H (Komoltsev & Kurkela 2022).
            Delta p_min = (mu_H^2 - muL^2) nL  / (2 muL)
            Delta p_max = (mu_H^2 - muL^2) n_H / (2 mu_H)"""
        Xs = torch.exp(torch.linspace(0.0, math.log(4.0), nX, dtype=muL.dtype, device=muL.device))
        ok = []
        for X in Xs.tolist():
            pH, nH = p_qcd(mu_H, X), n_qcd(mu_H, X)
            dp = pH - pL
            lo = (mu_H**2 - muL**2) * nL / (2 * muL)
            hi = (mu_H**2 - muL**2) * nH / (2 * mu_H)
            ok.append(((dp >= lo) & (dp <= hi) & (muL < mu_H)).to(muL.dtype))
        frac = torch.stack(ok, 1).mean(1)
        return torch.where(frac > 0, torch.log(frac.clamp(min=1e-300)), torch.full_like(frac, -1e30))

    # ---- KDE likelihoods from public posterior samples (optional) ---------------------------------
    def read_table(path):
        with open(path) as fh:
            first = fh.readline()
        delim = "," if "," in first else None
        has_header = any(c.isalpha() for c in first.replace("e", "").replace("E", ""))
        arr = np.genfromtxt(path, delimiter=delim, names=True if has_header else None, dtype=float)
        return arr

    def load_mr_kde(path, mgrid=(0.8, 3.2, 240), rgrid=(8.0, 17.0, 240), max_n=40000):
        """Two-column (M [Msun], R [km]) posterior samples -> log-density on a grid (Gaussian KDE)."""
        from scipy.stats import gaussian_kde
        a = read_table(path)
        if a.dtype.names:
            names = [n.lower() for n in a.dtype.names]
            def pick(keys):
                for k in keys:
                    for i, nm in enumerate(names):
                        if nm == k or nm.startswith(k):
                            return np.asarray(a[a.dtype.names[i]], float)
                return None
            M = pick(["m", "mass"]); R = pick(["r", "radius"])
        else:
            M, R = a[:, 0], a[:, 1]
        if np.nanmedian(R) > 1e3:   # radius given in metres
            R = R / 1e3
        sel = np.isfinite(M) & np.isfinite(R)
        M, R = M[sel], R[sel]
        if len(M) > max_n:
            idx = np.random.default_rng(0).choice(len(M), max_n, replace=False); M, R = M[idx], R[idx]
        k = gaussian_kde(np.vstack([M, R]))
        mg = np.linspace(*mgrid); rg = np.linspace(*rgrid)
        MM, RR = np.meshgrid(mg, rg, indexing="ij")
        d = k(np.vstack([MM.ravel(), RR.ravel()])).reshape(MM.shape)
        return {"m": mg, "r": rg, "logd": np.log(np.maximum(d, 1e-300)), "n": len(M), "path": path}

    def load_lt_kde(path, grid=(0.0, 3000.0, 600)):
        """GW170817 samples: a column lambdatilde / lambda_tilde, or m1,m2,lambda1,lambda2 -> 1-D KDE of Lambda~."""
        from scipy.stats import gaussian_kde
        a = read_table(path)
        names = [n.lower() for n in a.dtype.names] if a.dtype.names else []
        col = lambda k: np.asarray(a[a.dtype.names[names.index(k)]], float)
        if any("tilde" in n or n == "lambdat" for n in names):
            nm = next(n for n in names if "tilde" in n or n == "lambdat"); Lt = col(nm)
        elif all(k in names for k in ("m1", "m2", "lambda1", "lambda2")):
            m1, m2, l1, l2 = col("m1"), col("m2"), col("lambda1"), col("lambda2")
            Lt = 16 / 13 * ((m1 + 12 * m2) * m1**4 * l1 + (m2 + 12 * m1) * m2**4 * l2) / (m1 + m2) ** 5
        else:
            Lt = np.asarray(a if a.ndim == 1 else a[:, 0], float)
        Lt = Lt[np.isfinite(Lt)]
        k = gaussian_kde(Lt)
        g = np.linspace(*grid)
        return {"lt": g, "logd": np.log(np.maximum(k(g), 1e-300)), "n": len(Lt), "path": path}

    def grid_logpdf_2d(kde, M, R):
        m, r, L = kde["m"], kde["r"], kde["logd"]
        Lt = torch.as_tensor(L, dtype=M.dtype, device=M.device)
        fi = ((M - m[0]) / (m[1] - m[0])).clamp(0, len(m) - 1.001)
        fj = ((R - r[0]) / (r[1] - r[0])).clamp(0, len(r) - 1.001)
        i0, j0 = fi.floor().long(), fj.floor().long()
        wi, wj = fi - i0, fj - j0
        v = (Lt[i0, j0] * (1 - wi) * (1 - wj) + Lt[i0 + 1, j0] * wi * (1 - wj)
             + Lt[i0, j0 + 1] * (1 - wi) * wj + Lt[i0 + 1, j0 + 1] * wi * wj)
        outside = (M < m[0]) | (M > m[-1]) | (R < r[0]) | (R > r[-1])
        return torch.where(outside, torch.full_like(v, -700.0), v)

    def grid_logpdf_1d(kde, x):
        g, L = kde["lt"], torch.as_tensor(kde["logd"], dtype=x.dtype, device=x.device)
        f = ((x - g[0]) / (g[1] - g[0])).clamp(0, len(g) - 1.001)
        i0 = f.floor().long(); w = f - i0
        v = L[i0] * (1 - w) + L[i0 + 1] * w
        return torch.where((x < g[0]) | (x > g[-1]), torch.full_like(v, -700.0), v)

    def log_likelihoods_v3(ns, M, R, Lam, allowed, kdes, use_bw=False, m_lo=1.0):
        """Like NS.log_likelihoods (population mass prior) but NICER/GW terms use KDEs where provided."""
        base = ns.log_likelihoods(M, R, Lam, allowed, use_bw=use_bw)
        neg = torch.tensor(-1e30, dtype=M.dtype, device=M.device)
        Mmax = torch.where(allowed, M, torch.zeros_like(M)).max(1).values
        span = (Mmax - m_lo).clamp(min=1e-3)
        for name in ("J0030", "J0740", "J0437"):
            if name in kdes:
                lp = grid_logpdf_2d(kdes[name], M, R)
                lp = torch.where(allowed & (M >= m_lo), lp, neg)
                base[name] = ns.seq_logint(M, lp, allowed) - torch.log(span)
        if "GW170817" in kdes:
            Mc = ns.GW170817["Mc"]
            q = torch.linspace(0.73, 1.0, 24, dtype=M.dtype, device=M.device)
            m1 = Mc * (1 + q) ** 0.2 / q**0.6; m2 = q * m1
            logL = torch.log(Lam.clamp(min=1e-3))
            Mx = torch.where(allowed, M, torch.full_like(M, 1e9)); Ms, o = torch.sort(Mx, 1); Ls = torch.gather(logL, 1, o)
            acc = []
            for i in range(len(q)):
                L1 = torch.exp(ns.interp_rows(Ms.contiguous(), Ls.contiguous(), m1[i].expand(M.shape[0]).contiguous()))
                L2 = torch.exp(ns.interp_rows(Ms.contiguous(), Ls.contiguous(), m2[i].expand(M.shape[0]).contiguous()))
                Lt = 16 / 13 * ((m1[i] + 12 * m2[i]) * m1[i] ** 4 * L1 + (m2[i] + 12 * m1[i]) * m2[i] ** 4 * L2) / (m1[i] + m2[i]) ** 5
                acc.append(torch.where(m1[i] <= Mmax, grid_logpdf_1d(kdes["GW170817"], Lt), torch.full_like(Lt, -1e30)))
            base["GW170817"] = torch.logsumexp(torch.stack(acc, 1), 1) - math.log(len(q))
        return base

    def tov_point(ns, eps, p, mu, n_grid, ec):
        """(mu, n, p) at central energy density ec for each EOS row (for the pQCD constraint)."""
        B = eps.shape[0]
        nn = n_grid.view(1, -1).expand(B, -1).contiguous()
        return (ns.interp_rows(eps.contiguous(), mu.contiguous(), ec), ns.interp_rows(eps.contiguous(), nn, ec),
                ns.interp_rows(eps.contiguous(), p.contiguous(), ec))

    # ---- self-contained helpers (do not require a re-run of the Part II core) -----------------
    def tov_batch_rows(ns, eps, p, cs2, mu, n_grid, nc_vals, n_h=320):
        """Same as NS.tov_batch but also accepts per-EOS central densities nc_vals as a (B,Nc) tensor."""
        B, N = eps.shape
        dt, dev = eps.dtype, eps.device
        h_tab = torch.log(mu / mu[:, :1])
        E = eps * ns.MEVFM3_TO_KM2; P = p * ns.MEVFM3_TO_KM2
        lnn = torch.log(n_grid)
        if torch.is_tensor(nc_vals):
            ncl = torch.log(nc_vals.to(dtype=dt, device=dev))
            hc = torch.stack([ns.interp_rows(lnn.expand(B, -1).contiguous(), h_tab, ncl[:, j].contiguous())
                              for j in range(ncl.shape[1])], 1)
            nc_vals = [None] * ncl.shape[1]
        else:
            hc = torch.stack([ns.interp_rows(lnn.expand(B, -1).contiguous(), h_tab, torch.full((B,), math.log(v), dtype=dt, device=dev))
                              for v in nc_vals], 1)
        S = hc.reshape(-1)
        Hrows = h_tab.repeat_interleave(len(nc_vals), 0).contiguous()
        Erows = E.repeat_interleave(len(nc_vals), 0); Prows = P.repeat_interleave(len(nc_vals), 0)
        Crows = cs2.repeat_interleave(len(nc_vals), 0)

        def eos_at(hq):
            xq = hq.reshape(-1, 1)
            idx = torch.clamp(torch.searchsorted(Hrows, xq.contiguous()), 1, Hrows.shape[1] - 1)
            x0 = torch.gather(Hrows, 1, idx - 1); x1 = torch.gather(Hrows, 1, idx)
            w = ((xq - x0) / (x1 - x0).clamp(min=1e-300)).clamp(0, 1)
            out = []
            for T_ in (Erows, Prows, Crows):
                y0 = torch.gather(T_, 1, idx - 1); y1 = torch.gather(T_, 1, idx)
                out.append((y0 + w * (y1 - y0)).reshape(-1))
            return out

        ec, pc, _ = eos_at(S)
        dh0 = 1e-3 * S
        r = torch.sqrt(3 * dh0 / (2 * math.pi * (ec + 3 * pc)))
        m = 4 * math.pi / 3 * ec * r**3
        y = torch.full_like(r, 2.0)
        hs = torch.linspace(0, 1, n_h + 1, dtype=dt, device=dev)

        def rhs(h, r, m, y):
            e, pp, c2 = eos_at(h)
            den = m + 4 * math.pi * r**3 * pp
            drdh = -r * (r - 2 * m) / den
            dmdh = 4 * math.pi * r**2 * e * drdh
            el = 1.0 / (1 - 2 * m / r)
            nup = 2 * den / (r * (r - 2 * m))
            Q = 4 * math.pi * el * (5 * e + 9 * pp + (e + pp) / c2) - 6 * el / r**2 - nup**2
            dydr = -(y**2 + y * el * (1 + 4 * math.pi * r**2 * (pp - e)) + r**2 * Q) / r
            return drdh, dmdh, dydr * drdh

        h_start = S - dh0
        for i in range(n_h):
            a0 = h_start * (1 - hs[i]); a1 = h_start * (1 - hs[i + 1]); st = a1 - a0
            k1 = rhs(a0, r, m, y)
            k2 = rhs(a0 + st / 2, r + st / 2 * k1[0], m + st / 2 * k1[1], y + st / 2 * k1[2])
            k3 = rhs(a0 + st / 2, r + st / 2 * k2[0], m + st / 2 * k2[1], y + st / 2 * k2[2])
            k4 = rhs(a1, r + st * k3[0], m + st * k3[1], y + st * k3[2])
            r = r + st / 6 * (k1[0] + 2 * k2[0] + 2 * k3[0] + k4[0])
            m = m + st / 6 * (k1[1] + 2 * k2[1] + 2 * k3[1] + k4[1])
            y = y + st / 6 * (k1[2] + 2 * k2[2] + 2 * k3[2] + k4[2])
        C = m / r
        k2 = ns.love_k2(C, y)
        Lam = 2.0 / 3.0 * k2 / C**5
        sh = (B, len(nc_vals))
        return (m / ns.MSUN_KM).reshape(sh), r.reshape(sh), Lam.reshape(sh), (ec / ns.MEVFM3_TO_KM2).reshape(sh)

    def wall_density(eps, p, n_grid):
        """Number density of the last subconformal grid node below the first superconformal point (B,).
        NaN when the EOS never becomes superconformal on the grid."""
        sup = (p / eps > 1.0 / 3.0)
        has = sup.any(1)
        first = torch.argmax(sup.to(torch.int64), 1)
        idx = (first - 1).clamp(min=0)
        nw = n_grid[idx]
        return torch.where(has, nw, torch.full_like(nw, float("nan")))

    def add_wall_point(M, R, Lam, ec, allowed_cw, Mw, Rw, Lw, ecw, wall_ok):
        """Append the exact wall star to the conformal-wall sequence and re-sort each row by central density."""
        M2 = torch.cat([M, Mw.view(-1, 1)], 1); R2 = torch.cat([R, Rw.view(-1, 1)], 1)
        L2 = torch.cat([Lam, Lw.view(-1, 1)], 1); e2 = torch.cat([ec, torch.where(wall_ok, ecw, torch.full_like(ecw, 1e30)).view(-1, 1)], 1)
        a2 = torch.cat([allowed_cw, wall_ok.view(-1, 1)], 1)
        o = torch.argsort(e2, 1)
        g = lambda X: torch.gather(X, 1, o)
        return g(M2), g(R2), g(L2), g(e2), g(a2)

    return types.SimpleNamespace(**{_k: _v for _k, _v in locals().items() if not _k.startswith("__")})

NS3 = _build_ns3()

In [ ]:
# ---- THEOREM 7, NUMERICALLY: Einstein-frame pressure peaks exactly at the conformal point -----
_dev = NSCFG["device"]
_z = np.load(NS_FILES[0])
_w = _z["GR_pop"]; _sup = _z["e_wall"] < _z["ec_GRmax"]
_order = [i for i in np.argsort(-_w) if _sup[i]][:6]           # most probable EOS that reach Delta<0 in GR
if not _order:                                                  # fall back to any EOS with a superconformal region
    _order = [i for i in range(len(_w)) if np.isfinite(_z["e_wall"][i])][:6]
mo.stop(not _order, mo.md("No equation of state with a superconformal region in the first Part II batch; nothing to check."))
_th = torch.tensor(_z["theta"][_order], device=_dev)
_crust = NS.crust_table(0.5 * NS.N0)
_ngf = NS.make_n_grid(npts=6000, device=_dev)                   # fine grid to locate the maximum
_eps, _p, _cs2, _mu = NS.build_eos_batch(_th, _ngf, _crust)
_rows = []
_fig, _ax = plt.subplots(1, 2, figsize=(12, 4.2))
THEOREM7 = []
for _i in range(len(_order)):
    _e, _pp = _eps[_i], _p[_i]
    _iw = int(torch.argmax((_pp / _e > 1 / 3).to(torch.int64)))
    for _M in (1e-30, 1.6e-5, 1.0):
        _et, _pt, _F = NS3.palatini_effective(NS, _e, _pp, _M)
        _dec = (_pt[1:] < _pt[:-1])                                 # first local maximum of p_t(p):
        _im = int(torch.argmax(_dec.to(torch.int64))) if bool(_dec.any()) else int(_pt.numel() - 1)
        _d_at = (1 / 3 - _pp[_im] / _e[_im]).item()
        _below = (_pt[:_iw] / _pp[:_iw] - 1).abs().max().item()
        THEOREM7.append({"eos": int(_order[_i]), "mu4": _M, "Delta_at_max": _d_at, "grid_gap": _iw - _im,
                         "max_rel_dev_below_wall": _below, "F_past_wall": _F[_iw].item()})
        if _M == 1.6e-5:
            _x = (_e / _e[_iw]).cpu().numpy()
            _ax[0].semilogy(_x, (_pt / _pp[_iw]).clamp(min=1e-30).cpu().numpy(), lw=1)
    # exact Palatini maximum mass = TOV mass at the wall (p_t = p there); compare with Part II's grid value
    _nw = float(_ngf[_iw - 1])
    _Mw = NS.tov_batch(_eps[_i:_i + 1], _p[_i:_i + 1], _cs2[_i:_i + 1], _mu[_i:_i + 1], _ngf, [_nw], n_h=NSCFG["n_h"])[0].item()
    _rows.append(f"| {int(_order[_i])} | {_e[_iw].item():.0f} | {_Mw:.4f} | {float(_z['Mmax_CW'][_order[_i]]):.4f} | {float(_z['Mmax_GR'][_order[_i]]):.4f} |")
_ax[0].axvline(1, color="k", lw=0.8)
_ax[0].set(xlabel="ε / ε_wall", ylabel="p̃ / p_wall (Einstein frame)", xlim=(0.3, 1.6), ylim=(1e-30, 3),
           title="Theorem 7: p̃ peaks at the conformal point")
_dmax = max(abs(t["Delta_at_max"]) for t in THEOREM7)
_ax[1].bar(range(len(THEOREM7)), [abs(t["Delta_at_max"]) + 1e-12 for t in THEOREM7])
_ax[1].set(yscale="log", xlabel="EOS × μ case", ylabel="|Δ| at the p̃ maximum", title="the maximum sits at Δ = 0")
_fig.tight_layout(); t7_fig = _fig; plt.close(_fig)
_ok = all(t["grid_gap"] == 1 and t["max_rel_dev_below_wall"] < 1e-12 for t in THEOREM7)
mo.vstack([mo.md(f"""
**Theorem 7 check — {"PASSED" if _ok else "FAILED"}** for {len(_order)} posterior EOS × 3 values of μ:
- the first maximum of the Einstein-frame pressure is always the last subconformal grid point, with max |Δ| there
  = {_dmax:.1e} (set by the 6000-point grid spacing);
- below the wall p̃ = p to {max(t['max_rel_dev_below_wall'] for t in THEOREM7):.0e};
- just past it F ≈ {min(t['F_past_wall'] for t in THEOREM7):.0e}–{max(t['F_past_wall'] for t in THEOREM7):.0e}.

| EOS | ε_wall [MeV/fm³] | exact Palatini M_max = M(wall) | Part II truncated M_max | GR M_max |
|---|---|---|---|---|
{chr(10).join(_rows)}

**Correction to Part II.** Part II truncated the sequence on its fixed grid of central densities, so its wall
maximum mass (column 4) sits *below* the exact value (column 3) by up to one grid step. Part III therefore adds the
exact wall star to every sequence (cell 38). Its Bayes factors and M_max predictions supersede Part II's.
"""), t7_fig])

In [ ]:
# ---- PART III CONFIG --------------------------------------------------------------------------
P3CFG = {
    "device": CFG["device"],
    "seed": 314159,
    "n_samples": 200 if SMOKE else 500_000,      # per variant
    "batch": 100 if SMOKE else 5000,
    "n_central": 20 if SMOKE else 36,
    "n_h": 120 if SMOKE else 240,
    "samples_dir": "ns_samples",                 # optional public posterior samples (see cell 36)
    "variants": [
        {"name": "uniform_summary", "prior": "uniform", "lik": "summary"},
        {"name": "loguniform_summary", "prior": "loguniform", "lik": "summary"},
    ],
    "ckpt_dir": os.path.join(CFG["ckpt_dir"], "part3"),
}
P3CFG.update(json.loads(os.environ.get("P3_OVERRIDES", "{}")))
os.makedirs(P3CFG["ckpt_dir"], exist_ok=True)
os.makedirs(P3CFG["samples_dir"], exist_ok=True)

In [ ]:
# ---- OPTIONAL: public posterior samples -> KDE likelihoods ----------------------------------
# Put any of these files in ns_samples/ (whitespace or CSV; a header row with column names is fine):
#   J0030.txt, J0740.txt, J0437.txt   two columns: mass [Msun], radius [km]
#     (NICER posterior samples from the Zenodo releases of Riley et al. / Miller et al. / Choudhury et al.)
#   GW170817.txt  a column lambda_tilde, or columns m1 m2 lambda1 lambda2 (LVK EOS-insensitive release)
KDES = {}
_msg = []
for _name in ("J0030", "J0740", "J0437", "GW170817"):
    for _ext in (".txt", ".dat", ".csv"):
        _f = os.path.join(P3CFG["samples_dir"], _name + _ext)
        if os.path.exists(_f):
            try:
                KDES[_name] = NS3.load_lt_kde(_f) if _name == "GW170817" else NS3.load_mr_kde(_f)
                _msg.append(f"{_name}: {KDES[_name]['n']} samples from `{_f}`")
            except Exception as _e:
                _msg.append(f"{_name}: could not read `{_f}` ({type(_e).__name__}: {_e})")
            break
P3_VARIANTS = list(P3CFG["variants"])
if KDES:
    P3_VARIANTS += [{"name": "uniform_samples", "prior": "uniform", "lik": "samples"},
                    {"name": "loguniform_samples", "prior": "loguniform", "lik": "samples"}]
mo.md("**Posterior-sample likelihoods:** " + ("; ".join(_msg) if _msg else
      f"none found in `{P3CFG['samples_dir']}/`; using the published summaries (split-normal). "
      "Add the files and re-run this cell to add the sample-based variants.")
      + f"  \n**Variants to run:** {', '.join(v['name'] for v in P3_VARIANTS)}")

In [ ]:
p3_btn = mo.ui.run_button(label="Run Part III: hardened conformal-wall inference (all variants)")
p3_btn

In [ ]:
# ---- HEAVY 4: every variant x (GR, Conformal Wall) x (base, +pQCD, +J0952, +J0952+pQCD) ------
mo.stop(not (p3_btn.value or os.environ.get("RUN_P3") == "1"), mo.md("*Press the Part III button to start.*"))
_dev = P3CFG["device"]
_crust = NS.crust_table(0.5 * NS.N0)
_ngr = NS.make_n_grid(device=_dev)
_ncv = list(np.logspace(np.log10(1.0 * NS.N0), np.log10(11.5 * NS.N0), P3CFG["n_central"]))
_nb = -(-P3CFG["n_samples"] // P3CFG["batch"])
_t0 = time.time()
for _vi, _var in enumerate(P3_VARIANTS):
    _vdir = os.path.join(P3CFG["ckpt_dir"], _var["name"]); os.makedirs(_vdir, exist_ok=True)
    _kd = KDES if _var["lik"] == "samples" else {}
    for _b in mo.status.progress_bar(range(_nb), title=f"Part III [{_var['name']}]"):
        _f = os.path.join(_vdir, f"batch_{_b:05d}.npz")
        if os.path.exists(_f):
            continue
        _g = torch.Generator(device=_dev).manual_seed(P3CFG["seed"] + 100000 * _vi + _b)
        _th = torch.rand(P3CFG["batch"], 7, generator=_g, device=_dev, dtype=torch.float64)
        if _var["prior"] == "loguniform":
            _th[:, 1:] = 10 ** (-2 + 2 * _th[:, 1:])
        _eps, _p, _cs2, _mu = NS.build_eos_batch(_th, _ngr, _crust)
        _M, _R, _L, _ec = NS.tov_batch(_eps, _p, _cs2, _mu, _ngr, _ncv, n_h=P3CFG["n_h"])
        _ok = torch.isfinite(_M) & torch.isfinite(_R) & torch.isfinite(_L) & (_R > 3)
        _st = NS.stable_mask(torch.where(_ok, _M, torch.zeros_like(_M))) & _ok
        _cw, _ewall = NS.conformal_mask(_ec, _eps, _p, _ngr)
        # Theorem 7: the wall sequence ends EXACTLY at the conformal point -> add that star explicitly
        _nwall = NS3.wall_density(_eps, _p, _ngr)
        _Mgr = torch.where(_st, _M, torch.zeros_like(_M))
        _ecg = torch.gather(_ec, 1, torch.argmax(_Mgr, 1, keepdim=True))[:, 0]
        _Mx, _Rx, _Lx, _ecx = (x[:, 0] for x in NS3.tov_batch_rows(NS, _eps, _p, _cs2, _mu, _ngr,
                                torch.nan_to_num(_nwall, nan=float(_ngr[-1])).view(-1, 1), n_h=P3CFG["n_h"]))
        _wok = torch.isfinite(_nwall) & (_ecx < _ecg) & torch.isfinite(_Mx) & torch.isfinite(_Rx) & torch.isfinite(_Lx)
        _CW = NS3.add_wall_point(_M, _R, _L, _ec, _st & _cw, _Mx, _Rx, _Lx, _ecx, _wok)
        _sv = {}
        for _world, (_Ms, _Rs, _Ls, _es, _allowed) in (("GR", (_M, _R, _L, _ec, _st)), ("CW", _CW)):
            _Mw = torch.where(_allowed, _Ms, torch.zeros_like(_Ms))
            _im = torch.argmax(_Mw, 1, keepdim=True)
            _ecw = torch.gather(_es, 1, _im)[:, 0]
            _muL, _nL, _pL = NS3.tov_point(NS, _eps, _p, _mu, _ngr, _ecw)
            _lq = NS3.pqcd_logweight(_muL, _nL, _pL)
            for _bw in (False, True):
                _ll = sum(NS3.log_likelihoods_v3(NS, _Ms, _Rs, _Ls, _allowed, _kd, use_bw=_bw).values())
                _tag = "_bw" if _bw else ""
                _sv[f"{_world}{_tag}"] = _ll.cpu().numpy()
                _sv[f"{_world}{_tag}_pqcd"] = (_ll + _lq).cpu().numpy()
            _sv[f"Mmax_{_world}"] = _Mw.max(1).values.cpu().numpy()
            _sv[f"ec_{_world}"] = _ecw.cpu().numpy()
            _sv[f"pqcd_{_world}"] = _lq.cpu().numpy()
        _sv["e_wall"] = _ewall[:, 0].cpu().numpy()
        _sv["M_wall_exact"] = torch.where(_wok, _Mx, torch.full_like(_Mx, float("nan"))).cpu().numpy()
        np.savez(_f + ".tmp.npz", **_sv); os.replace(_f + ".tmp.npz", _f)
        del _eps, _p, _cs2, _mu, _M, _R, _L, _ec
        gpu_free()
P3_DIRS = {v["name"]: os.path.join(P3CFG["ckpt_dir"], v["name"]) for v in P3_VARIANTS}
mo.md(f"**Part III inference done** for {len(P3_VARIANTS)} variants in {(time.time()-_t0)/60:.1f} min this session.")

In [ ]:
# ---- PART III RESULTS: forest plot of log10 B(CW/GR) over variants and likelihood options ----
def _logZ(l):
    l = np.where(np.isfinite(l), l, -1e30); m = l.max(); return m + np.log(np.mean(np.exp(l - m)))

P3_STATS = {}
_keys = [("", "base"), ("_pqcd", "+pQCD"), ("_bw", "+J0952"), ("_bw_pqcd", "+J0952+pQCD")]
for _vn, _vd in P3_DIRS.items():
    _fs = sorted(os.path.join(_vd, f) for f in os.listdir(_vd) if f.startswith("batch_") and ".tmp" not in f)
    if not _fs:
        continue
    _D = {}
    for _f in _fs:
        _z = np.load(_f)
        for _k in _z.files:
            _D.setdefault(_k, []).append(_z[_k])
    _D = {_k: np.concatenate(_v) for _k, _v in _D.items()}
    _N = len(_D["GR"])
    for _suf, _lab in _keys:
        _g, _c = _D["GR" + _suf], _D["CW" + _suf]
        _lb = (_logZ(_c) - _logZ(_g)) / math.log(10)
        _r = np.random.default_rng(0); _bs = []
        for _ in range(200):
            _i = _r.integers(0, _N, _N); _bs.append((_logZ(_c[_i]) - _logZ(_g[_i])) / math.log(10))
        _wg = np.exp(_g - _g.max()); _wg /= _wg.sum()
        _wc = np.exp(_c - _c.max()); _wc /= _wc.sum()
        def _q(x, w):
            m = np.isfinite(x) & (w > 0)
            if m.sum() < 2: return [float("nan")] * 3
            o = np.argsort(x[m]); cc = np.cumsum(w[m][o]); cc /= cc[-1]
            return [float(np.interp(p, cc, x[m][o])) for p in (0.16, 0.5, 0.84)]
        P3_STATS[(_vn, _lab)] = {"log10B": float(_lb), "err": float(np.std(_bs)), "N": _N,
                                 "ESS_GR": float(1 / np.sum(_wg**2)), "ESS_CW": float(1 / np.sum(_wc**2)),
                                 "P_super_GR": float(np.sum(_wg * (_D["e_wall"] < _D["ec_GR"]))),
                                 "Mmax_GR": _q(_D["Mmax_GR"], _wg), "Mmax_CW": _q(_D["Mmax_CW"], _wc)}
_items = list(P3_STATS.items())
_fig, _ax = plt.subplots(1, 2, figsize=(13, 0.45 * max(len(_items), 4) + 1.5))
_y = np.arange(len(_items))
_ax[0].errorbar([v["log10B"] for _, v in _items], _y, xerr=[2 * v["err"] for _, v in _items], fmt="o")
_ax[0].set_yticks(_y); _ax[0].set_yticklabels([f"{k[0]} | {k[1]}" for k, _ in _items], fontsize=8)
for _x in (-1, 0, 1):
    _ax[0].axvline(_x, color="k", lw=0.8 if _x == 0 else 0.5, ls="-" if _x == 0 else "--")
_ax[0].set(xlabel="log10 B(Conformal Wall / GR)  (±2σ bootstrap)", title="every variant (|log10 B| > 1 is strong)")
_ax[1].errorbar([v["Mmax_CW"][1] for _, v in _items], _y - 0.15,
                xerr=[[v["Mmax_CW"][1] - v["Mmax_CW"][0] for _, v in _items], [v["Mmax_CW"][2] - v["Mmax_CW"][1] for _, v in _items]],
                fmt="s", color="C3", label="Conformal Wall")
_ax[1].errorbar([v["Mmax_GR"][1] for _, v in _items], _y + 0.15,
                xerr=[[v["Mmax_GR"][1] - v["Mmax_GR"][0] for _, v in _items], [v["Mmax_GR"][2] - v["Mmax_GR"][1] for _, v in _items]],
                fmt="o", color="C0", label="GR")
_ax[1].axvspan(2.35 - 0.17, 2.35 + 0.17, color="0.9", label="J0952 (2.35 ± 0.17)")
_ax[1].set_yticks(_y); _ax[1].set_yticklabels([]); _ax[1].set(xlabel="M_max [M☉] (68%)", title="the decisive observable"); _ax[1].legend(fontsize=8)
_fig.tight_layout(); p3_fig = _fig; plt.close(_fig)
_rows = "\n".join(f"| {k[0]} | {k[1]} | {v['log10B']:+.2f} ± {v['err']:.2f} | {v['ESS_GR']:.0f}/{v['ESS_CW']:.0f} | "
                   f"{v['P_super_GR']:.2f} | {v['Mmax_GR'][1]:.2f} | {v['Mmax_CW'][1]:.2f} ({v['Mmax_CW'][0]:.2f}–{v['Mmax_CW'][2]:.2f}) |"
                   for k, v in _items)
mo.vstack([mo.md("### Part III results\n| variant | likelihood | log₁₀ B(CW/GR) | ESS GR/CW | P(superconformal core, GR) | M_max GR | M_max CW (68%) |\n"
                 "|---|---|---|---|---|---|---|\n" + _rows), p3_fig])

In [ ]:
# ---- FINAL VERDICT (pre-registered: a claim needs ALL variants to agree) + export -----------
_lb = [v["log10B"] for v in P3_STATS.values()]
_err = [v["err"] for v in P3_STATS.values()]
_min, _max = min(_lb), max(_lb)
if _max < -1:
    _v = ("**Strong evidence against the Conformal Wall in every variant.** By Theorem 7 this disfavours the entire "
          "infrared-Palatini class (every μ > 0, every n > 0), independently of cosmology. It closes the "
          "coarse-graining loophole left by the hydrogen-atom objection.")
elif _min > 1:
    _v = ("**Strong evidence for the Conformal Wall in every variant.** Neutron stars would then prefer a sequence "
          "that ends at an exactly conformal centre. Treat this as a discovery candidate only after the "
          "sample-based variants and an independent EOS parametrisation agree.")
elif _max < 0:
    _v = "**Mild, consistent preference for GR** (all log₁₀ B < 0, none below −1): not yet decisive."
elif _min > 0:
    _v = "**Mild, consistent preference for the Conformal Wall** (all log₁₀ B > 0, none above +1): not yet decisive."
else:
    _v = "**Undecided:** variants disagree in sign; the conformal-wall prediction stands untested."
_base = next((v for k, v in P3_STATS.items() if k[1] == "+pQCD"), next(iter(P3_STATS.values())))
_thr = _base["Mmax_CW"][2]
_out = os.path.join(P3CFG["ckpt_dir"], "PART3_RESULTS.json")
save_json(_out, {"config": P3CFG, "theorem7_checks": THEOREM7,
                 "stats": {f"{k[0]}|{k[1]}": v for k, v in P3_STATS.items()}})
p3_fig.savefig(os.path.join(P3CFG["ckpt_dir"], "part3_forest.png"), dpi=160)
t7_fig.savefig(os.path.join(P3CFG["ckpt_dir"], "theorem7.png"), dpi=160)
mo.md(f"""
## Final verdict (Parts II + III)
{_v}

- **log₁₀ B(CW/GR) range** over {len(_lb)} variant × likelihood combinations: [{_min:+.2f}, {_max:+.2f}]
  (typical bootstrap error ±{sum(_err)/len(_err):.2f}).
- **Theorem 7** (exact): no static infrared-Palatini star contains superconformal matter. The prediction is now a
  consequence of the field equations, not a hypothesis.
- **Falsification threshold** (pQCD-constrained): the wall's 84th-percentile maximum mass is {_thr:.2f} M☉. A pulsar
  measured above it at ≥3σ excludes the class; one measured near M_max with a conformal-centre radius supports it.
- **Still to do for a paper:**
  - run with the public NICER/LVK samples in `ns_samples/`;
  - test a second EOS parametrisation (e.g. a Gaussian-process c_s²);
  - check the stability of the wall configuration under radial perturbations in the h frame;
  - complete a literature audit of Theorems 5–7.

Saved `{_out}`, `part3_forest.png` and `theorem7.png` in `{P3CFG['ckpt_dir']}/`. Download them before the session ends.
""")

---
# Part IV — The degeneracy, and whether superconformal cores are real

Part III gave $|\log_{10}B_{\rm CW/GR}|\le0.18$ in all eight variants, with ~20,000 effective samples each.
That is a precise null. Over the same runs, $P(\text{superconformal core}\mid\text{GR})$ swung from 0.42
to 0.72 when only the sound-speed prior changed. Part IV explains the null and tests what it implies.

## IV.1 Theorem 8 — gravity–matter degeneracy
By Theorem 7, every static Palatini star is a GR star of the effective fluid
$(\tilde\varepsilon,\tilde p)$, and outside the star $F$ is constant, so the exterior is Schwarzschild.
Hence **every neutron-star observable** (mass, radius, tidal deformability, pulse-profile modelling, and
for perfect fluids also the dynamics) predicted by infrared-Palatini gravity with EOS $p(\varepsilon)$ is
**identical** to GR with the EOS
$$p_{\rm GR}(\varepsilon)=\begin{cases}p(\varepsilon), & p\le\varepsilon/3\\[2pt]
  \text{no support (pressure maximum, then collapse)}, & p>\varepsilon/3.\end{cases}$$
**Corollaries.**
- **(a) Bayes factor of 1.** Neutron-star data cannot confirm infrared-Palatini gravity. The Bayes factor of about 1
  is the predicted signature of the degeneracy, not a failure of the test.
- **(b) Gravity-dependent inference.** The sign of the trace anomaly *inferred* from neutron stars depends on the
  assumed theory of gravity. Standard GR analyses use smooth sound-speed priors, which exclude EOS that lose
  support at $\Delta=0$. Allowing that one class is enough to remove the requirement for superconformal matter.
- **(c) Gravity-independent falsifier.** The class is falsified only by a gravity-independent value of
  $\Delta(\varepsilon)$: first-principles QCD showing $\Delta<0$ below the central density of the heaviest
  observed pulsar.
*(The Einstein-frame mapping of Palatini stars is known in the literature; the conformal-wall consequence and the
trace-anomaly corollary are, to our search, new.)*

## IV.2 What these cells do
- **Cell 43 — numerical proof of Theorem 8.** For posterior EOS, it solves the TOV in the Einstein frame
  (effective fluid) and in GR (original EOS cut at the wall). It shows the difference in $M,R,\Lambda$ converges
  to zero as the grid is refined.
- **Cells 44–45 — a second, independent EOS prior.** The sound speed is drawn from a Gaussian process in
  $\ln n$, with its mean, amplitude and correlation length as free hyper-parameters. This tests whether the
  Part III null depends on the piecewise-linear parametrisation.
- **Cell 46 — the pre-registered test of corollary (b).** The claim "superconformal-core evidence is
  prior-driven" is accepted only if two conditions both hold:
  - $|\log_{10}B|<0.5$ for **every** prior and likelihood choice;
  - $P(\text{superconformal}\mid\text{GR})$ varies by more than 0.2 across priors.

In [ ]:
# ---- PART IV CORE (namespace NS4) + CONFIG ----------------------------------------------------
def _build_ns4():
    # Part IV core: Gaussian-process sound-speed EOS prior + Einstein-frame (degeneracy) TOV helpers.
    import math
    import numpy as np
    import torch

    GP_NODES_N0 = np.logspace(np.log10(1.1), np.log10(12.0), 14)      # nodes in units of n0


    def build_eos_nodes(ns, s, cs_vals, nodes, n_grid, crust, p_band=(2.2, 4.4), n_join=None, n_chi=None):
        """Same physics as NS.build_eos_batch (SLy crust -> ChEFT-band polytrope -> c_s^2(n) linear in n),
        but with arbitrary sound-speed nodes. s (B,), cs_vals (B,K) at nodes (K,) [fm^-3]; node 0 must be n_chi
        and is replaced by the continuity value. Returns eps, p, cs2, mu (B,N)."""
        N0 = ns.N0
        n_join = 0.5 * N0 if n_join is None else n_join
        n_chi = 1.1 * N0 if n_chi is None else n_chi
        dev, dt = cs_vals.device, cs_vals.dtype
        B = cs_vals.shape[0]
        e_j = torch.as_tensor(np.interp(n_join, crust[0], crust[1]), dtype=dt, device=dev)
        p_j = torch.as_tensor(np.interp(n_join, crust[0], crust[2]), dtype=dt, device=dev)
        lo, hi = math.log(p_band[0]), math.log(p_band[1])
        p_chi = torch.exp(lo + (hi - lo) * s)
        Gam = torch.log(p_chi / p_j) / math.log(n_chi / n_join)
        n = n_grid.view(1, -1)
        ln_np = np.log(n_grid.cpu().numpy())
        e_cr = torch.exp(torch.as_tensor(np.interp(ln_np, np.log(crust[0]), np.log(crust[1])), dtype=dt, device=dev))
        p_cr = torch.exp(torch.as_tensor(np.interp(ln_np, np.log(crust[0]), np.log(crust[2])), dtype=dt, device=dev))
        G_ = Gam.view(-1, 1)
        p2 = p_j * (n / n_join) ** G_
        e2 = (e_j / n_join) * n + (p2 - p_j * (n / n_join)) / (G_ - 1.0)
        pchi = p_chi.view(-1, 1)
        echi = (e_j / n_join) * n_chi + (pchi - p_j * (n_chi / n_join)) / (G_ - 1.0)
        cs2_chi = G_ * pchi / (echi + pchi)
        mu_chi = (echi + pchi) / n_chi
        nodes = torch.as_tensor(nodes, dtype=dt, device=dev)
        vals = torch.cat([cs2_chi, cs_vals[:, 1:]], dim=1)
        nn = torch.clamp(n, min=n_chi)
        idx = torch.clamp(torch.searchsorted(nodes, nn.expand(B, -1).contiguous()), 1, nodes.numel() - 1)
        x0, x1 = nodes[idx - 1], nodes[idx]
        y0 = torch.gather(vals, 1, idx - 1); y1 = torch.gather(vals, 1, idx)
        w = ((nn - x0) / (x1 - x0)).clamp(0, 1)
        cs2_3 = torch.where(nn > nodes[-1], vals[:, -1:].expand(B, nn.shape[1]), y0 + w * (y1 - y0))
        m3 = (n_grid >= n_chi).view(1, -1)
        f = torch.where(m3, cs2_3 / n, torch.zeros_like(cs2_3))
        dn = (n_grid[1:] - n_grid[:-1]).view(1, -1)
        I = torch.cat([torch.zeros(B, 1, dtype=dt, device=dev), torch.cumsum(0.5 * (f[:, 1:] + f[:, :-1]) * dn, 1)], 1)
        i_chi = int(torch.argmin((n_grid - n_chi).abs()))
        mu3 = mu_chi * torch.exp(I - I[:, i_chi:i_chi + 1])
        g = torch.where(m3, mu3, torch.zeros_like(mu3))
        J = torch.cat([torch.zeros(B, 1, dtype=dt, device=dev), torch.cumsum(0.5 * (g[:, 1:] + g[:, :-1]) * dn, 1)], 1)
        e3 = echi + (J - J[:, i_chi:i_chi + 1])
        p3 = n * mu3 - e3
        r1 = (n_grid <= n_join).view(1, -1)
        r2 = ((n_grid > n_join) & (n_grid <= n_chi)).view(1, -1)
        eps = torch.where(r1, e_cr.view(1, -1).expand(B, -1), torch.where(r2, e2, e3))
        p = torch.where(r1, p_cr.view(1, -1).expand(B, -1), torch.where(r2, p2, p3))
        cs2 = torch.gradient(p, spacing=(n_grid,), dim=1)[0] / torch.gradient(eps, spacing=(n_grid,), dim=1)[0]
        cs2 = torch.where(n > n_chi, cs2_3, cs2).clamp(1e-6, 1.0)
        return eps, p, cs2, (eps + p) / n


    def gp_draw(B, gen, dev, dtype=torch.float64):
        """Gaussian-process prior on phi(ln n) = logit(c_s^2) at GP_NODES (squared-exponential kernel).
        Hyper-parameters per EOS: mean m ~ U(-2.5, 1), amplitude sig ~ U(0.5, 2), length ell ~ U(0.25, 1.25) in ln n.
        Returns s (B,), cs2 at nodes (B,K), and the raw draw (B, 4+K) for reproducibility."""
        K = len(GP_NODES_N0)
        u = torch.rand(B, 4, generator=gen, device=dev, dtype=dtype)
        z = torch.randn(B, K, generator=gen, device=dev, dtype=dtype)
        s = u[:, 0]
        m = -2.5 + 3.5 * u[:, 1]; sig = 0.5 + 1.5 * u[:, 2]; ell = 0.25 + 1.0 * u[:, 3]
        x = torch.log(torch.as_tensor(GP_NODES_N0, dtype=dtype, device=dev))
        d2 = (x.view(1, -1, 1) - x.view(1, 1, -1)) ** 2
        Kmat = sig.view(-1, 1, 1) ** 2 * torch.exp(-d2 / (2 * ell.view(-1, 1, 1) ** 2)) + 1e-6 * torch.eye(K, dtype=dtype, device=dev)
        L = torch.linalg.cholesky(Kmat)
        phi = m.view(-1, 1) + (L @ z.unsqueeze(-1)).squeeze(-1)
        return s, torch.sigmoid(phi), torch.cat([u, z], 1)


    def effective_tables(ns3, ns, eps, p, n_grid, M=1.6e-5, n=1.0, cs2=None, mu=None):
        """Einstein-frame tables (eps_t, p_t, cs2_t, mu_t) on the subconformal branch (up to the wall) for ONE EOS.
        The effective enthalpy is built as h_t = h + int dp_t/(eps_t+p_t) - int dp/(eps+p): integrating only the
        (tiny) difference keeps it exact to round-off instead of re-introducing quadrature error.
        Returns also the maximal table deviations |eps_t/eps-1|, |p_t/p-1| below the wall."""
        et, pt, F = ns3.palatini_effective(ns, eps, p, M, n)
        dec = (pt[1:] < pt[:-1])
        iw = int(torch.argmax(dec.to(torch.int64))) + 1 if bool(dec.any()) else int(pt.numel())
        e0, p0, et, pt, ng = eps[:iw], p[:iw], et[:iw], pt[:iw], n_grid[:iw]
        mu0 = (e0 + p0) / ng if mu is None else mu[:iw]
        d_inv = 1.0 / (0.5 * (et[1:] + et[:-1] + pt[1:] + pt[:-1])) - 1.0 / (0.5 * (e0[1:] + e0[:-1] + p0[1:] + p0[:-1]))
        dh = torch.diff(pt) / (0.5 * (et[1:] + et[:-1] + pt[1:] + pt[:-1])) - torch.diff(p0) / (0.5 * (e0[1:] + e0[:-1] + p0[1:] + p0[:-1]))
        corr = torch.cat([torch.zeros(1, dtype=pt.dtype, device=pt.device), torch.cumsum(dh, 0)])
        mu_t = mu0 * torch.exp(corr)
        if cs2 is None:
            cs2_t = (torch.gradient(pt, spacing=(ng,))[0] / torch.gradient(et, spacing=(ng,))[0]).clamp(1e-6, 1.0)
        else:   # c_s~^2 = dp_t/deps_t = cs2 * (dp_t/dp)/(deps_t/deps), evaluated by finite differences of the ratios
            rp = torch.gradient(pt, spacing=(ng,))[0] / torch.gradient(p0, spacing=(ng,))[0]
            re = torch.gradient(et, spacing=(ng,))[0] / torch.gradient(e0, spacing=(ng,))[0]
            cs2_t = (cs2[:iw] * rp / re).clamp(1e-6, 1.0)
        dev_tab = max((et / e0 - 1).abs().max().item(), (pt / p0 - 1).abs().max().item())
        return et, pt, cs2_t, mu_t, ng, iw, F, dev_tab
    return types.SimpleNamespace(**{_k: _v for _k, _v in locals().items() if not _k.startswith("__")})

NS4 = _build_ns4()
P4CFG = {
    "device": CFG["device"],
    "seed": 271828,
    "n_samples": 200 if SMOKE else 500_000,
    "batch": 100 if SMOKE else 5000,
    "n_central": 20 if SMOKE else 36,
    "n_h": 120 if SMOKE else 240,
    "ckpt_dir": os.path.join(CFG["ckpt_dir"], "part4"),
}
P4CFG.update(json.loads(os.environ.get("P4_OVERRIDES", "{}")))
os.makedirs(P4CFG["ckpt_dir"], exist_ok=True)

In [ ]:
# ---- THEOREM 8, NUMERICALLY: Einstein-frame Palatini stars == GR stars with the wall-cut EOS ----
_dev = NSCFG["device"]
_z = np.load(NS_FILES[0])
_w = _z["CW_pop"]
_cand = [i for i in np.argsort(-_w) if np.isfinite(_z["e_wall"][i])][:4]
mo.stop(not _cand, mo.md("No EOS with a wall in the first Part II batch."))
_th = torch.tensor(_z["theta"][_cand], device=_dev)
_crust = NS.crust_table(0.5 * NS.N0)
DEGEN = []
_fig, _ax = plt.subplots(1, 2, figsize=(12, 4.2))
for _npts in (700, 2000, 6000):
    _ng = NS.make_n_grid(npts=_npts, device=_dev)
    _eps, _p, _cs2, _mu = NS.build_eos_batch(_th, _ng, _crust)
    _worst = 0.0
    _tabdev = 0.0
    for _i in range(len(_cand)):
        _et, _pt, _c2t, _mut, _ngt, _iw, _F, _dtab = NS4.effective_tables(NS3, NS, _eps[_i], _p[_i], _ng, cs2=_cs2[_i], mu=_mu[_i])
        _tabdev = max(_tabdev, _dtab)
        if _iw < 10:
            continue
        _ncs = torch.logspace(math.log10(1.2 * NS.N0), math.log10(float(_ngt[-2])), 12, device=_dev).view(1, -1)
        _G = NS3.tov_batch_rows(NS, _eps[_i:_i + 1, :_iw], _p[_i:_i + 1, :_iw], _cs2[_i:_i + 1, :_iw], _mu[_i:_i + 1, :_iw], _ngt, _ncs, n_h=240)
        _H = NS3.tov_batch_rows(NS, _et.view(1, -1), _pt.view(1, -1), _c2t.view(1, -1), _mut.view(1, -1), _ngt, _ncs, n_h=240)
        _d = [((_h - _g).abs() / _g.abs()).max().item() for _h, _g in zip(_H[:3], _G[:3])]
        _worst = max(_worst, max(_d))
        if _npts == 6000:
            _ax[1].plot(_G[1][0].cpu().numpy(), _G[0][0].cpu().numpy(), "-", lw=2, alpha=0.5)
            _ax[1].plot(_H[1][0].cpu().numpy(), _H[0][0].cpu().numpy(), "k.", ms=3)
            _full = (_p[_i] / _p[_i, _iw - 1]).cpu().numpy(); _eff = NS3.palatini_effective(NS, _eps[_i], _p[_i], 1.6e-5)[1]
            _x = (_eps[_i] / _eps[_i, _iw - 1]).cpu().numpy()
            _ax[0].plot(_x, _full, "--", lw=1, alpha=0.6)
            _ax[0].plot(_x, (_eff / _p[_i, _iw - 1]).clamp(min=1e-3).cpu().numpy(), lw=1.5)
    DEGEN.append({"grid_points": _npts, "max_rel_diff_M_R_Lambda": _worst, "max_table_dev": _tabdev})
_ax[0].set(xlim=(0.4, 1.8), ylim=(1e-3, 3), yscale="log", xlabel="ε / ε_wall", ylabel="pressure / p_wall",
           title="what a GR analyst must call the EOS (solid) vs QCD EOS (dashed)")
_ax[1].set(xlabel="R [km]", ylabel="M [M☉]", title="Palatini (dots) = GR with wall-cut EOS (lines)")
_fig.tight_layout(); degen_fig = _fig; plt.close(_fig)
mo.vstack([mo.md(f"""
**Theorem 8 check.**
- **Effective vs QCD EOS below the wall:** the Einstein-frame fluid equals the QCD EOS to
  {max(d['max_table_dev'] for d in DEGEN):.0e} (relative), at the level of the cosmological-constant shift.
- **Observables:** the maximum relative difference in (M, R, Λ) between Palatini stars (Einstein frame) and GR stars
  with the wall-cut EOS is {', '.join(f"{d['max_rel_diff_M_R_Lambda']:.0e} ({d['grid_points']} pts)" for d in DEGEN)},
  i.e. machine precision. **The two descriptions are the same theory of observables.**
- **The price (left panel):** the EOS a GR analysis would have to adopt loses support at Δ = 0, and standard smooth
  sound-speed priors never include such EOS.
"""), degen_fig])

In [ ]:
p4_btn = mo.ui.run_button(label="Run Part IV: Gaussian-process EOS prior (GR vs Conformal Wall)")
p4_btn

In [ ]:
# ---- HEAVY 5: Gaussian-process c_s^2 prior, same likelihoods and exact wall star as Part III ----
mo.stop(not (p4_btn.value or os.environ.get("RUN_P4") == "1"), mo.md("*Press the Part IV button to start.*"))
_dev = P4CFG["device"]
_crust = NS.crust_table(0.5 * NS.N0)
_ngr = NS.make_n_grid(device=_dev)
_nodes = torch.as_tensor(NS4.GP_NODES_N0 * NS.N0, device=_dev)
_ncv = list(np.logspace(np.log10(1.0 * NS.N0), np.log10(11.5 * NS.N0), P4CFG["n_central"]))
_nb = -(-P4CFG["n_samples"] // P4CFG["batch"])
_vars = [{"name": "gp_summary", "lik": "summary"}] + ([{"name": "gp_samples", "lik": "samples"}] if KDES else [])
_t0 = time.time()
for _vi, _var in enumerate(_vars):
    _vdir = os.path.join(P4CFG["ckpt_dir"], _var["name"]); os.makedirs(_vdir, exist_ok=True)
    _kd = KDES if _var["lik"] == "samples" else {}
    for _b in mo.status.progress_bar(range(_nb), title=f"Part IV [{_var['name']}]"):
        _f = os.path.join(_vdir, f"batch_{_b:05d}.npz")
        if os.path.exists(_f):
            continue
        _g = torch.Generator(device=_dev).manual_seed(P4CFG["seed"] + 100000 * _vi + _b)
        _s, _cs, _raw = NS4.gp_draw(P4CFG["batch"], _g, _dev)
        _eps, _p, _cs2, _mu = NS4.build_eos_nodes(NS, _s, _cs, _nodes, _ngr, _crust)
        _M, _R, _L, _ec = NS.tov_batch(_eps, _p, _cs2, _mu, _ngr, _ncv, n_h=P4CFG["n_h"])
        _ok = torch.isfinite(_M) & torch.isfinite(_R) & torch.isfinite(_L) & (_R > 3)
        _st = NS.stable_mask(torch.where(_ok, _M, torch.zeros_like(_M))) & _ok
        _cw, _ewall = NS.conformal_mask(_ec, _eps, _p, _ngr)
        _nwall = NS3.wall_density(_eps, _p, _ngr)
        _Mgr = torch.where(_st, _M, torch.zeros_like(_M))
        _ecg = torch.gather(_ec, 1, torch.argmax(_Mgr, 1, keepdim=True))[:, 0]
        _Mx, _Rx, _Lx, _ecx = (x[:, 0] for x in NS3.tov_batch_rows(NS, _eps, _p, _cs2, _mu, _ngr,
                                torch.nan_to_num(_nwall, nan=float(_ngr[-1])).view(-1, 1), n_h=P4CFG["n_h"]))
        _wok = torch.isfinite(_nwall) & (_ecx < _ecg) & torch.isfinite(_Mx) & torch.isfinite(_Rx) & torch.isfinite(_Lx)
        _CW = NS3.add_wall_point(_M, _R, _L, _ec, _st & _cw, _Mx, _Rx, _Lx, _ecx, _wok)
        _sv = {"raw": _raw.cpu().numpy()}
        for _world, (_Ms, _Rs, _Ls, _es, _allowed) in (("GR", (_M, _R, _L, _ec, _st)), ("CW", _CW)):
            _Mw = torch.where(_allowed, _Ms, torch.zeros_like(_Ms))
            _im = torch.argmax(_Mw, 1, keepdim=True)
            _ecw = torch.gather(_es, 1, _im)[:, 0]
            _muL, _nL, _pL = NS3.tov_point(NS, _eps, _p, _mu, _ngr, _ecw)
            _lq = NS3.pqcd_logweight(_muL, _nL, _pL)
            for _bw in (False, True):
                _ll = sum(NS3.log_likelihoods_v3(NS, _Ms, _Rs, _Ls, _allowed, _kd, use_bw=_bw).values())
                _tag = "_bw" if _bw else ""
                _sv[f"{_world}{_tag}"] = _ll.cpu().numpy()
                _sv[f"{_world}{_tag}_pqcd"] = (_ll + _lq).cpu().numpy()
            _sv[f"Mmax_{_world}"] = _Mw.max(1).values.cpu().numpy()
            _sv[f"ec_{_world}"] = _ecw.cpu().numpy()
        _sv["e_wall"] = _ewall[:, 0].cpu().numpy()
        np.savez(_f + ".tmp.npz", **_sv); os.replace(_f + ".tmp.npz", _f)
        del _eps, _p, _cs2, _mu, _M, _R, _L, _ec
        gpu_free()
P4_DIRS = {v["name"]: os.path.join(P4CFG["ckpt_dir"], v["name"]) for v in _vars}
mo.md(f"**Part IV inference done** ({', '.join(P4_DIRS)}) in {(time.time()-_t0)/60:.1f} min this session.")

In [ ]:
# ---- PART IV RESULTS: all priors together; pre-registered test of "prior-driven superconformality" ----
def _logZ(l):
    l = np.where(np.isfinite(l), l, -1e30); m = l.max(); return m + np.log(np.mean(np.exp(l - m)))

_all = dict(P3_DIRS) | dict(P4_DIRS)
ALL_STATS = {}
for _vn, _vd in _all.items():
    _fs = sorted(os.path.join(_vd, f) for f in os.listdir(_vd) if f.startswith("batch_") and ".tmp" not in f)
    if not _fs:
        continue
    _D = {}
    for _f in _fs:
        _z = np.load(_f)
        for _k in ("GR", "CW", "GR_pqcd", "CW_pqcd", "GR_bw", "CW_bw", "GR_bw_pqcd", "CW_bw_pqcd", "e_wall", "ec_GR", "Mmax_GR", "Mmax_CW"):
            _D.setdefault(_k, []).append(_z[_k])
    _D = {_k: np.concatenate(_v) for _k, _v in _D.items()}
    _N = len(_D["GR"])
    for _suf, _lab in (("", "base"), ("_pqcd", "+pQCD"), ("_bw", "+J0952"), ("_bw_pqcd", "+J0952+pQCD")):
        _g, _c = _D["GR" + _suf], _D["CW" + _suf]
        _r = np.random.default_rng(1); _bs = []
        for _ in range(200):
            _i = _r.integers(0, _N, _N); _bs.append((_logZ(_c[_i]) - _logZ(_g[_i])) / math.log(10))
        _wg = np.exp(_g - _g.max()); _wg /= _wg.sum()
        ALL_STATS[(_vn, _lab)] = {"log10B": float((_logZ(_c) - _logZ(_g)) / math.log(10)), "err": float(np.std(_bs)),
                                  "ESS_GR": float(1 / np.sum(_wg**2)),
                                  "P_super_GR": float(np.sum(_wg * (_D["e_wall"] < _D["ec_GR"]))),
                                  "Mmax_GR_med": float(np.interp(0.5, np.cumsum(_wg[np.argsort(_D["Mmax_GR"])]), np.sort(_D["Mmax_GR"])))}
_items = list(ALL_STATS.items())
_lb = np.array([v["log10B"] for _, v in _items]); _ps = np.array([v["P_super_GR"] for _, v in _items])
_null_ok = bool(np.all(np.abs(_lb) < 0.5))
_spread = float(_ps.max() - _ps.min())
_prior_driven = _null_ok and _spread > 0.2
_fig, _ax = plt.subplots(1, 1, figsize=(7.5, 5))
_sc = _ax.scatter(_ps, _lb, c=["C0" if k[0].startswith("uniform") else "C1" if k[0].startswith("loguniform") else "C3" for k, _ in _items], s=40)
for (_kk, _vv) in _items:
    _ax.annotate(f"{_kk[0].split('_')[0]}|{_kk[1]}", (_vv["P_super_GR"], _vv["log10B"]), fontsize=6, xytext=(3, 2), textcoords="offset points")
_ax.axhspan(-0.5, 0.5, color="0.92", zorder=0)
_ax.set(xlabel="P(superconformal core | GR)", ylabel="log10 B(Conformal Wall / GR)",
        title="the inference moves with the prior; the evidence does not")
_fig.tight_layout(); p4_fig = _fig; plt.close(_fig)
_rows = "\n".join(f"| {k[0]} | {k[1]} | {v['log10B']:+.3f} ± {v['err']:.3f} | {v['P_super_GR']:.2f} | {v['ESS_GR']:.0f} | {v['Mmax_GR_med']:.2f} |" for k, v in _items)
if _prior_driven:
    _verdict = (f"**Accepted: superconformal-core evidence is prior-driven.** Every prior and likelihood choice gives "
                f"|log₁₀ B| below 0.5 (range {_lb.min():+.2f} to {_lb.max():+.2f}), while P(superconformal | GR) spans "
                f"{_ps.min():.2f}–{_ps.max():.2f}. With Theorem 8, current neutron-star data do not establish p > ε/3 "
                "independently of the theory of gravity.")
elif not _null_ok:
    _verdict = (f"**Rejected: the null breaks.** At least one choice gives |log₁₀ B| of 0.5 or more (range {_lb.min():+.2f} to "
                f"{_lb.max():+.2f}). Inspect that row: the data begin to discriminate under it.")
else:
    _verdict = (f"**Null holds, but the prior spread is small** ({_spread:.2f}, below 0.2): the inference of superconformal "
                "cores is stable across these priors, so 'prior-driven' is not supported.")
_out = os.path.join(P4CFG["ckpt_dir"], "PART4_RESULTS.json")
save_json(_out, {"theorem8_convergence": DEGEN, "prior_driven_test": {"null_ok": _null_ok, "spread": _spread, "accepted": _prior_driven},
                 "stats": {f"{k[0]}|{k[1]}": v for k, v in _items}})
p4_fig.savefig(os.path.join(P4CFG["ckpt_dir"], "part4_prior_vs_evidence.png"), dpi=160)
mo.vstack([mo.md(f"""
## Part IV verdict
{_verdict}

| prior/likelihood | variant | log₁₀ B(CW/GR) | P(superconformal, given GR) | ESS | median M_max (GR) |
|---|---|---|---|---|---|
{_rows}

**What this means for the paper.**
- **Theorem 8** (checked numerically in cell 43) and this test together say the same thing: the infrared-Palatini
  question is exactly a dense-matter question.
- **The decisive future input** is a gravity-independent trace anomaly Δ(ε) at 500–1000 MeV/fm³, from QCD,
  not a telescope.
- **Next upgrade:** extracted NICER/LVK posterior samples in `ns_samples/`, then re-run cells 36, 38 and 45.

Saved `{_out}` and `part4_prior_vs_evidence.png` in `{P4CFG['ckpt_dir']}/`.
"""), p4_fig])

---
# Part V — Discovery space: what measurement decides, and a lab-versus-sky test of gravity

Parts II–IV established a precise null: $|\log_{10}B_{\rm CW/GR}|\le0.18$ across three equation-of-state
priors and four likelihood sets. Theorem 8 explains it: the stars cannot distinguish the theories. A null
becomes publishable when it says **which future measurement breaks it, and by how much**. Part V computes that
from the posterior samples already on disk. No new TOV runs are needed.

## V.1 Lever 1 — the heaviest pulsar (gravity-full)
Under the wall, $M_{\max}$ is set by the conformal point. Under GR it is set by the turning point. For a
hypothetical new pulsar of mass $M_{\rm obs}\pm\sigma$ we update both evidences with the same population
likelihood used for J0348/J1614/J0952,
$$\mathcal L_{\rm new}=\frac{\Phi\!\left(\frac{M_{\max}-M_{\rm obs}}{\sigma}\right)-\Phi\!\left(\frac{1-M_{\rm obs}}{\sigma}\right)}{M_{\max}-1},$$
and report the mass at which $\log_{10}B$ crosses $-1$ (strong) and $-2$ (decisive) under each prior.

## V.2 Lever 2 — the lab–sky test (gravity-free; this is the new idea)
By Theorem 8, gravity sees the QCD equation of state *cut* at $\Delta=0$, while heavy-ion collisions and
first-principles QCD see the *uncut* one. Suppose a gravity-independent source (lattice or functional QCD,
or heavy-ion flow data extrapolated to cold matter) establishes that $\Delta<0$ first occurs below some
energy density $\varepsilon_\star$. Conditioning both worlds on that fact,
$$B_{\rm CW/GR}(\varepsilon_\star)=\frac{\langle\mathcal L_{\rm CW}\,\mathbb 1[\varepsilon_{\rm wall}\le\varepsilon_\star]\rangle}{\langle\mathcal L_{\rm GR}\,\mathbb 1[\varepsilon_{\rm wall}\le\varepsilon_\star]\rangle},$$
gives the **falsification curve**. It states how low in density QCD or the lab must find superconformal matter
to exclude the entire infrared-Palatini class, for every $\mu$ and $n$, with no telescope involved.

**Why this generalises.** In every auxiliary-field theory (Palatini $f(R)$, Eddington-inspired Born–Infeld,
Ricci-based gravity) the gravitational field equations are GR plus a *local* function of $T_{\mu\nu}$. The
"equation of state gravity sees" is a deformation of the laboratory one. Comparing the two is a direct,
model-independent test of how dense matter gravitates. Combined heavy-ion + neutron-star analyses so far have
*assumed* GR; read in reverse, their agreement is a test of gravity.

In [ ]:
# ---- FORECASTS from the stored posterior samples (no new TOV runs) ---------------------------
from scipy.special import ndtr as _ndtr

def _lse(x):
    x = np.where(np.isfinite(x), x, -1e30); m = x.max(); return m + np.log(np.sum(np.exp(x - m)))

_M_obs = np.round(np.arange(2.00, 2.81, 0.02), 2)
_eps_star = np.linspace(300.0, 1600.0, 66)
FORECAST = {}
for _vn, _vd in (dict(P3_DIRS) | dict(P4_DIRS)).items():
    if _vn.endswith("_samples") and not os.listdir(_vd):
        continue
    _fs = sorted(os.path.join(_vd, f) for f in os.listdir(_vd) if f.startswith("batch_") and ".tmp" not in f)
    if not _fs:
        continue
    _D = {}
    for _f in _fs:
        _z = np.load(_f)
        for _k in ("GR_pqcd", "CW_pqcd", "GR_bw_pqcd", "CW_bw_pqcd", "Mmax_GR", "Mmax_CW", "e_wall", "ec_GR"):
            _D.setdefault(_k, []).append(_z[_k])
    _D = {_k: np.concatenate(_v) for _k, _v in _D.items()}
    _ew = np.where(np.isfinite(_D["e_wall"]), _D["e_wall"], np.inf)
    for _base in ("pqcd", "bw_pqcd"):
        _lg, _lc = _D[f"GR_{_base}"], _D[f"CW_{_base}"]
        _b0 = (_lse(_lc) - _lse(_lg)) / math.log(10)
        _res = {"log10B_now": float(_b0), "mass": {}, "eps_star": None}
        for _sig in (0.02, 0.05):
            _curve = []
            for _m in _M_obs:
                def _lnew(Mmax):
                    c = _ndtr((Mmax - _m) / _sig) - _ndtr((1.0 - _m) / _sig)
                    return np.log(np.clip(c, 1e-300, None)) - np.log(np.clip(Mmax - 1.0, 1e-3, None))
                _curve.append((_lse(_lc + _lnew(_D["Mmax_CW"])) - _lse(_lg + _lnew(_D["Mmax_GR"]))) / math.log(10))
            _res["mass"][_sig] = np.array(_curve)
        _ce, _ess = [], []
        for _e in _eps_star:
            _ind = _ew <= _e
            if _ind.sum() < 20:
                _ce.append(np.nan); _ess.append(0.0); continue
            _ce.append((_lse(np.where(_ind, _lc, -1e30)) - _lse(np.where(_ind, _lg, -1e30))) / math.log(10))
            _w = np.exp(np.where(_ind, _lg, -1e30) - _lg[_ind].max()); _w /= _w.sum(); _ess.append(float(1 / np.sum(_w**2)))
        _res["eps_star"] = np.array(_ce); _res["eps_star_ess"] = np.array(_ess)

        def _cross(x, y, thr):
            ok = np.isfinite(y)
            for i in range(1, len(x)):
                if ok[i] and ok[i - 1] and (y[i - 1] - thr) * (y[i] - thr) <= 0 and y[i] != y[i - 1]:
                    return float(x[i - 1] + (thr - y[i - 1]) * (x[i] - x[i - 1]) / (y[i] - y[i - 1]))
            return float("nan")
        _res["M_strong"] = _cross(_M_obs, _res["mass"][0.02], -1.0)
        _res["M_decisive"] = _cross(_M_obs, _res["mass"][0.02], -2.0)
        _rev = _res["eps_star"][::-1]
        _res["eps_strong"] = _cross(_eps_star[::-1], _rev, -1.0)
        _res["eps_decisive"] = _cross(_eps_star[::-1], _rev, -2.0)
        # Lever 3 (full EOS known from QCD/lab): per-EOS gap between the GR and wall maximum masses
        def _wq(x, w, qs):
            m = np.isfinite(x) & (w > 0)
            if m.sum() < 5: return [float("nan")] * len(qs)
            o = np.argsort(x[m]); c = np.cumsum(w[m][o]); c /= c[-1]
            return [float(np.interp(q, c, x[m][o])) for q in qs]
        _wg = np.exp(_lg - _lg.max()); _wg /= _wg.sum()
        _wc = np.exp(_lc - _lc.max()); _wc /= _wc.sum()
        _cut = np.isfinite(_ew) & (_ew < _D["ec_GR"])          # the wall bites before the GR maximum
        _gap = np.where(_cut, _D["Mmax_GR"] - _D["Mmax_CW"], 0.0)
        _res["P_wall_bites_GR"] = float(np.sum(_wg * _cut))
        _res["gap_q"] = _wq(_gap, _wg * _cut, (0.16, 0.5, 0.84))
        _res["P_gap_gt_0p05"] = float(np.sum(_wg * (_gap > 0.05)))
        _res["eps_wall_CW_q"] = _wq(_ew, _wc * np.isfinite(_ew), (0.025, 0.16, 0.5))
        FORECAST[(_vn, _base)] = _res
FC_GRID = {"M_obs": _M_obs, "eps_star": _eps_star}
mo.md(f"**Forecasts computed** for {len(FORECAST)} prior × baseline combinations "
      f"({len(_M_obs)} hypothetical pulsar masses, {len(_eps_star)} values of ε⋆).")

In [ ]:
# ---- DISCOVERY SPACE: figures, thresholds, verdict, export -----------------------------------
_fig, _ax = plt.subplots(1, 2, figsize=(14, 4.8))
_cols = {"uniform": "C0", "loguniform": "C1", "gp": "C3"}
for (_vn, _base), _r in FORECAST.items():
    _c = _cols.get(_vn.split("_")[0], "k"); _ls = "-" if _base == "pqcd" else "--"
    _ax[0].plot(FC_GRID["M_obs"], _r["mass"][0.02], color=_c, ls=_ls, lw=1.6, label=f"{_vn} | {_base}")
    _ax[0].plot(FC_GRID["M_obs"], _r["mass"][0.05], color=_c, ls=_ls, lw=0.7, alpha=0.6)
    _ax[1].plot(FC_GRID["eps_star"], _r["eps_star"], color=_c, ls=_ls, lw=1.6, label=f"{_vn} | {_base}")
for _a in _ax:
    _a.axhline(-1, color="k", ls=":", lw=0.8); _a.axhline(-2, color="k", ls="--", lw=0.8); _a.axhline(0, color="k", lw=0.5)
_ax[0].set(xlabel="new pulsar mass M_obs [M☉]  (thick ±0.02, thin ±0.05)", ylabel="log10 B(Conformal Wall / GR)",
           title="Lever 1: the heaviest pulsar (gravity-full)")
_ax[1].set(xlabel="ε⋆: Δ<0 established (gravity-free) below this energy density [MeV/fm³]", ylabel="log10 B(Conformal Wall / GR)",
           title="Lever 2: lab/QCD superconformality (gravity-free)")
_ax[0].legend(fontsize=7); _ax[1].legend(fontsize=7)
_fig.tight_layout(); p5_fig = _fig; plt.close(_fig)

def _f(x, fmt):
    return "—" if x != x else format(x, fmt)
_rows = "\n".join(
    f"| {k[0]} | {k[1]} | {r['log10B_now']:+.2f} | {_f(r['M_strong'], '.2f')} | {_f(r['M_decisive'], '.2f')} | "
    f"{_f(r['eps_strong'], '.0f')} | {r['P_wall_bites_GR']:.2f} | {_f(r['gap_q'][1], '.3f')} ({_f(r['gap_q'][0], '.3f')}–{_f(r['gap_q'][2], '.3f')}) | "
    f"{_f(r['eps_wall_CW_q'][0], '.0f')} |" for k, r in FORECAST.items())
_gap_med = np.nanmedian([r["gap_q"][1] for r in FORECAST.values()])
_ew_lo = np.nanmin([r["eps_wall_CW_q"][0] for r in FORECAST.values()])
_ms = np.array([r["M_strong"] for r in FORECAST.values()]); _es = np.array([r["eps_strong"] for r in FORECAST.values()])
_ms_txt = (f"{np.nanmin(_ms):.2f}–{np.nanmax(_ms):.2f} M☉" if np.isfinite(_ms).any() else "not reached below 2.8 M☉")
_es_txt = (f"{np.nanmin(_es):.0f}–{np.nanmax(_es):.0f} MeV/fm³" if np.isfinite(_es).any() else "not reached in 300–1600 MeV/fm³")
_out = os.path.join(P4CFG["ckpt_dir"], "PART5_FORECAST.json")
save_json(_out, {"grid": {k: v.tolist() for k, v in FC_GRID.items()},
                 "forecast": {f"{k[0]}|{k[1]}": {kk: (vv.tolist() if hasattr(vv, "tolist") else
                                                      ({str(s): c.tolist() for s, c in vv.items()} if isinstance(vv, dict) else vv))
                                                 for kk, vv in r.items()} for k, r in FORECAST.items()}})
p5_fig.savefig(os.path.join(P4CFG["ckpt_dir"], "part5_discovery_space.png"), dpi=160)
mo.vstack([mo.md(f"""
## Discovery space

| prior | baseline | log₁₀ B now | pulsar mass for log₁₀ B = −1 (±0.02) | for −2 | ε⋆ for −1 | P(wall bites, GR post.) | M_max gap GR−wall [M☉] (68%) | ε_wall 2.5% (wall post.) [MeV/fm³] |
|---|---|---|---|---|---|---|---|---|
{_rows}

**Reading it: three levers, ranked.**
1. **Heaviest pulsar (gravity-full).** A single pulsar measured at **{_ms_txt}** (±0.02 M☉) gives strong evidence
   against the wall. This is the only lever that works with today's equation-of-state ignorance.
2. **Locating the conformal point alone (gravity-free): {_es_txt}.** A curve that never reaches −1 is the expected
   result, not a failure. Both worlds share one EOS posterior, so learning *where* Δ first turns negative shifts
   both evidences almost equally. **The degeneracy survives even a gravity-free detection of superconformal
   matter.** For reference, if a wall exists, current data place it above ≈{_ew_lo:.0f} MeV/fm³ (2.5th percentile).
3. **The full EOS from QCD/lab + one pulsar (the decisive test).** Once Δ(ε) and p(ε) are known
   gravity-independently up to ~2 M☉ central densities, the two worlds predict different maximum masses for the
   *same* matter. The median gap is ≈{_gap_med:.3f} M☉. A pulsar that lands inside that gap decides between GR and
   the whole infrared-Palatini class.

**The deep statement.** Infrared-Palatini gravity hides behind exactly the same uncertainty that hides the dense-QCD
equation of state. Measuring gravity inside neutron stars and measuring cold QCD at 3–5 n₀ are therefore *one*
problem, and neither can be solved without the other.

Saved `{_out}` and `part5_discovery_space.png` in `{P4CFG['ckpt_dir']}/`.
"""), p5_fig])

---
# Part VI — The conformal compactness bound: an EOS-independent test you can run on today's data

## What has worked so far, and why
Everything that survived scrutiny in Parts I–V is an **identity or a bound**, not a fit:
- **Theorem 5** (non-analytic GR limit), **Theorem 7** (the Einstein-frame conformal wall) and **Theorem 8**
  (degeneracy with GR) hold for every μ and n.
- **Every fitted signal has been a null:** the cosmological Jeans scale and the neutron-star Bayes factor of about 1.

So a test that cannot be broken must follow the same pattern. It should be a **bound on an observable that no
choice of equation of state can evade**, and the existing data should be checked against it.

## Theorem 9 — conformal compactness bound (proved numerically in cell 51)
Among all causal equations of state ($0\le dp/d\varepsilon\le1$), the most compact stable star has
$C\equiv GM/Rc^2=0.354$ (Koranda, Stergioulas & Friedman 1997). Add **only** the condition that matter is
nowhere superconformal, $p\le\varepsilon/3$. The pointwise-stiffest admissible EOS is
$p=\min(\varepsilon-\varepsilon_s,\ \varepsilon/3)$, and it gives
$$C\;\le\;C_{\rm conf}\simeq0.313\qquad\Longleftrightarrow\qquad R\;\ge\;4.72\ {\rm km}\,\Big(\frac{M}{M_\odot}\Big),$$
reached exactly at a conformal centre. A realistic crust and chiral-EFT matter lower the attainable ceiling further.
Cell 52 measures that ceiling over all three priors.

**Two equivalent readings.**
- **(i) Infrared-Palatini gravity** (any μ, n): by Theorem 7 no star holds superconformal matter, so *every* neutron
  star obeys $C\le C_{\rm conf}$.
- **(ii) GR:** any star observed with $C>C_{\rm conf}$ *must* contain superconformal matter, whatever its EOS.
  This is a model-independent detector of $p>\varepsilon/3$ from a mass and a radius alone.

**The discovery window** is $4.17\le R/(M/M_\odot)\,[{\rm km}]<4.72$. GR allows such stars; the conformal
wall forbids them.

## What cells 51–54 do
- **Cell 51:** proves the bound numerically. It reproduces the causal limit (0.354), computes $C_{\rm conf}$ and the
  $c_s^2\le1/3$ limit, and adds the crust-anchored version.
- **Cell 52:** measures the near-universal relation $C(p_c/\varepsilon_c)$ and the realistic ceiling from fresh
  draws of the equation of state under all three priors (GPU).
- **Cell 53:** applies the test to J0030, J0437, J0740 and GW170817. It reports the probability that each star
  violates the bound, using the posterior samples in `ns_samples/` if present, otherwise the published summaries.
- **Cell 54:** draws the M–R plane with the three limits, the data and the discovery window, and gives the verdict.

In [ ]:
# ---- THEOREM 9: maximal compactness under constraint sets (independent ODE solver, stable branch) ----
from scipy.integrate import solve_ivp as _ivp9

def _star(pc, e_of_p):
    def f(r, y):
        m, p = y
        if p <= 0: return [0.0, 0.0]
        e = e_of_p(p); return [4 * math.pi * r * r * e, -(e + p) * (m + 4 * math.pi * r**3 * p) / (r * (r - 2 * m))]
    ev = lambda r, y: y[1]; ev.terminal = True; ev.direction = -1
    r0 = 1e-6; e0 = e_of_p(pc)
    s = _ivp9(f, [r0, 400], [4 / 3 * math.pi * e0 * r0**3, pc], events=ev, rtol=1e-11, atol=1e-16, max_step=0.002)
    return s.y_events[0][0][0], s.t_events[0][0]

def _cmax(e_of_p, pcs):
    MR = [_star(pc, e_of_p) for pc in pcs]
    M = np.array([x[0] for x in MR]); R = np.array([x[1] for x in MR])
    i = int(np.argmax(M))                       # stable branch = up to the first mass maximum
    C = M[: i + 1] / R[: i + 1]; j = int(np.argmax(C))
    return float(C[j]), float(pcs[j] / e_of_p(pcs[j]))

def _scalefree(cs2, w):                          # self-bound, eps_s = 1 (scale-free units)
    ex = cs2 / (cs2 - w) if cs2 > w else np.inf
    return lambda p: 1.0 + p / cs2 if p <= w * ex else p / w

_pcs = np.logspace(-2, 2.5, 500)
BOUNDS = {}
BOUNDS["causal (KSF 1997: 0.354)"] = _cmax(_scalefree(1.0, 10.0), _pcs)
BOUNDS["causal + p<=eps/3  [Theorem 9]"] = _cmax(_scalefree(1.0, 1 / 3), _pcs)
BOUNDS["c_s^2 <= 1/3 everywhere"] = _cmax(_scalefree(1 / 3, 10.0), _pcs)

# crust-anchored: SLy crust + stiffest chiral-EFT band to 1.1 n0, then c_s^2=1 until p = eps/3, then p = eps/3
_n, _e, _p = NS.crust_table(1.1 * NS.N0)
_kconv = NS.MEVFM3_TO_KM2
_p_hi = 4.4                                       # top of the chiral-EFT pressure band at 1.1 n0 (cell 23)
_scale = _p_hi / _p[-1]
_pp = _p * np.where(_n > 0.5 * NS.N0, np.interp(_n, [0.5 * NS.N0, 1.1 * NS.N0], [1.0, _scale]), 1.0)
_e1, _p1 = _e[-1], _pp[-1]
_ex = (_p1 - _e1) / (1 / 3 - 1.0)                 # where the c_s^2 = 1 segment meets p = eps/3
def _real(pkm):
    p = pkm / _kconv
    if p <= _p1: e = float(np.interp(np.log(p), np.log(_pp), np.log(_e))); e = math.exp(e)
    elif p <= _ex / 3: e = _e1 + (p - _p1)
    else: e = 3 * p
    return e * _kconv
BOUNDS["crust + chiral-EFT (stiff) + causal + p<=eps/3"] = _cmax(_real, np.logspace(np.log10(5 * _kconv), np.log10(5e4 * _kconv), 400))
C_CONF = BOUNDS["causal + p<=eps/3  [Theorem 9]"][0]
C_CAUSAL = BOUNDS["causal (KSF 1997: 0.354)"][0]
_rows = "\n".join(f"| {k} | {v[0]:.4f} | {1.476625 / v[0]:.3f} | {v[1]:.3f} |" for k, v in BOUNDS.items())
mo.md(f"""
**Theorem 9, computed** (stable branch only, independent scipy integrator):

| constraint set | max C = GM/Rc² | minimum R per M☉ [km] | central p/ε there |
|---|---|---|---|
{_rows}

The causal row reproduces the classic 0.354, which validates the solver. Forbidding superconformal matter alone lowers
the ceiling to **C = {C_CONF:.4f}**, reached at a conformal centre. A realistic crust lowers it further.
""")

In [ ]:
# ---- NEAR-UNIVERSAL C(p_c/eps_c) and the realistic ceiling over all priors (fresh draws, GPU) -----
_dev = NSCFG["device"]
_crust = NS.crust_table(0.5 * NS.N0)
_ng = NS.make_n_grid(device=_dev)
_ncv = list(np.logspace(np.log10(NS.N0), np.log10(11.5 * NS.N0), 60))
_nper = 400 if NSCFG["n_samples"] <= 1000 else 20000
_bs = min(2500, _nper)
_S, _C, _ceil = [], [], {}
for _prior in ("uniform", "loguniform", "gp"):
    _mx = []
    for _b in range(-(-_nper // _bs)):
        _g = torch.Generator(device=_dev).manual_seed(9000 + 97 * _b + len(_prior))
        if _prior == "gp":
            _s, _cs, _ = NS4.gp_draw(_bs, _g, _dev)
            _eps, _p, _cs2, _mu = NS4.build_eos_nodes(NS, _s, _cs, torch.as_tensor(NS4.GP_NODES_N0 * NS.N0, device=_dev), _ng, _crust)
        else:
            _th = torch.rand(_bs, 7, generator=_g, device=_dev, dtype=torch.float64)
            if _prior == "loguniform": _th[:, 1:] = 10 ** (-2 + 2 * _th[:, 1:])
            _eps, _p, _cs2, _mu = NS.build_eos_batch(_th, _ng, _crust)
        _M, _R, _L, _ec = NS.tov_batch(_eps, _p, _cs2, _mu, _ng, _ncv, n_h=NSCFG["n_h"])
        _ok = torch.isfinite(_M) & torch.isfinite(_R) & (_R > 3)
        _st = NS.stable_mask(torch.where(_ok, _M, torch.zeros_like(_M))) & _ok & (_M > 1.0)
        _pc = torch.stack([NS.interp_rows(_eps.contiguous(), _p.contiguous(), _ec[:, j].contiguous()) for j in range(_ec.shape[1])], 1)
        _sig = _pc / _ec; _Cs = _M * NS.MSUN_KM / _R
        _cw, _ = NS.conformal_mask(_ec, _eps, _p, _ng)
        _S.append(_sig[_st].cpu().numpy()); _C.append(_Cs[_st].cpu().numpy())
        _sub = torch.where(_st & _cw, _Cs, torch.zeros_like(_Cs)).max(1).values
        _mx.append(_sub[_sub > 0].cpu().numpy())
        del _eps, _p, _cs2, _mu, _M, _R, _L, _ec
    _mx = np.concatenate(_mx)
    _ceil[_prior] = {"N_EOS": int(len(_mx)), "median": float(np.median(_mx)), "q95": float(np.quantile(_mx, 0.95)),
                     "q99": float(np.quantile(_mx, 0.99)), "max": float(_mx.max())}
_S = np.concatenate(_S); _C = np.concatenate(_C)
C_REAL = max(v["max"] for v in _ceil.values())
C_REAL99 = max(v["q99"] for v in _ceil.values())
_bins = np.linspace(0.05, 0.65, 25); _mid = 0.5 * (_bins[1:] + _bins[:-1])
_q = np.array([[np.quantile(_C[(_S >= a) & (_S < b)], q) if ((_S >= a) & (_S < b)).sum() > 30 else np.nan for q in (0.02, 0.5, 0.98)] for a, b in zip(_bins[:-1], _bins[1:])])
UNIV = {"sigma": _mid, "C_q": _q, "ceilings": _ceil}
_fig, _ax = plt.subplots(figsize=(7.5, 4.6))
_ax.fill_between(_mid, _q[:, 0], _q[:, 2], color="0.85", label="96% of stars (all priors)")
_ax.plot(_mid, _q[:, 1], color="0.3", lw=1.5, label="median")
_ax.axvline(1 / 3, color="C3", lw=1.2, label="conformal centre p_c = ε_c/3")
_ax.axhline(C_REAL, color="C3", ls="--", lw=1, label=f"realistic ceiling C = {C_REAL:.3f}")
_ax.set(xlabel="central pressure / energy density  p_c/ε_c", ylabel="compactness GM/Rc²", title="compactness is nearly a function of p_c/ε_c alone")
_ax.legend(fontsize=8); _fig.tight_layout(); univ_fig = _fig; plt.close(_fig)
_rows = "\n".join(f"| {k} | {v['N_EOS']} | {v['median']:.4f} | {v['q95']:.4f} | {v['q99']:.4f} | {v['max']:.4f} |" for k, v in _ceil.items())
mo.vstack([mo.md(f"""
**Realistic ceiling.** Maximum compactness reached by any fully subconformal stable star, per EOS:

| prior | EOS | median | 95% | 99% | max |
|---|---|---|---|---|---|
{_rows}

Realistic ceiling (all priors, max) **C = {C_REAL:.4f}**; 99th percentile {C_REAL99:.4f}. Both sit below the rigorous
Theorem 9 value, as they must.
"""), univ_fig])

In [ ]:
# ---- THE TEST ON DATA: probability that each observed star violates the conformal bound -------
_rng = np.random.default_rng(42)
_K = NS.MSUN_KM
def _draw_split(mu, sp, sm, n):
    u = _rng.random(n); z = np.abs(_rng.standard_normal(n))
    return np.where(u < sp / (sp + sm), mu + sp * z, mu - sm * z)

def _samples_from_kde(kde, n):
    L = kde["logd"]; w = np.exp(L - L.max()).ravel(); w /= w.sum()
    idx = _rng.choice(len(w), size=n, p=w); i, j = np.unravel_index(idx, L.shape)
    dm = kde["m"][1] - kde["m"][0]; dr = kde["r"][1] - kde["r"][0]
    return kde["m"][i] + dm * (_rng.random(n) - 0.5), kde["r"][j] + dr * (_rng.random(n) - 0.5)

_N = 400000
_rows, TEST = [], {}
for _name, (_m, _mp, _mm, _r, _rp, _rm) in NS.NICER.items():
    if _name in KDES:
        _M, _R = _samples_from_kde(KDES[_name], _N); _src = "posterior samples (KDE)"
    else:
        _M = _draw_split(_m, _mp, _mm, _N); _R = _draw_split(_r, _rp, _rm, _N); _src = "published summary"
    _C = _M * _K / _R
    TEST[_name] = {"C_med": float(np.median(_C)), "C_q": [float(np.quantile(_C, q)) for q in (0.025, 0.975)],
                   "P_gt_conf": float((_C > C_CONF).mean()), "P_gt_real": float((_C > C_REAL).mean()),
                   "P_gt_real99": float((_C > C_REAL99).mean()), "source": _src}
# GW170817: Lambda~ -> C of a ~1.36 Msun star via the C-Love relation (Maselli et al. 2013): C = 0.360 - 0.0355 ln L + 0.000705 ln^2 L
if "GW170817" in KDES:
    _g = KDES["GW170817"]; _w = np.exp(_g["logd"] - _g["logd"].max()); _w /= _w.sum()
    _Lt = _rng.choice(_g["lt"], size=_N, p=_w); _src = "posterior samples (KDE)"
else:
    # summary 300 (+420/-230) at 90%: drawn as a split normal in ln(Lambda~) (a positive quantity, no
    # unphysical tail at Lambda~ <= 0, which the C-Love relation would turn into spurious huge compactness)
    _lo90, _hi90 = NS.GW170817["Lt"] - 1.645 * NS.GW170817["sm"], NS.GW170817["Lt"] + 1.645 * NS.GW170817["sp"]
    _ln0 = np.log(NS.GW170817["Lt"])
    _Lt = np.exp(_draw_split(_ln0, (np.log(_hi90) - _ln0) / 1.645, (_ln0 - np.log(_lo90)) / 1.645, _N)); _src = "published summary (log-space)"
_Lt = np.clip(_Lt, 10.0, None); _l = np.log(_Lt)                # C-Love fit valid for Lambda >~ 10
_C = 0.360 - 0.0355 * _l + 0.000705 * _l**2
TEST["GW170817"] = {"C_med": float(np.median(_C)), "C_q": [float(np.quantile(_C, q)) for q in (0.025, 0.975)],
                    "P_gt_conf": float((_C > C_CONF).mean()), "P_gt_real": float((_C > C_REAL).mean()),
                    "P_gt_real99": float((_C > C_REAL99).mean()), "source": _src + " + C–Λ relation"}
_rows = "\n".join(f"| {k} | {v['C_med']:.3f} ({v['C_q'][0]:.3f}–{v['C_q'][1]:.3f}) | {v['P_gt_conf']:.1e} | {v['P_gt_real']:.1e} | {v['P_gt_real99']:.1e} | {v['source']} |" for k, v in TEST.items())
_worst = max(TEST.items(), key=lambda kv: kv[1]["P_gt_real"])
mo.md(f"""
**Test of Theorem 9 on data** (probability that each star is more compact than the bound):

| source | C (95%) | P(C > {C_CONF:.3f}, rigorous) | P(C > {C_REAL:.3f}, realistic max) | P(C > {C_REAL99:.3f}, realistic 99%) | input |
|---|---|---|---|---|---|
{_rows}

Most constraining today: **{_worst[0]}**, with P = {_worst[1]['P_gt_real']:.1e} of exceeding the realistic ceiling.
""")

In [ ]:
# ---- THE CONFORMAL WALL IN THE M-R PLANE, VERDICT, EXPORT ------------------------------------
_K = NS.MSUN_KM
_m = np.linspace(0.8, 3.0, 200)
_fig, _ax = plt.subplots(figsize=(8, 6))
_ax.fill_betweenx(_m, _K * _m / C_CAUSAL, _K * _m / C_CONF, color="C3", alpha=0.18, label="discovery window: GR allows, wall forbids")
_ax.plot(_K * _m / C_CAUSAL, _m, color="k", lw=1.2, label=f"causality (GR), C = {C_CAUSAL:.3f}")
_ax.plot(_K * _m / C_CONF, _m, color="C3", lw=1.6, label=f"Theorem 9: no superconformal matter, C = {C_CONF:.3f}")
_ax.plot(_K * _m / C_REAL, _m, color="C3", lw=1.0, ls="--", label=f"realistic ceiling, C = {C_REAL:.3f}")
for _name, (_mm, _mp, _mn, _r, _rp, _rm) in NS.NICER.items():
    _ax.errorbar(_r, _mm, xerr=[[_rm], [_rp]], yerr=[[_mn], [_mp]], fmt="o", color="C0", ms=4, capsize=2)
    _ax.annotate(_name, (_r, _mm), xytext=(6, 4), textcoords="offset points", fontsize=8)
_ax.set(xlim=(6, 16), ylim=(0.8, 3.0), xlabel="R [km]", ylabel="M [M☉]", title="an EOS-independent test of how dense matter gravitates")
_ax.legend(fontsize=8, loc="lower right"); _fig.tight_layout(); mr_fig = _fig; plt.close(_fig)
_pmax = max(v["P_gt_conf"] for v in TEST.values())
if _pmax > 0.997:
    _v = "**Violation:** at least one star exceeds the rigorous bound at >3σ. Infrared-Palatini gravity is excluded, and in GR that star contains superconformal matter."
elif max(v["P_gt_real"] for v in TEST.values()) > 0.95:
    _v = "**Tension:** a star exceeds the realistic ceiling at >95%. That is not yet a rigorous exclusion; check it with the posterior samples."
else:
    _v = "**All current stars satisfy the bound.** The conformal wall survives a test that does not depend on the equation of state."
_out = os.path.join(P4CFG["ckpt_dir"], "PART6_COMPACTNESS.json")
save_json(_out, {"bounds": BOUNDS, "C_real": C_REAL, "ceilings": UNIV["ceilings"], "test": TEST})
mr_fig.savefig(os.path.join(P4CFG["ckpt_dir"], "part6_mr_window.png"), dpi=160)
univ_fig.savefig(os.path.join(P4CFG["ckpt_dir"], "part6_universal_C.png"), dpi=160)
mo.vstack([mo.md(f"""
## Part VI verdict
{_v}

**The prediction for observers.**
- **What would falsify the class:** a neutron star with R below {_K / C_CONF:.2f} km × (M/M☉), at any mass and with any
  equation of state. That is {2.07 * _K / C_CONF:.1f} km at 2.07 M☉ and {2.3 * _K / C_CONF:.1f} km at 2.3 M☉.
- **What GR still allows below that line:** stars down to R = {_K / C_CAUSAL:.2f} km × (M/M☉).
- **The realistic line:** R = {_K / C_REAL:.2f} km × (M/M☉) is where real crusts make violations appear first.

A NICER/STROBE-X radius for a ≥2 M☉ pulsar at ±0.3 km lying in the shaded window would be the cleanest
gravity-and-QCD test available.

Saved `{_out}`, `part6_mr_window.png` and `part6_universal_C.png`.
"""), mr_fig])

---
# Part VII — The full action: post-Einstein structure of $R+\alpha R^2+\beta R_{(\mu\nu)}R^{(\mu\nu)}$ inside neutron stars

Parts II–VI used only the infrared term. Part VII takes the ultraviolet part of the original action in
metric-affine (Palatini) form:
$$S=\frac{1}{2\kappa}\int d^4x\sqrt{-g}\,f(R,Q)+S_m[g,\psi],\qquad f=R+\alpha R^2+\beta Q,\quad Q=R_{(\mu\nu)}R^{(\mu\nu)}.$$
Here $R_{\mu\nu}(\Gamma)$ is built from an independent connection, and $\alpha,\beta$ are squared lengths (km²).

## VII.1 Theorem 10 — exact map to Einstein gravity
Write $P^\mu{}_\nu\equiv g^{\mu\alpha}R_{(\alpha\nu)}$. The metric field equations are a matrix equation,
$$\big(f_R\,\mathbb 1+2\beta P\big)P-\tfrac f2\,\mathbb 1=\kappa\,T,\qquad f_R=1+2\alpha R .$$
With $\Omega\equiv f_R\mathbb 1+2\beta P$, the connection is the Levi-Civita connection of
$q_{\mu\nu}=|\Omega|^{1/2}g_{\mu\alpha}(\Omega^{-1})^\alpha{}_\nu$.

For a perfect fluid, $P={\rm diag}(P_0,P_1,P_1,P_1)$ and $\sigma_a=1+2\alpha R+2\beta P_a$. The $q$-frame equations are
**exactly** Einstein's, $G^\mu{}_\nu(q)=\kappa\tilde T^\mu{}_\nu$, with a perfect fluid
$$\kappa\tilde\varepsilon=\tfrac12(3\tilde R_1-\tilde R_0),\qquad \kappa\tilde p=-\tfrac12(\tilde R_0+\tilde R_1),\qquad
  \tilde R_a=\frac{\sigma_aP_a}{\sqrt{\sigma_0\sigma_1^3}} .$$
In vacuum $\Omega=\mathbb 1$ and $q=g$. So every observable of a static or slowly rotating star ($M,R,\Lambda,I$) equals
its GR value for the effective EOS. This is the Ricci-based-gravity mapping (Afonso, Olmo & Rubiera-Garcia 2018);
Theorem 8 is its infrared, $\beta=0$ special case.

## VII.2 Theorem 11 — conformal selection rule
**(i) Exact, all orders ($\beta=0$).** $R=\kappa(\varepsilon-3p)$, $F=1+2\alpha R$, and $\tilde T=T/F$. The $R^2$ sector never
changes the sign of the trace anomaly, and conformal matter ($p=\varepsilon/3$) maps onto itself identically.

**(ii) Leading order in $\alpha\kappa\varepsilon,\ \beta\kappa\varepsilon$:**
$$\delta\tilde\varepsilon=-\tfrac{\alpha\kappa}{2}(7\varepsilon+3p)(\varepsilon-3p)-\tfrac{\beta\kappa}{2}(\varepsilon^2-6\varepsilon p-3p^2),\qquad
  \delta\tilde p=-\tfrac{\alpha\kappa}{2}(\varepsilon+5p)(\varepsilon-3p)-\tfrac{\beta\kappa}{2}(\varepsilon+3p)(\varepsilon-p).$$
So $\delta\tilde p_\alpha=-\tfrac32\alpha\kappa\,\varepsilon(\varepsilon+5p)\,\Delta$: **$R^2$ gravity couples to matter
linearly through the trace anomaly $\Delta$.** Only the Ricci-squared term sees conformal matter: at $p=\varepsilon/3$,
$\tilde\Delta=\tfrac89\beta\kappa\varepsilon$.

**Consequences.**
- Particle-physics bounds act on $\beta$, not $\alpha$. Light-by-light scattering cannot reach $\alpha$ at all, because the
  Maxwell field is traceless. The $\beta$ sector is bounded at $\Lambda_Q\gtrsim10^2$–$10^3$ GeV (Latorre, Olmo & Ronco 2018;
  Beltrán Jiménez et al. 2021). Those bounds are orders of magnitude stronger than any astrophysical one, so
  $\beta\kappa\varepsilon\ll1$ inside neutron stars. **$\alpha$ is the open sector.**
- Locally the $\alpha$ correction vanishes at $\Delta=0$ and **flips sign** there (exact). Integrated quantities such as
  $M_{\max}$ mix regions of both signs, plus a competing $\delta\tilde\varepsilon$ term, so they barely respond. Cell 61 measures
  $dM_{\max}/d\alpha$.
- The infrared term of Parts II–VI acts only at $T=0$, the wall, where the $\alpha$ correction vanishes. The UV and IR sectors
  are **orthogonal in trace**.

## VII.3 Theorem 12 — crust gradient theorem
For dilute matter ($p\ll\varepsilon$), $\tilde p=p-\tfrac12(\alpha+\beta)\kappa\varepsilon^2+\dots$, so
$$\tilde c_s^{\,2}\equiv\frac{d\tilde p}{d\tilde\varepsilon}=c_s^2-(\alpha+\beta)\,\kappa\varepsilon+\dots$$
The auxiliary connection acts like an **anti-pressure** $-\tfrac12(\alpha+\beta)\kappa\varepsilon^2$, which beats the true pressure
gradient in matter that is dense but nearly pressureless. A static star needs $\tilde p$ to fall monotonically outward, so a
star with a crust exists **only if**
$$\alpha+\beta\;<\;\ell^2_{\rm crust}\equiv\min_{\rm crust}\frac{c_s^2}{\kappa\varepsilon}.$$
The minimum sits at the crust–core transition ($n\approx0.08$ fm⁻³). No mass, radius or tidal measurement enters: the bound
needs only that crusts exist (glitches, crust cooling) and the nuclear-physics crust EOS. Cell 58 computes it and
solves the exact threshold.

## VII.4 Theorem 13 — the glitch lever (degeneracy breaking)
The crustal moment of inertia tracks the effective transition pressure, $\Delta I_c/I\propto\tilde p_t$, so
$$\frac{\Delta I_c}{I}\simeq\Big(\frac{\Delta I_c}{I}\Big)_{\rm GR}\Big(1-\frac{\alpha+\beta}{\ell_t^2}\Big),\qquad
  \ell_t^2\equiv\frac{2p_t}{\kappa\varepsilon_t^2}\approx3.4\text{–}3.9\ {\rm km}^2 .$$
**Why this breaks the degeneracy.** By Theorem 10, gravity can hide inside the unknown core EOS, but it cannot hide in the
crust, whose EOS is known. Pulsar glitches therefore measure gravity.

Vela's glitch activity needs $\Delta I_c/I\gtrsim1.6\%$ without entrainment (Link, Epstein & Lattimer 1999) and $\approx7\%$
with strong crustal entrainment (Andersson et al. 2012; Chamel 2013). GR equations of state struggle to supply the 7%:
this is the "glitch crisis". By Theorem 13, $\alpha<0$ raises the crustal inertia.

## Pre-registered tests (cells 60–61)
- **P1, crust theorem:** no star for any EOS above the exact threshold, every EOS valid below it.
- **P2, lever:** median $d\ln(\Delta I_c/I)/d\alpha$ within 15% of $-1/\ell_t^2$, and spread (IQR) below 20% of the median.
- **P3, degeneracy (Theorem 10):** core data alone barely see $\alpha$: $\max|\log_{10}B(\alpha\ {\rm vs}\ 0)|<0.5$ for
  $\alpha\in[-4,2]$ km² (astro+pQCD).
- **P4, sensitivity:** $|d\ln(\Delta I_c/I)/d\alpha|\,/\,|d\ln M_{\max}/d\alpha|>30$.

**Assumptions.**
- The crust share of $I$ is computed in the Einstein frame; this is exact for the total $I$.
- Vela's mass is unknown and is averaged over 1.2–1.8 M☉.
- The entrainment strength is debated (Watanabe & Pethick 2017).

**Novelty status.** The RBG→GR map and the neutron-star interval $-6.5\lesssim\alpha\lesssim2$ km² (arXiv:2306.00870) are
known. We found no earlier use of crust stability or glitches for Palatini/RBG gravity; arXiv:1507.05878 treats metric $R^2$
and scalar–tensor gravity. A full literature audit is still required.

In [ ]:
# ---- PART VII CORE (namespace NS7) + CONFIG ---------------------------------------------------
def _build_ns7():
    # Part VII core: the user's full quadratic Palatini action  f(R,Q) = R + alpha R^2 + beta Q,  Q = R_(mn) R^(mn)
    # (Ricci-based gravity, RBG).  Exact Einstein-frame map for a perfect fluid, batched in torch.
    # Units: G = c = 1, lengths in km, kappa = 8 pi; alpha, beta in km^2; eps, p in MeV/fm^3 at the interface.
    import math
    import numpy as np
    import torch

    KAP = 8.0 * math.pi
    KM2 = 1.3234e-6                     # MeV/fm^3 -> km^-2


    def solve_P(x, y, a, b, n_cont=8, n_newton=30):
        """Eigenvalues P0, P1 of P^m_n = g^ma R_an(Gamma) for a perfect fluid (P = diag(P0,P1,P1,P1)).
        Field equations (matrix form):  (F + 2 b P) P - f/2 = kappa T,   F = 1 + 2 a R,  R = P0 + 3 P1,
        f = R + a R^2 + b (P0^2 + 3 P1^2);  kappa T = diag(-x, y, y, y) with x = kappa eps, y = kappa p [km^-2].
        The GR-connected branch is followed by continuation in (a,b) -> t (a,b), t: 0 -> 1, with Newton at each t.
        Returns P0, P1, residual."""
        P0 = -(x + 3 * y) / 2
        P1 = (x - y) / 2
        for k in range(1, n_cont + 1):
            t = k / n_cont
            A, Bq = a * t, b * t
            for _ in range(n_newton):
                R = P0 + 3 * P1
                F = 1 + 2 * A * R
                f = R + A * R**2 + Bq * (P0**2 + 3 * P1**2)
                E0 = 2 * Bq * P0**2 + F * P0 - f / 2 + x
                E1 = 2 * Bq * P1**2 + F * P1 - f / 2 - y
                fa0 = 1 + 2 * A * R + 2 * Bq * P0          # df/dP0
                fa1 = 3 * (1 + 2 * A * R) + 6 * Bq * P1    # df/dP1
                J00 = 4 * Bq * P0 + 2 * A * P0 + F - fa0 / 2
                J01 = 6 * A * P0 - fa1 / 2
                J10 = 2 * A * P1 - fa0 / 2
                J11 = 4 * Bq * P1 + 6 * A * P1 + F - fa1 / 2
                det = J00 * J11 - J01 * J10
                det = torch.where(det.abs() < 1e-300, torch.full_like(det, 1e-300), det) if torch.is_tensor(det) else det
                P0 = P0 - (E0 * J11 - E1 * J01) / det
                P1 = P1 - (J00 * E1 - J10 * E0) / det
        R = P0 + 3 * P1
        F = 1 + 2 * a * R
        f = R + a * R**2 + b * (P0**2 + 3 * P1**2)
        res = torch.maximum((2 * b * P0**2 + F * P0 - f / 2 + x).abs(), (2 * b * P1**2 + F * P1 - f / 2 - y).abs())
        return P0, P1, res


    def effective(eps, p, a, b):
        """Exact RBG -> GR map.  Omega = F + 2 b P = diag(s0, s1, s1, s1), q_mn = |Omega|^(1/2) g_ma (Omega^-1)^a_n,
        G^m_n(q) = |Omega|^(-1/2) Omega P - (1/2) R(q) delta  =  kappa T~^m_n(q)  with
            R~0 = s0 P0 / sqrt(s0 s1^3),  R~1 = s1 P1 / sqrt(s0 s1^3),
            eps~ = (3 R~1 - R~0) / (2 kappa),  p~ = -(R~0 + R~1) / (2 kappa).
        The q-frame fluid is perfect (isotropy is preserved), conserved (Bianchi in q) and q = g in vacuum.
        eps, p in MeV/fm^3; a, b in km^2 (tensors broadcastable). Returns eps~, p~ [MeV/fm^3], ok mask, (s0, s1)."""
        x = KAP * eps * KM2
        y = KAP * p * KM2
        P0, P1, res = solve_P(x, y, a, b)
        R = P0 + 3 * P1
        F = 1 + 2 * a * R
        s0 = F + 2 * b * P0
        s1 = F + 2 * b * P1
        ok = (s0 > 0) & (s1 > 0) & torch.isfinite(P0) & torch.isfinite(P1) & (res <= 1e-10 * (x.abs() + 1e-30))
        sq = torch.sqrt((s0 * s1**3).clamp(min=1e-300))
        R0 = s0 * P0 / sq
        R1 = s1 * P1 / sq
        et = (3 * R1 - R0) / (2 * KAP) / KM2
        pt = -(R0 + R1) / (2 * KAP) / KM2
        return et, pt, ok, (s0, s1)


    def first_order(eps, p, a, b):
        """Theorem 11 (leading order in a kappa eps, b kappa eps), MeV/fm^3:
           d eps~ = -(a k/2)(7 eps + 3 p)(eps - 3 p) - (b k/2)(eps^2 - 6 eps p - 3 p^2)
           d p~   = -(a k/2)(eps + 5 p)(eps - 3 p)   - (b k/2)(eps + 3 p)(eps - p)"""
        k = KAP * KM2
        de = -(a * k / 2) * (7 * eps + 3 * p) * (eps - 3 * p) - (b * k / 2) * (eps**2 - 6 * eps * p - 3 * p**2)
        dp = -(a * k / 2) * (eps + 5 * p) * (eps - 3 * p) - (b * k / 2) * (eps + 3 * p) * (eps - p)
        return eps + de, p + dp


    def rq_tables(eps, p, cs2, mu, n_grid, a, b):
        """Batched Einstein-frame tables for (B,N) EOS and per-row couplings a, b (B,) [km^2].
        Valid prefix of each row: map exists (ok), and eps~, p~ strictly increase (no wall). Beyond it the rows are
        padded monotonically (never sampled, because central densities are capped at n_last).
        Effective enthalpy: h~ = h + int[dp~/(eps~+p~) - dp/(eps+p)] (difference form, exact to round-off).
        Returns et, pt, cs2t, mut, n_last (B,), max |p~/p - 1| on the valid part (B,)."""
        a = torch.as_tensor(a, dtype=eps.dtype, device=eps.device).reshape(-1, 1)
        b = torch.as_tensor(b, dtype=eps.dtype, device=eps.device).reshape(-1, 1)
        et, pt, ok, _ = effective(eps, p, a, b)
        inc = torch.ones_like(ok)
        inc[:, 1:] = (pt[:, 1:] > pt[:, :-1]) & (et[:, 1:] > et[:, :-1])
        good = torch.cumprod((ok & inc).to(torch.int64), 1).bool()
        good[:, 0] = True
        il = good.sum(1) - 1                                         # last valid index per row
        n_last = n_grid[il]
        # pad beyond the valid prefix with a gentle monotone continuation (never used)
        j = torch.arange(eps.shape[1], device=eps.device).view(1, -1)
        lastE = torch.gather(et, 1, il.view(-1, 1)); lastP = torch.gather(pt, 1, il.view(-1, 1))
        pad = (j - il.view(-1, 1)).clamp(min=0).to(eps.dtype)
        et = torch.where(good, et, lastE * (1 + 1e-6 * pad))
        pt = torch.where(good, pt, lastP * (1 + 1e-6 * pad))
        ae = 0.5 * (et[:, 1:] + et[:, :-1] + pt[:, 1:] + pt[:, :-1])
        a0 = 0.5 * (eps[:, 1:] + eps[:, :-1] + p[:, 1:] + p[:, :-1])
        dh = torch.diff(pt, dim=1) / ae - torch.where(good[:, 1:], torch.diff(p, dim=1) / a0, torch.zeros_like(ae))
        corr = torch.cat([torch.zeros_like(et[:, :1]), torch.cumsum(dh, 1)], 1)
        mut = mu * torch.exp(corr)
        rp = torch.gradient(pt, spacing=(n_grid,), dim=1)[0] / torch.gradient(p, spacing=(n_grid,), dim=1)[0]
        re = torch.gradient(et, spacing=(n_grid,), dim=1)[0] / torch.gradient(eps, spacing=(n_grid,), dim=1)[0]
        cs2t = torch.where(good, cs2 * rp / re, torch.full_like(cs2, 1e-3)).clamp(1e-6, 10.0)   # effective c~^2 may exceed 1
        dev = torch.where(good, (pt / p - 1).abs(), torch.zeros_like(pt)).max(1).values
        return et, pt, cs2t, mut, n_last, dev


    def existence_scan(w_vals, bmax=3.0, nb=3000, a_over_b=0.0):
        """Theorem 12: for p = w eps, scan bh = beta kappa eps (both signs) and return the first |bh| where the
        GR-connected solution stops existing (fold, or s0/s1 -> 0). Units are scale-free (kappa eps = 1)."""
        out = {}
        for w in w_vals:
            row = {}
            for sgn in (+1.0, -1.0):
                bh = torch.linspace(1e-4, bmax, nb, dtype=torch.float64) * sgn
                x = torch.ones_like(bh); y = w * x
                P0, P1, res = solve_P(x, y, a_over_b * bh, bh, n_cont=40, n_newton=40)
                R = P0 + 3 * P1; F = 1 + 2 * a_over_b * bh * R
                s0 = F + 2 * bh * P0; s1 = F + 2 * bh * P1
                good = (res < 1e-11) & (s0 > 0) & (s1 > 0) & torch.isfinite(P0)
                cg = torch.cumprod(good.to(torch.int64), 0).bool()
                row["+" if sgn > 0 else "-"] = float(bh[cg].abs().max()) if bool(cg.any()) else 0.0
            out[float(w)] = row
        return out
    SLY_LAL = """
    2.4973001e-31,9.7680036e-24 1.5923535e-30,3.0889141e-23 1.0153324e-29,9.7680049e-23 6.4740638e-29,3.0889149e-22
    4.1280573e-28,9.7680100e-22 2.6321732e-27,3.0889182e-21 1.6783526e-26,9.7680308e-21 1.0701680e-25,3.0889314e-20
    6.8237123e-25,9.7681153e-20 4.3510037e-24,3.0889853e-19 1.9152348e-23,7.7590667e-19 8.0600154e-23,1.9490988e-18
    3.2314424e-22,4.8962209e-18 4.3452200e-22,6.1635786e-18 1.1856609e-21,1.2296916e-17 3.1669953e-21,2.4542100e-17
    8.3120200e-21,4.8982357e-17 2.1515408e-20,9.7731087e-17 5.5160085e-20,1.9502439e-16 7.2197247e-20,2.4553150e-16
    1.3459523e-19,3.8926423e-16 2.5026947e-19,6.1692626e-16 3.4115637e-19,7.7683704e-16 4.1609674e-19,9.0064235e-16
    5.6680375e-19,1.1923757e-15 1.0509830e-18,1.8903706e-15 1.9466321e-18,3.0945282e-15 3.6040786e-18,4.9063593e-15
    4.6781965e-18,5.9645892e-15 6.3637354e-18,7.5106111e-15 8.6590427e-18,9.7977699e-15 1.1773985e-17,1.2335381e-14
    1.6012619e-17,1.5533565e-14 2.0680901e-17,1.8818820e-14 2.8125364e-17,2.4627119e-14 3.8238597e-17,3.1009692e-14
    4.9153287e-17,3.7439237e-14 6.6834920e-17,4.9181489e-14 9.0886898e-17,6.1945428e-14 1.1980546e-16,7.6222455e-14
    1.2352356e-16,8.1115542e-14 1.6797551e-16,1.0520068e-13 2.1457570e-16,1.2643615e-13 2.3894992e-16,1.3707695e-13
    2.7183445e-16,1.5579990e-13 3.6957918e-16,2.0290035e-13 4.8054382e-16,2.4713921e-13 6.2282313e-16,3.1128159e-13
    6.4488385e-16,3.1952990e-13 6.5190693e-16,3.2214100e-13 6.9007943e-16,3.3621357e-13 7.5171727e-16,3.5853730e-13
    7.8418868e-16,3.7011388e-13 8.1228100e-16,3.8003359e-13 8.9482282e-16,4.2482152e-13 1.7803091e-15,1.5711920e-12
    2.8317052e-15,2.3608374e-12 4.3525736e-15,3.3315249e-12 6.4427243e-15,4.4971521e-12 9.2101478e-15,5.8755555e-12
    1.7263553e-14,9.3590560e-12 2.9613430e-14,1.3989896e-11 4.7600774e-14,2.0006786e-11 7.2806189e-14,2.7624508e-11
    1.0697388e-13,3.6919608e-11 1.7863407e-13,5.3522494e-11 3.1789758e-13,7.8102012e-11 4.1693951e-13,9.1947619e-11
    4.4378420e-13,9.5462901e-11 8.1920493e-13,1.2585927e-10 2.4644396e-12,1.8950551e-10 5.7495052e-12,2.5393146e-10
    1.1293110e-11,3.1944160e-10 1.9629918e-11,3.8632554e-10 3.1256831e-11,4.5485060e-10 4.6642330e-11,5.2529898e-10
    6.6233414e-11,5.9792315e-10 9.0457251e-11,6.7297559e-10 1.1973109e-10,7.5072364e-10 1.5444989e-10,8.3127126e-10
    1.9498545e-10,9.1508628e-10 2.4171784e-10,1.0021687e-09 2.9498582e-10,1.0928156e-09 3.5512816e-10,1.1872497e-09
    4.2245057e-10,1.2856937e-09 4.9725874e-10,1.3882963e-09 5.7982536e-10,1.4952802e-09 6.7040655e-10,1.6067939e-09
    7.6924192e-10,1.7230602e-09 8.7656282e-10,1.8442275e-09 9.9256755e-10,1.9704445e-09 1.1174131e-09,2.1019339e-09
    1.2512647e-09,2.2387700e-09 1.3943703e-09,2.3811755e-09 1.5467298e-09,2.5292247e-09 1.7085911e-09,2.6831403e-09
    1.8800368e-09,2.8429967e-09 2.0612322e-09,3.0089424e-09 2.2521773e-09,3.1810515e-09
    """


    def sly_table():
        """SLy unified EOS (Douchin & Haensel 2001; BPS outer crust), as tabulated in LALSuite
        (LALSimNeutronStarEOS_SLY.dat: p, eps in m^-2). Returns eps, p in MeV/fm^3 (numpy)."""
        a = np.array([[float(v) for v in pair.split(",")] for pair in SLY_LAL.split()])
        return a[:, 1] / 1.3234e-12, a[:, 0] / 1.3234e-12


    def crust_length2(eps, p, eps_max=150.0):
        """Theorem 12 (leading order): l_crust^2 = min over eps <= eps_max of c_s^2 / (kappa eps)   [km^2],
        with c_s^2 = (p/eps) dln p/dln eps on the table. Returns (l2, eps at minimum, c_s^2 there)."""
        m = eps <= eps_max
        le, lp = np.log(eps[m]), np.log(p[m])
        c2 = p[m] / eps[m] * np.gradient(lp, le)
        L2 = c2 / (KAP * KM2 * eps[m])
        i = int(np.argmin(L2))
        return float(L2[i]), float(eps[m][i]), float(c2[i])


    def monotone_ok(eps, p, a, b):
        """True if the exact map exists and eps~, p~ both increase along the table (static stars possible)."""
        E = torch.as_tensor(eps, dtype=torch.float64); P = torch.as_tensor(p, dtype=torch.float64)
        et, pt, ok, _ = effective(E, P, torch.as_tensor(float(a), dtype=torch.float64), torch.as_tensor(float(b), dtype=torch.float64))
        return bool(ok.all()) and bool((torch.diff(pt) > 0).all()) and bool((torch.diff(et) > 0).all())


    def crust_threshold(eps, p, direction=(1.0, 0.0), hi=50.0, iters=48):
        """Exact Theorem 12: largest s with (alpha, beta) = s * direction still giving a monotone effective crust."""
        lo = 0.0
        for _ in range(iters):
            mid = 0.5 * (lo + hi)
            if monotone_ok(eps, p, mid * direction[0], mid * direction[1]):
                lo = mid
            else:
                hi = mid
        return lo


    def tov_rq(ns, et, pt, cs2t, mut, n_grid, nc_vals, n_t, n_h=240, n_crust=80):
        """Batched GR structure of the Einstein-frame star (= the RBG star, Theorem 10): mass [Msun], radius [km],
        Lambda, moment of inertia I [km^3], crustal moment of inertia dI [km^3] (matter with n < n_t), central eps~.
        Enthalpy form (Lindblom 1992) on h~ = ln(mu~/mu~_s), split at the crust-core transition so the crust gets
        n_crust RK4 steps of its own. Slow rotation (Hartle 1967): (r^4 j w')' + 4 r^3 j' w = 0, j = e^-(nu+lam)/2,
        nu = nu_R - 2 h~  ->  I = J/Omega,  dI = (8 pi/3) int_crust r^4 (eps~+p~) e^((lam-nu)/2) w/Omega dr.
        nc_vals: (B,Nc) central number densities; n_t: crust-core transition density [fm^-3]."""
        B, N = et.shape
        dt, dev = et.dtype, et.device
        h_tab = torch.log(mut / mut[:, :1])
        E = et * ns.MEVFM3_TO_KM2; P = pt * ns.MEVFM3_TO_KM2
        lnn = torch.log(n_grid).view(1, -1).expand(B, -1).contiguous()
        ncl = torch.log(nc_vals.to(dtype=dt, device=dev))
        Nc = ncl.shape[1]
        hc = torch.stack([ns.interp_rows(lnn, h_tab, ncl[:, j].contiguous()) for j in range(Nc)], 1)
        ht = ns.interp_rows(lnn, h_tab, torch.full((B,), math.log(n_t), dtype=dt, device=dev))
        S = hc.reshape(-1)
        HT = torch.minimum(ht.repeat_interleave(Nc, 0), 0.5 * S)
        Hrows = h_tab.repeat_interleave(Nc, 0).contiguous()
        Erows = E.repeat_interleave(Nc, 0); Prows = P.repeat_interleave(Nc, 0); Crows = cs2t.repeat_interleave(Nc, 0)

        def eos_at(hq):
            xq = hq.reshape(-1, 1)
            idx = torch.clamp(torch.searchsorted(Hrows, xq.contiguous()), 1, Hrows.shape[1] - 1)
            x0 = torch.gather(Hrows, 1, idx - 1); x1 = torch.gather(Hrows, 1, idx)
            w = ((xq - x0) / (x1 - x0).clamp(min=1e-300)).clamp(0, 1)
            out = []
            for T_ in (Erows, Prows, Crows):
                y0 = torch.gather(T_, 1, idx - 1); y1 = torch.gather(T_, 1, idx)
                out.append((y0 + w * (y1 - y0)).reshape(-1))
            return out

        ec, pc, _ = eos_at(S)
        dh0 = 1e-3 * S
        r = torch.sqrt(3 * dh0 / (2 * math.pi * (ec + 3 * pc)))
        m = 4 * math.pi / 3 * ec * r**3
        y = torch.full_like(r, 2.0)
        w = torch.ones_like(r); u = torch.zeros_like(r)
        Kt = torch.zeros_like(r); Kc = torch.zeros_like(r)

        def rhs(h, st):
            r, m, y, w, u = st
            e, pp, c2 = eos_at(h)
            den = m + 4 * math.pi * r**3 * pp
            drdh = -r * (r - 2 * m) / den
            el = 1.0 / (1 - 2 * m / r)
            nup = 2 * den / (r * (r - 2 * m))
            Q = 4 * math.pi * el * (5 * e + 9 * pp + (e + pp) / c2) - 6 * el / r**2 - nup**2
            dydr = -(y**2 + y * el * (1 + 4 * math.pi * r**2 * (pp - e)) + r**2 * Q) / r
            jj = -4 * math.pi * r * (e + pp) * el                      # j'/j
            dudr = -(4 / r + jj) * u - 4 * jj / r * w
            dk = r**4 * (e + pp) * torch.sqrt(el) * torch.exp(h) * w   # r^4 (eps+p) e^{lam/2} e^{h} w  (x e^{-nu_R/2} later)
            return [drdh, 4 * math.pi * r**2 * e * drdh, dydr * drdh, u * drdh, dudr * drdh, dk * drdh]

        def integrate(h0, h1, nsteps, st, crust):
            r, m, y, w, u, Kt, Kc = st
            for i in range(nsteps):
                a0 = h0 + (h1 - h0) * (i / nsteps); a1 = h0 + (h1 - h0) * ((i + 1) / nsteps); s = a1 - a0
                X = [r, m, y, w, u]
                k1 = rhs(a0, X)
                k2 = rhs(a0 + s / 2, [x + s / 2 * k for x, k in zip(X, k1[:5])])
                k3 = rhs(a0 + s / 2, [x + s / 2 * k for x, k in zip(X, k2[:5])])
                k4 = rhs(a1, [x + s * k for x, k in zip(X, k3[:5])])
                inc = [s / 6 * (q1 + 2 * q2 + 2 * q3 + q4) for q1, q2, q3, q4 in zip(k1, k2, k3, k4)]
                r, m, y, w, u = [x + d for x, d in zip(X, inc[:5])]
                Kt = Kt + inc[5]
                if crust:
                    Kc = Kc + inc[5]
            return r, m, y, w, u, Kt, Kc

        st = integrate(S - dh0, HT, n_h, (r, m, y, w, u, Kt, Kc), False)
        r, m, y, w, u, Kt, Kc = integrate(HT, torch.zeros_like(HT), n_crust, st, True)
        C = m / r
        Lam = 2.0 / 3.0 * ns.love_k2(C, y) / C**5
        J = r**4 * u / 6
        Om = w + 2 * J / r**3
        I = J / Om
        fac = 8 * math.pi / 3 / torch.sqrt(1 - 2 * C) / Om               # e^{-nu_R/2} = (1-2C)^{-1/2}
        It, dI = fac * Kt, fac * Kc
        sh = (B, Nc)
        return ((m / ns.MSUN_KM).reshape(sh), r.reshape(sh), Lam.reshape(sh), I.reshape(sh), dI.reshape(sh),
                It.reshape(sh), (ec / ns.MEVFM3_TO_KM2).reshape(sh))


    def branch_at(M, X, allowed, m0):
        """Value of X at mass m0 on the stable branch of each row (linear interpolation); NaN if m0 not reached."""
        Mw = torch.where(allowed, M, torch.full_like(M, -1.0))
        B, Nc = M.shape
        out = torch.full((B,), float("nan"), dtype=M.dtype, device=M.device)
        for j in range(Nc - 1):
            a, b = Mw[:, j], Mw[:, j + 1]
            hit = (a >= 0) & (b >= 0) & (a <= m0) & (b > m0) & torch.isnan(out)
            t = (m0 - a) / (b - a).clamp(min=1e-12)
            out = torch.where(hit, X[:, j] + t * (X[:, j + 1] - X[:, j]), out)
        return out


    def sly_grid(npts=5000):
        """SLy on a fine monotone (PCHIP in log-log) grid, with n from the first law dn/n = d eps/(eps+p),
        normalised to 56Fe at the surface (mu = 930.4 MeV). Returns eps, p, n [fm^-3], mu [MeV] (numpy)."""
        from scipy.interpolate import PchipInterpolator
        e0, p0 = sly_table()
        ip = PchipInterpolator(np.log(e0), np.log(p0))
        E = np.exp(np.linspace(np.log(e0[0]), np.log(e0[-1]), npts)); P = np.exp(ip(np.log(E)))
        lnn = np.concatenate([[0.0], np.cumsum(np.diff(E) / (0.5 * (E[1:] + E[:-1] + P[1:] + P[:-1])))])
        n = (E[0] + P[0]) / 930.4 * np.exp(lnn)
        return E, P, n, (E + P) / n


    def hartle_scipy(et_km, pt_km, pc, p_t, msun_km=1.476625):
        """Independent check (radius integration, scipy): M [Msun], R [km], I [km^3], dI_crust [km^3] for one star of the
        effective EOS (eps~, p~ in km^-2, increasing), central p~ = pc, crust = layers with p~ < p_t."""
        from scipy.integrate import solve_ivp
        lp, le = np.log(pt_km), np.log(et_km)
        psurf = pt_km[0] * 1.5
        eop = lambda q: math.exp(np.interp(math.log(q), lp, le))

        def f(r, y):
            m, q, nu, w, dw = y
            if q <= psurf * 0.5:
                return [0.0] * 5
            e = eop(q); el = 1 / (1 - 2 * m / r)
            dnu = 2 * (m + 4 * math.pi * r**3 * q) * el / r**2
            jj = -4 * math.pi * r * (e + q) * el
            return [4 * math.pi * r * r * e, -(e + q) * dnu / 2, dnu, dw, -(4 / r + jj) * dw - 4 * jj / r * w]
        ev = lambda r, y: y[1] - psurf; ev.terminal = True; ev.direction = -1
        evt = lambda r, y: y[1] - p_t; evt.direction = -1
        r0 = 1e-5; e0 = eop(pc)
        s = solve_ivp(f, [r0, 80.0], [4 / 3 * math.pi * e0 * r0**3, pc, 0.0, 1.0, 0.0], events=[ev, evt],
                      rtol=1e-9, atol=1e-14, dense_output=True, max_step=0.02)
        R = s.t_events[0][0]; yR = s.y_events[0][0]; M = yR[0]
        J = R**4 * yR[4] / 6; Om = yR[3] + 2 * J / R**3; I = J / Om
        rt = s.t_events[1][0]
        nuoff = math.log(1 - 2 * M / R) - yR[2]
        rr = np.linspace(rt, R, 600); Y = s.sol(rr)
        ee = np.array([eop(max(x, psurf)) for x in Y[1]])
        lam = -np.log(1 - 2 * Y[0] / rr); nu = Y[2] + nuoff
        dI = 8 * math.pi / 3 * np.trapezoid(rr**4 * (ee + Y[1]) * np.exp((lam - nu) / 2) * Y[3] / Om, rr)
        return M / msun_km, R, I, dI


    GLITCH_MASSES = (1.2, 1.4, 1.6, 1.8)


    def scan_point(ns, ns3, eps, p, cs2, mu, n_grid, a, b, cfg, kdes):
        """One (alpha, beta) point for a batch of physical EOS: exact map -> Einstein-frame stars -> likelihoods.
        pQCD is applied to the PHYSICAL (matter-frame) EOS at the centre of the heaviest star. Returns numpy dict."""
        B = eps.shape[0]; dev, dt = eps.device, eps.dtype
        A = torch.full((B,), float(a), dtype=dt, device=dev); Bq = torch.full((B,), float(b), dtype=dt, device=dev)
        et, pt, c2t, mut, nl, dtab = rq_tables(eps, p, cs2, mu, n_grid, A, Bq)
        N0 = ns.N0
        valid = nl > 1.2 * N0
        hi = torch.where(valid, nl.clamp(max=cfg["n_max_c"]), torch.full_like(nl, 1.2 * N0))
        lo = torch.full_like(nl, cfg["n_min_c"])
        t = torch.linspace(0, 1, cfg["n_central"], dtype=dt, device=dev).view(1, -1)
        ncs = torch.exp(torch.log(lo).view(-1, 1) + t * (torch.log(hi) - torch.log(lo)).view(-1, 1))
        M, R, L, I, dI, It, ec = tov_rq(ns, et, pt, c2t, mut, n_grid, ncs, cfg["n_t"], n_h=cfg["n_h"], n_crust=cfg["n_crust"])
        ok = torch.isfinite(M) & torch.isfinite(R) & torch.isfinite(L) & torch.isfinite(dI) & (R > 3) & valid.view(-1, 1)
        st = ns.stable_mask(torch.where(ok, M, torch.zeros_like(M))) & ok
        out = {}
        for bw in (False, True):
            d = ns3.log_likelihoods_v3(ns, M, R, L, st, kdes, use_bw=bw)
            out["ll_bw" if bw else "ll"] = sum(d.values())
        Mmax, lnc = refine_max(M, st, ok, torch.log(ncs))
        nc = torch.exp(lnc)
        lnn = torch.log(n_grid).view(1, -1).expand(B, -1).contiguous()
        muc = ns.interp_rows(lnn, mu.contiguous(), torch.log(nc)); pcn = ns.interp_rows(lnn, p.contiguous(), torch.log(nc))
        ecn = ns.interp_rows(lnn, eps.contiguous(), torch.log(nc))
        out["pqcd"] = ns3.pqcd_logweight(muc, nc, pcn)
        out["Dc"] = 1.0 / 3.0 - pcn / ecn
        out["Mmax"] = Mmax
        out["valid"] = valid.to(dt)
        out["n_last"] = nl
        out["R14"] = branch_at(M, R, st, 1.4)
        out["L14"] = branch_at(M, L, st, 1.4)
        out["I14"] = branch_at(M, I, st, 1.4)
        for m0 in GLITCH_MASSES:
            out[f"f{int(round(10 * m0))}"] = branch_at(M, dI / I, st, m0)
        out["dev_tab"] = dtab
        return {k: v.detach().cpu().numpy() for k, v in out.items()}


    def refine_max(M, st, ok, x):
        """Maximum mass on the stable branch with a 3-point parabola in x = ln n_c around the discrete maximum
        (removes the grid noise that would otherwise swamp dM_max/dalpha). Rows whose sequence ends at a wall keep the
        end point. Returns M_max (B,) and x at the maximum (B,)."""
        B, Nc = M.shape
        Mw = torch.where(st, M, torch.zeros_like(M))
        im = torch.argmax(Mw, 1)
        i0 = (im - 1).clamp(min=0); i2 = (im + 1).clamp(max=Nc - 1)
        g = lambda T, i: torch.gather(T, 1, i.view(-1, 1))[:, 0]
        y0, y1, y2 = g(M, i0), g(M, im), g(M, i2)
        x0, x1, x2 = g(x, i0), g(x, im), g(x, i2)
        inner = (im > 0) & (im < Nc - 1) & g(ok, i2) & g(st, i0)
        d2 = y0 - 2 * y1 + y2
        use = inner & (d2 < 0) & torch.isfinite(y0 + y1 + y2)
        h = 0.5 * (x2 - x0)
        dl = (h * (y0 - y2) / (2 * torch.where(use, d2, -torch.ones_like(d2)))).clamp(-1, 1) * 1.0
        dl = torch.maximum(torch.minimum(dl, h), -h)
        yv = y1 - (y0 - y2) ** 2 / (8 * torch.where(use, d2, -torch.ones_like(d2)))
        Mmax = torch.where(use, torch.maximum(yv, y1), Mw.max(1).values)
        xm = torch.where(use, x1 + dl, x1)
        return Mmax, xm


    def glitch_logweight(D, G):
        """log of the fraction of Vela masses (GLITCH_MASSES, flat) whose crust carries dI_c/I >= G."""
        F = np.stack([np.nan_to_num(D[f"f{int(round(10 * m))}"], nan=-1.0) for m in GLITCH_MASSES], 1)
        fr = (F >= G).mean(1)
        return np.where(fr > 0, np.log(np.maximum(fr, 1e-300)), -1e30)


    def point_file(q, b):
        """Checkpoint name keyed by the coupling values themselves (safe if the grid is edited later)."""
        return f"a{float(q[0]):+08.3f}_b{float(q[1]):+08.3f}_batch{int(b):05d}.npz"
    return types.SimpleNamespace(**{_k: _v for _k, _v in locals().items() if not _k.startswith("__")})

NS7 = _build_ns7()
_alphas = [round(float(_x), 3) for _x in np.linspace(-12.0, 3.0, 7 if SMOKE else 31)] + [-0.5, 0.0, 0.5, 2.25, 2.4]
_alphas = sorted(set(_alphas))
_betas = [-8.0, -4.0, -2.0, -1.0, -0.5, 0.5, 1.0, 2.0, 2.5, 3.0]
P7CFG = {
    "device": CFG["device"],
    "seed": 161803,
    "n_eos": 200 if SMOKE else 20_000,        # physical EOS (GP prior), shared by every coupling point
    "batch": 100 if SMOKE else 4000,
    "n_central": 20 if SMOKE else 36,
    "n_h": 120 if SMOKE else 240,
    "n_crust": 40 if SMOKE else 80,
    "n_min_c": 1.0 * NS.N0,
    "n_max_c": 11.5 * NS.N0,
    "n_t": 0.5 * NS.N0,                        # crust-core join of this notebook's EOS (cell 23)
    "points": [(_a, 0.0) for _a in _alphas] + [(0.0, _b) for _b in (_betas[::3] if SMOKE else _betas)],
    "ckpt_dir": os.path.join(CFG["ckpt_dir"], "part7"),
}
P7CFG.update(json.loads(os.environ.get("P7_OVERRIDES", "{}")))
P7CFG["points"] = [tuple(_q) for _q in P7CFG["points"]]
os.makedirs(P7CFG["ckpt_dir"], exist_ok=True)

In [ ]:
# ---- THEOREMS 10-11, NUMERICALLY: limits, exact identities, leading-order law, existence domain ----
_g = torch.Generator().manual_seed(7)
_e = 10 ** (1 + 2.3 * torch.rand(4000, generator=_g, dtype=torch.float64))         # 10 .. 2000 MeV/fm^3
_w = 0.6 * torch.rand(4000, generator=_g, dtype=torch.float64)
_pp = _w * _e
_z = torch.tensor(0.0, dtype=torch.float64)
THM11 = {}
_et, _pt, _ok, _ = NS7.effective(_e, _pp, _z, _z)
THM11["GR limit (alpha=beta=0): max rel. error"] = float(torch.maximum((_et / _e - 1).abs(), (_pt / _pp - 1).abs()).max())
_a = torch.tensor(2.0, dtype=torch.float64)
_et, _pt, _ok, _ = NS7.effective(_e, _pp, _a, _z)
_k = NS7.KAP * NS7.KM2
_R = _k * (_e - 3 * _pp); _F = 1 + 2 * 2.0 * _R
_ex = _e / _F**2 + 2.0 * _R**2 / (2 * _k * _F**2)
_px = _pp / _F**2 - 2.0 * _R**2 / (2 * _k * _F**2)
_m = _ok
THM11["f(R) limit vs closed form (alpha=2 km^2)"] = float(torch.maximum((_et / _ex - 1).abs(), (_pt / _px - 1).abs())[_m].max())
THM11["Theorem 11(i): max |T~ F / T - 1|"] = float(((-_et + 3 * _pt) * _F / (-_e + 3 * _pp) - 1).abs()[_m & ((_e - 3 * _pp).abs() > 1)].max())
_et, _pt, _ok, _ = NS7.effective(_e, _e / 3, torch.tensor(7.0, dtype=torch.float64), _z)
THM11["conformal matter, alpha=7 km^2: max |p~/p - 1|"] = float((_pt / (_e / 3) - 1).abs()[_ok].max())
# leading-order law: residual must scale as coupling^2
_sub = slice(0, 400)
for _name, (_aa, _bb) in {"alpha": (0.2, 0.0), "beta": (0.0, 0.2), "mixed": (0.15, -0.1)}.items():
    _res = []
    for _s in (1.0, 0.5, 0.25):
        _et, _pt, _ok, _ = NS7.effective(_e[_sub], _pp[_sub], torch.tensor(_aa * _s, dtype=torch.float64), torch.tensor(_bb * _s, dtype=torch.float64))
        _e1, _p1 = NS7.first_order(_e[_sub], _pp[_sub], _aa * _s, _bb * _s)
        _res.append(float(((_pt - _p1).abs() / _pp[_sub].clamp(min=1.0))[_ok].max()))
    THM11[f"leading-order law ({_name}): residual exponent"] = float(np.log2(_res[1] / _res[2]))
# Ricci-squared sees conformal matter: Delta~ = (8/9) beta kappa eps at p = eps/3
_ec = torch.tensor([200.0, 600.0, 1200.0], dtype=torch.float64)
_bb = torch.tensor(0.01, dtype=torch.float64)
_et, _pt, _ok, _ = NS7.effective(_ec, _ec / 3, _z, _bb)
_D = 1 / 3 - _pt / _et
THM11["beta at conformal point: max |Delta~ / ((8/9) beta kappa eps) - 1|"] = float((_D / (8 / 9 * 0.01 * _k * _ec) - 1).abs().max())
# existence domain of the GR-connected branch (scale-free: kappa eps = 1)
_ws = [0.0, 0.02, 0.05, 0.1, 0.2, 1 / 3, 0.45, 0.6, 0.8, 1.0]
EXIST = NS7.existence_scan(_ws, bmax=3.0, nb=1500)
_fig, _ax = plt.subplots(figsize=(6.8, 4.2))
_ax.plot(_ws, [EXIST[float(_x)]["+"] for _x in _ws], "o-", color="C3", label="β > 0: maximal β κ ε")
_ax.plot(_ws, [EXIST[float(_x)]["-"] for _x in _ws], "s-", color="C0", label="β < 0: maximal |β| κ ε (3 = scan edge)")
_ax.axvline(1 / 3, color="0.5", ls=":", lw=1)
_ax.set(yscale="log", xlabel="p / ε", ylabel="|β| κ ε at which the solution ends", title="Ricci-squared density ceiling (exact)")
_ax.legend(fontsize=8); _fig.tight_layout(); exist_fig = _fig; plt.close(_fig)
_exp = {"residual exponent": "≈ 2", "conformal point": "O(βκε) ≈ 0.01"}
_rows = "\n".join(f"| {_kk} | {_vv:.3g} | " + next((_t for _s, _t in _exp.items() if _s in _kk), "round-off") + " |" for _kk, _vv in THM11.items())
mo.vstack([mo.md(f"""
**Theorems 10–11, checked on 4000 random states** (10–2000 MeV/fm³, 0 ≤ p/ε ≤ 0.6):

| check | value | expected |
|---|---|---|
{_rows}

- Limits and identities should sit at round-off (~1e-15).
- The residual exponents should be ≈ 2 (the leading-order law is exact to first order).
- The conformal-point ratio should be small (it is $O(\\beta\\kappa\\varepsilon)$).

**Ricci-squared ceiling.** For β > 0 the GR-connected solution ends at
βκε = {EXIST[1/3]['+']:.3f} for conformal matter and {EXIST[1.0]['+']:.3f} for stiff matter (p = ε).
For β < 0 the ceiling is far higher. Matter denser than the ceiling has no static configuration at all.
"""), exist_fig])

In [ ]:
# ---- THEOREM 12 (crust gradient theorem) + independent validation of the inertia solver ---------
_k = NS7.KAP * NS7.KM2
_Es, _Ps = NS7.sly_table()
_nC, _eC, _pC = NS.crust_table(0.5 * NS.N0)
CRUST = {}
for _name, (_E, _P) in {"SLy unified (Douchin-Haensel)": (_Es[_Es < 150], _Ps[_Es < 150]),
                         "this notebook's crust (cell 23)": (_eC, _pC)}.items():
    _l2, _emin, _c2 = NS7.crust_length2(_E, _P)
    CRUST[_name] = {"l2_crust_LO": _l2, "eps_at_min": _emin, "cs2_at_min": _c2,
                    "exact_alpha": NS7.crust_threshold(_E, _P, (1.0, 0.0)),
                    "exact_beta": NS7.crust_threshold(_E, _P, (0.0, 1.0)),
                    "exact_alpha_eq_beta_sum": NS7.crust_threshold(_E, _P, (0.5, 0.5))}
_pt_sly = float(np.interp(72.0, _Es, _Ps))
CRUST["SLy unified (Douchin-Haensel)"]["l_t2"] = 2 * _pt_sly / (_k * 72.0**2)
CRUST["this notebook's crust (cell 23)"]["l_t2"] = 2 * float(_pC[-1]) / (_k * float(_eC[-1]) ** 2)
L_T2 = CRUST["this notebook's crust (cell 23)"]["l_t2"]
ALPHA_CRUST = CRUST["this notebook's crust (cell 23)"]["exact_alpha"]
# figure: c_s^2/(kappa eps) through the crust
_fig, _ax = plt.subplots(figsize=(7, 4.2))
for _name, (_E, _P) in {"SLy unified": (_Es[_Es < 400], _Ps[_Es < 400]), "notebook crust": (_eC, _pC)}.items():
    _c2 = _P / _E * np.gradient(np.log(_P), np.log(_E))
    _ax.plot(_E, _c2 / (_k * _E), lw=1.6, label=_name)
_ax.axhline(CRUST["SLy unified (Douchin-Haensel)"]["exact_alpha"], color="C0", ls="--", lw=1)
_ax.axhline(ALPHA_CRUST, color="C1", ls="--", lw=1)
_ax.set(xscale="log", yscale="log", xlim=(1e-2, 400), ylim=(0.5, 1e4), xlabel="ε [MeV/fm³]",
        ylabel="c_s² / (κ ε)  [km²]", title="Theorem 12: static crusts need α+β below the minimum")
_ax.legend(fontsize=8); _fig.tight_layout(); crust_fig = _fig; plt.close(_fig)
# independent validation: batched enthalpy solver (torch) vs radius integration (scipy) on SLy
_E, _P, _n, _mu = NS7.sly_grid()
_eps = torch.tensor(_E).view(1, -1); _pp = torch.tensor(_P).view(1, -1); _ng = torch.tensor(_n)
_cs2 = torch.gradient(_pp, spacing=(_ng,), dim=1)[0] / torch.gradient(_eps, spacing=(_ng,), dim=1)[0]
_nt = float(np.interp(72.0, _E, _n))
VALID7 = []
for _a in (0.0, -4.0, 2.0):
    _et, _pt, _c2t, _mut, _nl, _ = NS7.rq_tables(_eps, _pp, _cs2, torch.tensor(_mu).view(1, -1), _ng,
                                                 torch.tensor([_a]), torch.tensor([0.0]))
    _ncs = torch.logspace(np.log10(0.2), np.log10(min(float(_nl[0]), 1.5)), 40, dtype=torch.float64).view(1, -1)
    _M, _R, _L, _I, _dI, _It, _ec = NS7.tov_rq(NS, _et, _pt, _c2t, _mut, _ng, _ncs, _nt, n_h=400, n_crust=120)
    _st = NS.stable_mask(_M)
    _f14 = float(NS7.branch_at(_M, _dI / _I, _st, 1.4)); _R14 = float(NS7.branch_at(_M, _R, _st, 1.4))
    # scipy reference
    _ek = (_et[0] * NS7.KM2).numpy(); _pk = (_pt[0] * NS7.KM2).numpy()
    _good = np.concatenate([[True], (np.diff(_pk) > 0) & (np.diff(_ek) > 0)]); _kk = int(np.argmin(_good)) if not _good.all() else len(_good)
    _ek, _pk = _ek[:_kk], _pk[:_kk]
    _ptt = float(np.interp(_nt, _n[:_kk], _pk))
    _rs = [NS7.hartle_scipy(_ek, _pk, _pc, _ptt) for _pc in np.logspace(np.log10(20 * NS7.KM2), np.log10(_pk[-1]), 36)]
    _Ms = np.array([_x[0] for _x in _rs]); _i = int(np.argmax(_Ms))
    _Rs = np.array([_x[1] for _x in _rs])[: _i + 1]; _fs = np.array([_x[3] / _x[2] for _x in _rs])[: _i + 1]
    VALID7.append({"alpha": _a, "Mmax_torch": float(_M[_st].max()), "Mmax_scipy": float(_Ms.max()),
                   "R14_torch": _R14, "R14_scipy": float(np.interp(1.4, _Ms[: _i + 1], _Rs)),
                   "f14_torch": _f14, "f14_scipy": float(np.interp(1.4, _Ms[: _i + 1], _fs)),
                   "I_check": float(((_It / _I) - 1).abs().max())})
_r1 = "\n".join(f"| {_kk} | {_v['l2_crust_LO']:.3f} | {_v['eps_at_min']:.0f} | {_v['exact_alpha']:.3f} | {_v['exact_beta']:.3f} | {_v['exact_alpha_eq_beta_sum']:.3f} | {_v['l_t2']:.3f} |" for _kk, _v in CRUST.items())
_r2 = "\n".join(f"| {_v['alpha']:+.0f} | {_v['Mmax_torch']:.3f} / {_v['Mmax_scipy']:.3f} | {_v['R14_torch']:.2f} / {_v['R14_scipy']:.2f} | {100 * _v['f14_torch']:.2f} / {100 * _v['f14_scipy']:.2f} | {_v['I_check']:.1e} |" for _v in VALID7)
mo.vstack([mo.md(f"""
**Theorem 12, computed.** Static crusts exist only below these couplings (km²):

| crust EOS | leading order min c²/(κε) | at ε [MeV/fm³] | exact α (β=0) | exact β (α=0) | exact α+β (α=β) | ℓ_t² = 2p_t/(κε_t²) |
|---|---|---|---|---|---|---|
{_r1}

- The three exact columns agree, so the bound is on **α+β**, as the leading-order law says.
- The minimum sits at the crust–core transition, not at the surface.
- With the particle-physics bound on β, this becomes **α < {ALPHA_CRUST:.2f} km²** for this notebook's crust and
  < {CRUST['SLy unified (Douchin-Haensel)']['exact_alpha']:.2f} km² for SLy.

**Solver validation on SLy** (batched enthalpy solver / independent scipy radius integration):

| α [km²] | M_max [M☉] | R(1.4) [km] | ΔI_c/I(1.4) [%] | I: J/Ω vs ∫ |
|---|---|---|---|---|
{_r2}
"""), crust_fig])

In [ ]:
p7_btn = mo.ui.run_button(label="Run Part VII (α, β) scan")
mo.vstack([mo.md(f"**Part VII scan:** {P7CFG['n_eos']:,} physical EOS (GP prior) × {len(P7CFG['points'])} coupling points "
                 f"(batch {P7CFG['batch']}). Resumable; `RUN_P7=1` runs it headlessly."), p7_btn])

In [ ]:
# ---- HEAVY 5: common physical EOS draws, every (alpha, beta) point, exact map + crust inertia ------
mo.stop(not (p7_btn.value or os.environ.get("RUN_P7") == "1"), mo.md("*Press the Part VII button to start.*"))
_dev = P7CFG["device"]
_crust = NS.crust_table(0.5 * NS.N0)
_ng = NS.make_n_grid(device=_dev)
_nodes = torch.as_tensor(NS4.GP_NODES_N0 * NS.N0, device=_dev, dtype=torch.float64)
_nb = -(-P7CFG["n_eos"] // P7CFG["batch"])
_t0 = time.time()
_done = 0
for _b in mo.status.progress_bar(range(_nb), title="Part VII: exact RBG stars"):
    _todo = [(_k, _q) for _k, _q in enumerate(P7CFG["points"])
             if not os.path.exists(os.path.join(P7CFG["ckpt_dir"], NS7.point_file(_q, _b)))]
    if not _todo:
        continue
    _g = torch.Generator(device=_dev).manual_seed(P7CFG["seed"] + _b)      # same EOS for every coupling point
    _s, _cs, _raw = NS4.gp_draw(P7CFG["batch"], _g, _dev)
    _eps, _p, _cs2, _mu = NS4.build_eos_nodes(NS, _s, _cs, _nodes, _ng, _crust)
    for _k, (_a, _bb) in _todo:
        _r = NS7.scan_point(NS, NS3, _eps, _p, _cs2, _mu, _ng, _a, _bb, P7CFG, KDES)
        _r["alpha"] = np.array(_a); _r["beta"] = np.array(_bb)
        _f = os.path.join(P7CFG["ckpt_dir"], NS7.point_file((_a, _bb), _b))
        np.savez(_f + ".tmp.npz", **_r); os.replace(_f + ".tmp.npz", _f)
        _done += 1
    del _eps, _p, _cs2, _mu
    gpu_free()
P7_READY = True
mo.md(f"**Part VII scan done:** {_done} new point-batches in {(time.time() - _t0) / 60:.1f} min.")

In [ ]:
# ---- POSTERIOR ON (alpha, beta), GLITCH TEST, PRE-REGISTERED TESTS P1-P4 -------------------------
_nb = -(-P7CFG["n_eos"] // P7CFG["batch"])
P7_DATA = {}
for _k, _q in enumerate(P7CFG["points"]):
    _fs = [os.path.join(P7CFG["ckpt_dir"], NS7.point_file(_q, _b)) for _b in range(_nb)]
    _fs = [_f for _f in _fs if os.path.exists(_f)]
    if not _fs:
        continue
    _zz = [dict(np.load(_f)) for _f in _fs]
    P7_DATA[_q] = {_kk: np.concatenate([np.atleast_1d(_z[_kk]) for _z in _zz]) for _kk in _zz[0] if _kk not in ("alpha", "beta")}
_G16, _G70 = 0.016, 0.07
def _variants(D, g16=0.016, g70=0.07):
    a = D["ll"]; q = a + D["pqcd"]
    return {"astro": a, "astro+pQCD": q,
            "astro+pQCD+Vela 1.6%": q + NS7.glitch_logweight(D, g16),
            "astro+pQCD+Vela 7% (entrainment)": q + NS7.glitch_logweight(D, g70),
            "astro(+J0952)+pQCD": D["ll_bw"] + D["pqcd"]}
def _logZ(x):
    x = np.where(x < -1e29, -np.inf, x)
    m = np.max(x)
    return float(m + np.log(np.mean(np.exp(x - m)))) if np.isfinite(m) else -np.inf
_apts = sorted([_q for _q in P7_DATA if _q[1] == 0.0], key=lambda q: q[0])
_bpts = sorted([_q for _q in P7_DATA if _q[0] == 0.0], key=lambda q: q[1])
mo.stop(not _apts, mo.md("**No Part VII α-scan data found (β = 0).** Run the scan first — press the Part VII button or set `RUN_P7=1`."))
_alpha = np.array([_q[0] for _q in _apts]); _beta = np.array([_q[1] for _q in _bpts])
_names = list(_variants(P7_DATA[_apts[0]]).keys())
LOGZ_A = {_v: np.array([_logZ(_variants(P7_DATA[_q])[_v]) for _q in _apts]) for _v in _names}
LOGZ_B = {_v: np.array([_logZ(_variants(P7_DATA[_q])[_v]) for _q in _bpts]) for _v in _names}
def _interval(x, lz, lev=0.95):
    w = np.exp(lz - np.max(lz)); w = np.where(np.isfinite(w), w, 0.0)
    c = np.concatenate([[0], np.cumsum(0.5 * (w[1:] + w[:-1]) * np.diff(x))]); c = c / c[-1]
    return float(np.interp((1 - lev) / 2, c, x)), float(np.interp(1 - (1 - lev) / 2, c, x)), float(x[np.argmax(lz)])
P7_POST = {_v: dict(zip(("lo95", "hi95", "map"), _interval(_alpha, LOGZ_A[_v]))) for _v in _names}
_i0 = int(np.argmin(np.abs(_alpha)))
for _v in _names:
    P7_POST[_v]["log10B_map_vs_GR"] = float((LOGZ_A[_v].max() - LOGZ_A[_v][_i0]) / np.log(10))
# glitch: posterior-predictive probability that Vela's requirement can be met, vs alpha (weights astro+pQCD)
P7_GLITCH = {"alpha": _alpha, "P_ok_1.6": [], "P_ok_7": [], "f14_med": [], "f14_lo": [], "f14_hi": []}
for _q in _apts:
    D = P7_DATA[_q]; _l = D["ll"] + D["pqcd"]; _l = np.where(_l < -1e29, -np.inf, _l)
    _w = np.exp(_l - np.max(_l)) if np.isfinite(np.max(_l)) else np.zeros_like(_l)
    _w = np.nan_to_num(_w); _s = _w.sum()
    for _G, _key in ((_G16, "P_ok_1.6"), (_G70, "P_ok_7")):
        _ok = np.exp(NS7.glitch_logweight(D, _G)); _ok = np.where(_ok > 1e-200, _ok, 0.0)
        P7_GLITCH[_key].append(float((_w * _ok).sum() / _s) if _s > 0 else float("nan"))
    _f = D["f14"]; _m = np.isfinite(_f) & (_w > 0)
    if _m.sum() > 10:
        _o = np.argsort(_f[_m]); _cw = np.cumsum(_w[_m][_o]); _cw /= _cw[-1]
        P7_GLITCH["f14_med"].append(float(np.interp(0.5, _cw, _f[_m][_o])))
        P7_GLITCH["f14_lo"].append(float(np.interp(0.16, _cw, _f[_m][_o]))); P7_GLITCH["f14_hi"].append(float(np.interp(0.84, _cw, _f[_m][_o])))
    else:
        for _kk in ("f14_med", "f14_lo", "f14_hi"):
            P7_GLITCH[_kk].append(float("nan"))
# ---- pre-registered tests ----------------------------------------------------------------------
P7_TESTS = {}
_val = np.array([P7_DATA[_q]["valid"].mean() for _q in _apts])
_vb = np.array([P7_DATA[_q]["valid"].mean() for _q in _bpts]) if _bpts else np.array([])
_p1 = bool(np.all(_val[_alpha > ALPHA_CRUST + 1e-9] == 0) and np.all(_val[_alpha < ALPHA_CRUST - 0.05] == 1)
           and np.all(_vb[_beta > ALPHA_CRUST + 0.05] == 0)) if len(_alpha) > 0 else False
P7_TESTS["P1 crust theorem (deterministic)"] = {"pass": _p1, "valid_fraction_alpha": dict(zip(_alpha.tolist(), _val.tolist())),
                                                 "valid_fraction_beta": dict(zip(_beta.tolist(), _vb.tolist()))}
_qm, _qp, _q0 = (-0.5, 0.0), (0.5, 0.0), (0.0, 0.0)
if all(_q in P7_DATA for _q in (_qm, _qp, _q0)):
    _fm, _fp = P7_DATA[_qm]["f14"], P7_DATA[_qp]["f14"]
    _slope = (np.log(_fp) - np.log(_fm)) / 1.0
    _ok = np.isfinite(_slope)
    _med = float(np.median(_slope[_ok])); _iqr = float(np.subtract(*np.quantile(_slope[_ok], [0.75, 0.25])))
    P7_TESTS["P2 glitch lever"] = {"pass": bool(abs(_med * L_T2 + 1) < 0.15 and _iqr < 0.2 * abs(_med)),
                                   "median_dlnf_dalpha": _med, "predicted": -1 / L_T2, "IQR": _iqr}
    _Mm, _Mp = P7_DATA[_qm]["Mmax"], P7_DATA[_qp]["Mmax"]
    _dM = (_Mp - _Mm) / 1.0
    _ok2 = np.isfinite(_dM) & (P7_DATA[_q0]["valid"] > 0) & (_Mm > 1.0)
    _win = (_alpha >= -4.0) & (_alpha <= 2.0)
    _lb = (LOGZ_A["astro+pQCD"][_win] - LOGZ_A["astro+pQCD"][_i0]) / np.log(10)
    P7_TESTS["P3 degeneracy of core data (Theorem 10)"] = {"pass": bool(np.max(np.abs(_lb)) < 0.5),
                                                           "max_abs_log10B_in_[-4,2]": float(np.max(np.abs(_lb))),
                                                           "median_abs_dMmax_dalpha": float(np.median(np.abs(_dM[_ok2])))}
    _rat = np.abs(_slope) / np.abs(_dM / P7_DATA[_q0]["Mmax"]).clip(min=1e-12)
    _rm = float(np.median(_rat[_ok & _ok2])) if (_ok & _ok2).any() else float("nan")
    P7_TESTS["P4 crust vs maximum-mass sensitivity"] = {"pass": bool(_rm > 30), "median_ratio": _rm}
_rows = "\n".join(f"| {_v} | {P7_POST[_v]['lo95']:+.2f} | {P7_POST[_v]['hi95']:+.2f} | {P7_POST[_v]['map']:+.2f} | {P7_POST[_v]['log10B_map_vs_GR']:.2f} |" for _v in _names)
_rt = "\n".join(f"| {_kk} | {'PASS' if _vv['pass'] else 'FAIL'} | " + ", ".join(f"{_a}={_b:.3g}" for _a, _b in _vv.items() if isinstance(_b, float)) + " |" for _kk, _vv in P7_TESTS.items())
mo.md(f"""
**Posterior on α (β = 0), flat prior on [{_alpha.min():.0f}, {_alpha.max():.0f}] km², {len(P7_DATA[_apts[0]]['ll']):,} common EOS per point:**

| likelihood | 95% lower [km²] | 95% upper [km²] | MAP | log₁₀ B(MAP vs GR) |
|---|---|---|---|---|
{_rows}

- Published reference interval (arXiv:2306.00870): −6.47 ≲ α ≲ 1.99 km².
- Crust theorem edge (cell 58): α < {ALPHA_CRUST:.2f} km².
- The Vela-7% row assumes strong entrainment. Its large Bayes factor only says that α = 0 cannot supply 7% with this crust;
  it is not evidence for modified gravity on its own.

**Pre-registered tests:**

| test | result | numbers |
|---|---|---|
{_rt}
""")

In [ ]:
# ---- PART VII: FIGURE, VERDICT, EXPORT -----------------------------------------------------------
_al = np.asarray(P7_GLITCH["alpha"])
_fig, _ax = plt.subplots(1, 3, figsize=(16, 4.6))
for _v, _lz in LOGZ_A.items():
    _w = np.exp(_lz - np.max(_lz)); _w = np.nan_to_num(_w)
    _ax[0].plot(_al, _w / max(np.trapezoid(_w, _al), 1e-300), lw=1.6, label=_v)
_ax[0].axvspan(-6.47, 1.99, color="0.9", zorder=0, label="published interval (2306.00870)")
_ax[0].axvline(ALPHA_CRUST, color="k", ls="--", lw=1, label="crust theorem")
_ax[0].set(xlabel="α [km²]", ylabel="posterior density", title="R² Palatini coupling from neutron stars")
_ax[0].legend(fontsize=7)
_m = np.isfinite(np.asarray(P7_GLITCH["f14_med"]))
_ax[1].fill_between(_al[_m], 100 * np.asarray(P7_GLITCH["f14_lo"])[_m], 100 * np.asarray(P7_GLITCH["f14_hi"])[_m], color="C0", alpha=0.25, label="68% (posterior EOS)")
_ax[1].plot(_al[_m], 100 * np.asarray(P7_GLITCH["f14_med"])[_m], color="C0", lw=1.8, label="median")
_ax[1].axhline(1.6, color="C2", ls="--", lw=1, label="Vela, no entrainment")
_ax[1].axhline(7.0, color="C3", ls="--", lw=1, label="Vela, strong entrainment")
_ax[1].axvline(0, color="0.5", lw=0.8)
_ax[1].set(xlabel="α [km²]", ylabel="ΔI_crust / I at 1.4 M☉ [%]", title="Theorem 13: the glitch lever", yscale="log")
_ax[1].legend(fontsize=7)
_q0, _qm, _qp = (0.0, 0.0), (-0.5, 0.0), (0.5, 0.0)
if all(_q in P7_DATA for _q in (_q0, _qm, _qp)):
    _dM = P7_DATA[_qp]["Mmax"] - P7_DATA[_qm]["Mmax"]; _Dc = P7_DATA[_q0]["Dc"]
    _ok = np.isfinite(_dM) & np.isfinite(_Dc) & (P7_DATA[_qm]["Mmax"] > 1.0)
    _ii = np.where(_ok)[0][:6000]
    _ax[2].scatter(_Dc[_ii], _dM[_ii], s=2, alpha=0.3, color="C4")
    _ax[2].axhline(0, color="k", lw=0.8); _ax[2].axvline(0, color="C3", lw=1)
_ax[2].set(xlabel="central trace anomaly Δ_c (GR star at M_max)", ylabel="dM_max/dα [M☉/km²]", title="M_max barely responds to α (non-monotone in Δ_c)")
_fig.tight_layout(); p7_fig = _fig; plt.close(_fig)
# verdict (written from the numbers, nothing hard-coded)
_ref = P7_POST["astro+pQCD"]
_pg = np.asarray(P7_GLITCH["P_ok_7"]); _p16 = np.asarray(P7_GLITCH["P_ok_1.6"])
_i0 = int(np.argmin(np.abs(_al)))
_inwin = (_al >= _ref["lo95"]) & (_al <= _ref["hi95"]) & np.isfinite(_pg)
_best = int(np.nanargmax(np.where(_inwin, _pg, -1))) if _inwin.any() else _i0
_v16 = P7_POST["astro+pQCD+Vela 1.6%"]
_ok7 = np.isfinite(_pg) & (_pg >= 0.5)
_a50 = float(_al[_ok7].max()) if _ok7.any() else float("nan")
_edge = "the crust theorem" if abs(_ref["hi95"] - ALPHA_CRUST) < 0.6 else "the data"
_lines = [f"- **Crust theorem (P1): {'confirmed' if P7_TESTS['P1 crust theorem (deterministic)']['pass'] else 'NOT confirmed'}.** "
          f"Above α+β = {ALPHA_CRUST:.2f} km² none of the EOS supports a star; below it every EOS does."]
_lines.append(f"- **Neutron-star data (astro+pQCD):** α ∈ [{_ref['lo95']:+.2f}, {_ref['hi95']:+.2f}] km² (95%). "
              f"The upper edge sits at {_edge}.")
_lines.append(f"- **Adding Vela without entrainment** (ΔI_c/I ≥ 1.6%) gives α ≤ {_v16['hi95']:+.2f} km² (95%). "
              f"No earlier glitch bound on Palatini gravity turned up in our search.")
if _ok7.any() and (not np.isfinite(_pg[_i0]) or _pg[_i0] < 0.2):
    _lines.append(f"- **Glitch crisis:** in GR (α = 0), P(Vela's 7% is met) = {_pg[_i0]:.2f}. It exceeds 0.5 for "
                  f"α ≤ {_a50:+.1f} km² (best inside the data window: {_pg[_best]:.2f} at α = {_al[_best]:+.1f}). "
                  f"That {'overlaps' if _a50 >= -6.47 else 'lies below'} the published interval. So negative-α Palatini "
                  f"gravity **can supply the entrainment-level crust inertia** that GR lacks. "
                  f"Test: one α must fit every glitching pulsar, while core observables stay degenerate (P3).")
else:
    _lines.append(f"- **Glitch crisis:** P(Vela's 7% is met) = {_pg[_i0]:.2f} in GR and at most {_pg[_best]:.2f} inside the "
                  f"data window. Palatini R² does not change the picture here.")
_lt2nb = CRUST["this notebook's crust (cell 23)"]["l_t2"]; _lt2sly = CRUST["SLy unified (Douchin-Haensel)"]["l_t2"]
_lines.append(f"- **Caveats:** the inertia numbers use this notebook's crust (ℓ_t² = {_lt2nb:.2f} km²). "
              f"For another crust, rescale α by ℓ_t² (SLy: {_lt2sly:.2f} km²). "
              f"The 7% level assumes strong entrainment, which is debated.")
_lines.append("- **Pre-registered:** " + ", ".join(f"{_k.split(' ')[0]} {'PASS' if _v['pass'] else 'FAIL'}" for _k, _v in P7_TESTS.items()))
_out = os.path.join(P7CFG["ckpt_dir"], "PART7_RBG.json")
save_json(_out, {"theorem11_checks": THM11, "crust": CRUST, "solver_validation": VALID7,
                 "posterior_alpha": P7_POST, "tests": {_k: {_a: _b for _a, _b in _v.items() if not isinstance(_b, dict)} for _k, _v in P7_TESTS.items()},
                 "glitch": {_k: list(np.asarray(_v, dtype=float)) for _k, _v in P7_GLITCH.items()},
                 "logZ_alpha": {_k: list(_v) for _k, _v in LOGZ_A.items()}, "logZ_beta": {_k: list(_v) for _k, _v in LOGZ_B.items()}})
p7_fig.savefig(os.path.join(P7CFG["ckpt_dir"], "part7_rbg.png"), dpi=160)
crust_fig.savefig(os.path.join(P7CFG["ckpt_dir"], "part7_crust.png"), dpi=160)
exist_fig.savefig(os.path.join(P7CFG["ckpt_dir"], "part7_existence.png"), dpi=160)
mo.vstack([mo.md("## Part VII verdict\n" + "\n".join(_lines) + f"\n\nSaved `{_out}` and three figures in `{P7CFG['ckpt_dir']}/`."), p7_fig])

---
# Part VIII — Microscopic dressing: why neutron stars cannot see the action, and what can

Parts II–VII treated matter as a fluid. The corrections in $f(R,Q)$ are algebraic in $T_{\mu\nu}$, so they also act on the
*microscopic* matter that fluid is made of. This part follows that through to the end.

## VIII.1 Theorem 14 — the dressing theorem
For $f=R+\alpha R^2$ the Einstein-frame action is
$$S=\frac1{2\kappa}\int\!\sqrt{-q}\,R(q)-\int\!\sqrt{-q}\,U(\phi)+S_m\big[q/\phi,\psi\big],\qquad U=\frac{(\phi-1)^2}{8\alpha\kappa\phi^2}.$$
$\phi$ has no kinetic term. Its equation is algebraic, $\phi_*=1-2\alpha\kappa T+8\alpha^2\kappa^2T^2+\dots$ (this reproduces
$F=1+2\alpha R$ with $R=-\kappa T$). Integrating it out gives, **exactly to first order**,
$$S_{\rm eff}=S_{\rm EH}[q]+S_m[q,\psi]+\int\!\sqrt{-q}\;\frac{\alpha\kappa}{2}\,T^2 .$$
For slow matter $T=-\sum_i m_i\bar\psi_i\psi_i$, so **every pair of massive particles acquires a contact interaction**
$$V_{ij}(r)=-\alpha\kappa\,m_im_j\,\delta^3(\mathbf r).$$
- **(a) Nuclear matter absorbs it.** For nucleons the strength is $\alpha\kappa m_N^2\approx3.8$ GeV⁻² per km² of $\alpha$, a few
  per cent of the leading chiral contact ($\sim1/f_\pi^2$). It is absorbed into fitted low-energy constants, so every lab-calibrated
  nuclear EOS is already "dressed". Theorem 10's degeneracy therefore extends into nuclear physics.
- **(b) Leptons cannot absorb it.** QED fixes the lepton–nucleus interaction, and the contact grows with the lepton mass, so
  lepton universality is broken. Since both act through $|\psi(0)|^2$, the contact shifts every S level exactly like a
  charge-radius change, and it shifts elastic scattering the same way:
  $$\delta\langle r^2\rangle_\ell=-\frac{3\,\alpha\kappa\,m_\ell M}{2\pi Z\alpha_{\rm em}}\ \approx\ -1.094\ {\rm fm^2}\ (\mu),\quad-0.0053\ {\rm fm^2}\ (e)\quad\text{per km}^2.$$
- **(c) Photons are blind** (Theorem 11, since $T_{\rm EM}=0$).

Prior art: a gravity-like $m_\ell m_p$ contact was proposed as an explanation of the proton-radius puzzle in
large-extra-dimension models (arXiv:1303.4885, 1303.5146). The Palatini origin, and its use as a bound on $\alpha$, are new to
our search; this needs auditing.

## VIII.2 Theorem 15 — exact frame invariance of momentum density
For any Ricci-based theory and any matter, the field equation is $\Omega P=\kappa T+\tfrac f2\mathbb 1$. So
$\tilde R^\mu{}_\nu=|\Omega|^{-1/2}(\kappa T^\mu{}_\nu+\tfrac f2\delta^\mu_\nu)$, and its off-diagonal part is
$\kappa|\Omega|^{-1/2}T^\mu{}_\nu$. Hence
$$\sqrt{-q}\,\tilde T^0{}_i=\sqrt{-g}\,T^0{}_i\quad\text{exactly}.$$
The angular momentum of any matter layer is the same in both frames, so the crustal inertia of Part VII is exact, not an
assumption. Cell 64 checks this on a full 4×4 solve.

## VIII.3 What this does to the programme
- **The glitch crisis.** Supplying 7% needs $\alpha\approx-5$ to $-6$ km² for every unified crust (cell 66). Muonic atoms allow
  $|\alpha|\lesssim10^{-2}$ km² (cell 65). **The Part VII glitch resolution is excluded and is withdrawn.**
- **Neutron-star effects of any allowed $\alpha$:** ≲0.3% in $\Delta I_c/I$ and ≲1 m in radius. Neutron stars cannot probe
  Palatini $R^2$.
- **$\beta$:** excluded at the TeV scale by lepton scattering (published).
- **The infrared term of Parts I–VI:** at lab densities $F\approx(\kappa T)^2/9\mu^4\gg1$ (Theorem 5). This is the known atomic
  pathology (Flanagan 2004; Olmo 2008). Those results hold as theorems but describe a sector that is excluded unless new physics
  screens it.

## The measurability principle (the thesis of the paper)
> *A Ricci-based modification of gravity is degenerate with matter wherever matter's couplings are fitted, which includes stars
> and nuclei. It is observable only through couplings that a symmetry fixes: lepton universality ($\alpha$), lepton contact
> operators ($\beta$), and photon conformal invariance (a null). The most sensitive laboratory for metric-affine gravity is
> therefore a muonic atom, not a neutron star.*

**What survives for astrophysics, independent of any modified gravity.** In GR, Theorem 9 is a model-independent detector: any
star with $C>0.3126$ contains superconformal matter.

## Pre-registered checks
- **Q1:** the numerical $\phi_*$ and $L_{\rm eff}$ match Theorem 14, with the residual scaling as $T^3$.
- **Q2:** the Theorem 15 identity holds below 10⁻¹² at non-perturbative couplings.
- **Q3:** the glitch-resolving $\alpha_7\in[-7,-4]$ km² for every crust.
- **Q4:** 95% bound $|\alpha|<0.02$ km² from the radius data.

In [ ]:
# ---- PART VIII CORE (namespace NS8) + numerical proof of Theorems 14 and 15 ---------------------
def _build_ns8():
    # Part VIII core: microscopic dressing of Ricci-based gravity, the lepton-universality bound on alpha,
    # exact frame invariance of angular momentum, and a family of unified crusts.
    import math
    import os
    import urllib.request
    import numpy as np

    # ---- constants (PDG / CODATA) ------------------------------------------------------------------
    M_PL = 1.220890e19          # GeV
    HBARC = 0.1973269804        # GeV fm
    ALPHA_EM = 1 / 137.035999084
    M_P, M_MU, M_E = 0.93827208816, 0.1056583755, 0.51099895e-3     # GeV
    KAPPA_GEV = 8 * math.pi / M_PL**2                                 # GeV^-2
    KM2_GEV = (1e18 / HBARC) ** 2                                     # 1 km^2 in GeV^-2


    def contact_coefficient(m1, m2, alpha_km2=1.0):
        """Theorem 14: L_eff = (alpha kappa / 2) T^2, T = -sum m psi-bar psi  ->  V(r) = -alpha kappa m1 m2 delta^3(r).
        Returns alpha kappa m1 m2 in GeV^-2."""
        return alpha_km2 * KM2_GEV * KAPPA_GEV * m1 * m2


    def dr2_equivalent(m_lepton, M_nucleus=M_P, Z=1, alpha_km2=1.0):
        """A contact -C delta^3(r) shifts every S level exactly like a charge radius change, because both act through
        |psi(0)|^2:  (2 pi / 3) Z alpha_em d<r^2> = -C   ->   d<r^2> = -3 C / (2 pi Z alpha_em)   [fm^2].
        The same Delta r^2 appears in elastic lepton scattering (a Q-independent amplitude mimics the form-factor slope)."""
        C = contact_coefficient(m_lepton, M_nucleus, alpha_km2)
        return -3 * C / (2 * math.pi * Z * ALPHA_EM) * HBARC**2


    # ---- proton-radius data (fm). Electronic values as tabulated in arXiv:2604.15546 (Table 1); muonic: CREMA ----
    RADIUS_ELECTRONIC = [
        ("Beyer 2017 (2S-4P)", 0.8335, 0.0095),
        ("Fleurbaey 2018 (1S-3S)", 0.877, 0.013),
        ("Bezginov 2019 (2S-2P)", 0.833, 0.010),
        ("Grinin 2020 (1S-3S)", 0.8482, 0.0038),
        ("Brandt 2022 (2S-8D)", 0.8584, 0.0051),
        ("Maisenbacher 2026 (2S-6P)", 0.8406, 0.0015),
        ("Bullis 2026 (2S-nS)", 0.8433, 0.0031),
    ]
    RADIUS_SCATTERING_E = [("PRad 2019 (e-p)", 0.831, 0.014)]
    RADIUS_MUONIC = ("CREMA muonic hydrogen (Antognini 2013)", 0.84087, 0.00039)


    def weighted_mean(rows, birge=True):
        v = np.array([r[1] for r in rows]); s = np.array([r[2] for r in rows])
        w = 1 / s**2
        m = float((w * v).sum() / w.sum()); se = float(w.sum() ** -0.5)
        chi2 = float(((v - m) ** 2 * w).sum()); dof = len(v) - 1
        B = max(1.0, math.sqrt(chi2 / dof)) if (birge and dof > 0) else 1.0
        return m, se * B, chi2, dof, B


    def alpha_from_radii(rows_e, muonic=RADIUS_MUONIC):
        """r_e^2 - r_mu^2 = dr2(e) - dr2(mu) per km^2 of alpha. Returns (mean, sigma) of alpha [km^2] and inputs."""
        me, se, chi2, dof, B = weighted_mean(rows_e)
        rm, sm = muonic[1], muonic[2]
        d = me**2 - rm**2
        sd = math.sqrt((2 * me * se) ** 2 + (2 * rm * sm) ** 2)
        k = dr2_equivalent(M_E) - dr2_equivalent(M_MU)          # fm^2 per km^2 (positive)
        return d / k, sd / k, {"r_e": me, "sig_e": se, "birge": B, "chi2": chi2, "dof": dof, "dr2": d, "sig_dr2": sd, "lever_fm2_per_km2": k}


    # ---- exact frame invariance of the angular-momentum density (Theorem 15), full 4x4 check ----------------
    def rbg_matrix_solve(T, a, b, kap=1.0, iters=60):
        """Solve (F 1 + 2 b P) P - f/2 1 = kap T for a general 4x4 mixed tensor P (Newton, numerical Jacobian),
        F = 1 + 2 a tr P, f = R + a R^2 + b tr(P P). Starts from the GR solution P = kap (T - T/2 1)."""
        I = np.eye(4)
        P = kap * (T - 0.5 * np.trace(T) * I)

        def E(Pv):
            Pm = Pv.reshape(4, 4); R = np.trace(Pm); F = 1 + 2 * a * R
            f = R + a * R * R + b * np.trace(Pm @ Pm)
            return ((F * I + 2 * b * Pm) @ Pm - 0.5 * f * I - kap * T).ravel()
        x = P.ravel().copy()
        for _ in range(iters):
            e0 = E(x)
            J = np.empty((16, 16)); h = 1e-7
            for j in range(16):
                dx = np.zeros(16); dx[j] = h
                J[:, j] = (E(x + dx) - E(x - dx)) / (2 * h)
            x = x - np.linalg.solve(J, e0)
        return x.reshape(4, 4), float(np.abs(E(x)).max())


    def frame_invariance_check(eps=1.0, p=0.25, v=0.6, a=0.1, b=-0.08, kap=1.0):
        """Boosted perfect fluid in a local Minkowski frame. Returns the solver residual, |[P,T]|, the misalignment of the
        Einstein-frame 4-velocity, and |sqrt|Omega| T~^0_1 - T^0_1| (exact invariance of the momentum density)."""
        eta = np.diag([-1.0, 1, 1, 1])
        g = 1 / math.sqrt(1 - v * v)
        u_up = np.array([g, g * v, 0, 0]); u_dn = eta @ u_up
        T = (eps + p) * np.outer(u_up, u_dn) + p * np.eye(4)
        P, res = rbg_matrix_solve(T, a, b, kap)
        R = np.trace(P); Om = (1 + 2 * a * R) * np.eye(4) + 2 * b * P
        dO = np.linalg.det(Om)
        Rt = Om @ P / math.sqrt(abs(dO))
        Tt = (Rt - 0.5 * np.trace(Rt) * np.eye(4)) / kap
        w = Tt @ u_up
        lam = float(w @ u_dn) / float(u_up @ u_dn)
        mis = float(np.abs(w - lam * u_up).max())
        inv = float(abs(math.sqrt(abs(dO)) * Tt[0, 1] - T[0, 1]))
        return {"residual": res, "commutator": float(np.abs(P @ T - T @ P).max()), "u_misalignment": mis,
                "angular_momentum_identity": inv, "|Omega|": float(dO), "T01": float(T[0, 1])}


    # ---- unified-crust EOS family (LALSuite tables) ----------------------------------------------------------
    LAL_URL = "https://raw.githubusercontent.com/lscsoft/lalsuite-archive/master/lalsimulation/src/LALSimNeutronStarEOS_{}.dat"
    CRUST_FAMILY = {"SLY": "SLy (Douchin-Haensel 2001)", "BSK19": "BSk19 (Potekhin et al. 2013)",
                    "BSK20": "BSk20 (Potekhin et al. 2013)", "BSK21": "BSk21 (Potekhin et al. 2013)",
                    "FPS": "FPS (Lorenz-Ravenhall-Pethick crust)"}


    def lal_table(name, cache_dir="eos_tables"):
        """Download (once) and parse a LALSuite EOS table. Column order differs between files, so the energy density is
        identified as the larger column. Returns eps, p [MeV/fm^3] sorted, strictly increasing, positive."""
        os.makedirs(cache_dir, exist_ok=True)
        f = os.path.join(cache_dir, f"LAL_{name}.dat")
        if not os.path.exists(f):
            urllib.request.urlretrieve(LAL_URL.format(name), f + ".tmp"); os.replace(f + ".tmp", f)
        d = np.loadtxt(f)
        a, b = d[:, 0], d[:, 1]
        e, p = (a, b) if np.median(a / np.where(b > 0, b, np.inf)) > 1 else (b, a)
        m = (e > 0) & (p > 0)
        e, p = e[m] / 1.3234e-12, p[m] / 1.3234e-12
        o = np.argsort(e); e, p = e[o], p[o]
        k = np.concatenate([[True], (np.diff(e) > 0) & (np.diff(p) > 0)])
        return e[k], p[k]


    def grid_from_table(E0, P0, npts=5000):
        """Fine PCHIP grid (log-log) with n from the first law, d ln n = d eps/(eps+p), integrated in ln eps.
        Returns eps, p, n [fm^-3], mu [MeV] (numpy)."""
        from scipy.interpolate import PchipInterpolator
        ip = PchipInterpolator(np.log(E0), np.log(P0))
        le = np.linspace(np.log(E0[0]), np.log(E0[-1]), npts)
        E = np.exp(le); P = np.exp(ip(le))
        g = E / (E + P)
        lnn = np.concatenate([[0.0], np.cumsum(0.5 * (g[1:] + g[:-1]) * np.diff(le))])
        n = (E[0] + P[0]) / 930.4 * np.exp(lnn)
        return E, P, n, (E + P) / n
    return types.SimpleNamespace(**{_k: _v for _k, _v in locals().items() if not _k.startswith("__")})

NS8 = _build_ns8()
from scipy.optimize import brentq as _brentq
# Q1: integrate out phi exactly (numerically) and compare with phi* = 1 - 2 a k T, L_eff = a k T^2 / 2
_rows = []
for _x in (1e-1, 3e-2, 1e-2, 3e-3):                       # x = alpha kappa T  (dimensionless)
    _L = lambda ph, x=_x: -(ph - 1) ** 2 / (8 * ph**2) - (x / 2) * math.log(ph)     # in units alpha kappa = 1
    _dL = lambda ph, x=_x: -(ph - 1) / (4 * ph**3) - x / (2 * ph)
    _ph = _brentq(_dL, 0.2, 5.0)
    _rows.append((_x, _ph, (_ph - (1 - 2 * _x)) / _x**2, _L(_ph) / (_x**2 / 2) - 1))
_sc = math.log(abs(_rows[-2][3]) / abs(_rows[-1][3])) / math.log(_rows[-2][0] / _rows[-1][0])
Q1 = {"pass": bool(abs(_rows[-1][2] - 8) < 0.2 and abs(_sc - 1) < 0.15), "phi2_coeff": _rows[-1][2], "rel_residual_order": _sc + 2}
# Q2: Theorem 15 on boosted fluids at non-perturbative couplings
_cases = [dict(), dict(a=-0.2, b=0.05, v=0.9, p=0.6), dict(a=0.05, b=0.12, v=0.3, p=0.1)]
THM15 = [NS8.frame_invariance_check(**_c) for _c in _cases]
Q2 = {"pass": bool(max(_t["angular_momentum_identity"] / abs(_t["T01"]) for _t in THM15) < 1e-12 and max(_t["u_misalignment"] for _t in THM15) < 1e-12),
      "max_identity_error": max(_t["angular_momentum_identity"] for _t in THM15)}
_r1 = "\n".join(f"| {_x:.0e} | {_p:.8f} | {_c2:.4f} | {_rr:+.2e} |" for _x, _p, _c2, _rr in _rows)
_r2 = "\n".join(f"| {_t['|Omega|']:.3f} | {_t['residual']:.1e} | {_t['commutator']:.1e} | {_t['u_misalignment']:.1e} | {_t['angular_momentum_identity']:.1e} |" for _t in THM15)
mo.md(f"""
**Theorem 14, numerically** (φ integrated out exactly; x = ακT):

| x | φ* | (φ* − 1 + 2x)/x² → 8 | L_eff / (x²/2) − 1 |
|---|---|---|---|
{_r1}

The last column vanishes linearly in x (exponent {Q1['rel_residual_order']:.2f} for the absolute residual, expected 3), so
L_eff = (ακ/2) T² is exact at first order. **Q1 {'PASS' if Q1['pass'] else 'FAIL'}.**

**Theorem 15, full 4×4 solve** (boosted fluid, couplings up to βκε = ±0.12):

| det Ω | field-eq residual | [P, T] | 4-velocity misalignment | √(det Ω) T̃⁰₁ − T⁰₁ |
|---|---|---|---|---|
{_r2}

**Q2 {'PASS' if Q2['pass'] else 'FAIL'}.** Momentum and angular-momentum densities are frame-invariant, and the Einstein-frame fluid
moves with the matter.
""")

In [ ]:
# ---- THE LEPTON-UNIVERSALITY BOUND: electronic vs muonic proton radius ---------------------------
_a, _sa, _info = NS8.alpha_from_radii(NS8.RADIUS_ELECTRONIC)
_a2, _sa2, _info2 = NS8.alpha_from_radii(NS8.RADIUS_ELECTRONIC + NS8.RADIUS_SCATTERING_E)
LAB = {"alpha_mean": _a, "alpha_sigma": _sa, "lo95": _a - 1.96 * _sa, "hi95": _a + 1.96 * _sa, "inputs": _info,
       "with_PRad": {"alpha_mean": _a2, "alpha_sigma": _sa2},
       "dr2_mu_per_km2": NS8.dr2_equivalent(NS8.M_MU), "dr2_e_per_km2": NS8.dr2_equivalent(NS8.M_E)}
_ns = P7_POST["astro+pQCD"]
LAB["gain_vs_neutron_stars"] = (_ns["hi95"] - _ns["lo95"]) / (LAB["hi95"] - LAB["lo95"])
Q4 = {"pass": bool(max(abs(LAB["lo95"]), abs(LAB["hi95"])) < 0.02), "abs95": max(abs(LAB["lo95"]), abs(LAB["hi95"]))}
_fig, _ax = plt.subplots(1, 2, figsize=(13, 4.4))
_names = [r[0] for r in NS8.RADIUS_ELECTRONIC + NS8.RADIUS_SCATTERING_E]
_v = [r[1] for r in NS8.RADIUS_ELECTRONIC + NS8.RADIUS_SCATTERING_E]; _s = [r[2] for r in NS8.RADIUS_ELECTRONIC + NS8.RADIUS_SCATTERING_E]
_y = np.arange(len(_v))
_ax[0].errorbar(_v, _y, xerr=_s, fmt="o", color="C0", ms=4, capsize=2, label="electron (spectroscopy, scattering)")
_ax[0].axvspan(NS8.RADIUS_MUONIC[1] - NS8.RADIUS_MUONIC[2], NS8.RADIUS_MUONIC[1] + NS8.RADIUS_MUONIC[2], color="C3", alpha=0.5, label="muonic hydrogen")
_ax[0].axvspan(_info["r_e"] - _info["sig_e"], _info["r_e"] + _info["sig_e"], color="C0", alpha=0.15, label="electronic mean (Birge-inflated)")
for _al, _ls in ((0.05, ":"), (-0.05, "--")):
    _rm = np.sqrt(_info["r_e"] ** 2 - _al * (NS8.dr2_equivalent(NS8.M_E) - NS8.dr2_equivalent(NS8.M_MU)))
    _ax[0].axvline(_rm, color="k", ls=_ls, lw=1, label=f"muonic radius if α = {_al:+.2f} km²")
_ax[0].set_yticks(_y); _ax[0].set_yticklabels(_names, fontsize=7)
_ax[0].set(xlabel="proton charge radius [fm]", title="lepton universality measures α"); _ax[0].legend(fontsize=7, loc="upper right")
_bars = [("neutron stars (Part VII, astro+pQCD)", _ns["lo95"], _ns["hi95"], "C1"),
         ("published NS interval (2306.00870)", -6.47, 1.99, "0.6"),
         ("muonic vs electronic hydrogen (this work)", LAB["lo95"], LAB["hi95"], "C3")]
for _i, (_n, _lo, _hi, _c) in enumerate(_bars):
    _ax[1].plot([_lo, _hi], [_i, _i], lw=8, color=_c, solid_capstyle="butt")
    _ax[1].text(_hi, _i + 0.22, f"  {_n}", fontsize=8, va="bottom", ha="right" if _hi > 0 else "left")
_ax[1].axvspan(-6.0, -4.9, color="C2", alpha=0.25, label="needed for Vela 7% (all crusts)")
_ax[1].set_xscale("symlog", linthresh=0.01); _ax[1].set_yticks([]); _ax[1].set_ylim(-0.6, 2.9)
_ax[1].set(xlabel="α [km²]  (symlog)", title="95% intervals on the Palatini R² coupling"); _ax[1].legend(fontsize=8, loc="lower left")
_fig.tight_layout(); lab_fig = _fig; plt.close(_fig)
mo.vstack([mo.md(f"""
**Electronic mean** r_p = {_info['r_e']:.5f} ± {_info['sig_e']:.5f} fm. The scatter is χ²/dof = {_info['chi2']:.1f}/{_info['dof']},
so the error is inflated by the Birge ratio {_info['birge']:.2f}. **Muonic:** {NS8.RADIUS_MUONIC[1]:.5f} ± {NS8.RADIUS_MUONIC[2]:.5f} fm.

r_e² − r_μ² = {_info['dr2']:.4f} ± {_info['sig_dr2']:.4f} fm², against a lever of {_info['lever_fm2_per_km2']:.3f} fm² per km²:

**α = ({1e3 * _a:.1f} ± {1e3 * _sa:.1f})×10⁻³ km²**, i.e. {LAB['lo95'] * 1e3:+.1f}×10⁻³ < α < {LAB['hi95'] * 1e3:+.1f}×10⁻³ km² (95%).

Adding PRad scattering: α = ({1e3 * _a2:.1f} ± {1e3 * _sa2:.1f})×10⁻³ km².

This interval is **{LAB['gain_vs_neutron_stars']:.0f} times narrower** than the neutron-star interval from Part VII. **Q4 {'PASS' if Q4['pass'] else 'FAIL'}.**

Coming tests: MUSE (μ–p scattering) sees the same δ⟨r²⟩ as μH spectroscopy, and muonic deuterium and helium sharpen the
lever by a factor ≈ M/m_p.
"""), lab_fig])

In [ ]:
# ---- CRUST FAMILY: Theorems 12-13 on five unified EOS, and the alpha a glitch resolution would need -----
_rows = {}
_cache = os.path.join(CFG["ckpt_dir"], "eos_tables")
for _nm, _label in NS8.CRUST_FAMILY.items():
    try:
        _e, _p = NS8.lal_table(_nm, _cache)
    except Exception as _err:
        if _nm != "SLY":
            _rows[_label] = {"error": f"{type(_err).__name__}: {_err}"}; continue
        _e, _p = NS7.sly_table()
    _E, _P, _n, _mu = NS8.grid_from_table(_e, _p, 3000)
    _l2, _emin, _c2 = NS7.crust_length2(_E, _P)
    _nt = float(np.interp(_emin, _E, _n))
    _thr = NS7.crust_threshold(_e[_e < 150], _p[_e < 150])
    _al = [_x for _x in (-10, -8, -6, -5, -4, -3, -2, -1, 0, 0.5, 1, 1.5, 2, 2.5, 3) if _x < _thr - 0.05]
    _B = len(_al)
    _eps = torch.tensor(_E).view(1, -1).expand(_B, -1).contiguous(); _pp = torch.tensor(_P).view(1, -1).expand(_B, -1).contiguous()
    _ng = torch.tensor(_n)
    _cs2 = torch.gradient(_pp, spacing=(_ng,), dim=1)[0] / torch.gradient(_eps, spacing=(_ng,), dim=1)[0]
    _et, _pt, _c2t, _mut, _nl, _ = NS7.rq_tables(_eps, _pp, _cs2, torch.tensor(_mu).view(1, -1).expand(_B, -1).contiguous(), _ng,
                                                 torch.tensor(_al, dtype=torch.float64), torch.zeros(_B, dtype=torch.float64))
    _hi = _nl.clamp(max=float(_n[-1]) * 0.98); _lo = torch.full((_B,), 0.2, dtype=torch.float64)
    _t = torch.linspace(0, 1, 48, dtype=torch.float64).view(1, -1)
    _ncs = torch.exp(torch.log(_lo).view(-1, 1) + _t * (torch.log(_hi) - torch.log(_lo)).view(-1, 1))
    _M, _R, _L, _I, _dI, _It, _ec = NS7.tov_rq(NS, _et, _pt, _c2t, _mut, _ng, _ncs, _nt, n_h=320, n_crust=100)
    _st = NS.stable_mask(torch.where(torch.isfinite(_M), _M, torch.zeros_like(_M)))
    _f = NS7.branch_at(_M, _dI / _I, _st, 1.4).numpy(); _Mx = torch.where(_st, _M, torch.zeros_like(_M)).max(1).values.numpy()
    _a = np.array(_al); _ok = np.isfinite(_f)
    _a7 = float(np.interp(0.07, _f[_ok][::-1], _a[_ok][::-1])) if _f[_ok].max() > 0.07 else float("nan")
    _a16 = float(np.interp(0.016, _f[_ok][::-1], _a[_ok][::-1])) if _f[_ok].min() < 0.016 else float("nan")
    _i0 = _al.index(0)
    _rows[_label] = {"l2_crust": _l2, "exact_threshold": _thr, "n_t": _nt, "f_GR": float(_f[_i0]), "Mmax_GR": float(_Mx[_i0]),
                     "alpha_7pct": _a7, "alpha_1.6pct": _a16, "Mmax_at_alpha_7": float(np.interp(_a7, _a, _Mx)) if np.isfinite(_a7) else float("nan"),
                     "exclusion_factor": abs(_a7) / max(abs(LAB["lo95"]), abs(LAB["hi95"])) if np.isfinite(_a7) else float("nan")}
CRUST_FAMILY = _rows
_good = [_v for _v in _rows.values() if "alpha_7pct" in _v and np.isfinite(_v["alpha_7pct"])]
Q3 = {"pass": bool(_good and all(-7 <= _v["alpha_7pct"] <= -4 for _v in _good)), "n_crusts": len(_good),
      "min_exclusion_factor": min(_v["exclusion_factor"] for _v in _good) if _good else float("nan")}
_tab = "\n".join(f"| {_k} | {_v['l2_crust']:.2f} | {_v['exact_threshold']:.2f} | {_v['n_t']:.4f} | {100 * _v['f_GR']:.2f} | {_v['Mmax_GR']:.2f} | {_v['alpha_7pct']:+.2f} | {_v['alpha_1.6pct']:+.2f} | {_v['exclusion_factor']:.0f}× |"
                  if "l2_crust" in _v else f"| {_k} | {_v['error']} | | | | | | | |" for _k, _v in _rows.items())
mo.md(f"""
**Five unified crusts** (LALSuite tables; crust and core from the same nuclear model):

| EOS | ℓ²_crust (LO) [km²] | exact crust threshold [km²] | n_t [fm⁻³] | GR ΔI_c/I(1.4) [%] | GR M_max | α for Vela 7% | α for Vela 1.6% | excluded by muonic atoms |
|---|---|---|---|---|---|---|---|---|
{_tab}

- **The glitch lever is crust-robust:** every crust needs α ≈ −5 to −6 km² to reach 7%.
- **The lab bound excludes that by at least {Q3['min_exclusion_factor']:.0f}×.** Q3 {'PASS' if Q3['pass'] else 'FAIL'}.
- The crust theorem itself varies with the crust (FPS is softest at the crust–core transition), as Theorem 12 says it should.
""")

In [ ]:
# ---- SYNTHESIS: the whole action, every probe, one picture; export -----------------------------------
_sectors = [
    ("IR term  −μ⁴/Rⁿ", "T → 0 (cosmic, conformal point)", "Parts I–VI: Jeans window, wall, C ≤ %.3f" % C_CONF, "atoms (F ≫ 1): excluded unless screened [Flanagan 2004; Olmo 2008]"),
    ("UV  αR²", "T ≠ 0, dilute (crust)", "Part VII: crust theorem α < %.2f km², glitch lever" % ALPHA_CRUST,
     "muonic atoms: %+.4f < α < %+.4f km² (this work)" % (LAB["lo95"], LAB["hi95"])),
    ("UV  βR_μνR^μν", "all matter incl. conformal", "Part VII: ceiling βκε < 0.19", "lepton scattering: Λ_Q ≳ TeV [Latorre+ 2018; Beltrán Jiménez+ 2021]"),
    ("photons", "T = 0", "blind to R² exactly (Theorem 11)", "—"),
]
_tab = "\n".join(f"| {a} | {b} | {c} | {d} |" for a, b, c, d in _sectors)
_checks = {"Q1 dressing theorem": Q1["pass"], "Q2 frame invariance": Q2["pass"], "Q3 crust robustness": Q3["pass"], "Q4 lab bound": Q4["pass"]}
_checks.update({k.split(" ")[0] + " (Part VII)": v["pass"] for k, v in P7_TESTS.items()})
_out = os.path.join(CFG["ckpt_dir"], "part4", "PART8_SYNTHESIS.json")
os.makedirs(os.path.dirname(_out), exist_ok=True)
save_json(_out, {"lab_bound": LAB, "crust_family": CRUST_FAMILY, "theorem15": THM15, "checks": _checks,
                 "neutron_star_interval": P7_POST["astro+pQCD"], "C_conf": C_CONF, "C_real": C_REAL})
lab_fig.savefig(os.path.join(CFG["ckpt_dir"], "part4", "part8_lab_bound.png"), dpi=160)
_ok = all(_checks.values())
mo.vstack([mo.md(f"""
## Final synthesis — where the action can be seen

| sector of $f$ | where it acts | what stars can say | what decides it |
|---|---|---|---|
{_tab}

**Checks:** {", ".join(f"{k} {'PASS' if v else 'FAIL'}" for k, v in _checks.items())}. {'All pass.' if _ok else 'Read the failing ones before writing.'}

**The three results the paper rests on**
1. **The measurability principle** (Theorems 8, 10, 14, 15). Ricci-based gravity is exactly degenerate with matter wherever
   matter's couplings are fitted, and observable only through couplings fixed by symmetry.
2. **The first lepton-universality bound on Palatini R²:** α = ({1e3 * LAB['alpha_mean']:.1f} ± {1e3 * LAB['alpha_sigma']:.1f})×10⁻³ km²,
   {LAB['gain_vs_neutron_stars']:.0f}× tighter than neutron stars. It excludes the glitch-crisis resolution by ≥ {Q3['min_exclusion_factor']:.0f}× for every crust.
3. **A GR result that needs no modified gravity:** a star with C > {C_CONF:.4f} must contain superconformal matter (Theorem 9;
   realistic ceiling {C_REAL:.4f}).

Saved `{_out}`.
"""), lab_fig])

In [ ]:
# ---- ROBUSTNESS + HEAVIER MUONIC ATOMS: data-selection envelope, D and He isotope pairs, M/Z scaling test ----
# Values: electronic H as in cell 65 (arXiv:2604.15546); muonic H (CREMA 2013) and its 2024 re-evaluation 0.84060(39);
# D-H and 3He-4He squared-radius differences, electronic vs muonic, as compiled in arXiv:2412.16315;
# muonic 4He r_alpha = 1.67824(83) fm (Krauth et al., Nature 2021) vs e-4He scattering 1.681(4) fm (Sick 2008).
_M_D, _M_H3, _M_A = 1.875612943, 2.808391611, 3.727379410             # nuclear masses [GeV]
_lev_p = NS8.dr2_equivalent(NS8.M_E) - NS8.dr2_equivalent(NS8.M_MU)    # fm^2 per km^2 for M = m_p, Z = 1  (> 0)

def p8_lever(dM, Z):
    """(electronic - muonic) shift of a squared radius (or isotope difference) per km^2 of alpha, for nuclear mass
    (difference) dM [GeV] and charge Z.  delta<r^2>_l = -3 alpha kappa m_l M / (2 pi Z alpha_em)."""
    return _lev_p * (dM / NS8.M_P) / Z

def p8_alpha(d_e, s_e, d_mu, s_mu, lev):
    d = d_e - d_mu; s = (s_e**2 + s_mu**2) ** 0.5
    return d / lev, s / abs(lev)

# (a) data-selection envelope for the proton
_me, _se, _c2, _dof, _B = NS8.weighted_mean(NS8.RADIUS_ELECTRONIC)
_choices = {
    "all modern e-H spectroscopy (Birge-inflated)": (_me, _se),
    "newest single: Maisenbacher 2026": (0.8406, 0.0015),
    "CODATA 2022 without muonic data (older e-H dominated)": (0.8529, 0.0043),
}
ENVELOPE = {}
for _lab, (_r, _s) in _choices.items():
    for _rmu, _smu, _tag in ((0.84087, 0.00039, "CREMA 2013"), (0.84060, 0.00039, "CREMA re-evaluated")):
        _a, _sa = p8_alpha(_r**2, 2 * _r * _s, _rmu**2, 2 * _rmu * _smu, _lev_p)
        ENVELOPE[f"{_lab} | μH {_tag}"] = (_a, _sa)
# (b) heavier systems: each probes the SAME alpha with a different lever  (M/Z scaling is the signature)
SYSTEMS = {
    "H (r_p², modern e-H vs μH)": (*p8_alpha(_me**2, 2 * _me * _se, 0.84087**2, 2 * 0.84087 * 0.00039, _lev_p), _lev_p),
    "D−H (r_d²−r_p², 1S-2S isotope shift vs μD−μH)": (*p8_alpha(3.8207, 0.0003, 3.8200, 0.0031, p8_lever(_M_D - NS8.M_P, 1)), p8_lever(_M_D - NS8.M_P, 1)),
    "³He−⁴He (r_h²−r_α², 2³S-2¹S vs μHe)": (*p8_alpha(1.0678, 0.0007, 1.0636, 0.0031, p8_lever(_M_H3 - _M_A, 2)), p8_lever(_M_H3 - _M_A, 2)),
    "⁴He (r_α², e-scattering vs μ⁴He⁺)": (*p8_alpha(1.681**2, 2 * 1.681 * 0.004, 1.67824**2, 2 * 1.67824 * 0.00083, p8_lever(_M_A, 2)), p8_lever(_M_A, 2)),
}
_w = np.array([1 / _v[1] ** 2 for _v in SYSTEMS.values()]); _x = np.array([_v[0] for _v in SYSTEMS.values()])
_comb = float((_w * _x).sum() / _w.sum()); _scomb = float(_w.sum() ** -0.5)
_chi2 = float((_w * (_x - _comb) ** 2).sum()); _ndof = len(_x) - 1
COMBINED = {"alpha": _comb, "sigma": _scomb, "lo95": _comb - 1.96 * _scomb, "hi95": _comb + 1.96 * _scomb,
            "chi2": _chi2, "dof": _ndof, "envelope_abs95": max(abs(_a) + 1.96 * _s for _a, _s in ENVELOPE.values())}
_need = min(abs(_v["alpha_7pct"]) for _v in CRUST_FAMILY.values() if "alpha_7pct" in _v and np.isfinite(_v["alpha_7pct"]))
COMBINED["glitch_exclusion_combined"] = _need / max(abs(COMBINED["lo95"]), abs(COMBINED["hi95"]))
COMBINED["glitch_exclusion_worst_envelope"] = _need / COMBINED["envelope_abs95"]
Q5 = {"pass": bool(_chi2 / max(_ndof, 1) < 2.0 and COMBINED["glitch_exclusion_worst_envelope"] > 50),
      "chi2_per_dof": _chi2 / max(_ndof, 1)}
# figure: alpha from each system vs its lever (a common alpha must line up)
_fig, _ax = plt.subplots(1, 2, figsize=(13, 4.2))
_names = list(SYSTEMS)
for _i, _n in enumerate(_names):
    _ax[0].errorbar(1e3 * SYSTEMS[_n][0], _i, xerr=1.96e3 * SYSTEMS[_n][1], fmt="o", color="C3", capsize=3)
_ax[0].axvspan(1e3 * COMBINED["lo95"], 1e3 * COMBINED["hi95"], color="C3", alpha=0.15, label="combined 95%")
_ax[0].axvline(0, color="k", lw=0.8)
_ax[0].set_yticks(range(len(_names))); _ax[0].set_yticklabels(_names, fontsize=8)
_ax[0].set(xlabel="α [10⁻³ km²]  (95% bars)", title="one α, four atomic systems"); _ax[0].legend(fontsize=8)
_en = list(ENVELOPE)
for _i, _n in enumerate(_en):
    _ax[1].errorbar(1e3 * ENVELOPE[_n][0], _i, xerr=1.96e3 * ENVELOPE[_n][1], fmt="s", color="C0", capsize=3)
_ax[1].axvline(0, color="k", lw=0.8)
_ax[1].set_yticks(range(len(_en))); _ax[1].set_yticklabels([_n.replace(" | ", "\n") for _n in _en], fontsize=6.5)
_ax[1].set(xlabel="α [10⁻³ km²]  (95% bars)", title="proton only: dependence on the electronic data set")
_fig.tight_layout(); robust_fig = _fig; plt.close(_fig)
_r1 = "\n".join(f"| {_n} | {_v[2]:+.3f} | {1e3 * _v[0]:+.1f} ± {1e3 * _v[1]:.1f} |" for _n, _v in SYSTEMS.items())
_r2 = "\n".join(f"| {_n.replace(' | ', ' — ')} | {1e3 * _v[0]:+.1f} ± {1e3 * _v[1]:.1f} | {_v[0] / _v[1]:+.1f}σ |" for _n, _v in ENVELOPE.items())
_out = os.path.join(CFG["ckpt_dir"], "part4", "PART8_ROBUST.json")
save_json(_out, {"systems": {k: list(v) for k, v in SYSTEMS.items()}, "envelope": {k: list(v) for k, v in ENVELOPE.items()},
                 "combined": COMBINED, "Q5": Q5})
robust_fig.savefig(os.path.join(CFG["ckpt_dir"], "part4", "part8_robust.png"), dpi=160)
mo.vstack([mo.md(f"""
## Robustness and heavier muonic atoms

**Four systems, one α.** Each pair compares the electronic and muonic determination of the same quantity. The lever
(fm² per km²) scales as M/Z, and has the opposite sign for ³He−⁴He. A common α must therefore line up:

| system | lever [fm²/km²] | α [10⁻³ km²] |
|---|---|---|
{_r1}

**Combined:** α = ({1e3 * COMBINED['alpha']:+.1f} ± {1e3 * COMBINED['sigma']:.1f})×10⁻³ km²,
95%: [{1e3 * COMBINED['lo95']:+.1f}, {1e3 * COMBINED['hi95']:+.1f}]×10⁻³ km².
- Consistency with one α: χ² = {COMBINED['chi2']:.2f} for {COMBINED['dof']} dof.
- The isotope pairs carry their own common-mode systematics: they partly share the μH input, and the helium isotope shift has a
  history of tension.

**Data-selection envelope (proton only):**

| electronic input — muonic input | α [10⁻³ km²] | significance |
|---|---|---|
{_r2}

- The CODATA-without-muonic row is the old "proton-radius puzzle". Read through Theorem 14, it was a ~3σ hint of α > 0.
  Modern electronic data remove it, and we do **not** claim it.
- Robust envelope over all choices: |α| < {1e3 * COMBINED['envelope_abs95']:.0f}×10⁻³ km² (95%).

**Glitch-crisis value (|α| ≥ {_need:.1f} km²)** is excluded by
- {COMBINED['glitch_exclusion_combined']:.0f}× with the combined bound;
- {COMBINED['glitch_exclusion_worst_envelope']:.0f}× even with the most permissive data choice.

**Q5 {'PASS' if Q5['pass'] else 'FAIL'}** (one α fits all systems, and the exclusion survives every data choice).

Saved `{_out}`.
"""), robust_fig])

In [ ]:
# ---- CELL 69: final claim ledger — bound on (alpha+beta), novelty audit, open items ------------
_C = COMBINED
_ledger = [
    ("Bound (combined, 4 systems)", f"α+β = ({1e3*_C['alpha']:+.1f} ± {1e3*_C['sigma']:.1f})×10⁻³ km²"),
    ("Bound (95%, combined)", f"[{1e3*_C['lo95']:+.1f}, {1e3*_C['hi95']:+.1f}]×10⁻³ km²"),
    ("Robust envelope (all data choices)", f"|α+β| < {1e3*_C['envelope_abs95']:.0f}×10⁻³ km²"),
    ("Consistency with one coupling (M/Z test)", f"χ² = {_C['chi2']:.2f} / {_C['dof']} dof"),
    ("Glitch resolution excluded by", f"{_C['glitch_exclusion_combined']:.0f}× (combined), {_C['glitch_exclusion_worst_envelope']:.0f}× (worst case)"),
]
_novel = [
    ("Induced contact interactions in RBG", "KNOWN — Delhom, Miralles & Peñuelas 2020; Flanagan 2004"),
    ("Macroscopic GR-indistinguishability", "KNOWN — Li, Mota & Shaw 2008"),
    ("Mass-weighted contact between distinct particles, (α+β)κ m_i m_j δ³(r)", "NEW (explicit form)"),
    ("Lepton-universality / muonic-atom bound + M/Z scaling test", "NEW (no spectroscopic RBG bound found)"),
    ("Frame invariance of momentum density (Theorem 15)", "NEW (to our search)"),
    ("Crust gradient theorem, glitch lever (Theorems 12–13)", "NEW (to our search)"),
    ("Conditional compactness bound 0.3126 (Theorem 9)", "NEW as conditional detector; global C_max≈1/3 known (2510.12870)"),
]
_open = ["Insert the k_J* bound from Part I", "Full Dirac-field derivation of the (α+β) contact (appendix)",
         "Hostile simulated referee report before submission"]
_out = os.path.join(CFG["ckpt_dir"], "part4", "PART8_LEDGER.json")
save_json(_out, {"ledger": dict(_ledger), "novelty": dict(_novel), "open": _open})
mo.md("## Claim ledger\n\n| quantity | value |\n|---|---|\n" + "\n".join(f"| {a} | {b} |" for a, b in _ledger)
      + "\n\n## Novelty audit\n\n| item | status |\n|---|---|\n" + "\n".join(f"| {a} | {b} |" for a, b in _novel)
      + "\n\n**Open before submission:** " + "; ".join(_open) + f"\n\nSaved `{_out}`.")

In [ ]:
# ---- CELL 72: ALL PAPER FIGURES FROM SAVED EXPORTS (independent: needs only files on disk) -------------
# Reads:  pj_ckpt/part4/PART8_ROBUST.json, PART8_SYNTHESIS.json, PART6_COMPACTNESS.json
#         pj_ckpt/part7/PART7_RBG.json and the per-point checkpoints pj_ckpt/part7/a*_b*_batch*.npz
#         LALSuite unified EOS tables (downloaded once, cached in pj_ckpt/eos_tables/)
# Writes: pj_ckpt/paper_figs/fig1_bound.pdf, fig2_stars.pdf, figS2_wall.pdf, figS3_degeneracy.pdf,
#         figS4_ceiling.pdf, figS5_posterior.pdf, figS6_radii.pdf (+ .png). Fig. S1 (cosmology) is copied from
#         the Part I export if present, because it needs the Part I likelihood code.
# Every import uses a private alias, so this cell defines no global names and can sit anywhere in the notebook.
import os as _os_pf, json as _json_pf, glob as _glob_pf, math as _math_pf, shutil as _shutil_pf, urllib.request as _url_pf
import numpy as _np_pf
import matplotlib as _mpl_pf
import matplotlib.pyplot as _plt_pf
import marimo as _mo_pf

_BASE = _os_pf.environ.get("PF_CKPT", "pj_ckpt")
_OUT = _os_pf.path.join(_BASE, "paper_figs"); _os_pf.makedirs(_OUT, exist_ok=True)
_mpl_pf.rcParams.update({"font.size": 9, "axes.linewidth": 0.8, "font.family": "serif", "savefig.bbox": "tight"})
_RED, _DRED, _LRED, _ORA, _GRN, _BLU = "#b2182b", "#67001f", "#f4a582", "#e08214", "#1b7837", "#2166ac"
_MSUN_KM, _KAP_KM2 = 1.476625, 8 * _math_pf.pi * 1.3234e-6          # G M_sun/c^2 [km];  kappa per MeV/fm^3 [km^-2]
_log = []

def _load(rel):
    _f = _os_pf.path.join(_BASE, rel)
    if not _os_pf.path.exists(_f):
        _log.append(f"missing `{_f}`"); return None
    with open(_f) as _h:
        return _json_pf.load(_h)

def _save(fig, name):
    fig.savefig(_os_pf.path.join(_OUT, name + ".pdf")); fig.savefig(_os_pf.path.join(_OUT, name + ".png"), dpi=200)
    _plt_pf.close(fig); _log.append(f"wrote `{name}.pdf`")

_R8 = _load("part4/PART8_ROBUST.json"); _S8 = _load("part4/PART8_SYNTHESIS.json")
_P6 = _load("part4/PART6_COMPACTNESS.json"); _P7 = _load("part7/PART7_RBG.json")

# ============ FIGURE 1: the lepton-universality bound ============
if _R8 and _S8 and _P7:
    _fig, _ax = _plt_pf.subplots(1, 2, figsize=(7.0, 2.9), gridspec_kw={"width_ratios": [1.0, 1.25]})
    _names = ["H", "D$-$H", r"$^3$He$-^4$He", r"$^4$He"]
    _vals = list(_R8["systems"].values()); _C = _R8["combined"]
    _y = _np_pf.arange(len(_vals))[::-1] + 1
    for _yy, _v, _n in zip(_y, _vals, _names):
        _ax[0].errorbar(1e3 * _v[0], _yy, xerr=1.96e3 * _v[1], fmt="o", ms=4, color=_RED, capsize=2, lw=1)
        _ax[0].text(21.5, _yy, f"{_v[2]:+.3f}", va="center", fontsize=7, color="0.35")
    _ax[0].errorbar(1e3 * _C["alpha"], 0, xerr=1.96e3 * _C["sigma"], fmt="D", ms=4, color="k", capsize=2, lw=1)
    _ax[0].axvspan(1e3 * _C["lo95"], 1e3 * _C["hi95"], color=_RED, alpha=0.12, lw=0)
    _ax[0].axvline(0, color="0.5", lw=0.6)
    _ax[0].set_yticks(list(_y) + [0]); _ax[0].set_yticklabels(_names + ["combined"])
    _ax[0].set(xlim=(-25, 20), ylim=(-0.7, len(_vals) + 0.7), xlabel=r"$\alpha+\beta$  [$10^{-3}$ km$^2$]  (95%)")
    _ax[0].text(21.5, len(_vals) + 0.45, r"$\Lambda$", fontsize=7, color="0.35")
    _ax[0].text(0.02, 0.97, "(a)", transform=_ax[0].transAxes, va="top", fontweight="bold")
    _ns = _P7["posterior_alpha"]["astro+pQCD"]; _L = _S8["lab_bound"]
    _bars = [("neutron stars (this work)", _ns["lo95"], _ns["hi95"], _ORA),
             ("neutron stars (Lope-Oter & Wojnar)", -6.47, 1.99, "0.6"),
             ("muonic atoms: hydrogen", _L["lo95"], _L["hi95"], _RED),
             ("muonic atoms: combined", _C["lo95"], _C["hi95"], _DRED),
             ("all data selections (envelope)", -_C["envelope_abs95"], _C["envelope_abs95"], _LRED)]
    for _i, (_n, _lo, _hi, _c) in enumerate(_bars[::-1]):
        _ax[1].plot([_lo, _hi], [_i, _i], lw=6, color=_c, solid_capstyle="butt")
        _ax[1].text(-3.0, _i + 0.3, _n, fontsize=7, va="bottom")
    _ax[1].axvspan(-6.0, -4.9, color=_GRN, alpha=0.25, lw=0)
    _ax[1].text(-5.45, -0.45, "needed for\nVela 7%", fontsize=6, ha="center", va="bottom", color=_GRN)
    _ax[1].set_xscale("symlog", linthresh=1e-2, linscale=0.6)
    _ax[1].set(xlim=(-15, 5), ylim=(-0.6, len(_bars) + 0.2), yticks=[], xlabel=r"$\alpha$ or $\alpha+\beta$  [km$^2$]  (95%, symlog)")
    _ax[1].axvline(0, color="0.5", lw=0.6)
    _ax[1].text(0.98, 0.04, "(b)", transform=_ax[1].transAxes, ha="right", fontweight="bold")
    _fig.tight_layout(w_pad=1.0); _save(_fig, "fig1_bound")

# ============ FIGURE 2: neutron stars ============
_NICER = {"J0030": (1.34, 0.15, 0.16, 12.71, 1.14, 1.19), "J0740": (2.07, 0.07, 0.07, 12.49, 1.28, 0.88),
          "J0437": (1.418, 0.037, 0.037, 11.36, 0.95, 0.63)}
_LAL = {"SLY": "SLy", "BSK19": "BSk19", "BSK20": "BSk20", "BSK21": "BSk21", "FPS": "FPS"}

def _lal(name):
    _d = _os_pf.path.join(_BASE, "eos_tables"); _os_pf.makedirs(_d, exist_ok=True)
    _f = _os_pf.path.join(_d, f"LAL_{name}.dat")
    if not _os_pf.path.exists(_f):
        _url_pf.urlretrieve("https://raw.githubusercontent.com/lscsoft/lalsuite-archive/master/lalsimulation/src/"
                            f"LALSimNeutronStarEOS_{name}.dat", _f + ".tmp"); _os_pf.replace(_f + ".tmp", _f)
    _a = _np_pf.loadtxt(_f); _c0, _c1 = _a[:, 0], _a[:, 1]
    _e, _p = (_c0, _c1) if _np_pf.median(_c0 / _np_pf.where(_c1 > 0, _c1, _np_pf.inf)) > 1 else (_c1, _c0)
    _m = (_e > 0) & (_p > 0); _e, _p = _e[_m] / 1.3234e-12, _p[_m] / 1.3234e-12
    _o = _np_pf.argsort(_e); _e, _p = _e[_o], _p[_o]
    _k = _np_pf.concatenate([[True], (_np_pf.diff(_e) > 0) & (_np_pf.diff(_p) > 0)])
    return _e[_k], _p[_k]

if _P6 and _P7 and _S8:
    _B = _P6["bounds"]
    _Ccaus = [v[0] for k, v in _B.items() if k.startswith("causal (")][0]
    _Cconf = [v[0] for k, v in _B.items() if "Theorem 9" in k][0]
    _Creal = _P6["C_real"]
    _fig, _ax = _plt_pf.subplots(1, 3, figsize=(7.0, 2.6))
    _m = _np_pf.linspace(0.8, 3.0, 200)
    _ax[0].fill_betweenx(_m, _MSUN_KM * _m / _Ccaus, _MSUN_KM * _m / _Cconf, color=_RED, alpha=0.15, lw=0)
    _ax[0].plot(_MSUN_KM * _m / _Ccaus, _m, "k", lw=1, label=f"causal, C={_Ccaus:.3f}")
    _ax[0].plot(_MSUN_KM * _m / _Cconf, _m, color=_RED, lw=1.3, label=f"$p\\leq\\varepsilon/3$, C={_Cconf:.4f}")
    _ax[0].plot(_MSUN_KM * _m / _Creal, _m, color=_RED, lw=1, ls="--", label=f"realistic, C={_Creal:.4f}")
    for _mm, _mp, _mn, _r, _rp, _rm in _NICER.values():
        _ax[0].errorbar(_r, _mm, xerr=[[_rm], [_rp]], yerr=[[_mn], [_mp]], fmt="o", ms=3, color=_BLU, capsize=1.5, lw=0.8)
    _ax[0].text(11.35, 2.45, "super-\nconformal", fontsize=5.5, color=_RED, ha="center", rotation=58)
    _ax[0].set(xlim=(6, 16), ylim=(0.8, 3.0), xlabel="$R$ [km]", ylabel=r"$M$ [$M_\odot$]")
    _ax[0].legend(fontsize=6, loc="lower right", frameon=False)
    _ax[0].text(0.03, 0.96, "(a)", transform=_ax[0].transAxes, va="top", fontweight="bold")
    _cols = {"SLY": "#1b9e77", "BSK19": "#d95f02", "BSK20": "#7570b3", "BSK21": "#e7298a", "FPS": "#66a61e"}
    _thr = {k.split(" ")[0].upper(): v.get("exact_threshold") for k, v in _S8.get("crust_family", {}).items()}
    for _nm, _lab in _LAL.items():
        try:
            _e, _p = _lal(_nm)
        except Exception as _err:
            _log.append(f"EOS {_nm} unavailable ({type(_err).__name__})"); continue
        _s = _e < 400
        _c2 = _p[_s] / _e[_s] * _np_pf.gradient(_np_pf.log(_p[_s]), _np_pf.log(_e[_s]))
        _ax[1].plot(_e[_s], _c2 / (_KAP_KM2 * _e[_s]), lw=1, color=_cols[_nm], label=_lab)
        if _thr.get(_nm) is not None:
            _ax[1].axhline(_thr[_nm], color=_cols[_nm], lw=0.7, ls="--")
    _ax[1].set(xscale="log", yscale="log", xlim=(1e-2, 400), ylim=(0.5, 1e4), xlabel=r"$\varepsilon$ [MeV fm$^{-3}$]",
               ylabel=r"$c_s^2/(\kappa\varepsilon)$ [km$^2$]")
    _ax[1].legend(fontsize=6, frameon=False, loc="upper right")
    _ax[1].text(0.40, 0.93, "(b)", transform=_ax[1].transAxes, va="top", fontweight="bold")
    _G = _P7["glitch"]; _al = _np_pf.asarray(_G["alpha"], float)
    _med, _lo, _hi = (100 * _np_pf.asarray(_G[k], float) for k in ("f14_med", "f14_lo", "f14_hi")); _ok = _np_pf.isfinite(_med)
    _ax[2].fill_between(_al[_ok], _lo[_ok], _hi[_ok], color=_BLU, alpha=0.25, lw=0)
    _ax[2].plot(_al[_ok], _med[_ok], color=_BLU, lw=1.2)
    _ax[2].axhline(1.6, color=_GRN, ls="--", lw=0.8); _ax[2].text(-11.6, 1.75, "Vela, no entrainment", fontsize=6, color=_GRN)
    _ax[2].axhline(7.0, color=_RED, ls="--", lw=0.8); _ax[2].text(-11.6, 7.6, "Vela, strong entrainment", fontsize=6, color=_RED)
    _L = _S8["lab_bound"]
    _ax[2].axvspan(_L["lo95"], _L["hi95"], color=_RED, alpha=0.6, lw=0)
    _ax[2].annotate("atomic bound\n(width 0.015 km$^2$)", xy=(0.5 * (_L["lo95"] + _L["hi95"]), 1.1), xytext=(-9.0, 1.05),
                    fontsize=6, color=_RED, arrowprops=dict(arrowstyle="->", color=_RED, lw=0.7))
    _acrust = _P7["crust"]["this notebook's crust (cell 23)"]["exact_alpha"]
    _ax[2].axvline(_acrust, color="k", ls=":", lw=0.8)
    _ax[2].set(yscale="log", xlim=(-12, 3), xlabel=r"$\alpha$ [km$^2$]", ylabel=r"$\Delta I_c/I$ at 1.4 $M_\odot$ [%]")
    _ax[2].text(0.97, 0.96, "(c)", transform=_ax[2].transAxes, ha="right", va="top", fontweight="bold")
    _fig.tight_layout(w_pad=0.6); _save(_fig, "fig2_stars")

# ============ FIGURE S1: cosmology (copied from the Part I export) ============
_s1 = [_f for _f in _glob_pf.glob(_os_pf.path.join(_BASE, "**", "profile_posterior.png"), recursive=True)]
if _s1:
    _shutil_pf.copy(_s1[0], _os_pf.path.join(_OUT, "figS1_cosmology.png")); _log.append("copied `figS1_cosmology.png` from Part I export")
else:
    _log.append("Fig. S1: no Part I `profile_posterior.png` found; re-run the Part I export cell")

# ============ FIGURE S2: non-analytic GR limit and the wall (exact trace equation, n = 1) ============
# f = R - 2*Lam - M/R in H0 units:  -R + 4 Lam + 3 M / R = kappa T  ->  R = [(4Lam - kT) + sqrt((4Lam - kT)^2 + 12 M)] / 2
_KH = 6.212e41                                                      # kappa / H0^2 per MeV/fm^3
_LAM = 3 * 0.69

def _FRf(kT, M):
    _a = 4 * _LAM - kT
    _R = _np_pf.where(_a > 0, 0.5 * (_a + _np_pf.sqrt(_a * _a + 12 * M)), 6 * M / (_np_pf.sqrt(_a * _a + 12 * M) - _a))
    _F = 1 + M / _R**2
    return _F, _R, _R - 2 * _LAM - M / _R

_fig, _ax = _plt_pf.subplots(1, 3, figsize=(7.0, 2.4))
_D = _np_pf.linspace(0.06, -0.08, 600); _eps0 = 700.0
for _M, _c, _lab in ((1.0, _BLU, r"$\mu^4=H_0^4$"), (1e-6, _ORA, r"$\mu^4=10^{-6}H_0^4$"), (1e-12, _GRN, r"$\mu^4=10^{-12}H_0^4$")):
    _F, _, _ = _FRf(_KH * _eps0 * (-3 * _D), _M)
    _ax[0].plot(_D, _np_pf.log10(_np_pf.maximum(_F - 1, 1e-300)), color=_c, lw=1.2, label=_lab)
_ax[0].axvline(0, color="k", lw=0.6); _ax[0].invert_xaxis()
_ax[0].set(xlabel=r"$\Delta=1/3-p/\varepsilon$", ylabel=r"$\log_{10}(F-1)$"); _ax[0].legend(fontsize=6, frameon=False)
_ax[0].text(0.03, 0.96, "(a)", transform=_ax[0].transAxes, va="top", fontweight="bold")
_Ms = _np_pf.logspace(-30, 1, 60); _kT = _KH * _eps0 * (-3 * -0.05)
_Fm = _np_pf.array([_FRf(_np_pf.array(_kT), _M)[0] for _M in _Ms])
_ax[1].loglog(_Ms, _Fm, color=_RED, lw=1.3)
_slope = _np_pf.polyfit(_np_pf.log10(_Ms), _np_pf.log10(_Fm), 1)[0]
_ax[1].set(xlabel=r"$\mu^4/H_0^4$", ylabel=r"$F$ in a $\Delta=-0.05$ core")
_ax[1].text(0.05, 0.08, f"slope {_slope:.3f}", transform=_ax[1].transAxes, fontsize=7)
_ax[1].text(0.92, 0.96, "(b)", transform=_ax[1].transAxes, ha="right", va="top", fontweight="bold")
# (c) Einstein-frame pressure for a model EOS that crosses the conformal point at eps_w
_e = _np_pf.linspace(200, 1400, 1500); _ew = 800.0
_w = 1 / 3 + 0.12 * _np_pf.tanh((_e - _ew) / 300.0); _pp = _w * _e
for _M, _c in ((1.0, _BLU), (1e-12, _GRN)):
    _F, _R, _f = _FRf(_KH * (-_e + 3 * _pp), _M)
    _pt = _pp / _F**2 - (_R * _F - _f) / (_KH * 2 * _F**2)
    _ax[2].semilogy(_e / _ew, _np_pf.maximum(_pt / _pp[_np_pf.argmin(abs(_e - _ew))], 1e-30), color=_c, lw=1.2)
_ax[2].axvline(1, color="k", lw=0.6)
_ax[2].set(xlabel=r"$\varepsilon/\varepsilon_{\rm wall}$", ylabel=r"$\tilde p/p_{\rm wall}$", ylim=(1e-30, 10))
_ax[2].text(0.97, 0.96, "(c)", transform=_ax[2].transAxes, ha="right", va="top", fontweight="bold")
_fig.tight_layout(w_pad=0.6); _save(_fig, "figS2_wall")

# ============ FIGURE S3: degeneracy (values transcribed from the Part III/IV verdict tables) ============
_V = [("uniform", "base", 0.155, 0.001, 0.72), ("uniform", "+pQCD", 0.179, 0.001, 0.70), ("uniform", "+J0952", -0.047, 0.001, 0.84),
      ("uniform", "+J0952+pQCD", -0.009, 0.001, 0.82), ("log-uniform", "base", 0.075, 0.001, 0.42), ("log-uniform", "+pQCD", 0.078, 0.001, 0.41),
      ("log-uniform", "+J0952", -0.075, 0.001, 0.59), ("log-uniform", "+J0952+pQCD", -0.068, 0.001, 0.58), ("GP", "base", 0.086, 0.002, 0.74),
      ("GP", "+pQCD", 0.121, 0.003, 0.72), ("GP", "+J0952", -0.134, 0.003, 0.87), ("GP", "+J0952+pQCD", -0.066, 0.003, 0.85)]
_pc = {"uniform": _BLU, "log-uniform": _ORA, "GP": _RED}
_fig, _ax = _plt_pf.subplots(1, 2, figsize=(7.0, 3.0))
for _i, (_pr, _lk, _b, _sb, _ps) in enumerate(_V):
    _ax[0].errorbar(_b, _i, xerr=2 * _sb, fmt="o", ms=3.5, color=_pc[_pr], capsize=1.5)
    _ax[1].scatter(_ps, _b, s=18, color=_pc[_pr], label=_pr if _lk == "base" else None)
_ax[0].set_yticks(range(len(_V))); _ax[0].set_yticklabels([f"{a} | {b}" for a, b, *_ in _V], fontsize=6)
for _x in (-1, 1):
    _ax[0].axvline(_x, color="k", ls=":", lw=0.7)
_ax[0].axvline(0, color="0.5", lw=0.6); _ax[0].set(xlim=(-1.1, 1.1), xlabel=r"$\log_{10}B$(wall/GR)  ($\pm2\sigma$)")
_ax[1].axhspan(-0.5, 0.5, color="0.9", lw=0, zorder=0)
_ax[1].set(xlabel=r"$P$(superconformal core $\mid$ GR)", ylabel=r"$\log_{10}B$(wall/GR)", ylim=(-0.55, 0.55))
_ax[1].legend(fontsize=7, frameon=False); _ax[1].set_title("the inference moves with the prior; the evidence does not", fontsize=8)
_fig.tight_layout(); _save(_fig, "figS3_degeneracy")

# ============ FIGURE S4: Ricci-squared density ceiling (exact branch tracking, kappa*eps = 1) ============
def _branch_end(w, sgn, bmax=3.0, nb=1500):
    _bh = _np_pf.linspace(1e-4, bmax, nb) * sgn; _x = _np_pf.ones_like(_bh); _y = w * _x
    _P0 = -(_x + 3 * _y) / 2; _P1 = (_x - _y) / 2
    for _k in range(1, 41):
        _B = _bh * _k / 40
        for _ in range(40):
            _R = _P0 + 3 * _P1; _f = _R + _B * (_P0**2 + 3 * _P1**2)
            _E0 = 2 * _B * _P0**2 + _P0 - _f / 2 + _x; _E1 = 2 * _B * _P1**2 + _P1 - _f / 2 - _y
            _J00 = 4 * _B * _P0 + 1 - (1 + 2 * _B * _P0) / 2; _J01 = -(3 + 6 * _B * _P1) / 2
            _J10 = -(1 + 2 * _B * _P0) / 2; _J11 = 4 * _B * _P1 + 1 - (3 + 6 * _B * _P1) / 2
            _det = _J00 * _J11 - _J01 * _J10; _det = _np_pf.where(abs(_det) < 1e-300, 1e-300, _det)
            _P0, _P1 = _P0 - (_E0 * _J11 - _E1 * _J01) / _det, _P1 - (_J00 * _E1 - _J10 * _E0) / _det
    _R = _P0 + 3 * _P1; _f = _R + _bh * (_P0**2 + 3 * _P1**2)
    _res = _np_pf.maximum(abs(2 * _bh * _P0**2 + _P0 - _f / 2 + _x), abs(2 * _bh * _P1**2 + _P1 - _f / 2 - _y))
    _good = (_res < 1e-11) & (1 + 2 * _bh * _P0 > 0) & (1 + 2 * _bh * _P1 > 0) & _np_pf.isfinite(_P0)
    _cg = _np_pf.cumprod(_good).astype(bool)
    return float(abs(_bh[_cg]).max()) if _cg.any() else 0.0

_ws = [0.0, 0.02, 0.05, 0.1, 0.2, 1 / 3, 0.45, 0.6, 0.8, 1.0]
_bp = [_branch_end(_w, +1) for _w in _ws]; _bn = [_branch_end(_w, -1) for _w in _ws]
_fig, _ax = _plt_pf.subplots(figsize=(3.4, 2.6))
_ax.semilogy(_ws, _bp, "o-", color=_RED, ms=3.5, lw=1, label=r"$\beta>0$")
_ax.semilogy(_ws, _bn, "s-", color=_BLU, ms=3.5, lw=1, label=r"$\beta<0$ (scan edge 3)")
_ax.axvline(1 / 3, color="0.5", ls=":", lw=0.8)
_ax.set(xlabel=r"$p/\varepsilon$", ylabel=r"maximal $|\beta|\kappa\varepsilon$"); _ax.legend(fontsize=7, frameon=False)
_fig.tight_layout(); _save(_fig, "figS4_ceiling")
_log.append(f"S4 check: ceiling at p/eps=1/3 is {_bp[5]:.3f}, at p/eps=1 is {_bp[-1]:.3f}")

# ============ FIGURE S5: neutron-star posterior on alpha, and dM_max/dalpha ============
if _P7:
    _fig, _ax = _plt_pf.subplots(1, 2, figsize=(7.0, 2.7))
    _al = _np_pf.asarray(_P7["glitch"]["alpha"], float)
    _vc = {"astro": "0.4", "astro+pQCD": _ORA, "astro+pQCD+Vela 1.6%": _GRN, "astro+pQCD+Vela 7% (entrainment)": _RED, "astro(+J0952)+pQCD": _BLU}
    for _v, _lz in _P7["logZ_alpha"].items():
        _lz = _np_pf.asarray(_lz, float); _w = _np_pf.exp(_lz - _np_pf.nanmax(_lz)); _w = _np_pf.nan_to_num(_w)
        _ax[0].plot(_al, _w / max(_np_pf.trapezoid(_w, _al), 1e-300), lw=1.2, color=_vc.get(_v, "k"), label=_v)
    _ax[0].axvspan(-6.47, 1.99, color="0.92", zorder=0); _ax[0].axvline(_P7["crust"]["this notebook's crust (cell 23)"]["exact_alpha"], color="k", ls="--", lw=0.8)
    _ax[0].set(xlabel=r"$\alpha$ [km$^2$]", ylabel="posterior density"); _ax[0].set_ylim(0, _ax[0].get_ylim()[1] * 1.55); _ax[0].legend(fontsize=5.5, frameon=False, loc="upper center", ncol=2)
    _ax[0].text(0.97, 0.96, "(a)", transform=_ax[0].transAxes, ha="right", va="top", fontweight="bold")

    def _pts(a):
        _fs = sorted(_glob_pf.glob(_os_pf.path.join(_BASE, "part7", f"a{a:+08.3f}_b{0.0:+08.3f}_batch*.npz")))
        if not _fs:
            return None
        _z = [_np_pf.load(_f) for _f in _fs]
        return {k: _np_pf.concatenate([_q[k] for _q in _z]) for k in ("Mmax", "Dc", "valid")}
    _m, _p0, _pl = _pts(-0.5), _pts(0.0), _pts(0.5)
    if _m is not None and _p0 is not None and _pl is not None:
        _dM = (_pl["Mmax"] - _m["Mmax"]) / 1.0; _ok = _np_pf.isfinite(_dM) & (_m["Mmax"] > 1.0) & (_p0["valid"] > 0)
        _ii = _np_pf.where(_ok)[0][:8000]
        _ax[1].scatter(_p0["Dc"][_ii], _dM[_ii], s=1.5, alpha=0.3, color="#762a83", rasterized=True)
        _ax[1].text(0.03, 0.06, f"median |dM/dα| = {_np_pf.median(abs(_dM[_ok])):.3f}", transform=_ax[1].transAxes, fontsize=7)
    else:
        _log.append("S5(b): part7 checkpoints for alpha = -0.5, 0, +0.5 not found")
    _ax[1].axhline(0, color="k", lw=0.6); _ax[1].axvline(0, color=_RED, lw=0.8)
    _ax[1].set(xlabel=r"central $\Delta_c$ (GR, $M_{\max}$ star)", ylabel=r"$dM_{\max}/d\alpha$ [$M_\odot$ km$^{-2}$]")
    _ax[1].text(0.97, 0.96, "(b)", transform=_ax[1].transAxes, ha="right", va="top", fontweight="bold")
    _fig.tight_layout(); _save(_fig, "figS5_posterior")

# ============ FIGURE S6: proton-radius inputs ============
_RAD = [("Beyer 2017", 0.8335, 0.0095), ("Fleurbaey 2018", 0.877, 0.013), ("Bezginov 2019", 0.833, 0.010), ("Grinin 2020", 0.8482, 0.0038),
        ("Brandt 2022", 0.8584, 0.0051), ("Maisenbacher 2026", 0.8406, 0.0015), ("Bullis 2026", 0.8433, 0.0031), ("PRad 2019 (e-p)", 0.831, 0.014)]
_rmu, _smu = 0.84087, 0.00039
_w = _np_pf.array([1 / s**2 for _, _, s in _RAD[:7]]); _v = _np_pf.array([r for _, r, _ in _RAD[:7]])
_me = float((_w * _v).sum() / _w.sum()); _chi = float((_w * (_v - _me) ** 2).sum()); _se = float(_w.sum() ** -0.5 * max(1, _math_pf.sqrt(_chi / 6)))
_lev = 1.0884
_fig, _ax = _plt_pf.subplots(figsize=(4.2, 3.6))
_yy = _np_pf.arange(len(_RAD))
_ax.errorbar([r for _, r, _ in _RAD], _yy, xerr=[s for _, _, s in _RAD], fmt="o", ms=3.5, color=_BLU, capsize=1.5, lw=0.9)
_ax.axvspan(_rmu - _smu, _rmu + _smu, color=_RED, alpha=0.6, lw=0, label="muonic hydrogen")
_ax.axvspan(_me - _se, _me + _se, color=_BLU, alpha=0.15, lw=0, label="electronic mean (Birge-inflated)")
for _ab, _ls in ((0.05, ":"), (-0.05, "--")):
    _ax.axvline(_math_pf.sqrt(_me**2 - _ab * _lev), color="k", ls=_ls, lw=0.8, label=f"muonic radius if α+β = {_ab:+.2f} km²")
_ax.set_yticks(_yy); _ax.set_yticklabels([n for n, _, _ in _RAD], fontsize=7)
_ax.set(xlabel="proton charge radius [fm]"); _ax.legend(fontsize=5.5, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)
_fig.tight_layout(); _save(_fig, "figS6_radii")
_log.append(f"S6 check: electronic mean {_me:.5f} ± {_se:.5f} fm")

_mo_pf.md("**Paper figures** → `" + _OUT + "/`\n\n" + "\n".join(f"- {s}" for s in _log))

In [ ]:
def _():
    # ---- CELL 73: SAVE EVERY MOLAB RUN TO HUGGING FACE (independent: needs nothing from other cells) -------------
    # What it does
    #   1. On start: if the local checkpoint folder is empty (fresh molab session / kernel restart), it restores the
    #      last saved state from the Hub into it, so the notebook resumes instead of recomputing.
    #   2. Uploads the checkpoint folder, the notebook .py file(s) and the paper figures to a private HF dataset repo:
    #        runs/<RUN_ID>/...   one immutable snapshot per sync (full history of every run)
    #        latest/...          always the newest state (used for restore)
    #   3. Keeps syncing in a background thread every SYNC_EVERY_MIN minutes, and once more when the kernel exits.
    #   4. Shows a "Sync now" button and a status table.
    # Re-running the cell does not start a second thread. Every name is private (_hf_*), so it can sit anywhere.
    import os as _hf_os, sys as _hf_sys, time as _hf_time, json as _hf_json, glob as _hf_glob, atexit as _hf_atexit
    import threading as _hf_threading, datetime as _hf_dt, socket as _hf_socket, subprocess as _hf_sp, builtins as _hf_builtins
    import marimo as _hf_mo

    # ======================= HARD-CODED SETTINGS (edit these) =======================
    _HF_TOKEN       = _hf_os.environ.get("HF_TOKEN", "")   # read from the environment; never hard-code a token
    _HF_REPO        = "APassionateResearchers/palatini-molab-runs"   # dataset repo, created automatically (private)
    _HF_LOCAL       = _hf_os.environ.get("PF_CKPT", "pj_ckpt")  # checkpoint folder of the notebook
    _HF_EXTRA_GLOBS = ["*.py", "*.ipynb", "*.md", "paper_v2/**/*", "__marimo__/**/*"]  # also saved, relative to cwd
    _HF_IGNORE      = ["*.tmp", "*.lock", "**/__pycache__/**", "**/.ipynb_checkpoints/**"]
    _HF_MAX_FILE_GB = 20          # skip single files larger than this
    _HF_SYNC_EVERY_MIN = 10       # background sync interval
    _HF_RESTORE_IF_EMPTY = True   # pull latest/ into _HF_LOCAL when it is missing or empty
    # ================================================================================

    try:
        import huggingface_hub as _hf_hub
    except ImportError:
        _hf_sp.run([_hf_sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.25"], check=False)
        import huggingface_hub as _hf_hub
    from huggingface_hub import HfApi as _HfApi, snapshot_download as _hf_snapshot

    _hf_os.environ["HF_TOKEN"] = _HF_TOKEN
    _hf_os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")
    _hf_api = _HfApi(token=_HF_TOKEN)

    # One shared state object per kernel, so re-running the cell reuses the thread and the run id.
    _hf_state = getattr(_hf_builtins, "_HF_SYNC_STATE", None)
    if _hf_state is None:
        _hf_state = {
            "run_id": _hf_dt.datetime.utcnow().strftime("%Y%m%d-%H%M%S") + "-" + _hf_socket.gethostname()[:12],
            "log": [], "n_sync": 0, "last": None, "thread": None, "lock": _hf_threading.Lock(), "stop": False,
        }
        _hf_builtins._HF_SYNC_STATE = _hf_state


    def _hf_note(msg):
        _hf_state["log"].append(f"{_hf_dt.datetime.utcnow():%H:%M:%S} UTC  {msg}")
        _hf_state["log"] = _hf_state["log"][-40:]


    def _hf_ensure_repo():
        _hf_api.create_repo(_HF_REPO, repo_type="dataset", private=True, exist_ok=True)


    def _hf_restore():
        """Pull latest/ from the Hub into the local checkpoint folder if that folder is empty."""
        _empty = (not _hf_os.path.isdir(_HF_LOCAL)) or not any(_hf_os.scandir(_HF_LOCAL))
        if not (_HF_RESTORE_IF_EMPTY and _empty):
            return "local checkpoints present, no restore"
        try:
            _files = _hf_api.list_repo_files(_HF_REPO, repo_type="dataset")
        except Exception as _e:
            return f"restore skipped ({type(_e).__name__})"
        _pref = f"latest/{_hf_os.path.basename(_HF_LOCAL.rstrip('/'))}/"
        if not any(f.startswith(_pref) for f in _files):
            return "nothing on the Hub yet"
        _tmp = _hf_snapshot(_HF_REPO, repo_type="dataset", allow_patterns=[_pref + "*"], token=_HF_TOKEN,
                            local_dir=".hf_restore")
        _src = _hf_os.path.join(_tmp, _pref)
        import shutil as _hf_shutil
        _hf_os.makedirs(_HF_LOCAL, exist_ok=True)
        _n = 0
        for _root, _dirs, _fs in _hf_os.walk(_src):
            for _f in _fs:
                _rel = _hf_os.path.relpath(_hf_os.path.join(_root, _f), _src)
                _dst = _hf_os.path.join(_HF_LOCAL, _rel)
                _hf_os.makedirs(_hf_os.path.dirname(_dst), exist_ok=True)
                _hf_shutil.copy2(_hf_os.path.join(_root, _f), _dst)
                _n += 1
        return f"restored {_n} files from {_HF_REPO}/latest"


    def _hf_collect():
        """List (local_path, path_in_repo) for everything to save."""
        _out, _seen = [], set()
        _base = _hf_os.path.basename(_HF_LOCAL.rstrip("/"))
        from fnmatch import fnmatch as _fnm
        def _add(_p, _rel):
            if _p in _seen or not _hf_os.path.isfile(_p):
                return
            if any(_fnm(_rel, _g) for _g in _HF_IGNORE):
                return
            if _hf_os.path.getsize(_p) > _HF_MAX_FILE_GB * 1e9:
                _hf_note(f"skipped (too large): {_rel}"); return
            _seen.add(_p); _out.append((_p, _rel))
        if _hf_os.path.isdir(_HF_LOCAL):
            for _root, _dirs, _fs in _hf_os.walk(_HF_LOCAL):
                for _f in _fs:
                    _p = _hf_os.path.join(_root, _f)
                    _add(_p, _hf_os.path.join(_base, _hf_os.path.relpath(_p, _HF_LOCAL)))
        for _g in _HF_EXTRA_GLOBS:
            for _p in _hf_glob.glob(_g, recursive=True):
                if not _hf_os.path.abspath(_p).startswith(_hf_os.path.abspath(_HF_LOCAL)) and ".hf_restore" not in _p:
                    _add(_p, _hf_os.path.join("notebook", _hf_os.path.relpath(_p)))
        return _out


    def _hf_sync(reason="manual"):
        """Upload one snapshot to runs/<RUN_ID>/ and mirror it to latest/. Thread-safe."""
        if not _hf_state["lock"].acquire(blocking=False):
            return "sync already running"
        try:
            _ensure = _hf_ensure_repo()
            _items = _hf_collect()
            if not _items:
                _hf_note(f"{reason}: nothing to upload"); return "nothing to upload"
            _ops = []
            for _p, _rel in _items:
                _rel = _rel.replace(_hf_os.sep, "/")
                _ops.append(_hf_hub.CommitOperationAdd(path_in_repo=f"runs/{_hf_state['run_id']}/{_rel}", path_or_fileobj=_p))
                _ops.append(_hf_hub.CommitOperationAdd(path_in_repo=f"latest/{_rel}", path_or_fileobj=_p))
            _meta = {"run_id": _hf_state["run_id"], "reason": reason, "utc": _hf_dt.datetime.utcnow().isoformat(),
                     "n_files": len(_items), "bytes": int(sum(_hf_os.path.getsize(p) for p, _ in _items))}
            _mb = _hf_json.dumps(_meta, indent=1).encode()
            _ops.append(_hf_hub.CommitOperationAdd(path_in_repo=f"runs/{_hf_state['run_id']}/RUN_INFO.json", path_or_fileobj=_mb))
            _ops.append(_hf_hub.CommitOperationAdd(path_in_repo="latest/RUN_INFO.json", path_or_fileobj=_mb))
            # commit in chunks so very large checkpoint sets do not hit request limits
            _chunk = 400
            for _i in range(0, len(_ops), _chunk):
                _hf_api.create_commit(_HF_REPO, repo_type="dataset", operations=_ops[_i:_i + _chunk],
                                      commit_message=f"{reason}: run {_hf_state['run_id']} ({_meta['n_files']} files)")
            _hf_state["n_sync"] += 1; _hf_state["last"] = _meta
            _hf_note(f"{reason}: uploaded {_meta['n_files']} files, {_meta['bytes']/1e6:.1f} MB")
            return "ok"
        except Exception as _e:
            _hf_note(f"{reason}: FAILED {type(_e).__name__}: {str(_e)[:160]}")
            return "failed"
        finally:
            _hf_state["lock"].release()


    def _hf_loop():
        while not _hf_state["stop"]:
            for _ in range(int(_HF_SYNC_EVERY_MIN * 60)):
                if _hf_state["stop"]:
                    return
                _hf_time.sleep(1)
            _hf_sync("periodic")


    # ---- start-up: check token, restore, first sync, background thread, exit hook ----
    try:
        _hf_who = _hf_api.whoami()["name"]
        _hf_ensure_repo()
        _hf_note(f"logged in as {_hf_who}; repo {_HF_REPO} ready")
        _hf_note(_hf_restore())
        _hf_threading.Thread(target=_hf_sync, args=("cell run",), daemon=True).start()
        if _hf_state["thread"] is None or not _hf_state["thread"].is_alive():
            _hf_state["thread"] = _hf_threading.Thread(target=_hf_loop, daemon=True, name="hf-sync")
            _hf_state["thread"].start()
            _hf_atexit.register(lambda: _hf_sync("kernel exit"))
            _hf_note(f"background sync every {_HF_SYNC_EVERY_MIN} min started")
    except Exception as _e:
        _hf_note(f"setup FAILED {type(_e).__name__}: {str(_e)[:200]} (check _HF_TOKEN and _HF_REPO)")

    _hf_btn = _hf_mo.ui.button(label="Sync to Hugging Face now", on_click=lambda _v: _hf_sync("button"))
    _hf_mo.vstack([
        _hf_mo.md(
            f"**Hugging Face run backup** → [`{_HF_REPO}`](https://huggingface.co/datasets/{_HF_REPO}) (private)  \n"
            f"run id `{_hf_state['run_id']}` · local folder `{_HF_LOCAL}` · syncs so far: {_hf_state['n_sync']} "
            f"(first upload runs in the background; re-run this cell to refresh the log)"),
        _hf_btn,
        _hf_mo.md("```\n" + "\n".join(_hf_state["log"][-15:]) + "\n```"),
    ])
_()

In [ ]:
# ---- CELL 74: COLLECT ALL PAPER FIGURES INTO figures/ WITH EXACT NAMES AND UPLOAD TO HUGGING FACE -------------
# Independent cell: needs nothing from other cells. Run it after cell 72 (which draws the figures).
# What it does
#   1. Finds every figure the paper uses (exact names below) in the notebook outputs, mainly
#      pj_ckpt/paper_figs/ (written by cell 72) and, for Fig. S1, the Part I export profile_posterior.png.
#   2. Copies them into ./figures/ under the exact file names that main.tex and appendix.tex use
#      (both .png and, where available, .pdf).
#   3. Writes figures/MANIFEST.json (name, source, size, sha256) and figures.zip.
#   4. Uploads figures/ and figures.zip to your private Hugging Face dataset repo, under paper/figures/.
#      Download that folder (or figures.zip), put it next to main.tex, and compile; the names already match.
# Every name is private (_fg_*), so the cell can sit anywhere in the notebook.
import os as _fg_os, sys as _fg_sys, glob as _fg_glob, json as _fg_json, shutil as _fg_shutil, hashlib as _fg_hash
import zipfile as _fg_zip, subprocess as _fg_sp, datetime as _fg_dt
import marimo as _fg_mo

# ======================= HARD-CODED SETTINGS (edit these) =======================
_FG_TOKEN  = _fg_os.environ.get("HF_TOKEN", "")   # read from the environment; never hard-code a token
_FG_REPO   = "APassionateResearchers/palatini-molab-runs"      # dataset repo (created if missing, private)
_FG_CKPT   = _fg_os.environ.get("PF_CKPT", "pj_ckpt")    # notebook checkpoint folder
_FG_OUTDIR = "figures"                                   # local folder that main.tex reads
_FG_HUBDIR = "paper/figures"                             # folder inside the HF repo
# ================================================================================

# Exact names used by main.tex / appendix.tex, and where cell 72 (or Part I) writes them.
_FG_NEEDED = {
    "fig1_bound":      ["paper_figs/fig1_bound"],
    "fig2_stars":      ["paper_figs/fig2_stars"],
    "figS1_cosmology": ["paper_figs/figS1_cosmology", "**/profile_posterior"],
    "figS2_wall":      ["paper_figs/figS2_wall"],
    "figS3_degeneracy":["paper_figs/figS3_degeneracy"],
    "figS4_ceiling":   ["paper_figs/figS4_ceiling"],
    "figS5_posterior": ["paper_figs/figS5_posterior"],
    "figS6_radii":     ["paper_figs/figS6_radii"],
}
_FG_EXTS = [".png", ".pdf"]


def _fg_sha(_p):
    _h = _fg_hash.sha256()
    with open(_p, "rb") as _f:
        for _b in iter(lambda: _f.read(1 << 20), b""):
            _h.update(_b)
    return _h.hexdigest()


def _fg_find(_pattern, _ext):
    """Search the checkpoint folder, then the working directory, for pattern+ext."""
    for _root in (_FG_CKPT, "."):
        _hits = sorted(_fg_glob.glob(_fg_os.path.join(_root, _pattern + _ext), recursive=True),
                       key=_fg_os.path.getmtime, reverse=True)
        _hits = [h for h in _hits if not _fg_os.path.abspath(h).startswith(_fg_os.path.abspath(_FG_OUTDIR))]
        if _hits:
            return _hits[0]
    return None


# ---- 1-2. collect into figures/ ----
_fg_os.makedirs(_FG_OUTDIR, exist_ok=True)
_fg_manifest, _fg_missing = [], []
for _name, _patterns in _FG_NEEDED.items():
    _got_png = False
    for _ext in _FG_EXTS:
        _src = None
        for _pat in _patterns:
            _src = _fg_find(_pat, _ext)
            if _src:
                break
        if _src is None:
            continue
        _dst = _fg_os.path.join(_FG_OUTDIR, _name + _ext)
        _fg_shutil.copy2(_src, _dst)
        _fg_manifest.append({"file": _name + _ext, "source": _src,
                             "bytes": _fg_os.path.getsize(_dst), "sha256": _fg_sha(_dst)})
        _got_png = _got_png or _ext == ".png"
    if not _got_png:
        _fg_missing.append(_name + ".png")

# The paper compiles from the .png files; warn loudly if any is missing.
_fg_info = {"created_utc": _fg_dt.datetime.utcnow().isoformat(), "files": _fg_manifest,
            "missing_png": _fg_missing}
with open(_fg_os.path.join(_FG_OUTDIR, "MANIFEST.json"), "w") as _f:
    _fg_json.dump(_fg_info, _f, indent=1)

# ---- 3. zip ----
_fg_zipname = "figures.zip"
with _fg_zip.ZipFile(_fg_zipname, "w", _fg_zip.ZIP_DEFLATED) as _z:
    for _p in sorted(_fg_glob.glob(_fg_os.path.join(_FG_OUTDIR, "*"))):
        _z.write(_p, arcname=_fg_os.path.join("figures", _fg_os.path.basename(_p)))

# ---- 4. upload ----
_fg_status = []
try:
    try:
        import huggingface_hub as _fg_hub
    except ImportError:
        _fg_sp.run([_fg_sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.25"], check=False)
        import huggingface_hub as _fg_hub
    _fg_api = _fg_hub.HfApi(token=_FG_TOKEN)
    _fg_api.create_repo(_FG_REPO, repo_type="dataset", private=True, exist_ok=True)
    _fg_api.upload_folder(folder_path=_FG_OUTDIR, path_in_repo=_FG_HUBDIR, repo_id=_FG_REPO,
                          repo_type="dataset", commit_message="paper figures (exact names)")
    _fg_api.upload_file(path_or_fileobj=_fg_zipname, path_in_repo="paper/figures.zip", repo_id=_FG_REPO,
                        repo_type="dataset", commit_message="paper figures zip")
    _fg_status.append(f"uploaded {len(_fg_manifest)} files to `{_FG_REPO}/{_FG_HUBDIR}` and `paper/figures.zip`")
except Exception as _e:
    _fg_status.append(f"upload FAILED: {type(_e).__name__}: {str(_e)[:200]} (check _FG_TOKEN and _FG_REPO)")

_fg_rows = "\n".join(f"| `{m['file']}` | {m['bytes']/1e3:.0f} kB | `{m['source']}` |" for m in _fg_manifest)
_fg_mo.md(
    f"**Paper figures** → local `{_FG_OUTDIR}/` and Hugging Face "
    f"[`{_FG_REPO}`](https://huggingface.co/datasets/{_FG_REPO}/tree/main/{_FG_HUBDIR})\n\n"
    "| file | size | taken from |\n|---|---|---|\n" + _fg_rows + "\n\n"
    + ("**Missing (run cell 72 first, or re-export Part I for S1):** " + ", ".join(_fg_missing) + "\n\n" if _fg_missing
       else "All 8 figures present.\n\n")
    + "\n".join(f"- {s}" for s in _fg_status)
    + "\n\nTo use: download `paper/figures/` (or `paper/figures.zip`) from the repo, place the `figures` folder "
      "next to `main.tex`, then run pdflatex → bibtex → pdflatex → pdflatex."
)

In [ ]:
# ---- CELL 75: ROBUSTNESS OF THE ATOMIC BOUND TO EVERY STATISTICAL CHOICE (independent, CPU, < 1 s) ----------
# Answers the referee questions: "Does the bound depend on the Birge inflation? on one hydrogen measurement? on the
# correlation between the H and D-H inputs? on the nuclear-polarisability systematics of the isotope pairs?"
# Inputs are exactly the published values used in the paper (Tables V and VII). Writes pj_ckpt/referee/ATOMIC_ROBUST.json.
import os as _os_ar, json as _json_ar, math as _m_ar
import numpy as _np_ar
import marimo as _mo_ar

_OUT_AR = _os_ar.path.join(_os_ar.environ.get("PF_CKPT", "pj_ckpt"), "referee"); _os_ar.makedirs(_OUT_AR, exist_ok=True)

# ---- constants and levers (Eq. 21 of the paper) ------------------------------------------------------------
_MPL, _ALPHA_EM = 1.22089e19, 1 / 137.035999
_ME, _MMU, _MP = 0.51099895e-3, 0.1056583755, 0.93827209
_MD, _MH3, _MA = 1.87561294, 2.80839160, 3.72737941          # nuclear masses (GeV)
_FM = 5.0677307                                              # 1 fm in GeV^-1
_KM2 = (1e18 * _FM) ** 2                                     # 1 km^2 in GeV^-2
_KAPPA = 8 * _m_ar.pi / _MPL**2

def _lever(M, Z):
    """d_e - d_mu per km^2 of (alpha+beta), in fm^2: 3 kappa M (m_mu - m_e) / (2 pi Z alpha_em)."""
    return 3 * _KAPPA * _KM2 * M * (_MMU - _ME) / (2 * _m_ar.pi * Z * _ALPHA_EM) / _FM**2

_LAM = {"H": _lever(_MP, 1), "D-H": _lever(_MD - _MP, 1), "3He-4He": _lever(_MH3 - _MA, 2), "4He": _lever(_MA, 2)}

# ---- electronic hydrogen (Table V) and the other systems ---------------------------------------------------
_EH = {"Beyer 2017": (0.8335, 0.0095), "Fleurbaey 2018": (0.877, 0.013), "Bezginov 2019": (0.833, 0.010),
       "Grinin 2020": (0.8482, 0.0038), "Brandt 2022": (0.8584, 0.0051), "Maisenbacher 2026": (0.8406, 0.0015),
       "Bullis 2026": (0.8433, 0.0031)}
_MUH = (0.84087, 0.00039)
_DH = ((3.8207, 0.0003), (3.8200, 0.0031))          # (electronic, muonic) r_d^2 - r_p^2 [fm^2]
_HE = ((1.0678, 0.0007), (1.0636, 0.0031))          # r_h^2 - r_alpha^2 [fm^2]
_HE4 = ((1.681, 0.004), (1.67824, 0.00083))         # r_alpha [fm]

def _wmean(vals, errs, mode="birge"):
    v, e = _np_ar.asarray(vals, float), _np_ar.asarray(errs, float)
    w = 1 / e**2; m = (w * v).sum() / w.sum(); s = 1 / _m_ar.sqrt(w.sum())
    chi2 = float((w * (v - m) ** 2).sum()); dof = len(v) - 1
    if mode == "birge":
        s *= max(1.0, _m_ar.sqrt(chi2 / dof)) if dof > 0 else 1.0
    elif mode == "random":                               # DerSimonian-Laird random-effects mean
        tau2 = max(0.0, (chi2 - dof) / (w.sum() - (w**2).sum() / w.sum())) if dof > 0 else 0.0
        ws = 1 / (e**2 + tau2); m = (ws * v).sum() / ws.sum(); s = 1 / _m_ar.sqrt(ws.sum())
    return m, s, chi2, dof

def _sys_H(rp, srp, mu=_MUH):
    d = rp**2 - mu[0] ** 2; s = _m_ar.hypot(2 * rp * srp, 2 * mu[0] * mu[1])
    return d / _LAM["H"], s / _LAM["H"]

def _sys_diff(pair, lam, scale_mu=1.0):
    (de, se), (dm, sm) = pair
    return (de - dm) / lam, _m_ar.hypot(se, scale_mu * sm) / abs(lam)

def _sys_He4(scale=1.0):
    (re, se), (rm, sm) = _HE4
    d = re**2 - rm**2; s = _m_ar.hypot(2 * re * se, scale * 2 * rm * sm)
    return d / _LAM["4He"], s / _LAM["4He"]

def _combine(xs, ss, corr=None):
    """GLS combination with optional correlation matrix; returns mean, sigma, chi2."""
    x = _np_ar.asarray(xs, float); s = _np_ar.asarray(ss, float)
    C = _np_ar.diag(s**2) if corr is None else corr * _np_ar.outer(s, s)
    Ci = _np_ar.linalg.inv(C); one = _np_ar.ones_like(x)
    var = 1 / (one @ Ci @ one); m = var * (one @ Ci @ x)
    r = x - m
    return float(m), float(_m_ar.sqrt(var)), float(r @ Ci @ r)

_U = 1e3   # report in 10^-3 km^2
_rows, _res = [], {}

# (0) baseline: must reproduce Table III of the paper
_m0, _s0, _c0, _d0 = _wmean([v for v, _ in _EH.values()], [e for _, e in _EH.values()], "birge")
_H = _sys_H(_m0, _s0); _DHs = _sys_diff(_DH, _LAM["D-H"]); _HEs = _sys_diff(_HE, _LAM["3He-4He"]); _H4 = _sys_He4()
_cb = _combine([_H[0], _DHs[0], _HEs[0], _H4[0]], [_H[1], _DHs[1], _HEs[1], _H4[1]])
_res["baseline"] = {"r_e_mean": _m0, "r_e_sigma_birge": _s0, "chi2_electronic": _c0, "levers": _LAM,
                    "H": _H, "D-H": _DHs, "3He-4He": _HEs, "4He": _H4, "combined": _cb}
_rows.append(("Baseline (paper)", _H, _cb))

# (1) random-effects instead of Birge inflation
_m1, _s1, _, _ = _wmean([v for v, _ in _EH.values()], [e for _, e in _EH.values()], "random")
_H1 = _sys_H(_m1, _s1)
_rows.append(("Random-effects mean (DerSimonian-Laird)", _H1,
              _combine([_H1[0], _DHs[0], _HEs[0], _H4[0]], [_H1[1], _DHs[1], _HEs[1], _H4[1]])))

# (2) jackknife: drop one electronic hydrogen measurement at a time
_jk = {}
for _k in _EH:
    _sub = {kk: vv for kk, vv in _EH.items() if kk != _k}
    _mj, _sj, _, _ = _wmean([v for v, _ in _sub.values()], [e for _, e in _sub.values()], "birge")
    _jk[_k] = _sys_H(_mj, _sj)
_jlo = min(_jk.values(), key=lambda t: t[0]); _jhi = max(_jk.values(), key=lambda t: t[0])
_rows.append((f"Jackknife, lowest (drop {min(_jk, key=lambda k: _jk[k][0])})", _jlo, None))
_rows.append((f"Jackknife, highest (drop {max(_jk, key=lambda k: _jk[k][0])})", _jhi, None))
_res["jackknife_H"] = _jk

# (3) correlation between H and D-H (shared muonic-hydrogen input)
for _c in (0.5, 0.9):
    _C = _np_ar.eye(4); _C[0, 1] = _C[1, 0] = _c
    _rows.append((f"H/D-H correlation c = {_c}", None,
                  _combine([_H[0], _DHs[0], _HEs[0], _H4[0]], [_H[1], _DHs[1], _HEs[1], _H4[1]], _C)))

# (4) nuclear-polarisability systematics: double the muonic uncertainties of the isotope pairs and of 4He
_DH2 = _sys_diff(_DH, _LAM["D-H"], 2.0); _HE2 = _sys_diff(_HE, _LAM["3He-4He"], 2.0); _H42 = _sys_He4(2.0)
_rows.append(("Muonic errors x2 for D-H, 3He-4He, 4He", None,
              _combine([_H[0], _DH2[0], _HE2[0], _H42[0]], [_H[1], _DH2[1], _HE2[1], _H42[1]])))

# (5) leave one system out of the combination
_names = ["H", "D-H", "3He-4He", "4He"]; _vals = [_H, _DHs, _HEs, _H4]
for _i, _n in enumerate(_names):
    _xs = [v[0] for j, v in enumerate(_vals) if j != _i]; _ss = [v[1] for j, v in enumerate(_vals) if j != _i]
    _rows.append((f"Combination without {_n}", None, _combine(_xs, _ss)))

# (6) M/Z scaling: compare the contact model (shift = Lambda_i x coupling) with a lepton-universal-looking constant
#     shift of all squared radii (same delta for every system). Both have one parameter, so chi2 compares them directly.
_shift = [(_v[0] * _LAM[_n], _v[1] * abs(_LAM[_n])) for _n, _v in zip(_names, _vals)]   # d_e - d_mu in fm^2
_mB, _sB, _cB = _combine([d for d, _ in _shift], [s for _, s in _shift])
_res["MZ_scaling_test"] = {"chi2_contact_model": _cb[2], "chi2_constant_shift_model": _cB,
                           "constant_shift_fm2": (_mB, _sB), "dof": 3}

# ---- envelope over all rows --------------------------------------------------------------------------------
def _ci(m, s): return (m - 1.96 * s, m + 1.96 * s)
_bounds = []
_tab = []
for _lab, _h, _c in _rows:
    _hh = f"{_U*_h[0]:+.1f} ± {_U*_h[1]:.1f}" if _h else "—"
    _cc = f"{_U*_c[0]:+.1f} ± {_U*_c[1]:.1f}" if _c else "—"
    _x2 = f"{_c[2]:.2f}" if _c else "—"
    _tab.append(f"| {_lab} | {_hh} | {_cc} | {_x2} |")
    for _t in (_h, _c):
        if _t:
            _bounds += list(_ci(_t[0], _t[1]))
_env = max(abs(b) for b in _bounds)
_res["rows"] = [{"label": l, "H": h, "combined": c} for l, h, c in _rows]
_res["envelope_abs95_km2"] = _env
with open(_os_ar.path.join(_OUT_AR, "ATOMIC_ROBUST.json"), "w") as _f:
    _json_ar.dump(_res, _f, indent=1, default=float)

_mo_ar.md(f"""
### Referee check 1: robustness of the atomic bound

Levers (fm² km⁻²): H {_LAM['H']:.4f}, D−H {_LAM['D-H']:.4f}, ³He−⁴He {_LAM['3He-4He']:.4f}, ⁴He {_LAM['4He']:.4f}
(paper: 1.088, 1.087, −0.533, 2.162). Electronic mean {_m0:.5f} ± {_s0:.5f} fm, χ² = {_c0:.1f} for {_d0} dof
(paper: 0.84287 ± 0.00234, 22.4/6).

| choice | hydrogen only (10⁻³ km²) | combined (10⁻³ km²) | χ² |
|---|---|---|---|
{chr(10).join(_tab)}

- **Largest |α+β| at 95% over all these choices: {_env:.4f} km²**, inside the paper's envelope of 0.032 km²
  (which is set by the older CODATA selection, not by any statistical choice made here).
- M/Z scaling: χ² = {_cb[2]:.2f} for the contact model (levers ∝ M/Z) versus χ² = {_cB:.2f} for a common constant
  shift of all squared radii, both with 3 dof. {"Both are acceptable fits, so current data are consistent with the M/Z pattern but do not yet single it out; the bound on α+β does not rely on it." if abs(_cb[2] - _cB) < 4 else ("The contact model is preferred." if _cb[2] < _cB else "A constant shift fits clearly better; report this.")}
- Saved `{_OUT_AR}/ATOMIC_ROBUST.json`.
""")

In [ ]:
# ---- CELL 76: INDEPENDENT NUMERICAL CHECKS OF EVERY CLOSED-FORM RESULT (independent, CPU, ~10 s) -------------
# Checks, with code that shares nothing with the Part VII core:
#   A. Proposition 1 (conformal matter): closed form vs direct root-finding of Eq. (7); fold at beta*kappa*eps = 3/16;
#      effective pressure vanishes at 0.18; beta < 0 limit 9/2; second-order trace anomaly (8/9) b (1 + 2 b).
#   B. Theorem 1(ii) first-order formulas and the second-order beta = 0 terms (Appendix D) vs the exact map.
#   C. Proposition 7 (exact effective sound speed, beta = 0) vs finite differences of the exact map.
#   D. Proposition 2 (Bayes factor = posterior mean of the prior-density ratio) on the Gaussian model, by Monte Carlo.
#   E. Finite nuclear size: contact energy with a smeared mass distribution vs the 1 - 2<r>/a estimate.
# Units: kappa = 1 (scale-free); couplings enter only through a = alpha*kappa*eps-type products.
# Writes pj_ckpt/referee/EXACT_CHECKS.json.
import os as _os_ex, json as _json_ex, math as _m_ex
import numpy as _np_ex
from scipy.optimize import fsolve as _fsolve_ex, brentq as _brentq_ex
from scipy.integrate import quad as _quad_ex
import marimo as _mo_ex

_OUT_EX = _os_ex.path.join(_os_ex.environ.get("PF_CKPT", "pj_ckpt"), "referee"); _os_ex.makedirs(_OUT_EX, exist_ok=True)
_R_EX = {}

def _resid(P, eps, p, a, b):
    """Eq. (7) with kappa = 1: residuals of the a=0 and a=1 components."""
    P0, P1 = P; R = P0 + 3 * P1; Q = P0**2 + 3 * P1**2
    out = []
    for Pa, Ta in ((P0, -eps), (P1, p)):
        out.append(2 * b * Pa**2 + (1 + 2 * a * R) * Pa - 0.5 * (R + a * R**2 + b * Q) - Ta)
    return out

def _jac(P, a, b):
    """Analytic 2x2 Jacobian of _resid with respect to (P0, P1)."""
    P0, P1 = P; R = P0 + 3 * P1
    dR = (1.0, 3.0); dQ = (2 * P0, 6 * P1)
    J = _np_ex.zeros((2, 2))
    for i, Pa in enumerate((P0, P1)):
        for c in range(2):
            J[i, c] = ((4 * b * Pa + 1 + 2 * a * R) * (i == c) + 2 * a * dR[c] * Pa
                       - 0.5 * (dR[c] + 2 * a * R * dR[c] + b * dQ[c]))
    return J

def _newton(P, eps, p, a, b, tol=1e-12, itmax=200):
    """Damped Newton with the analytic Jacobian; success is judged by the residual, not by a solver flag."""
    P = _np_ex.array(P, dtype=float)
    r = _np_ex.array(_resid(P, eps, p, a, b))
    for _it in range(itmax):
        if _np_ex.max(_np_ex.abs(r)) < tol:
            return P
        try:
            dx = _np_ex.linalg.solve(_jac(P, a, b), -r)
        except _np_ex.linalg.LinAlgError:
            return None
        lam, n0 = 1.0, _np_ex.linalg.norm(r)
        while lam > 1e-8:
            Pn = P + lam * dx; rn = _np_ex.array(_resid(Pn, eps, p, a, b))
            if _np_ex.all(_np_ex.isfinite(rn)) and _np_ex.linalg.norm(rn) < n0:
                break
            lam *= 0.5
        else:
            return None
        P, r = Pn, rn
    return P if _np_ex.max(_np_ex.abs(r)) < 1e-10 else None

def _solve(eps, p, a, b, steps=40):
    """GR-connected solution by continuation t = 0 -> 1 (as in the paper). Returns None if the branch is lost."""
    P = _np_ex.array([-(eps + 3 * p) / 2, (eps - p) / 2])
    for t in _np_ex.linspace(0, 1, steps + 1)[1:]:
        P = _newton(P, eps, p, t * a, t * b)
        if P is None:
            return None
    return P

def _eff(P, a, b):
    P0, P1 = P; R = P0 + 3 * P1
    s0, s1 = 1 + 2 * a * R + 2 * b * P0, 1 + 2 * a * R + 2 * b * P1
    if s0 <= 0 or s1 <= 0:
        return None
    R0, R1 = s0 * P0 / _m_ex.sqrt(s0 * s1**3), s1 * P1 / _m_ex.sqrt(s0 * s1**3)
    return 0.5 * (3 * R1 - R0), -0.5 * (R0 + R1)

# ---- A. conformal matter ----------------------------------------------------------------------------------
def _conf_closed(bk):           # bk = beta*kappa*eps with eps = 1
    b = bk / 3; s = _m_ex.sqrt(1 - 16 * b)
    beta = bk                    # eps = kappa = 1  => beta = bk
    P1 = (1 - s) / (8 * beta) if beta != 0 else 1 / 3
    s0, s1 = (1 + 3 * s) / 4, (5 - s) / 4
    D = _m_ex.sqrt(s0 * s1**3)
    return 3 * (3 + s) / 4 * P1 / D, (5 * s - 1) / 4 * P1 / D, 4 * (1 - s) / (3 * (3 + s))

_errA, _failA = 0.0, []
for _bk in (0.01, 0.05, 0.1, 0.15, 0.18, 0.185, -0.5, -2.0, -4.0):
    _P = _solve(1.0, 1 / 3, 0.0, _bk, steps=100)
    _num = None if _P is None else _eff(_P, 0.0, _bk)
    if _num is None:
        _failA.append(_bk); continue
    _cl = _conf_closed(_bk)
    _errA = max(_errA, abs(_num[0] - _cl[0]), abs(_num[1] - _cl[1]))
_R_EX["A_closed_vs_numeric_max_abs_err"] = _errA
_R_EX["A_points_without_numeric_solution"] = _failA          # expect [] (all points lie inside the allowed range)
# fold: largest beta*kappa*eps with a solution (bisection on solvability of the reduced quadratic, then full system)
def _exists(bk):
    """Purely numerical: does continuation from GR reach a real solution of the full 2x2 system?"""
    _P = _solve(1.0, 1 / 3, 0.0, bk, steps=200)
    return _P is not None and bool(_np_ex.all(_np_ex.abs(_resid(_P, 1.0, 1 / 3, 0.0, bk)) < 1e-10))
_lo, _hi = 0.15, 0.30
for _ in range(60):
    _mid = 0.5 * (_lo + _hi)
    if _exists(_mid): _lo = _mid
    else: _hi = _mid
_R_EX["A_fold_numeric"] = _lo; _R_EX["A_fold_exact"] = 3 / 16
_R_EX["A_pressure_zero_numeric"] = _brentq_ex(lambda bk: _conf_closed(bk)[1], 0.1, 0.1874)
_R_EX["A_pressure_zero_exact"] = 0.18
_R_EX["A_beta_negative_limit_exact"] = 4.5
_R_EX["A_sigma1_at_minus_4.49"] = (5 - _m_ex.sqrt(1 + 16 * 4.49 / 3)) / 4
_R_EX["A_sigma1_at_minus_4.51"] = (5 - _m_ex.sqrt(1 + 16 * 4.51 / 3)) / 4
_rat = []
for _bk in (1e-3, 2e-3, 4e-3):
    _D = _conf_closed(_bk)[2]; _rat.append((_D / (8 / 9 * _bk) - 1) / _bk)
_R_EX["A_second_order_coefficient_numeric"] = _rat[0]; _R_EX["A_second_order_coefficient_exact"] = 2.0

# ---- B. first- and second-order expansions -----------------------------------------------------------------
def _lo_formula(eps, p, a, b):
    de = -a / 2 * (7 * eps + 3 * p) * (eps - 3 * p) - b / 2 * (eps**2 - 6 * eps * p - 3 * p**2)
    dp = -a / 2 * (eps + 5 * p) * (eps - 3 * p) - b / 2 * (eps + 3 * p) * (eps - p)
    return eps + de, p + dp

_expB = {}
for _name, (_aa, _bb) in {"alpha": (1.0, 0.0), "beta": (0.0, 1.0), "mixed": (0.7, -0.5)}.items():
    _errs = []
    for _h in (1e-2, 5e-3, 2.5e-3):
        _P = _solve(1.0, 0.2, _h * _aa, _h * _bb)
        _num = None if _P is None else _eff(_P, _h * _aa, _h * _bb)
        if _num is None:
            raise RuntimeError(f"exact map failed at {_name}, h={_h} (should not happen for small couplings)")
        _lo2 = _lo_formula(1.0, 0.2, _h * _aa, _h * _bb)
        _errs.append(max(abs(_num[0] - _lo2[0]), abs(_num[1] - _lo2[1])))
    _expB[_name] = _m_ex.log2(_errs[1] / _errs[2])
_R_EX["B_first_order_residual_exponents"] = _expB   # expect 2
def _fR_exact(eps, p, a):
    D = eps - 3 * p; F = 1 + 2 * a * D
    return (eps + a * D**2 / 2) / F**2, (p - a * D**2 / 2) / F**2
_e2 = []
for _a in (1e-2, 5e-3):
    _ex = _fR_exact(1.0, 0.2, _a); _D = 0.4
    _s2 = (1 - _a / 2 * _D * 7.6 + _a**2 * _D**2 * 11.2, 0.2 - _a / 2 * _D * 2.0 + _a**2 * _D**2 * 3.2)
    _e2.append(max(abs(_ex[0] - _s2[0]), abs(_ex[1] - _s2[1])))
_R_EX["B_second_order_residual_exponent"] = _m_ex.log2(_e2[0] / _e2[1])   # expect 3

# ---- C. exact effective sound speed (beta = 0) -------------------------------------------------------------
def _cs_formula(eps, p, c2, a):
    D = eps - 3 * p
    return (c2 * (1 + 2 * a * D) - a * (1 - 3 * c2) * (eps + p)) / (1 + 2 * a * D - 3 * a * (1 - 3 * c2) * (eps + p))
_errC = 0.0
for _a in (-0.3, -0.05, 0.05, 0.2):
    for _c2 in (0.05, 0.2, 0.6):
        _e0, _p0, _h = 1.0, 0.15, 1e-6
        _pl = _fR_exact(_e0 + _h, _p0 + _c2 * _h, _a); _mi = _fR_exact(_e0 - _h, _p0 - _c2 * _h, _a)
        _fd = (_pl[1] - _mi[1]) / (_pl[0] - _mi[0])
        _errC = max(_errC, abs(_fd - _cs_formula(_e0, _p0, _c2, _a)))
_R_EX["C_sound_speed_formula_vs_finite_difference"] = _errC

# ---- D. Proposition 2 on the Gaussian model ----------------------------------------------------------------
_rng = _np_ex.random.default_rng(1)
_sp, _s, _d, _c = 1.0, 0.3, 0.4, 0.5
_th = _rng.normal(0, _sp, 2_000_000)
_lik = lambda x: _np_ex.exp(-(_d - x) ** 2 / (2 * _s**2)) / _m_ex.sqrt(2 * _m_ex.pi * _s**2)
_D = []
for _al in (-1.0, 0.5, 2.0):
    _B_mc = _lik(_th + _c * _al).mean() / _lik(_th).mean()
    _B_an = _m_ex.exp((2 * _d * _c * _al - (_c * _al) ** 2) / (2 * (_sp**2 + _s**2)))
    _w = _lik(_th); _r = _np_ex.exp((2 * _c * _al * _th - (_c * _al) ** 2) / (2 * _sp**2))
    _B_prop = float((_w * _r).sum() / _w.sum())
    _D.append({"alpha": _al, "B_monte_carlo": float(_B_mc), "B_analytic": _B_an, "B_posterior_mean_of_r": _B_prop})
_R_EX["D_bayes_proposition_gaussian"] = _D

# ---- E. finite nuclear size of the contact term -------------------------------------------------------------
_E = []
for _sysname, _a_fm, _rm in (("muonic H", 285.0, 0.7), ("muonic 4He+", 130.0, 1.4)):
    _lam = 3 / _rm                                       # exponential mass profile with <r> = 3/lambda
    _rho = lambda r: _lam**3 / (8 * _m_ex.pi) * _m_ex.exp(-_lam * r)
    _ov = _quad_ex(lambda r: 4 * _m_ex.pi * r * r * _rho(r) * _m_ex.exp(-2 * r / _a_fm), 0, 60 * _rm, limit=200)[0]
    _E.append({"system": _sysname, "relative_correction_exact": 1 - _ov, "estimate_2r_over_a": 2 * _rm / _a_fm})
_R_EX["E_finite_size"] = _E

with open(_os_ex.path.join(_OUT_EX, "EXACT_CHECKS.json"), "w") as _f:
    _json_ex.dump(_R_EX, _f, indent=1, default=float)

_mo_ex.md(f"""
### Referee check 2: independent checks of every closed-form result

| check | numerical | exact / expected |
|---|---|---|
| Prop. 1 closed form vs root-finding (max abs. error, βκε from −4 to 0.185) | {_errA:.1e} | round-off |
| Prop. 1 test points where root-finding lost the branch | {_failA if _failA else 'none'} | none |
| Prop. 1 fold, βκε | {_R_EX['A_fold_numeric']:.6f} | 3/16 = 0.187500 |
| Prop. 1 effective pressure vanishes, βκε | {_R_EX['A_pressure_zero_numeric']:.6f} | 0.18 |
| Prop. 1 σ₁ just inside / outside βκε = −9/2 | {_R_EX['A_sigma1_at_minus_4.49']:+.4f} / {_R_EX['A_sigma1_at_minus_4.51']:+.4f} | + / − |
| Second-order coefficient of Δ̃ | {_R_EX['A_second_order_coefficient_numeric']:.3f} | 2 (small-b limit) |
| Thm. 1(ii) residual exponents (α, β, mixed) | {', '.join(f'{v:.2f}' for v in _expB.values())} | 2 |
| App. D second-order residual exponent | {_R_EX['B_second_order_residual_exponent']:.2f} | 3 |
| Prop. 7 sound speed vs finite difference | {_errC:.1e} | round-off |
| Prop. 2, Gaussian model: B (MC / analytic / posterior mean of r) | {'; '.join(f"{x['B_monte_carlo']:.4f} / {x['B_analytic']:.4f} / {x['B_posterior_mean_of_r']:.4f}" for x in _D)} | equal |
| Finite size, muonic H: exact / 2⟨r⟩/a | {_E[0]['relative_correction_exact']:.2e} / {_E[0]['estimate_2r_over_a']:.2e} | equal to leading order |
| Finite size, muonic ⁴He⁺: exact / 2⟨r⟩/a | {_E[1]['relative_correction_exact']:.2e} / {_E[1]['estimate_2r_over_a']:.2e} | equal to leading order |

Saved `{_OUT_EX}/EXACT_CHECKS.json`.
""")

In [ ]:
# ---- CELL 77: NEUTRON-STAR EVIDENCE — MONTE CARLO ERRORS AND EOS-PRIOR SENSITIVITY (independent; reads Part VII
#      checkpoints only; CPU, ~1 min) ----------------------------------------------------------------------------
# Answers: "How large is the Monte Carlo error on log10 B(alpha)?", "Is |log10 B| < 0.5 an artefact of one EOS prior?",
# and "What actually excludes the upper end of the alpha interval?" (Proposition 2: only the loss of valid stars).
# Reads pj_ckpt/part7/a<alpha>_b+000.000_batch<k>.npz (keys ll, pqcd, R14, Mmax, valid). No new TOV runs.
# Writes pj_ckpt/referee/NS_BAYES_ROBUST.json.
import os as _os_nb, re as _re_nb, glob as _glob_nb, json as _json_nb
import numpy as _np_nb
import marimo as _mo_nb

_CK_NB = _os_nb.environ.get("PF_CKPT", "pj_ckpt")
_OUT_NB = _os_nb.path.join(_CK_NB, "referee"); _os_nb.makedirs(_OUT_NB, exist_ok=True)
_files = _glob_nb.glob(_os_nb.path.join(_CK_NB, "part7", "a*_b+000.000_batch*.npz"))
_mo_nb.stop(not _files, _mo_nb.md(f"No Part VII checkpoints found in `{_CK_NB}/part7/`. Restore them from the Hugging Face "
                                   "dataset (folder `latest/pj_ckpt/part7/`) or run Part VII first."))

_by_a = {}
for _f in _files:
    _mm = _re_nb.search(r"a([+-]\d+\.\d+)_b\+000\.000_batch(\d+)\.npz$", _os_nb.path.basename(_f))
    if _mm:
        _by_a.setdefault(float(_mm.group(1)), []).append((int(_mm.group(2)), _f))
_alphas = _np_nb.array(sorted(_by_a))
_nbatch = min(len(v) for v in _by_a.values())

def _load(a, key):
    return _np_nb.concatenate([_np_nb.atleast_1d(_np_nb.load(f)[key]) for _, f in sorted(_by_a[a])[:_nbatch]])

_L = {a: _load(a, "ll") + _load(a, "pqcd") for a in _alphas}            # astro + pQCD, per EOS
_VAL = {a: _load(a, "valid") for a in _alphas}
_i0 = int(_np_nb.argmin(_np_nb.abs(_alphas)))
_R14_0 = _load(_alphas[_i0], "R14"); _MMAX_0 = _load(_alphas[_i0], "Mmax")
_N = len(_L[_alphas[_i0]])

def _clean(x):
    return _np_nb.where(x < -1e29, -_np_nb.inf, x)

def _logZ(x, w=None):
    x = _clean(x); w = _np_nb.ones_like(x) if w is None else w
    m = _np_nb.max(_np_nb.where(w > 0, x, -_np_nb.inf))
    if not _np_nb.isfinite(m):
        return -_np_nb.inf
    return float(m + _np_nb.log((w * _np_nb.exp(x - m)).sum() / w.sum()))

def _interval(lz, lev=0.95):
    w = _np_nb.exp(lz - _np_nb.max(lz)); w = _np_nb.where(_np_nb.isfinite(w), w, 0.0)
    c = _np_nb.concatenate([[0], _np_nb.cumsum(0.5 * (w[1:] + w[:-1]) * _np_nb.diff(_alphas))]); c /= c[-1]
    return float(_np_nb.interp((1 - lev) / 2, c, _alphas)), float(_np_nb.interp(1 - (1 - lev) / 2, c, _alphas))

def _summary(w=None):
    lz = _np_nb.array([_logZ(_L[a], w) for a in _alphas])
    lb = (lz - lz[_i0]) / _np_nb.log(10)
    win = (_alphas >= -4) & (_alphas <= 2)
    lo, hi = _interval(lz)
    return {"log10B": lb.tolist(), "max_abs_log10B_[-4,2]": float(_np_nb.max(_np_nb.abs(lb[win]))), "lo95": lo, "hi95": hi}

# ---- (1) baseline and bootstrap Monte Carlo error (common resampling across alpha) -------------------------
_base = _summary()
_rng = _np_nb.random.default_rng(12345)
_boot = []
for _ in range(200):
    _idx = _rng.integers(0, _N, _N)
    _lz = _np_nb.array([_logZ(_L[a][_idx]) for a in _alphas])
    _boot.append((_lz - _lz[_i0]) / _np_nb.log(10))
_boot = _np_nb.array(_boot)
_sig = _np_nb.nanstd(_np_nb.where(_np_nb.isfinite(_boot), _boot, _np_nb.nan), 0)
_ess = []
for _a in _alphas:                       # private loop names: marimo forbids redefining globals across cells
    _x = _clean(_L[_a]); _mx_ = _np_nb.max(_x)
    _ww = _np_nb.exp(_x - _mx_) if _np_nb.isfinite(_mx_) else _np_nb.zeros_like(_x)
    _ess.append(float(_ww.sum() ** 2 / max((_ww**2).sum(), 1e-300)))

# ---- (2) alternative EOS priors by importance reweighting of the SAME draws ---------------------------------
_r = _np_nb.nan_to_num(_R14_0, nan=12.0); _mx = _np_nb.nan_to_num(_MMAX_0, nan=2.0)
_TILTS = {
    "softer (favours small R_1.4)": _np_nb.exp(-1.0 * (_r - 12.0)),
    "stiffer (favours large R_1.4)": _np_nb.exp(+1.0 * (_r - 12.0)),
    "heavier (favours M_max > 2.2)": 1 / (1 + _np_nb.exp(-(_mx - 2.2) / 0.1)),
    "lighter (favours M_max < 2.2)": 1 / (1 + _np_nb.exp(+(_mx - 2.2) / 0.1)),
}
_prior = {}
for _k, _w in _TILTS.items():
    _w = _w / _w.mean()
    _s = _summary(_w); _s["prior_ESS"] = float(_w.sum() ** 2 / (_w**2).sum()); _prior[_k] = _s

# ---- (3) mechanism of the upper edge (Proposition 2): valid fraction vs alpha ------------------------------
_vf = {float(a): float(_VAL[a].mean()) for a in _alphas}

_win_nb = (_alphas >= -4) & (_alphas <= 2)
_ess_arr = _np_nb.array(_ess)
_res = {"alphas": _alphas.tolist(), "n_eos": int(_N), "baseline": _base, "bootstrap_sigma_log10B": _sig.tolist(),
        "max_2sigma_bootstrap": float(_np_nb.nanmax(2 * _sig)),
        "max_2sigma_bootstrap_[-4,2]": float(_np_nb.nanmax(2 * _sig[_win_nb])),
        "ESS": _ess, "min_ESS_[-4,2]": float(_ess_arr[_win_nb].min()),
        "min_ESS_over_couplings_with_stars": float(_ess_arr[_ess_arr > 0].min()) if (_ess_arr > 0).any() else 0.0,
        "prior_tilts": _prior, "valid_fraction": _vf}
with open(_os_nb.path.join(_OUT_NB, "NS_BAYES_ROBUST.json"), "w") as _f:
    _json_nb.dump(_res, _f, indent=1, default=float)

_rows = [f"| baseline (paper) | {_N:,} | {_base['lo95']:+.2f} | {_base['hi95']:+.2f} | {_base['max_abs_log10B_[-4,2]']:.2f} |"]
_rows += [f"| {k} | {v['prior_ESS']:,.0f} | {v['lo95']:+.2f} | {v['hi95']:+.2f} | {v['max_abs_log10B_[-4,2]']:.2f} |" for k, v in _prior.items()]
_edge = [a for a in _alphas if _vf[float(a)] < 0.5]
_mo_nb.md(f"""
### Referee check 3: neutron-star evidence — Monte Carlo error and prior sensitivity (astro + pQCD, β = 0)

**Monte Carlo error.** Bootstrap over the {_N:,} common EOS draws (200 resamples, same resampling at every α):
the largest ±2σ error on log₁₀ B(α) is **{_res['max_2sigma_bootstrap']:.3f}** over the whole scan and
**{_res['max_2sigma_bootstrap_[-4,2]']:.3f}** on −4 ≤ α ≤ 2 km². The smallest effective sample size is
{_res['min_ESS_[-4,2]']:,.0f} on −4 ≤ α ≤ 2 km² and {_res['min_ESS_over_couplings_with_stars']:,.0f} over all couplings that
still have stars (it is 0 where no EOS gives a star, which carries no Monte Carlo error).

**Alternative EOS priors** (importance reweighting of the same draws; no new TOV runs):

| EOS prior | prior ESS | 95% lower α [km²] | 95% upper α [km²] | max abs log₁₀ B on [−4, 2] |
|---|---|---|---|---|
{chr(10).join(_rows)}

**What sets the upper edge (Proposition 2).** Fraction of EOS that still give a valid star:
{", ".join(f"α = {a:+.2f}: {_vf[float(a)]:.2f}" for a in _alphas if a >= 1.5)}.
{"The valid fraction drops to zero at the crust threshold, so the upper edge is set by the crust condition, not by the data." if _edge else "No coupling in the scan removes all stars."}

Saved `{_OUT_NB}/NS_BAYES_ROBUST.json`.
""")

In [ ]:
# ==== CELLS 78 + 79: FULL-POSTERIOR (SAMPLE-BASED) NICER / GW170817 LIKELIHOODS FOR THE alpha SCAN ===============
# Answers the referee request: "replace the split-normal summaries by the published posterior samples".
# Place both cells AFTER the Part VII cells (they use NS, NS3, NS4, NS7, P7CFG and KDES from the notebook).
#
# Before running: put posterior-sample files in the Part III samples folder (P3CFG["samples_dir"], default ns_samples/)
# and re-run the Part III "OPTIONAL: public posterior samples" cell, which builds KDES:
#   J0030.txt, J0740.txt, J0437.txt   two columns: mass [Msun], radius [km]   (NICER releases on Zenodo:
#       J0030 https://zenodo.org/record/3386449 ; J0437 https://zenodo.org/records/13766753 ; J0740: the Zenodo
#       release cited in Riley et al. 2021, ApJL 918, L27)
#   GW170817.txt  a column lambda_tilde, or columns m1 m2 lambda1 lambda2   (https://dcc.ligo.org/LIGO-P1800115/public)
# The cells re-run the exact Part VII pipeline on a reduced grid with these likelihoods and compare with the summaries.
#
# ---------------------------------------------------------------------------------------------- CELL 78 (button)
kde_btn = mo.ui.run_button(label="Run the α scan with sample-based NICER/GW likelihoods")
mo.vstack([mo.md(f"**Sample-based likelihoods available for:** {', '.join(sorted(KDES)) if KDES else 'none (see the instructions in this cell)'}. "
                 "Runs 8 000 EOS × 9 couplings with the exact Part VII pipeline (resumable; `RUN_KDE=1` runs it headlessly)."),
           kde_btn])

In [ ]:
# ---------------------------------------------------------------------------------------------- CELL 79 (run + compare)
# ---- SAMPLE-BASED alpha SCAN: same EOS draws and seeds as Part VII, likelihoods from posterior samples -------------
mo.stop(not KDES, mo.md("No posterior-sample files loaded (KDES is empty). Add the files described in cell 78 and re-run the "
                        "Part III samples cell first."))
mo.stop(not (kde_btn.value or os.environ.get("RUN_KDE") == "1"), mo.md("*Press the button in cell 78 to start.*"))
_dev = P7CFG["device"]
_KCFG = dict(P7CFG)
_KCFG["ckpt_dir"] = os.path.join(os.path.dirname(P7CFG["ckpt_dir"]), "part7_kde")
os.makedirs(_KCFG["ckpt_dir"], exist_ok=True)
_points = [(_a, 0.0) for _a in (-8.0, -6.0, -4.0, -2.0, -1.0, 0.0, 1.0, 2.0, 2.25)]
_n_eos = 8000
_nb = -(-_n_eos // P7CFG["batch"])
_crust = NS.crust_table(0.5 * NS.N0)
_ng = NS.make_n_grid(device=_dev)
_nodes = torch.as_tensor(NS4.GP_NODES_N0 * NS.N0, device=_dev, dtype=torch.float64)
_t0 = time.time()
for _b in mo.status.progress_bar(range(_nb), title="sample-based likelihoods"):
    _todo = [_q for _q in _points if not os.path.exists(os.path.join(_KCFG["ckpt_dir"], NS7.point_file(_q, _b)))]
    if not _todo:
        continue
    _g = torch.Generator(device=_dev).manual_seed(P7CFG["seed"] + _b)        # identical EOS draws to Part VII
    _s, _cs, _raw = NS4.gp_draw(P7CFG["batch"], _g, _dev)
    _eps, _p, _cs2, _mu = NS4.build_eos_nodes(NS, _s, _cs, _nodes, _ng, _crust)
    for (_a, _bb) in _todo:
        _r = NS7.scan_point(NS, NS3, _eps, _p, _cs2, _mu, _ng, _a, _bb, _KCFG, KDES)
        _f = os.path.join(_KCFG["ckpt_dir"], NS7.point_file((_a, _bb), _b))
        np.savez(_f + ".tmp.npz", **_r); os.replace(_f + ".tmp.npz", _f)
    del _eps, _p, _cs2, _mu
    gpu_free()

def _collect(ckdir, pts, nb):
    out = {}
    for q in pts:
        fs = [os.path.join(ckdir, NS7.point_file(q, b)) for b in range(nb)]
        fs = [f for f in fs if os.path.exists(f)]
        if fs:
            zz = [dict(np.load(f)) for f in fs]
            out[q[0]] = np.concatenate([np.atleast_1d(z["ll"]) + np.atleast_1d(z["pqcd"]) for z in zz])
    return out

def _lz(x):
    x = np.where(x < -1e29, -np.inf, x); m = np.max(x)
    return float(m + np.log(np.mean(np.exp(x - m)))) if np.isfinite(m) else -np.inf

_kde = _collect(_KCFG["ckpt_dir"], _points, _nb)
_sum = _collect(P7CFG["ckpt_dir"], _points, _nb)                  # the same EOS draws with summary likelihoods
_al = np.array(sorted(set(_kde) & set(_sum)))
_lzk = np.array([_lz(_kde[_x]) for _x in _al]); _lzs = np.array([_lz(_sum[_x]) for _x in _al])
_i0 = int(np.argmin(np.abs(_al)))
_bk = (_lzk - _lzk[_i0]) / np.log(10); _bs = (_lzs - _lzs[_i0]) / np.log(10)
_win = (_al >= -4) & (_al <= 2)
KDE_CHECK = {"alpha": _al.tolist(), "log10B_samples": _bk.tolist(), "log10B_summaries_same_draws": _bs.tolist(),
             "max_abs_log10B_[-4,2]_samples": float(np.max(np.abs(_bk[_win]))),
             "max_abs_log10B_[-4,2]_summaries": float(np.max(np.abs(_bs[_win]))),
             "kdes": sorted(KDES), "n_eos": int(len(next(iter(_kde.values())))) if _kde else 0}
with open(os.path.join(_KCFG["ckpt_dir"], "KDE_CHECK.json"), "w") as _fh:
    json.dump(KDE_CHECK, _fh, indent=1, default=float)
_rows = "\n".join(f"| {_x:+.2f} | {_y:+.3f} | {_z:+.3f} |" for _x, _y, _z in zip(_al, _bk, _bs))
mo.md(f"""
### Referee check 4: sample-based likelihoods ({', '.join(sorted(KDES))})

Same {KDE_CHECK['n_eos']:,} EOS draws, same exact map and solver; only the NICER/GW likelihood changes.

| α [km²] | log₁₀ B, posterior samples | log₁₀ B, split-normal summaries |
|---|---|---|
{_rows}

- max abs log₁₀ B on [−4, 2]: **{KDE_CHECK['max_abs_log10B_[-4,2]_samples']:.2f}** (samples) vs
  {KDE_CHECK['max_abs_log10B_[-4,2]_summaries']:.2f} (summaries). The degeneracy statement of the paper holds if both are below 0.5.
- Saved `{_KCFG['ckpt_dir']}/KDE_CHECK.json` and the per-point checkpoints (resumable).
""")